# Session analysis
Turns what the finger rehab game logged into the numbers and figures for my thesis.
Fold the Setup heading below, pick a save, then hit Run All.

## Setup
Everything the sections below call. Fold this heading and you get results only.

In [ ]:
# Setup. Every import, constant and function the notebook uses, then
# the catalogue and the picker. This is the one long cell: everything
# below it is a heading and a call, so this is the only scroll.
#
# Nothing here reads a session or draws anything. Pick a save from the
# dropdown at the bottom of this cell, then press Run All.
#
# Everything this notebook needs is defined in this one cell. There is
# no module to import and no other file to keep next to it, so it can
# be handed to someone on its own and still run.
#
# A note on force, because it is the easiest number here to misread.
# Each sensor pad reads a different number of counts for the same real
# force, so raw counts cannot be compared between fingers. Force
# therefore comes in three columns:
#
#     peak_force_n    raw counts as recorded
#     peak_force_N    newtons, absolute, for the Demouche comparison
#     peak_force_cal  counts divided by that finger's OWN calibration
#                     press
#
# peak_force_cal is NOT a strength ranking. The calibration press is the
# patient's own light press, not a known physical force, so a weak
# finger recorded a small reference press and dividing by it cancels the
# real weakness along with the pad difference. Four identical pads with
# a true four to one weakness gradient across the hand all come out near
# 1.0. Read peak_force_cal as effort against that finger's own
# reference, which is the right measure for consistency within a finger
# and for change over time. For "which finger is stronger", read the
# newton column and accept that the pad differences are not corrected
# there. Separating pad sensitivity from finger strength needs a known
# physical reference on each pad, a weight or a load cell, which this
# device does not have.
#
# Sessions recorded before the in-app calibration existed get the raw
# and newton columns only, and are said to be uncorrected rather than
# quietly pooled with the rest.

from __future__ import annotations

%matplotlib inline
import warnings; warnings.filterwarnings("ignore")

import json
import re
from dataclasses import dataclass, field
from pathlib import Path

# Imported up front, but with a readable failure. A bare
# ModuleNotFoundError out of the first cell tells someone who has just
# been handed this file nothing about what to install.
try:
    import numpy as np
    import pandas as pd
    import matplotlib
    import matplotlib.pyplot as plt
    from matplotlib.ticker import MaxNLocator
except ImportError as _e:
    raise ImportError(
        f"{_e.name} is not installed, so this notebook cannot run.\n"
        f"Install everything it needs with:\n"
        f"    pip install pandas numpy matplotlib scipy ipywidgets\n"
    ) from _e

# The dropdown is a convenience, not a requirement. Every section below
# runs from `pick`, and pick is an ordinary variable, so a kernel with
# no ipywidgets should still be able to analyse a save. Making the
# widget import mandatory turned a missing convenience package into a
# notebook where nothing at all could run.
try:
    import ipywidgets as W
    from IPython.display import display, clear_output

    HAVE_WIDGETS = True
except ImportError:
    W = None
    clear_output = None
    display = print
    HAVE_WIDGETS = False


# ---------------------------------------------------------------- config

# Where the recordings live. Looked for next to this notebook first,
# then up the tree, so the file runs whether it sits in analysis/ inside
# the repo or on its own beside a copy of sessions/. If the recordings
# are somewhere else entirely, replace the call with the path:
#     SESSIONS_DIR = Path("/full/path/to/sessions")
def _find_sessions(start=None) -> Path:
    here = Path(start or Path.cwd()).resolve()
    for base in (here, *list(here.parents)[:4]):
        if (base / "sessions").is_dir():
            return base / "sessions"
    return here / "sessions"


SESSIONS_DIR = _find_sessions()


# Outputs land in the session folder they describe, not here. These two
# start as placeholders and prepare() points them into the newest
# analysed session (sessions/<date>/<session>/analysis/).
FIGDIR = Path("figures")
OUTDIR = Path(".")


FINGERS = ["Index", "Middle", "Ring", "Pinky"]

# Chords constants, copied verbatim from finger_rehab/game/modes/chords.py
# (FINGER_LETTERS, ENSLAVABILITY, ADJACENT, SIZE_PENALTY) so
# chord_difficulty here reproduces the mode's own D exactly. The
# healthy band matches the 8-15% (light force) figure the cross-talk
# panel already prints (Abolins et al. 2020).
CHORD_LETTERS = ("I", "M", "R", "P")
CHORD_ENSLAVABILITY = (1.0, 2.0, 3.0, 2.0)
CHORD_ADJACENT = {0: (1,), 1: (0, 2), 2: (1, 3), 3: (2,)}
CHORD_SIZE_PENALTY = 1.5
HEALTHY_ER_BAND = (0.08, 0.15)


FINGER_COLOUR = {"Index": "#ea580c", "Middle": "#0ea5e9",
                 "Ring": "#0f172a", "Pinky": "#ca8a04"}


HAND_COLOUR = {"right": "#2563eb", "left": "#a855f7"}


MODE_COLOUR = {"classic": "#2563eb", "adaptive": "#16a34a",
               "rhythm": "#a855f7", "mirror": "#ea580c",
               "reaction": "#0ea5e9", "pattern": "#f59e0b",
               "chords": "#dc2626", "syllables": "#14b8a6",
               "force_pilot": "#7c3aed", "lighthouse": "#f97316",
               "buzz_hunt": "#0d9488",
               "unknown": "#94a3b8"}


# The adaptive controller holds ONE shared BPM regulated off the
# overall hit rate; it does not (and cannot) push each finger
# individually into this band -- weak fingers just get practiced
# more often (lane weighting), not made individually easier. Band
# is a design choice (Guadagnoli and Lee's challenge-point framework
# motivates having a target band; they do not report this number).
# Wilson and colleagues put the optimum nearer 85 percent, so both
# get drawn.
BAND_LO, BAND_HI, WILSON = 0.65, 0.80, 0.85


NUMERIC = ["block_t_s", "trial", "lane", "time_difference_ms", "points",
           "num_presses", "first_incorrect_ms", "first_incorrect_lane",
           "bpm_at_trial", "streak_at_trial", "song_time_s", "peak_force_n",
           "impulse_n", "timeout_ms", "force_window_sum"]


BOOLISH = ["had_incorrect_press", "in_recovery", "loud_trial", "stim_delivered",
           "pattern_trial"]


# What the TRUE/FALSE columns can arrive as. read_csv turns a column of
# plain TRUE/FALSE into real booleans on its own, so the text forms only
# survive when the column also holds blanks and comes through as object.
_TRUE_TEXT = {"true", "t", "yes", "y", "1", "1.0"}


_FALSE_TEXT = {"false", "f", "no", "n", "0", "0.0"}


# --------------------------------------------- figures from past theses
# Read off the past Curtin theses in this project's lineage (Lim 2023,
# Palmer and Lew 2024, Nakayama, Lee, Demouche and Dixon 2025). The
# sections that use them are in the later parts; the numbers live here
# so there is one place to correct them.

# SingleTact conversion. The manufacturer equation is
#     Load(N) = (counts - baseline) / 512 * sensor rating
# so on a 45 N part one count is about 0.0879 N.
SENSOR_RATING_N = 45.0


COUNTS_FULL_SCALE = 512.0


N_PER_COUNT = SENSOR_RATING_N / COUNTS_FULL_SCALE


# Healthy peak fingertip force measured by Demouche on the 2025 button
# device, 7 participants. Different button geometry so not a direct
# read-across, but the only same-lineage human data available.
DEMOUCHE_2025 = {"index_mean": 3.11, "index_max": 6.56,
                 "little_mean": 2.66, "little_max": 5.60}


# Li et al. via Lew: enslavement, the share of force appearing on the
# fingers that were not asked to move.
ENSLAVEMENT_REF = {"unimpaired": 0.13, "stroke": 0.251}


# Lang's clinical figure for repetitions per therapy session, the number
# Basil's dose argument is built on.
LANG_REPS_PER_SESSION = 32


# ------------------------------------------------ the calibrated column
# Wording, not maths. Nearly every section prints one of these, so they
# sit here rather than in the calibration part, and there is one copy of
# the caveat instead of six paraphrases of it.

# Name for the calibrated measure. Deliberately not called a force: it
# is a ratio against a reference the patient produced. The old label
# "force (x calibration press)" read as a force comparable between
# fingers, which is exactly the wrong reading.
NORM_UNIT = "x own reference press"


NORM_LABEL = "relative effort (x that finger's own reference press)"


# One sentence for every table and axis where the ratio turns up. The
# reader should never meet the number without meeting this.
NORM_SHORT = ("effort against that finger's own reference press, "
              "not a strength ranking")


def print_norm_short(indent="   "):
    """The short version, wrapped to the width everything else here
    prints at. Every table carrying the ratio gets this."""
    for line in (f"{NORM_LABEL}.",
                 "Effort against that finger's own reference press: good",
                 "for consistency within a finger and for change over",
                 "time, NOT a between-finger strength ranking. For which",
                 "finger is stronger, read the newton column and its",
                 "caveat."):
        print(indent + line)


def normalisation_note(indent="") -> str:
    """The full statement of what the calibrated ratio can and cannot
    answer, printed once wherever force is the subject.

    Written out rather than summarised because the failure it guards
    against is silent: peak_force_cal looks like a corrected force and
    reads like one, and a reader who ranks fingers on it gets a number
    that has had the ranking divided out of it.
    """
    lines = [
        "WHAT THE CALIBRATED COLUMN MEANS",
        f"  {NORM_LABEL}.",
        "  Each finger's force divided by the light press THAT finger gave",
        "  at calibration. 1.0 means the same effort as that reference.",
        "",
        "  It CAN answer: was this finger consistent, did its effort change",
        "  across the block, did it change between sessions.",
        "",
        "  It CANNOT answer: which finger is stronger. The reference press",
        "  is the patient's own light press, not a known physical force, so",
        "  a weak finger recorded a small reference and dividing by it",
        "  cancels the weakness along with the pad difference. Four",
        "  identical pads with a real four to one weakness gradient across",
        "  the hand all come out near 1.0 on this measure.",
        "",
        "  For a between-finger strength comparison, read the newton",
        "  figures. Those are absolute, but they are NOT corrected for pad",
        "  sensitivity, so part of any difference between fingers there is",
        "  the hardware.",
        "",
        "  This device cannot separate pad sensitivity from finger strength.",
        "  Doing that needs a known physical reference on each pad, a weight",
        "  or a load cell, and there is none here. Report the limitation",
        "  rather than picking whichever number looks cleaner.",
    ]
    return "\n".join(indent + ln if ln else ln for ln in lines)


def as_bool(series: pd.Series) -> pd.Series:
    """A TRUE/FALSE column as real booleans, NaN where unreadable.

    read_csv has already parsed most of these to bool, so mapping the
    literal strings a second time turns every value into NaN. That is
    silent: the checks built on these columns all test `== True`,
    `!= False` or `.dropna()`, so an all-NaN column reads as "nothing to
    report" rather than as an error, and a block where every cue failed
    to reach the device comes out looking clean.
    """
    if pd.api.types.is_bool_dtype(series):
        return series

    def one(v):
        if isinstance(v, (bool, np.bool_)):
            return bool(v)
        if v is None or v is pd.NA:
            return np.nan
        if isinstance(v, float) and np.isnan(v):
            return np.nan
        text = str(v).strip().lower()
        if text in _TRUE_TEXT:
            return True
        if text in _FALSE_TEXT:
            return False
        return np.nan

    return series.map(one)


def use_style():
    plt.rcParams.update({
        "figure.dpi": 110, "savefig.dpi": 160,
        "axes.grid": True, "grid.color": "#e2e8f0", "grid.linewidth": 0.8,
        "axes.axisbelow": True,
        "axes.spines.top": False, "axes.spines.right": False,
        "font.size": 10, "axes.titlesize": 11, "axes.titleweight": "bold",
        "axes.titlelocation": "left", "figure.autolayout": True,
    })


_FIGS_CLEARED = False


def _clear_figures():
    """Wipe figures/ once per run.

    Sections only write the plots their data supports, so a selection with
    no rhythm block leaves the previous selection's rhythm.png sitting
    there looking current. Picking that up for the report would put
    someone else's chart under this participant's heading.
    """
    global _FIGS_CLEARED
    if _FIGS_CLEARED or not FIGDIR.is_dir():
        _FIGS_CLEARED = True
        return
    for old in FIGDIR.glob("*.png"):
        try:
            old.unlink()
        except OSError:
            pass
    _FIGS_CLEARED = True


def _save(fig, name):
    FIGDIR.mkdir(parents=True, exist_ok=True)
    _clear_figures()
    fig.savefig(FIGDIR / f"{name}.png", bbox_inches="tight")


def _nbins(series, want=28):
    """matplotlib throws if every value is identical, which happens with
    small or very consistent samples."""
    s = pd.Series(series).dropna()
    if len(s) < 2 or float(s.max()) == float(s.min()):
        return 1
    return min(want, max(5, int(len(s) ** 0.5) * 2))


def _show(obj):
    """display() in a notebook, print() anywhere else."""
    try:
        from IPython.display import display as _d
        _d(obj)
    except Exception:
        print(obj)


def _order(df, col="finger"):
    return [f for f in FINGERS if f in df[col].unique()]


def _nothing(*lines):
    """Say why a section has nothing to show. Every section prints
    something: a section that returns in silence reads as one that
    crashed, and the reader cannot tell an empty result from a broken
    cell."""
    for ln in lines:
        print(ln)


def _boxes(ax, data, order, ylabel, title):
    bp = ax.boxplot(data, labels=order, patch_artist=True, widths=.6)
    for p, f in zip(bp["boxes"], order):
        p.set_facecolor(FINGER_COLOUR[f]); p.set_alpha(.65)
    for m in bp["medians"]:
        m.set_color("white"); m.set_linewidth(2)
    ax.set_ylabel(ylabel); ax.set_title(title)
    return bp


# ---------------------------------------------------------------- loading

def read_meta(folder: Path) -> dict:
    p = Path(folder) / "metadata.json"
    if not p.exists():
        return {}
    try:
        return json.loads(p.read_text())
    except json.JSONDecodeError:
        return {}


CATALOGUE_COLS = ["day", "time", "who", "mode", "hand", "trials",
                  "hit_rate", "mean_rt", "status", "folder", "session"]


def build_catalogue(root: Path | str = None) -> pd.DataFrame:
    """One row per game folder. A game is one block of one mode; a
    session is one person on one day."""
    root = Path(root or SESSIONS_DIR)
    if not root.exists():
        # No recordings folder. Say so plainly and hand back an empty
        # frame rather than raising: a traceback on the first cell is a
        # rotten way to meet someone who has just opened the file, and
        # every section below already knows how to say it has nothing.
        print(f"No recordings found. Looked for a sessions folder at\n"
              f"  {root.resolve()}\n"
              f"Put one beside this notebook, or set SESSIONS_DIR near the\n"
              f"top of this cell to wherever yours lives.")
        return pd.DataFrame(columns=CATALOGUE_COLS)
    rows = []
    for trials_csv in sorted(root.rglob("trials.csv")):
        folder = trials_csv.parent
        meta = read_meta(folder)
        bs = meta.get("block_summary", {}) or {}
        day = (folder.parent.name
               if re.fullmatch(r"\d{4}-\d{2}-\d{2}", folder.parent.name)
               else meta.get("started_at", "")[:10])
        m = re.match(r"^(.*)_(\d{6})(?:_(.*))?$", folder.name)
        clock = m.group(2) if m else ""
        clock = f"{clock[:2]}:{clock[2:4]}" if len(clock) == 6 else ""
        try:
            n = sum(1 for _ in open(trials_csv)) - 1
        except OSError:
            n = 0
        rows.append({
            "day": day, "time": clock,
            "who": meta.get("participant",
                            m.group(1) if m else folder.name),
            "mode": bs.get("block",
                           m.group(3) if m and m.group(3) else "unknown"),
            "hand": meta.get("hand", "?"),
            "trials": int(bs.get("trials", n) or n),
            "hit_rate": bs.get("hit_rate"), "mean_rt": bs.get("avg_rt_ms"),
            "status": bs.get("status", "?"), "folder": str(folder),
        })
    # Columns even when there is nothing on disk. An empty frame with no
    # columns turns every later cat["session"] into KeyError('session'),
    # which cascades through the whole notebook and looks like a broken
    # install rather than an empty sessions folder.
    cat = pd.DataFrame(rows, columns=CATALOGUE_COLS[:-1])
    if cat.empty:
        cat["session"] = pd.Series(dtype="object")
        return cat
    cat = cat.sort_values(["day", "time", "who"]).reset_index(drop=True)
    cat["session"] = cat["day"] + "  " + cat["who"]
    return cat


def catalogue(root=None) -> pd.DataFrame:
    """Print everything on disk with an id per game, and return it."""
    cat = build_catalogue(root)
    if cat.empty:
        print("Nothing recorded yet. Play a block and come back.")
        return cat
    print(f"{len(cat)} game(s) across {cat['session'].nunique()} session(s)\n")
    show = cat[["day", "time", "who", "mode", "hand",
                "trials", "hit_rate", "status"]].copy()
    show.index.name = "id"
    _show(show)
    print("\nSESSIONS  (one person, one day)")
    for s, g in cat.groupby("session"):
        modes = ", ".join(f"{m} x{c}" if c > 1 else m
                          for m, c in g["mode"].value_counts().items())
        print(f'   "{s}"   {len(g)} game(s), {g["trials"].sum()} trials   {modes}')
    print("\nPick one from the dropdown at the top of this notebook and")
    print("press Run All in the toolbar, or set pick to one of those folder")
    print("names by hand in the setup cell.")
    return cat


def resolve(pick, cat: pd.DataFrame) -> pd.DataFrame:
    """Turn a pick into catalogue rows. Accepts an id, a list of ids, a
    name, a date, a mode, a session label, 'latest' or 'all'."""
    if cat.empty:
        return cat
    if isinstance(pick, str) and pick.lower() == "all":
        return cat
    if isinstance(pick, str) and pick.lower() == "cohort":
        # The study participants: a code as who. The visit and
        # demo screening happen in cohort_catalogue, which reads
        # the metadata this frame does not hold.
        return cat[cat["who"].astype(str).map(is_study_code)]
    if isinstance(pick, str) and pick.lower() == "latest":
        return cat.tail(1)
    items = pick if isinstance(pick, (list, tuple, set)) else [pick]
    ids = [i for i in items if isinstance(i, (int, np.integer))
           and not isinstance(i, bool)]
    if ids:
        bad = [i for i in ids if i not in cat.index]
        if bad:
            raise KeyError(f"no game with id {bad}. "
                           f"Valid ids are 0 to {cat.index.max()}.")
        return cat.loc[ids]
    sel = cat
    for term in items:
        t = str(term).strip()
        # Exact matches on the real columns first. The folder fallback
        # only looks at the folder's OWN name, never the path above it.
        #
        # It used to match anywhere in the absolute path, which quietly
        # pooled other people's data into a result. On this machine the
        # repository sits under "Basil's Brain", so asking for Basil's
        # games matched every folder on disk and returned P01's blocks as
        # well. Anyone whose name appears in a parent directory hits the
        # same thing, and nothing about the output says it happened.
        folder_name = sel["folder"].map(lambda f: Path(str(f)).name)
        # The day-qualified game key first: it names exactly one game.
        # A bare folder name can collide across days (game_key's
        # docstring), so on its own it keeps every match and a
        # one-game pick loads two days pooled.
        gkey = sel["folder"].map(game_key)
        exact = gkey.eq(t)
        if not exact.any():
            exact = folder_name.eq(t)
        if exact.any():
            # An exact name wins outright. A substring test here
            # would also match the de-duplicated sibling, because
            # P01_120000_classic is a prefix of P01_120000_classic_1.
            sel = sel[exact]
            continue
        hit = (sel["session"].eq(t) | sel["day"].eq(t) | sel["who"].eq(t)
               | sel["mode"].eq(t) | sel["hand"].eq(t)
               | folder_name.str.contains(re.escape(t), case=False))
        if not hit.any():
            raise KeyError(
                f"nothing matches {t!r}. Try a day, a name, a mode, a "
                f"session label, or an id from catalogue().")
        sel = sel[hit]
    return sel


def game_key(folder) -> str:
    """A name that identifies exactly one game folder.

    The folder name on its own is not enough. sessions/<day>/<game> can
    hold the same block name on two different days, and every per-game
    lookup here is keyed on that name: the metadata, the calibration,
    the force factors. A collision quietly applies one game's
    calibration profile to another game's trials, which is the same
    class of error as normalising the left hand with the right hand's
    gaps. The day folder disambiguates, and two folders inside one day
    cannot share a name.
    """
    folder = Path(folder)
    parent = folder.parent.name
    return f"{parent}/{folder.name}" if parent else folder.name


def load_games(folders, cat: pd.DataFrame) -> pd.DataFrame:
    frames = []
    for folder in folders:
        folder = Path(folder)
        df = pd.read_csv(folder / "trials.csv")
        for c in NUMERIC:
            if c in df.columns:
                df[c] = pd.to_numeric(df[c], errors="coerce")
        for c in BOOLISH:
            if c in df.columns:
                df[c] = as_bool(df[c])
        meta = read_meta(folder)
        bs = meta.get("block_summary", {}) or {}
        row = cat[cat["folder"] == str(folder)]
        df["game"] = game_key(folder)
        df["game_label"] = (f"{row['time'].iloc[0]} {row['mode'].iloc[0]}"
                            if len(row) else folder.name)
        df["session"] = row["session"].iloc[0] if len(row) else folder.name
        df["mode"] = bs.get("block", "unknown")
        df["hand_mode"] = meta.get("hand", "right")
        df["participant"] = meta.get("participant", "NA")
        df["folder"] = str(folder)
        lane0 = df["lane"] - 1
        df["finger"] = [FINGERS[int(l) % 4] if pd.notna(l) else None
                        for l in lane0]
        df["side"] = ["left" if (pd.notna(l) and l >= 4) else "right"
                      for l in lane0]
        if df["hand_mode"].iloc[0] == "left":
            df["side"] = "left"
        frames.append(df)
    return _unique_game_labels(pd.concat(frames, ignore_index=True))


def _unique_game_labels(trials: pd.DataFrame) -> pd.DataFrame:
    """Make game_label name exactly one game.

    The label is clock time plus mode, which two games can share: two
    people recorded in the same minute, or the same mode started twice
    on two boards. Sections group on this label, so a collision silently
    merges two games into one row and sec_compare then reports there is
    nothing to compare when there is.
    """
    if trials.empty or "game_label" not in trials.columns:
        return trials
    for extra in ("participant", "game"):
        pairs = trials[["game", "game_label"]].drop_duplicates()
        clashing = pairs["game_label"].duplicated(keep=False)
        if not clashing.any():
            break
        mask = trials["game_label"].isin(set(pairs.loc[clashing, "game_label"]))
        if extra == "game":
            trials.loc[mask, "game_label"] = trials.loc[mask, "game"]
        else:
            trials.loc[mask, "game_label"] = (
                trials.loc[mask, "game_label"] + "  "
                + trials.loc[mask, extra].astype(str))
    return trials


def load_raw(folder: Path):
    p = Path(folder) / "raw.csv"
    if not p.exists() or p.stat().st_size < 200:
        return None
    df = pd.read_csv(p)
    for c in ["t_perf"] + [f"fsr{i}" for i in range(1, 9)]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    return df


def force_unit(metas) -> str:
    for m in _meta_list(metas):
        u = (m.get("block_summary", {}) or {}).get("force_unit")
        if u:
            return u
    return "sensor counts"


def load_metas(folders) -> dict:
    """Read every metadata.json for a selection, keyed by game folder
    name. Same shape prepare() builds internally, so a cell can get one
    without loading a whole selection."""
    return {game_key(f): read_meta(Path(f)) for f in folders}


# ------------------------------------------------- stale-selection guard
# The notebook keeps its selection in a global that the dropdown writes
# to. Change the dropdown, re-run some cells and not others, and the
# headline table ends up built from `trials` for one save and `rt` or
# `force` for another. Nothing warns, the numbers look plausible, and
# they belong to no single session.
#
# So prepare() stamps what it loaded into the context, every cell checks
# that stamp against the dropdown before touching anything, and results
# are filed under the stamp they were computed on. A stale run raises
# instead of printing a mixed answer.


class StaleSelection(RuntimeError):
    """The cell was run against a selection the context was not built
    from. Raised rather than warned: a warning scrolls past and the
    wrong number still gets written to the CSV."""


def selection_key(pick, sel) -> str:
    """Short text naming exactly what was loaded.

    Built from the resolved folders rather than from `pick` alone,
    because two different picks can name the same games and the same
    pick can name different games after a rescan.
    """
    if sel is None or getattr(sel, "empty", True):
        return f"{pick!r} -> nothing"
    folders = sorted(str(f) for f in sel["folder"])
    return f"{pick!r} -> {len(folders)} game(s): " + "; ".join(folders)


def describe_pick(pick, cat) -> str:
    """What `pick` would load right now, for the error message."""
    try:
        return selection_key(pick, resolve(pick, cat))
    except (KeyError, TypeError) as e:
        return f"{pick!r} -> cannot be resolved ({e})"


def check_selection(ctx, pick):
    """Stop the cell unless ctx was built from the current dropdown value.

    Call this at the top of every cell that reads anything prepare()
    built. It costs a catalogue lookup and it is the only thing standing
    between a changed dropdown and a headline table made of two
    different sessions.
    """
    if not isinstance(ctx, dict) or "selection" not in ctx:
        raise StaleSelection(
            "the context has not been built yet. Run the 'Load the "
            "selection' cell, then the cells above this one, before "
            "running this cell.")
    now = describe_pick(pick, ctx.get("cat"))
    if now != ctx["selection"]:
        raise StaleSelection(
            "this cell is about to mix two selections.\n"
            f"   loaded : {ctx['selection']}\n"
            f"   dropdown now : {now}\n"
            "   Re-run the cells above, starting at 'Load the selection', "
            "then run this one again.")
    return True


# ================================================== report capture
# One HTML per run holding everything this notebook printed and drew.
#
# How the capture works, because it has to survive people adding
# sections later: prepare() wraps every sec_ function in this cell so
# each one runs with its stdout redirected into a buffer, then prints
# that buffer straight back out (the cell shows exactly what it always
# showed) and hands the text to keep(), which files it with any figure
# that appeared while the section ran.
#
# THE CONTRACT: a section added later lands in the report with no
# extra work, provided it is a function named sec_* and its cell calls
# keep(ctx, name, sec_x(...)) like every other section. Anything that
# prints outside that pattern is not in the report, and the export
# names it rather than dropping it silently.

import base64
import contextlib
import html as _html
import io
import sys
from datetime import datetime

# Anchored to the resolved sessions folder, never a relative
# path: the kernel runs in the notebook's own directory, so a
# relative "sessions" would create a second empty sessions tree
# beside the notebook and shadow the real recordings next run.
PATIENT_RESULTS = Path(SESSIONS_DIR) / "individual_patient_results"

_CAPTURE = {"pending": "", "figs": {}, "sections": [], "started": ""}


def _fig_state():
    """Filename to mtime for everything currently in FIGDIR."""
    if not FIGDIR.is_dir():
        return {}
    out = {}
    for p in sorted(FIGDIR.glob("*.png")):
        try:
            out[p.name] = p.stat().st_mtime_ns
        except OSError:
            pass
    return out


def _wrap_sections():
    """Give every sec_ function a stdout buffer, once."""
    for name, fn in list(globals().items()):
        if not name.startswith("sec_") or not callable(fn):
            continue
        if getattr(fn, "_captured", False):
            continue

        def make(inner):
            def wrapper(*args, **kwargs):
                buf = io.StringIO()
                try:
                    with contextlib.redirect_stdout(buf):
                        result = inner(*args, **kwargs)
                finally:
                    # Print it back so the cell output is unchanged
                    # whether or not anything is capturing.
                    text = buf.getvalue()
                    if text:
                        print(text, end="")
                    _CAPTURE["pending"] = text
                return result

            wrapper._captured = True
            wrapper.__doc__ = inner.__doc__
            wrapper.__name__ = getattr(inner, "__name__", "sec")
            return wrapper

        globals()[name] = make(fn)


def _capture_reset():
    """Start a fresh report. Called by prepare(), so re-picking a save
    never mixes two selections into one report."""
    _wrap_sections()
    _CAPTURE["pending"] = ""
    _CAPTURE["figs"] = _fig_state()
    _CAPTURE["sections"] = []
    _CAPTURE["started"] = datetime.now().strftime("%Y-%m-%d %H:%M")


def _capture_section(name):
    """File the last section's text and whatever it drew."""
    text = _CAPTURE["pending"]
    _CAPTURE["pending"] = ""
    now = _fig_state()
    before = _CAPTURE["figs"]
    fresh = [n for n, m in now.items() if before.get(n) != m]
    _CAPTURE["figs"] = now
    # Re-running one section replaces its entry instead of doubling it.
    _CAPTURE["sections"] = [s for s in _CAPTURE["sections"]
                            if s["name"] != name]
    # Absolute paths: the cohort sections point FIGDIR at their
    # own folder, and the per-session report written after them
    # must still find the figures its sections drew.
    _CAPTURE["sections"].append({"name": name, "text": text.strip(),
                                 "figures": [str(FIGDIR / n)
                                             for n in fresh]})


def _safe_name(text) -> str:
    """A folder name that survives whatever was typed at setup."""
    clean = re.sub(r"[^A-Za-z0-9._-]+", "_", str(text or "").strip())
    return clean.strip("._-") or "unnamed"


def _img_tag(png: Path) -> str:
    """A figure embedded in the page, so the HTML needs no side files."""
    try:
        b64 = base64.b64encode(png.read_bytes()).decode("ascii")
    except OSError:
        return ""
    return (f'<img alt="{_html.escape(png.stem)}" '
            f'src="data:image/png;base64,{b64}">')


# The report files a section under its keep() key ("srtt", "on_task"),
# which is an internal handle, not a heading. These are the headings
# the notebook's own markdown cells use, so the exported page reads
# like the notebook rather than like its variable names. A section
# missing here falls back to its key instead of vanishing.
SECTION_TITLES = {
    "calibration": "Calibration this data was recorded under",
    "on_task": "Overview",
    "quality": "Data quality",
    "compare": "Comparing the games",
    "mode_compare": "Comparing the training modes",
    "cadence_ras": "Fixed cadence against RAS",
    "rt": "Reaction time",
    "reaction_mode": "Reaction mode",
    "srtt": "Muscle memory (patterns)",
    "fatigue": "Fatigue across blocks",
    "accuracy": "Accuracy and the challenge point",
    "judgements": "Timing judgement bands",
    "outcome_rates": "Outcome rates per finger",
    "force": "Force",
    "drift": "Baseline drift",
    "ind": "Finger individuation",
    "chords": "Chord mode",
    "crosstalk": "Cross-talk between fingers",
    "rhythm": "Rhythm",
    "tap_variability": "Tap variability and beat phase",
    "syllables": "Syllable beats",
    "bilateral": "Both hands",
    "inter_hand": "Inter-hand correlation",
    "affected": "Affected against unaffected hand",
    "raw_stream": "Raw sample stream",
    "continuous": "Continuous force modes",
    "force_tracking": "Force tracking",
    "precision_hold": "Precision hold and force sense",
    "tactile": "Tactile perception",
    "onset": "Movement onset and rate of force development",
    "objective_one": "Objective 1, per-finger hit rate",
    "flagged": "Trial exclusions",
    "phase": "Pretest to aftertest",
    "threshold_audit": "Press thresholds in newtons",
    "cues": "Cue modality",
    "dose": "Dose",
    "sampling_note": "Sampling",
    "startup": "Startup latency",
    "constraints": "Constraint budget from the logs",
    "participant_progress": "Progress per participant",
    "convergence": "Convergence across sessions",
    "hand_size": "Hand size against the sizing range",
    "statistics": "Intervals and tests",
    "summary": "Headline numbers",
}


def _build_report_html(title, subtitle, sections) -> str:
    css = ("body{font:15px/1.5 -apple-system,Segoe UI,Roboto,sans-serif;"
           "margin:0 auto;max-width:1000px;padding:32px 20px;color:#111}"
           "h1{margin:0 0 4px;font-size:26px}"
           ".sub{color:#555;margin-bottom:24px}"
           "h2{margin:36px 0 8px;font-size:19px;border-top:1px solid #ddd;"
           "padding-top:16px}"
           "pre{background:#f6f7f9;padding:12px;border-radius:6px;"
           "white-space:pre-wrap;word-wrap:break-word;font-size:13px}"
           "img{max-width:100%;height:auto;margin:10px 0;"
           "border:1px solid #eee;border-radius:6px}"
           "ol{columns:2;font-size:14px}a{color:#0b5cd5;"
           "text-decoration:none}")
    toc = "".join(
        f'<li><a href="#s{i}">'
        f'{_html.escape(SECTION_TITLES.get(s["name"], s["name"]))}'
        f"</a></li>"
        for i, s in enumerate(sections))
    body = []
    for i, s in enumerate(sections):
        heading = SECTION_TITLES.get(s["name"], s["name"])
        body.append(f'<h2 id="s{i}">{_html.escape(heading)}</h2>')
        if s["text"]:
            body.append(f"<pre>{_html.escape(s['text'])}</pre>")
        for fig in s["figures"]:
            tag = _img_tag(Path(fig))
            if tag:
                body.append(tag)
        if not s["text"] and not s["figures"]:
            body.append("<p><i>nothing for this selection</i></p>")
    return ("<!doctype html><html><head><meta charset='utf-8'>"
            f"<title>{_html.escape(title)}</title><style>{css}</style>"
            f"</head><body><h1>{_html.escape(title)}</h1>"
            f"<p class='sub'>{_html.escape(subtitle)}</p>"
            f"<ol>{toc}</ol>{''.join(body)}</body></html>")


def write_report(ctx):
    """Write this run's HTML: one beside the session data, and for a
    single participant one more under
    sessions/individual_patient_results/<who>/summary.html.

    The standing summary is replaced only when the pick covers EVERY
    game that participant has on disk: it promises the latest full
    read of the person, and a one-game pick used to overwrite it with
    a one-block view under the same name. A partial pick still writes
    report.html beside the session and still lands in runs.txt, with a
    note that the standing summary was left alone."""
    # The cohort sections have their own report under
    # sessions/cohort_results; they never belong in one
    # participant's summary.
    sections = [s for s in _CAPTURE["sections"]
                if not s["name"].startswith("cohort_")]
    if not sections:
        print("Nothing captured. Run the notebook from the top.")
        return None
    if not (ctx.get("folders") or []):
        # Nothing was selected, so OUTDIR still points at the working
        # directory. Writing there would drop a stray report next to
        # the notebook instead of beside a session.
        print("Nothing selected, so there is no session to write a "
              "report into. Pick a save and run again.")
        return None
    sel = ctx.get("sel")
    who = []
    if sel is not None and not getattr(sel, "empty", True) \
            and "who" in getattr(sel, "columns", []):
        who = sorted({str(w) for w in sel["who"]})
    games = len(ctx.get("folders") or [])
    title = ("Finger Rehab analysis" if len(who) != 1
             else f"Finger Rehab analysis: {who[0]}")
    subtitle = (f"{games} game(s) | run {_CAPTURE['started']} | "
                f"{len(sections)} section(s) | pick {ctx.get('pick')!r}")
    page = _build_report_html(title, subtitle, sections)

    OUTDIR.mkdir(parents=True, exist_ok=True)
    here = OUTDIR / "report.html"
    here.write_text(page, encoding="utf-8")
    written = [here]

    # One participant means this run is a read of that person. It only
    # becomes their STANDING summary when it covers every game they
    # have on disk: a one-game pick is a one-block view, and replacing
    # the full read with it under the same filename breaks the file's
    # promise. A mixed selection never writes one: there is no single
    # person it would belong to.
    if len(who) == 1:
        # Under the tree this run read, not the tree found at import:
        # a run pointed at another sessions folder (a copy, a test's
        # temp tree) must not plant its summary in the default one.
        pdir = (Path(ctx.get("root") or SESSIONS_DIR)
                / "individual_patient_results" / _safe_name(who[0]))
        pdir.mkdir(parents=True, exist_ok=True)
        cat_all = ctx.get("cat")
        sel_set = {str(f) for f in (ctx.get("folders") or [])}
        theirs = (set(cat_all.loc[cat_all["who"].astype(str) == who[0],
                                  "folder"].astype(str))
                  if cat_all is not None
                  and not getattr(cat_all, "empty", True) else set())
        full_read = bool(theirs) and theirs <= sel_set
        note = "" if full_read else "  (partial, standing summary kept)"
        if full_read:
            summary = pdir / "summary.html"
            summary.write_text(page, encoding="utf-8")
            written.append(summary)
        else:
            print(f"partial pick of {who[0]}: "
                  f"{pdir / 'summary.html'} left as the last full read")
        with (pdir / "runs.txt").open("a", encoding="utf-8") as fh:
            fh.write(f"{_CAPTURE['started']}  {games} game(s)  "
                     f"{len(sections)} section(s)  "
                     f"pick={ctx.get('pick')!r}{note}\n")

    for p in written:
        print(f"report -> {p}")
    ran = {s["name"] for s in sections}
    missing = [n for n in ctx.get("results", {})
               if n not in ran and not n.startswith("cohort_")]
    if missing:
        print("not captured (did not go through a sec_ function): "
              + ", ".join(missing))
    return here


def keep(ctx, name, value):
    """File a section's result under the selection it was computed on,
    and capture what that section printed and drew for the report."""
    ctx.setdefault("results", {})[name] = (ctx.get("selection"), value)
    _capture_section(name)
    return value


def need(ctx, *names):
    """The stored results, or a clear refusal.

    The summary cell reads what the section cells produced. Without this,
    running the summary after re-running prepare() but not the sections
    quietly reuses the previous selection's tables.
    """
    out, missing, stale = {}, [], []
    stored = ctx.get("results", {})
    for name in names:
        if name not in stored:
            missing.append(name)
            continue
        key, value = stored[name]
        if key != ctx.get("selection"):
            stale.append(name)
            continue
        out[name] = value
    if missing or stale:
        lines = ["the sections this cell needs have not been run for the "
                 "current selection."]
        if missing:
            lines.append(f"   never run here : {', '.join(missing)}")
        if stale:
            lines.append(f"   run on an older selection : {', '.join(stale)}")
        lines.append("   Run every cell from 'Load the selection' down to "
                     "this one, in order.")
        raise StaleSelection("\n".join(lines))
    return out


# ---------------------------------------------------------------- picker

def _friendly_day(day: str) -> str:
    """'today', 'yesterday', or the date itself."""
    from datetime import date, timedelta
    try:
        d = date.fromisoformat(day)
    except (ValueError, TypeError):
        return day or "unknown date"
    today = date.today()
    if d == today:
        return "today"
    if d == today - timedelta(days=1):
        return "yesterday"
    if (today - d).days < 7:
        return d.strftime("%A")          # Monday, Tuesday, ...
    return d.strftime("%d %b")           # 28 Jul


def menu_options(cat: pd.DataFrame):
    """Dropdown entries, newest first. Returns [(label, pick), ...].

    Three groupings, which is every scope the analysis is actually read
    at: one game, one session, or one person across every day they
    played. Headings come back with a pick of None.

    "Most recent game" and "everything together" are deliberately absent.
    The first row of the first group is the most recent game, so an extra
    entry for it was the same analysis under two names, and pooling
    different people into one set of figures is not a result anyone can
    read.
    """
    if cat.empty:
        return []
    newest = cat.iloc[::-1]              # catalogue is oldest first
    opts = [("---  one game, newest first  ---", None)]
    for idx, r in newest.iterrows():
        hit = f"{r['hit_rate']:.0%}" if pd.notna(r["hit_rate"]) else "  ?"
        # Value is the day-qualified game key, not the row number and
        # not the bare folder name. Row numbers move whenever a game
        # is added that sorts earlier, and a bare folder name collides
        # across days (two games at the same clock time on different
        # days share one), so a one-game pick would quietly load both.
        opts.append((f"   {_friendly_day(r['day'])} {r['time']}   "
                     f"{r['who']}   {r['mode']}   "
                     f"{int(r['trials'])} trials, {hit} hit",
                     game_key(r["folder"])))

    sessions = list(dict.fromkeys(newest["session"]))
    if sessions:
        opts.append(("---  one session, a person on one day  ---", None))
        for s in sessions:
            g = cat[cat["session"] == s]
            day = _friendly_day(g["day"].iloc[0])
            modes = ", ".join(dict.fromkeys(g["mode"]))
            n = len(g)
            opts.append((f"   {day}   {g['who'].iloc[0]}   "
                         f"{n} game{'s' if n != 1 else ''} ({modes})", s))

    people = list(dict.fromkeys(newest["who"]))
    if people:
        opts.append(("---  one person, every day  ---", None))
        for p in people:
            n = int((cat["who"] == p).sum())
            opts.append((f"   {p}   all {n} game{'s' if n != 1 else ''}", p))
    return opts


# ---------------------------------------------------------------- health

def check(root=None, verbose=True) -> bool:
    """Confirm the notebook can actually run before anything else.

    Checks the packages, that the sessions folder exists, and that there
    is something in it. Prints what to do about anything missing rather
    than failing halfway through an analysis.
    """
    ok = True
    print("CHECKING SETUP")
    print("-" * 52)

    for mod, why, install in (
        ("pandas", "reading the CSVs", "pandas"),
        ("numpy", "the maths", "numpy"),
        ("matplotlib", "the plots", "matplotlib"),
        ("ipywidgets", "the dropdown picker", "ipywidgets"),
    ):
        try:
            __import__(mod)
            if verbose:
                print(f"   ok    {mod:12} {why}")
        except ImportError:
            ok = False
            print(f"   MISSING {mod:12} {why}")
            print(f"           fix: pip install {install}")

    folder = Path(root or SESSIONS_DIR)
    if not folder.exists():
        ok = False
        print(f"\n   MISSING sessions folder. Looked next to this notebook")
        print(f"           and up the tree from {Path.cwd().resolve()}")
        print("           and found nothing. Put a sessions folder beside")
        print("           this file, or set the path in the setup cell:")
        print("             SESSIONS_DIR = Path('/full/path/to/sessions')")
    else:
        cat = build_catalogue(folder)
        if cat.empty:
            print(f"\n   sessions folder found at {folder.resolve()}")
            print("   but there are no recordings in it yet.")
            print("   Play a block in the game, then run this again.")
            ok = False
        else:
            print(f"\n   ok    {len(cat)} game(s) found in {folder.resolve()}")
            newest = cat.iloc[-1]
            print(f"         newest: {newest['day']} {newest['time']} "
                  f"{newest['who']} {newest['mode']}")

    print("-" * 52)
    print("   ready to go" if ok else "   fix the above, then run check() again")
    return ok


# -------------------------------------------------- what counts as a
# reaction time
#
# time_difference_ms is two different measurements under one name. In
# classic, adaptive and mirror it is the delay from the cue to the press,
# a reaction time, and it cannot be negative. In rhythm it is a SIGNED
# offset from the beat: the note is known in advance, so pressing early
# is normal and the column is negative about half the time.
#
# Pooling the two produces numbers that are not wrong so much as
# meaningless. A rhythm block's mean of -95 ms mixed with a classic
# block's 158 ms gave a left-right "asymmetry" of -8.027 on the shipped
# data, and a coefficient of variation of -0.475, which cannot exist.
# Every reaction-time aggregate goes through here instead.
#
# reaction mode belongs in this tuple: it is the PVT-style press-after-
# randomised-wait task (reaction.py's module docstring), and its
# time_difference_ms is a genuine cue-to-press latency by the same
# reasoning as classic and adaptive. Leaving it out was audit finding
# #106: a selection with reaction blocks measured real RTs in its own
# chapter while sec_bilateral, sec_statistics and convergence reported
# nothing, because "no cued trials" meant "no trials in CUED_MODES", not
# "no reaction times". pattern and chords are left out still: both are
# cue-to-press tasks too, but each carries its own trained/probe or
# multi-target structure that a plain pooled mean would flatten, and
# each already has its own dedicated section for exactly that reason.

CUED_MODES = ("classic", "adaptive", "mirror", "reaction")


def is_cued(trials) -> pd.Series:
    """True on trials whose time_difference_ms is a reaction time."""
    if trials.empty or "mode" not in trials.columns:
        return pd.Series(False, index=trials.index, dtype=bool)
    return trials["mode"].isin(CUED_MODES)


def is_scorable(trials) -> pd.Series:
    """True on rows a generic hit rate may score.

    reaction and buzz_hunt log EVENTS in the same CSV as trials: false
    starts, catch outcomes and the simple sub-mode's free wrong-finger
    retry. None of those is a hit or a miss, but every one of them
    passes (early_late != "Miss"), so a generic hit rate over raw rows
    counts a survived catch (CatchOk), a free retry (Wrong) and a
    false start (Early) as HITS and drifts above the mode's own
    accuracy. Same split reaction_frame's is_event applies, available
    to every table rather than one chapter.
    """
    if trials is None or trials.empty:
        idx = trials.index if trials is not None else []
        return pd.Series(True, index=idx, dtype=bool)
    err = (trials["error_type"].fillna("").astype(str)
           if "error_type" in trials.columns
           else pd.Series("", index=trials.index))
    label = (trials["early_late"].fillna("").astype(str)
             if "early_late" in trials.columns
             else pd.Series("", index=trials.index))
    never = (err.isin(REACTION_NEVER_SCORABLE)
             | (label == "CatchOk")
             | ((err == "wrong_finger") & (label == "Wrong")))
    return ~never


def reaction_times(trials, per=None):
    """Reaction times only: cued modes, misses removed, blanks removed.

    Returns the values as a Series when `per` is None, otherwise the
    rows, so callers that need to group keep their other columns.
    """
    if trials is None or trials.empty:
        empty = trials if trials is not None else pd.DataFrame()
        return empty if per == "rows" else pd.Series(dtype="float64")
    rows = trials[is_cued(trials)]
    if "time_difference_ms" not in rows.columns:
        return rows.iloc[0:0] if per == "rows" else pd.Series(dtype="float64")
    rows = rows[rows["time_difference_ms"].notna()]
    if "early_late" in rows.columns:
        rows = rows[rows["early_late"] != "Miss"]
    if per == "rows":
        return rows
    return rows["time_difference_ms"].astype(float)


def rt_stats(trials) -> dict:
    """n, mean and CV over reaction times only. CV is None rather than
    negative: a negative CV means beat offsets got in, not a fast
    participant."""
    v = reaction_times(trials)
    if v.empty:
        return {"n": 0, "mean_rt": np.nan, "rt_cv": np.nan}
    mean = float(v.mean())
    cv = float(v.std() / mean) if mean > 0 and len(v) > 1 else np.nan
    return {"n": int(len(v)), "mean_rt": round(mean, 1),
            "rt_cv": round(cv, 3) if pd.notna(cv) else np.nan}


def censoring_caveat(trials) -> str | None:
    """Warn when a reaction-time pool mixes adaptive trials logged
    under different response windows.

    adaptive varies timeout_ms with the cadence -- observed as low as
    ~0.39s at 140 BPM and as high as ~5.4s at bpm_min -- while classic
    stays at a fixed window (game.timeout_s, 1000ms by default). Any
    RT above the window is censored as a Miss and drops out of this
    pool entirely, so a fast-pace adaptive mean is biased low by
    survivorship (the slow responders on that trial never entered the
    RT column at all) and is not the same measurement as a classic RT.
    Returns a one-line caveat, or None when the pool has no adaptive
    trials or they all shared one window (nothing to warn about).
    """
    if trials is None or trials.empty or "mode" not in trials.columns:
        return None
    ad = trials[trials["mode"] == "adaptive"]
    if ad.empty or "timeout_ms" not in ad.columns:
        return None
    tw = pd.to_numeric(ad["timeout_ms"], errors="coerce").dropna()
    if len(tw) < 2 or (tw.max() - tw.min()) < 50:
        return None
    return (f"CAVEAT: includes adaptive trials logged under response "
            f"windows from {tw.min():.0f} to {tw.max():.0f} ms "
            "(cadence-dependent). An RT above the window is censored "
            "as a Miss, so fast-pace RTs here are biased low by "
            "survivorship and are not directly comparable to a fixed-"
            "window mode's RTs.")


def rhythm_rows(trials):
    """Rhythm trials with a usable beat offset."""
    if trials.empty or "mode" not in trials.columns:
        return trials.iloc[0:0] if not trials.empty else trials
    rhy = trials[(trials["mode"] == "rhythm")
                 & trials["time_difference_ms"].notna()]
    if "early_late" in rhy.columns:
        rhy = rhy[rhy["early_late"] != "Miss"]
    return rhy


# --------------------------------------------------- trial exclusions
# Two kinds of trial cannot be analysed. A trial where the cue command
# never reached the device was never presented, so the participant had
# nothing to react to. A press under 100 ms is faster than a cued
# reaction and is anticipation, not a response.
#
# The headline table and the exported CSV used to be built from every
# recorded trial, including these, while the exclusions section printed
# how many there were. On the shipped default that meant a hit rate and
# a mean reaction time computed from 48 trials the same notebook said
# could not be analysed.

ANTICIPATION_MS = 100.0


def exclusion_flags(trials) -> pd.DataFrame:
    """Copy of `trials` with why each trial can or cannot be analysed."""
    df = trials.copy()
    if df.empty:
        for c in ("no_cue", "anticipation", "excluded"):
            df[c] = pd.Series(dtype="bool")
        df["exclusion_reason"] = pd.Series(dtype="object")
        return df
    rt = df["time_difference_ms"] if "time_difference_ms" in df.columns \
        else pd.Series(np.nan, index=df.index)
    mode = df["mode"] if "mode" in df.columns \
        else pd.Series("", index=df.index)
    # stim_delivered False means the BUZZ channel failed (no
    # Arduino, keyboard fallback, or a hardware stim failure), but the
    # cue can still have been PRESENTED through another channel -- the
    # visual lane highlight (cue_target_shown) or the audio tone. A
    # keyboard-only mirror block under the default config has
    # buzz_before on and no Arduino, so stim_delivered is FALSE on
    # every row even though cue_target_shown is TRUE and the patient
    # plainly had something to react to. Only exclude as "no cue at
    # all" when NEITHER the visual nor the buzz channel got through;
    # cue_target_shown missing (older saves that predate the column)
    # counts as shown, matching sec_cue_modality's own truthy() rule.
    def _shown(v):
        if v is None or (isinstance(v, float) and v != v):
            return True
        if isinstance(v, str):
            return v.strip().lower() not in ("false", "0", "no", "")
        return bool(v)
    cue_shown = (df["cue_target_shown"].map(_shown)
                if "cue_target_shown" in df.columns
                else pd.Series(True, index=df.index))
    df["no_cue"] = (df.get("stim_delivered") == False) & ~cue_shown
    # Rhythm and syllables log SIGNED offsets in this column, so a
    # small value there is a good tap, not an anticipation.
    df["anticipation"] = (rt.notna() & (rt < ANTICIPATION_MS)
                          & ~mode.isin(("rhythm", "syllables")))
    df["excluded"] = df["no_cue"] | df["anticipation"]
    df["exclusion_reason"] = np.where(
        df["no_cue"], "cue never delivered",
        np.where(df["anticipation"],
                 f"faster than {ANTICIPATION_MS:.0f} ms", ""))
    return df


def analysable(trials):
    """(kept trials, flagged trials, counts). The kept frame is what any
    headline number should be built from."""
    flagged = exclusion_flags(trials)
    if flagged.empty:
        counts = {"recorded": 0, "no_cue": 0, "anticipation": 0,
                  "analysed": 0}
        return flagged, flagged, counts
    counts = {"recorded": int(len(flagged)),
              "no_cue": int(flagged["no_cue"].sum()),
              "anticipation": int(flagged["anticipation"].sum()),
              "analysed": int((~flagged["excluded"]).sum())}
    return flagged[~flagged["excluded"]].copy(), flagged, counts


# ==================================================== Load the selection
# Reads the trials, metadata and calibration for the save picked
# above, then bolts the calibrated force columns onto the trials.

# ------------------------------------------------------------- calibration
# Each pad reads a different number of counts for the same real force,
# because of where it sits under the finger. On this device one light
# press gives about 49 counts on the index and 115 on the pinky. Raw
# counts are therefore not comparable between fingers, and every
# cross-finger force number built from them is reporting the hardware as
# much as the patient.
#
# The in-app calibration records what a press was worth on each pad on
# the day. Dividing by that finger's own gap removes the pad, but it
# also removes the finger: the gap is the patient's own light press, so
# a weak finger has a small divisor and the ratio hides the weakness.
# See NORM_MEANING below. The ratio answers "how hard did this finger
# push compared with itself", not "which finger is stronger".
#
# Calibration is per hand. A bilateral block sits on eight pads and each
# hand has its own profile, saved as config/calibration/current_<hand>.json
# and stamped into the session metadata with a "hand" field. Lanes are
# normalised with their own hand's profile, and a lane whose hand was
# never calibrated stays uncorrected rather than borrowing the other
# hand's numbers.

# Per-finger lists a calibration carries, all index order
# index/middle/ring/pinky, matching FINGERS.
CAL_LISTS = ("empty", "resting", "press", "press_all",
             "preload", "gap", "on_delta", "off_delta")


# The hands a lane can belong to.
HANDS = ("right", "left")


# Counts between resting and pressing below which the calibration press
# was too weak to divide by: sensor noise would come out as a large
# normalised force. Same floor the app refuses to save a profile under.
MIN_USABLE_GAP = 20.0


# Columns add_force_columns writes.
FORCE_COLS = ("peak_force_cal", "impulse_cal", "force_window_sum_cal",
              "peak_force_N", "impulse_Ns", "force_calibrated")


def _meta_list(metas):
    """metas turns up as a dict keyed by game name, a list, or None."""
    return [m for _, m in _meta_items(metas)]


def _meta_items(metas):
    """(game name, metadata) pairs. An unreadable metadata.json comes
    through as an empty dict so the game is still counted and still
    reported as having no calibration, rather than vanishing."""
    if metas is None:
        return []
    pairs = (metas.items() if hasattr(metas, "items")
             else enumerate(metas))
    return [(str(k), v if isinstance(v, dict) else {}) for k, v in pairs]


def _finger_index(finger):
    """Accepts 'Index', 0, or a 0-based lane number. None when it makes
    no sense, which happens on keyboard rows with no lane."""
    if isinstance(finger, str):
        for i, f in enumerate(FINGERS):
            if f.lower() == finger.strip().lower():
                return i
        return None
    if finger is None or (isinstance(finger, float) and np.isnan(finger)):
        return None
    try:
        return int(finger) % len(FINGERS)
    except (TypeError, ValueError):
        return None


def lane_side(lane, hand_mode="right"):
    """Which hand a 0-based lane belongs to, or None when it makes no
    sense.

    A bilateral block puts the right hand on lanes 0 to 3 and the left on
    4 to 7. A one-handed block only ever uses 0 to 3, and those belong to
    whichever hand was played, so the lane number alone does not say
    which hand a reading came from.
    """
    mode = str(hand_mode or "right").strip().lower()
    if mode in ("left", "right"):
        return mode
    try:
        i = int(lane)
    except (TypeError, ValueError):
        return None
    if isinstance(lane, float) and np.isnan(lane):
        return None
    return "left" if i >= len(FINGERS) else "right"


def normalise_hand(hand) -> str | None:
    h = str(hand or "").strip().lower()
    return h if h in HANDS else None


def read_calibrations(meta) -> dict:
    """Every calibration a session recorded, keyed by hand.

    Calibration is measured one hand at a time, so a bilateral block can
    carry two profiles, one, or none. Three shapes turn up and all three
    have to be read, because a session written by an older build has to
    keep working:

        {"hand": "right", "gap": [...], ...}      one profile
        {"right": {...}, "left": {...}}            one per hand
        [{...}, {...}]                             a list of profiles

    A profile with no hand field at all is taken as the right hand,
    which is what the app defaults to, and that assumption is reported
    by sec_calibration rather than buried here.
    """
    raw = (meta or {}).get("calibration")
    out = {}

    def take(cal, fallback=None):
        if not isinstance(cal, dict) or not cal:
            return
        hand = normalise_hand(cal.get("hand")) or normalise_hand(fallback)
        if hand is None:
            hand = "right"
            cal = {**cal, "hand": hand, "hand_assumed": True}
        out.setdefault(hand, cal)

    if isinstance(raw, dict):
        by_hand = {k: v for k, v in raw.items()
                   if normalise_hand(k) and isinstance(v, dict)}
        if by_hand:
            for key, cal in by_hand.items():
                take(cal, key)
        else:
            take(raw)
    elif isinstance(raw, (list, tuple)):
        for cal in raw:
            take(cal)
    return out


def read_calibration(meta, hand=None) -> dict:
    """One hand's calibration, empty dict when that hand has none.

    Kept for callers that only ever look at one hand. With no hand asked
    for it returns the single profile when there is exactly one, and an
    empty dict when the session carries two, so nothing can pick up the
    wrong hand's numbers by accident.
    """
    cals = read_calibrations(meta)
    if hand is not None:
        return cals.get(normalise_hand(hand) or "", {})
    return list(cals.values())[0] if len(cals) == 1 else {}


def calibration_problems(cal) -> list:
    """Everything wrong with one calibration, empty when it is sound.

    A truncated list is the case worth catching: a calibration holding
    two of four fingers is not a calibration, and silently reading the
    missing entries as zero prints a full table that was never measured.
    """
    if not cal:
        return ["no calibration recorded"]
    problems = []
    for key in ("resting", "press", "gap"):
        seq = cal.get(key)
        if not isinstance(seq, (list, tuple)):
            problems.append(f"{key}: missing")
        elif len(seq) < len(FINGERS):
            problems.append(
                f"{key}: {len(seq)} of {len(FINGERS)} fingers recorded")
    gaps = cal.get("gap")
    if isinstance(gaps, (list, tuple)):
        for i, finger in enumerate(FINGERS):
            if i >= len(gaps):
                continue
            try:
                g = float(gaps[i])
            except (TypeError, ValueError):
                problems.append(f"{finger}: gap is not a number")
                continue
            if g <= 0:
                problems.append(
                    f"{finger}: gap is {g:.0f} counts, so that pad never "
                    f"moved between resting and pressing")
            elif g < MIN_USABLE_GAP:
                problems.append(
                    f"{finger}: gap is only {g:.0f} counts, under the "
                    f"{MIN_USABLE_GAP:.0f} a usable press needs")
    return problems


def calibration_gaps(cal) -> list:
    """Per-finger resting-to-press gap in counts, None where the entry is
    missing, unreadable or too small to divide by."""
    out = [None] * len(FINGERS)
    seq = (cal or {}).get("gap")
    if not isinstance(seq, (list, tuple)):
        return out
    for i in range(len(FINGERS)):
        if i >= len(seq):
            continue
        try:
            g = float(seq[i])
        except (TypeError, ValueError):
            continue
        if g >= MIN_USABLE_GAP:
            out[i] = g
    return out


def calibration_signature(cal) -> tuple:
    """What a calibration actually measured, for telling two of them
    apart.

    created_at is not enough on its own. Two profiles saved inside the
    same second, or one metadata.json copied into another session, share
    a timestamp while holding different numbers, and grouping on the
    timestamp then prints one table of the first game's figures over the
    lot.
    """
    out = []
    for key in CAL_LISTS:
        seq = cal.get(key)
        if not isinstance(seq, (list, tuple)):
            out.append((key, None))
            continue
        vals = []
        for v in seq:
            try:
                vals.append(round(float(v), 3))
            except (TypeError, ValueError):
                vals.append(None)
        out.append((key, tuple(vals)))
    return tuple(out)


@dataclass
class CalibrationSet:
    """Every calibration behind one selection, kept apart rather than
    collapsed.

    A selection can span games recorded under different calibrations, or
    under none at all, so there is no single set of numbers to hand back.
    Callers ask per game and per hand and get None when that game has
    nothing usable for that hand and finger.

    Per hand matters. Lanes 4 to 7 are the left hand, and normalising
    them with the right hand's gaps mixes two sets of pads into one
    number that looks corrected. A hand with no profile stays
    uncorrected instead.
    """

    # game -> {hand -> calibration}
    per_game: dict = field(default_factory=dict)
    # game -> {hand -> [gap|None] x4}
    gaps: dict = field(default_factory=dict)
    # game -> {hand -> [str]}
    problems: dict = field(default_factory=dict)
    units: dict = field(default_factory=dict)        # game -> logged unit
    played: dict = field(default_factory=dict)       # game -> {hand}
    # Per game, how many raw sensor counts one logged unit is, and how
    # many newtons. Both are 1.0 and N_PER_COUNT for the ordinary case
    # where force was logged in counts.
    counts_per_unit: dict = field(default_factory=dict)
    newtons_per_unit: dict = field(default_factory=dict)
    unit: str = "sensor counts"
    n_per_unit: float = 1.0                          # newtons per unit

    @property
    def mixed_units(self) -> bool:
        """True when the selection pools games logged in different force
        units, which no single axis label can describe honestly."""
        return len(set(self.units.values())) > 1

    def counts(self, game, value):
        """A logged force value back in raw sensor counts, which is the
        unit the calibration gaps are in."""
        try:
            return float(value) * self.counts_per_unit.get(str(game), 1.0)
        except (TypeError, ValueError):
            return float("nan")

    @property
    def calibrated_games(self) -> list:
        return sorted(k for k, v in self.per_game.items() if v)

    @property
    def uncalibrated_games(self) -> list:
        return sorted(k for k, v in self.per_game.items() if not v)

    def cals(self, game) -> dict:
        """{hand: calibration} for one game, empty when it has none."""
        return self.per_game.get(str(game)) or {}

    def hands(self, game) -> set:
        """Hands of one game that have at least one usable finger gap."""
        out = set()
        for hand, seq in (self.gaps.get(str(game)) or {}).items():
            if any(g is not None for g in seq):
                out.add(hand)
        return out

    def missing_hands(self, game, needed) -> list:
        """Hands this game has trials on but no usable calibration for."""
        have = self.hands(game)
        return sorted(h for h in needed if h and h not in have)

    @property
    def all_cals(self) -> list:
        """(game, hand, calibration) for every profile in the selection."""
        return [(g, h, c)
                for g in sorted(self.per_game)
                for h, c in sorted((self.per_game[g] or {}).items())
                if c]

    @property
    def stamps(self) -> dict:
        """Label -> the (game, hand) pairs recorded under it, oldest key
        first.

        Grouped by what each calibration measured as well as by when it
        was taken, so two different profiles sharing a timestamp stay in
        two blocks instead of one block showing the first one's numbers.
        The hand is part of the grouping too: a left and a right profile
        saved in the same second are two different measurements of two
        different sets of pads.
        """
        groups = {}
        for game, hand, cal in self.all_cals:
            key = (cal.get("created_at") or "unknown", hand,
                   calibration_signature(cal))
            groups.setdefault(key, []).append((game, hand))

        out, seen = {}, {}
        for (created, hand, _sig), pairs in sorted(groups.items(),
                                                   key=lambda kv: str(kv[0])):
            base = f"{created}  ({hand} hand)"
            seen[base] = seen.get(base, 0) + 1
            label = (base if seen[base] == 1
                     else f"{base}  (distinct calibration {seen[base]})")
            out[label] = pairs
        return out

    @property
    def status(self) -> str:
        """none, single, partial or multiple.

        Counted per hand. One bilateral game with a left and a right
        profile is one calibration of each hand, not two calibrations of
        the selection, so it must not raise the "these games span more
        than one calibration" warning.
        """
        n_cal = len(self.calibrated_games)
        if n_cal == 0:
            return "none"
        per_hand = {}
        for _game, hand, cal in self.all_cals:
            per_hand.setdefault(hand, set()).add(calibration_signature(cal))
        if any(len(sigs) > 1 for sigs in per_hand.values()):
            return "multiple"
        if n_cal < len(self.per_game):
            return "partial"
        if any(self.missing_hands(g, self.played.get(g, set()))
               for g in self.per_game):
            return "partial"
        return "single"

    @property
    def usable(self) -> bool:
        """Whether any finger of any hand of any game can be normalised."""
        return any(g is not None
                   for by_hand in self.gaps.values()
                   for seq in by_hand.values()
                   for g in seq)

    def gap(self, game, finger, hand=None):
        """Counts between resting and a light press, for one finger of
        one hand of one game. None when that pad has no usable
        calibration.

        `hand` is required whenever the game could have both. With no
        hand given it falls back to the game's only profile, and returns
        None when the game carries two, so a lane can never pick up the
        other hand's numbers by omission.
        """
        by_hand = self.gaps.get(str(game)) or {}
        if not by_hand:
            return None
        hand = normalise_hand(hand)
        if hand is None:
            if len(by_hand) != 1:
                return None
            seq = list(by_hand.values())[0]
        else:
            seq = by_hand.get(hand)
        if not seq:
            return None
        i = _finger_index(finger)
        if i is None or i >= len(seq):
            return None
        return seq[i]

    def lane_gap(self, game, lane, hand_mode="right"):
        """Gap for a 0-based lane, taken from the hand that lane sits on."""
        return self.gap(game, lane % len(FINGERS),
                        lane_side(lane, hand_mode))

    def factor(self, game, finger, hand=None):
        """Multiply raw counts above baseline by this to get force as a
        fraction of that finger's own reference press."""
        g = self.gap(game, finger, hand)
        return None if not g else 1.0 / g

    def newtons(self, value, game=None):
        """A logged force value in newtons, whichever unit it was logged
        in. Absolute, so it can be checked against Demouche."""
        try:
            per = self.newtons_per_unit.get(str(game), self.n_per_unit)
            return float(value) * per
        except (TypeError, ValueError):
            return float("nan")


def _is_newtons(unit) -> bool:
    return str(unit).strip().upper() in ("N", "NEWTON", "NEWTONS")


def played_hands(meta) -> set:
    """The hands a block was played with, from its metadata."""
    mode = str((meta or {}).get("hand") or "").strip().lower()
    if mode == "both":
        return set(HANDS)
    h = normalise_hand(mode)
    return {h} if h else set()


def hand_profiles_on_disk(root=None) -> dict:
    """The per-hand profiles sitting in config/calibration right now.

    Only used to say which hands have ever been calibrated on this
    machine, so the caveat text can name the file that is missing. These
    are NOT applied to any session: current_left.json is whatever was
    measured last, not what a past block ran under, and using it would
    put a corrected-looking number on data it never covered.
    """
    base = Path(root) if root else Path(SESSIONS_DIR).parent
    out = {}
    for hand in HANDS:
        p = base / "config" / "calibration" / f"current_{hand}.json"
        if not p.exists():
            continue
        try:
            data = json.loads(p.read_text())
        except (json.JSONDecodeError, OSError):
            continue
        if isinstance(data, dict):
            out[hand] = {"path": str(p),
                         "created_at": data.get("created_at", "unknown")}
    return out


def calibration_factors(metas, unit=None) -> CalibrationSet:
    """Per-finger normalisation factors for a selection.

    Handles the three cases that actually turn up: nothing calibrated,
    some games calibrated and some not, and several games under different
    calibrations. Nothing is collapsed to a single set of numbers,
    because collapsing is exactly what lets the oldest calibration stand
    in for the whole report.
    """
    items = _meta_items(metas)
    unit = unit or force_unit(metas)
    cs = CalibrationSet(unit=unit,
                        n_per_unit=1.0 if _is_newtons(unit) else N_PER_COUNT)
    for name, meta in items:
        cals = read_calibrations(meta)
        cs.per_game[name] = cals
        cs.gaps[name] = {h: calibration_gaps(c) for h, c in cals.items()}
        cs.problems[name] = {h: calibration_problems(c)
                             for h, c in cals.items()}
        # Which hands the block actually played, so sec_calibration can
        # say a hand was used but never calibrated rather than only
        # listing what was measured.
        cs.played[name] = played_hands(meta)

        game_unit = ((meta.get("block_summary", {}) or {}).get("force_unit")
                     or unit)
        cs.units[name] = game_unit
        # The app only logs newtons when a counts-to-newtons constant is
        # configured, and it snapshots that config with the session, so
        # the constant is there to undo when it is needed.
        snap = ((meta.get("config_snapshot") or {}).get("fsr") or {})
        try:
            const = float(snap.get("force_calibration_n_per_count") or 0)
        except (TypeError, ValueError):
            const = 0.0
        const = const or None
        if _is_newtons(game_unit):
            per_n = const or N_PER_COUNT
            cs.counts_per_unit[name] = 1.0 / per_n
            cs.newtons_per_unit[name] = 1.0
        else:
            cs.counts_per_unit[name] = 1.0
            cs.newtons_per_unit[name] = const or N_PER_COUNT
    return cs


def counts_to_newtons(counts) -> float:
    return float(counts) * N_PER_COUNT


def parse_peaks(cell) -> dict:
    """force_window_peaks looks like '1:50.000;4:200.000', lanes 1-indexed."""
    out = {}
    if not isinstance(cell, str) or not cell.strip():
        return out
    for part in cell.split(";"):
        if ":" in part:
            lane, val = part.split(":", 1)
            try:
                out[int(lane) - 1] = float(val)
            except ValueError:
                pass
    return out


def parse_peaks_normalised(cell, game=None, calset=None,
                           hand_mode="right") -> dict:
    """parse_peaks with every lane divided by that lane's own reference
    press, taken from the profile for the hand that lane sits on.

    Lanes with no usable gap are left out, so a caller can spot a partial
    result by comparing the length against parse_peaks. `hand_mode` is
    the block's hand setting: without it, lanes 4 to 7 would be divided
    by the right hand's gaps, which is eight pads normalised by four.
    """
    raw = parse_peaks(cell)
    if calset is None or not raw:
        return {}
    out = {}
    for lane0, val in raw.items():
        g = calset.lane_gap(game, lane0, hand_mode)
        if g:
            out[lane0] = calset.counts(game, val) / g
    return out


def add_force_columns(trials, calset=None) -> pd.DataFrame:
    """Copy of `trials` carrying the calibrated and the newton force
    columns next to the raw counts.

    peak_force_cal, impulse_cal and force_window_sum_cal are counts above
    baseline divided by the light press THAT finger of THAT hand gave at
    calibration, so 1.0 means the same effort as its own reference. They
    are not a between-finger strength comparison: see normalisation_note.

    peak_force_N and impulse_Ns stay absolute, because Demouche's healthy
    fingertip forces are in newtons and a ratio cannot be checked against
    them. They are not corrected for pad sensitivity either.

    Each row is normalised with the profile for the hand its lane sits
    on. A row on a hand with no profile comes back blank in the
    calibrated columns with force_calibrated False, rather than borrowing
    the other hand's gaps.

    Safe to call twice. With no calibration the calibrated columns come
    back all NaN and force_calibrated is False everywhere, which is what
    the sections check before claiming a correction was applied.
    """
    df = trials.copy()
    if df.empty:
        for c in FORCE_COLS:
            if c not in df.columns:
                df[c] = pd.Series(dtype="bool" if c == "force_calibrated"
                                  else "float64")
        return df

    peak_cal, imp_cal, win_cal, flag = [], [], [], []
    peak_n, imp_n = [], []
    for _, r in df.iterrows():
        game = r.get("game")
        hand_mode = r.get("hand_mode", "right")
        side = r.get("side") or lane_side(_lane0(r), hand_mode)
        g = (calset.gap(game, r.get("finger"), side)
             if calset is not None else None)
        pk, im = r.get("peak_force_n"), r.get("impulse_n")
        if calset is None:
            peak_n.append(float(pk) * N_PER_COUNT if pd.notna(pk) else np.nan)
            imp_n.append(float(im) * N_PER_COUNT if pd.notna(im) else np.nan)
        else:
            peak_n.append(calset.newtons(pk, game) if pd.notna(pk) else np.nan)
            imp_n.append(calset.newtons(im, game) if pd.notna(im) else np.nan)
        peak_cal.append(calset.counts(game, pk) / g
                        if g and pd.notna(pk) else np.nan)
        imp_cal.append(calset.counts(game, im) / g
                       if g and pd.notna(im) else np.nan)
        cell = r.get("force_window_peaks")
        norm = parse_peaks_normalised(cell, game, calset, hand_mode)
        # Only sum when every lane that registered has a gap to divide
        # by. A partial sum mixes normalised lanes with dropped ones and
        # is not comparable with anything.
        raw_n = len(parse_peaks(cell))
        win_cal.append(sum(norm.values())
                       if raw_n and len(norm) == raw_n else np.nan)
        flag.append(bool(g))

    df["peak_force_cal"] = peak_cal
    df["impulse_cal"] = imp_cal
    df["force_window_sum_cal"] = win_cal
    df["force_calibrated"] = flag
    df["peak_force_N"] = peak_n
    df["impulse_Ns"] = imp_n
    return df


def _lane0(row):
    """A trial row's 0-based lane, or None when it has none."""
    lane = row.get("lane")
    try:
        if pd.isna(lane):
            return None
        return int(lane) - 1
    except (TypeError, ValueError):
        return None


def ensure_force_columns(trials, calset=None) -> pd.DataFrame:
    """Add the force columns unless they are already there. Lets every
    section be called on its own with just trials and a calset, without
    recomputing when the load cell has already done it."""
    if all(c in trials.columns for c in FORCE_COLS):
        return trials
    return add_force_columns(trials, calset)


# ---------------------------------------------------------------- entry

def _point_outputs_at(folders):
    """Send figures and CSV exports into the newest analysed session.

    Analysis outputs live with the session they describe, so a session
    folder is self-contained: data, figures and exports together. With
    several sessions selected the newest one carries the run's report,
    and the print says exactly where everything went.
    """
    global FIGDIR, OUTDIR, _FIGS_CLEARED
    if not folders:
        return
    newest = max(folders, key=lambda p: str(p))
    OUTDIR = Path(newest) / "analysis"
    FIGDIR = OUTDIR / "figures"
    _FIGS_CLEARED = False
    print(f"outputs -> {OUTDIR}")


def prepare(pick="latest", root=None) -> dict:
    """Everything the sections need, built once.

    Returns a dict with cat, sel, folders, metas, sessions, trials, unit
    and calset. A notebook cell can unpack it and then call any sec_
    function on its own, in any order, with no hidden state between
    cells. `trials` already carries the calibrated and newton force
    columns.

    The selection is stamped into the context. Every later cell checks
    that stamp against the dropdown before using anything, so changing
    the pick and re-running only some cells cannot blend two selections
    into one headline table. See check_selection.
    """
    _capture_reset()
    cat = build_catalogue(root)
    if cat.empty:
        ctx = {"cat": cat, "sel": cat, "folders": [], "metas": {},
               "sessions": {}, "trials": pd.DataFrame(),
               "unit": "sensor counts", "calset": CalibrationSet(),
               "root": root, "pick": pick}
        ctx["selection"] = selection_key(pick, cat)
        ctx["results"] = {}
        return ctx
    sel = resolve(pick, cat)
    folders = [Path(p) for p in sel["folder"]]
    _point_outputs_at(folders)
    metas = load_metas(folders)
    sessions = {game_key(p): s
                for p, s in zip(sel["folder"], sel["session"])}
    trials = load_games(folders, cat)
    unit = force_unit(metas)
    calset = calibration_factors(metas, unit)
    trials = add_force_columns(trials, calset)
    return {"cat": cat, "sel": sel, "folders": folders, "metas": metas,
            "sessions": sessions, "trials": trials, "unit": unit,
            "calset": calset, "root": root, "pick": pick,
            "selection": selection_key(pick, sel), "results": {}}


# ============================== Calibration this data was recorded under
# What one light press was worth on each pad that day. Without it,
# force can't be compared between fingers.

def multi_finger_deficit(cal) -> dict:
    """Recompute the calibration's multi-finger deficit and say what can
    and cannot be read off it.

    Three problems with the saved number, all of which have to be
    surfaced rather than fixed silently:

    The guard on the saved value is any(press_all), so one sensor reading
    non-zero passes it while the other three read zero. A dead sensor
    then fabricates a large deficit that looks like a clinical finding.

    It can come out negative, meaning more force appeared with all four
    pressing than with each alone. That is not a deficit, it is either
    noise or an uneven effort, and printing it as a percentage lost is
    wrong.

    It is computed from submaximal presses that nobody asked the
    participant to match for effort, whereas the multi-finger deficit in
    the literature is defined on maximal voluntary force. So it is not
    the published measure and must not be compared against published
    values.

    Returns a dict: value (clamped, None when not measurable), signed
    (the raw figure including negatives), dead (finger names with no
    reading), notes (plain-language caveats), measurable (bool).
    """
    out = {"value": None, "signed": None, "dead": [], "notes": [],
           "measurable": False}
    if not cal:
        out["notes"].append("no calibration recorded")
        return out
    gaps = [None] * len(FINGERS)
    raw_gaps = cal.get("gap") or []
    for i in range(len(FINGERS)):
        if i < len(raw_gaps):
            try:
                gaps[i] = float(raw_gaps[i])
            except (TypeError, ValueError):
                gaps[i] = None
    press_all = cal.get("press_all") or []
    resting = cal.get("resting") or []
    if len(press_all) < len(FINGERS) or len(resting) < len(FINGERS):
        out["notes"].append(
            "the all-fingers step was skipped or only partly recorded, so "
            "there is no multi-finger measurement here")
        return out

    together = []
    for i in range(len(FINGERS)):
        try:
            together.append(max(0.0, float(press_all[i]) - float(resting[i])))
        except (TypeError, ValueError):
            together.append(0.0)
    if not any(together):
        # Nothing at all read during the all-fingers step, which means it
        # was skipped rather than that all four sensors died.
        out["notes"].append(
            "the all-fingers step was skipped, so there is no multi-finger "
            "measurement here")
        return out

    # A pad that read nothing while all four pressed, but that did move
    # for its own single press, is a dead sensor rather than a finger
    # producing no force. Either way its zero cannot be counted as lost
    # force, so the deficit is not measurable.
    for i, finger in enumerate(FINGERS):
        if together[i] <= 0 and (gaps[i] or 0) > 0:
            out["dead"].append(finger)
    empty = cal.get("empty") or []
    for i, finger in enumerate(FINGERS):
        if i < len(empty):
            try:
                if float(empty[i]) <= 1 and finger not in out["dead"]:
                    out["dead"].append(finger)
                    out["notes"].append(
                        f"{finger}: reads zero with nothing touching it, "
                        f"which is an I2C fault, not a weak finger")
            except (TypeError, ValueError):
                pass

    singles = sum(g for g in gaps if g)
    if singles <= 0:
        out["notes"].append("no usable single-finger presses to compare "
                            "against")
        return out
    signed = (singles - sum(together)) / singles
    out["signed"] = round(signed, 4)

    if out["dead"]:
        out["notes"].append(
            "sensors reading nothing during the all-fingers press: "
            + ", ".join(out["dead"])
            + ". Their zeros would be counted as lost force, so this "
              "figure is not usable until that is fixed")
        return out
    if signed < 0:
        out["notes"].append(
            f"came out at {signed * 100:+.0f} percent, meaning more force "
            f"appeared with all four fingers than with each alone. That is "
            f"not a deficit, it is an uneven effort between the two steps")
        return out

    out["value"] = round(signed, 4)
    out["measurable"] = True
    out["notes"].append(
        "measured from light submaximal presses, not from maximal "
        "voluntary force, which is how the stroke literature defines the "
        "multi-finger deficit. Treat it as a device-specific number and "
        "do not compare it against published values")
    return out


def _calibration_table(cal):
    """One per-finger table for one calibration. Missing entries come
    back as NaN, never as zero: a truncated calibration printed as a
    full table of zeros looks like a measurement that was never taken.
    """
    rows = []
    for i, finger in enumerate(FINGERS):
        def at(key):
            seq = cal.get(key)
            if not isinstance(seq, (list, tuple)) or i >= len(seq):
                return np.nan
            try:
                return float(seq[i])
            except (TypeError, ValueError):
                return np.nan
        on, gap = at("on_delta"), at("gap")
        rows.append({
            "finger": finger,
            "rest_load_counts": at("preload"),
            "press_gap_counts": gap,
            "trigger_counts": on,
            "trigger_N": (round(counts_to_newtons(on), 2)
                          if pd.notna(on) else np.nan),
            "pct_of_gap": (round(100 * on / gap, 0)
                           if pd.notna(on) and pd.notna(gap) and gap else
                           np.nan),
        })
    return pd.DataFrame(rows)


def sec_calibration(metas, sessions=None, calset=None):
    """What a press meant on the day, taken from the calibration each
    game recorded rather than from whatever the config says now.

    One block per distinct calibration per hand. Nothing is collapsed: a
    selection can span several, and printing the first one as though it
    covered the lot reports the OLDEST calibration and asserts the newer
    games ran under it. A left profile does not stand in for the right
    hand either, because they are eight different pads.

    `sessions` maps game folder name to session label, so the counts
    below can say games and sessions separately. A game is one block of
    one mode; a session is one person on one day. Without it only games
    are counted, because game folders are all this function can see.

    A game recorded before the in-app calibration existed carries
    nothing here. Its force numbers are still valid in counts, but they
    cannot be compared across fingers, and the counts-to-newtons
    conversion rests on the datasheet figure alone.
    """
    print("\n" + "=" * 62)
    print("CALIBRATION THIS DATA WAS RECORDED UNDER")
    print("=" * 62)

    cs = calset if calset is not None else calibration_factors(metas)
    n_games = len(cs.per_game)
    if sessions:
        n_sessions = len({sessions.get(g, g) for g in cs.per_game})
        scope = f"{n_games} game(s) across {n_sessions} session(s)"
    else:
        scope = f"{n_games} game(s)"
    print(f"{scope}   (a game is one block, a session is one person on "
          f"one day)\n")

    if cs.status == "none":
        print("None of these games recorded a calibration.")
        print("Force stays in raw counts, which are NOT comparable between")
        print("fingers, and the newton conversion comes from the SingleTact")
        print("datasheet rather than from this device.")
        return None

    if cs.uncalibrated_games:
        missing = cs.uncalibrated_games
        line = f"{len(missing)} of {n_games} game(s) have no calibration"
        if sessions:
            miss_sessions = sorted({sessions.get(g, g) for g in missing})
            line += f", covering {len(miss_sessions)} session(s)"
        print(line + ":")
        for g in missing:
            print(f"   {g}" + (f"   ({sessions.get(g, '?')})"
                               if sessions else ""))
        print("Their force values cannot be normalised, so they are left")
        print("out of the calibrated columns rather than pooled with the")
        print("rest.\n")

    # A hand that was played but never calibrated. This is the case the
    # per-hand split exists for: the bilateral force numbers used to be
    # normalised entirely with whichever hand happened to be measured.
    gaps_missing = {g: cs.missing_hands(g, cs.played.get(g, set()))
                    for g in cs.per_game if cs.per_game[g]}
    gaps_missing = {g: m for g, m in gaps_missing.items() if m}
    if gaps_missing:
        print("HANDS PLAYED WITH NO USABLE CALIBRATION:")
        for game, miss in gaps_missing.items():
            print(f"   {game}: no profile for the {', '.join(miss)} hand")
        print("Those lanes are left uncorrected. They are NOT normalised")
        print("with the other hand's gaps, because the two hands sit on")
        print("eight different pads and borrowing four of them produces a")
        print("corrected-looking number off the wrong sensors.")
        on_disk = hand_profiles_on_disk()
        for hand in HANDS:
            if hand in on_disk:
                print(f"   config/calibration/current_{hand}.json exists, "
                      f"taken {on_disk[hand]['created_at']}. It is NOT")
                print("      applied here: it is whatever was measured last,")
                print("      not what these blocks ran under.")
            else:
                print(f"   config/calibration/current_{hand}.json does not "
                      f"exist.")
        print("Run Calibrate once per hand before the next bilateral "
              "session.\n")

    if cs.status == "multiple":
        print("WARNING: one or both hands were calibrated more than once")
        print("across these games. A press did not mean the same thing in")
        print("each, so a force change across them is not necessarily a")
        print("change in the patient. Normalising by each game's own")
        print("calibration makes the pads comparable within a finger, but a")
        print("like-for-like comparison over time still has to stay inside")
        print("one calibration.\n")

    tables = {}
    for stamp, pairs in cs.stamps.items():
        first_game, hand = pairs[0]
        games = [g for g, _ in pairs]
        cal = cs.per_game[first_game][hand]
        print("-" * 62)
        print(f"calibration taken {stamp} on "
              f"{cal.get('device_port') or 'an unrecorded port'}")
        if cal.get("hand_assumed"):
            print("   this profile carries no hand field, so it is taken as")
            print("   the right hand, which is what the app defaults to")
        label = f"{len(games)} game(s)"
        if sessions:
            label += (f" across "
                      f"{len({sessions.get(g, g) for g in games})} session(s)")
        print(f"used by {label}: {', '.join(games)}")
        problems = cs.problems[first_game][hand]
        if problems:
            print("\n   INCOMPLETE CALIBRATION, do not read the blanks as")
            print("   measurements of zero:")
            for p in problems:
                print(f"      {p}")
        tbl = _calibration_table(cal)
        print()
        _show(tbl)
        tables[stamp] = tbl

        mfd = multi_finger_deficit(cal)
        print("\nMulti-finger force deficit")
        if mfd["measurable"]:
            print(f"   {mfd['value'] * 100:.0f} percent of the single-finger "
                  f"force went missing when all four pressed together.")
        elif mfd["signed"] is not None:
            print(f"   not usable. Raw figure {mfd['signed'] * 100:+.0f} "
                  f"percent.")
        else:
            print("   not measured.")
        for note in mfd["notes"]:
            print(f"   caveat: {note}")
    print("-" * 62)
    print("What these gaps can and cannot be used for:")
    print(normalisation_note("   "))
    return tables[list(tables)[0]] if len(tables) == 1 else tables


# ============================================================== Overview
# A row per game, then time on task with the pauses taken out.

def sec_overview(trials, folders, metas):
    print("=" * 62)
    print("OVERVIEW")
    print("=" * 62)
    if not folders:
        _nothing("Nothing is selected, so there is no game to summarise.",
                 "Record a block, then run the cells above again.")
        return 0.0
    rows = []
    for f in folders:
        m = metas.get(game_key(f), {})
        bs = m.get("block_summary", {}) or {}
        sub = (trials[trials["folder"] == str(f)]
               if "folder" in trials.columns else trials.iloc[0:0])
        rows.append({
            "game": (sub["game_label"].iloc[0] if len(sub)
                     else game_key(f)),
            "mode": bs.get("block", "?"), "hand": m.get("hand", "?"),
            "trials": bs.get("trials", len(sub)), "hit_rate": bs.get("hit_rate"),
            "mean_rt_ms": bs.get("avg_rt_ms"), "score": bs.get("final_score"),
            "duration_s": bs.get("duration_s"),
            "paused_s": bs.get("paused_total_s", 0),
            "input": m.get("source_name", "?")})
    ov = pd.DataFrame(rows)
    _show(ov)
    on_task = (ov["duration_s"].fillna(0).sum() - ov["paused_s"].fillna(0).sum())
    print(f"\ngames        : {len(ov)}")
    print(f"total trials : {ov['trials'].sum():.0f}")
    print(f"time on task : {on_task/60:.1f} min (pauses removed)")
    if on_task > 0:
        print(f"presses/min  : {ov['trials'].sum()/(on_task/60):.1f}")
    return on_task


# ========================================================== Data quality
# Cues that never reached the device, how many trials carry a force
# reading, pauses, and any sensor drift worth a look.

def sec_quality(trials, folders, metas):
    print("\n" + "=" * 62)
    print("DATA QUALITY")
    print("=" * 62)
    if trials.empty:
        _nothing("No trials are loaded, so there is nothing to check.",
                 "Pick a save from the dropdown and run the load cell.")
        return
    if "stim_delivered" in trials.columns:
        sd = trials["stim_delivered"].dropna()
        if len(sd):
            # Compare against False rather than inverting. Concatenating
            # games leaves this column as object dtype, where `~` is
            # integer bitwise negation and turns the count negative.
            failed = int((sd == False).sum())
            print(f"cue commands not delivered : {failed} of {len(sd)}")
            if failed:
                print("   ^ no cue on those trials. Not ordinary misses.")
    n_force = trials["peak_force_n"].notna().sum()
    print(f"trials with force data     : {n_force} of {len(trials)}")
    if n_force == 0:
        # Blaming keyboard mode is a guess. A serial device that was
        # connected but never crossed a trigger leaves the same empty
        # column, and that is a hardware fault worth chasing rather than
        # a choice of input worth ignoring.
        sources = {str(metas.get(game_key(f), {}).get("source_name", "?"))
                   for f in folders}
        if all(s == "?" or s.lower().startswith("keyboard") for s in sources):
            print("   ^ keyboard mode, so force and individuation are empty.")
        else:
            print(f"   ^ input was {', '.join(sorted(sources))}, which is not")
            print("     keyboard mode. The sensors were connected and never")
            print("     registered a press, so check the wiring and the press")
            print("     thresholds before reading any force number below.")
    for f in folders:
        bs = metas.get(game_key(f), {}).get("block_summary", {}) or {}
        if bs.get("pauses"):
            print(f"{game_key(f)}: paused {bs['pauses']}x "
                  f"for {bs.get('paused_total_s', 0):.0f}s")
        drift = bs.get("drift_units_per_min") or {}
        vals = [(abs(v), k) for k, v in drift.items() if v is not None]
        if vals:
            worst = max(vals)
            flag = "   <- large, check the sensor" if worst[0] > 10 else ""
            print(f"{game_key(f)}: worst drift {worst[1]} "
                  f"{worst[0]:.2f}/min{flag}")


# =================================================== Comparing the games
# Hit rate, speed and consistency side by side. Stays quiet on a
# single game, because there's nothing to compare it with.

def sec_compare(trials):
    if trials.empty:
        print("No trials loaded, so there's nothing to compare.")
        return None
    games = trials["game_label"].nunique()
    if games < 2:
        print("Only one game in this selection, so there's nothing\n"
              "to compare it against. Pick a session or a participant\n"
              "from the dropdown to line games up side by side.")
        return None
    print("\n" + "=" * 62)
    print("COMPARING GAMES")
    print("=" * 62)
    rows = []
    for g, sub in trials.groupby("game_label", sort=False):
        # Reaction time only from cued modes. A rhythm block's column is
        # a signed offset from the beat, so putting it in a mean_rt
        # cell would print a negative reaction time next to real ones.
        s = rt_stats(sub)
        mode = sub["mode"].iloc[0]
        # Scorable rows only: a survived catch or a free retry is an
        # event, not a hit, and would inflate this rate.
        scored = sub[is_scorable(sub)]
        rows.append({"game": g, "mode": mode, "trials": len(sub),
                     "hit_rate": (round(float((scored["early_late"] != "Miss")
                                              .mean()), 3)
                                  if len(scored) else np.nan),
                     "rt_trials": s["n"],
                     "mean_rt": s["mean_rt"], "rt_cv": s["rt_cv"]})
    comp = pd.DataFrame(rows)
    _show(comp)
    if (comp["mode"] == "rhythm").any():
        print("mean_rt and rt_cv are blank for rhythm blocks. Their")
        print("time_difference_ms is a signed offset from the beat, not a")
        print("reaction time, so it cannot go in the same column. The")
        print("rhythm section below reports those offsets on their own.")
    cols = [MODE_COLOUR.get(m, "#94a3b8") for m in comp["mode"]]
    x = np.arange(len(comp))
    fig, ax = plt.subplots(1, 3, figsize=(14, 3.6))
    ax[0].bar(x, comp["hit_rate"], color=cols, width=.6)
    ax[0].axhspan(BAND_LO, BAND_HI, color="#16a34a", alpha=.15)
    ax[0].set_ylim(0, 1.02); ax[0].set_ylabel("hit rate")
    ax[0].set_title("Accuracy per game")
    ax[1].bar(x, comp["mean_rt"], color=cols, width=.6)
    ax[1].set_ylabel("mean reaction time (ms)"); ax[1].set_title("Speed per game")
    ax[2].bar(x, comp["rt_cv"], color=cols, width=.6)
    ax[2].set_ylabel("reaction time CV")
    ax[2].set_title("Consistency (lower is steadier)")
    for a in ax:
        a.set_xticks(x)
        a.set_xticklabels(comp["game"], rotation=35, ha="right", fontsize=8)
    _save(fig, "game_comparison"); plt.show()
    return comp


# ========================================== Comparing the training modes
# One row per mode instead of one per block, which is the comparison
# the abstract promises.

# The Abstract commits to this one: "the collected finger pressure and
# event-based data will be analysed to compare performance across the
# bilateral, rhythmic and adaptive training modes, looking at reaction
# speed and response consistency". The per-game table above cannot
# answer it. That one is a row per recorded block with mode as a label,
# so a sitting of four adaptive blocks and one rhythm block reads as
# five results rather than two conditions.


def mode_measures(trials):
    """One row per training mode, pooled over every block and person.

    Speed is reported twice, on purpose. time_difference_ms is two
    different clocks: in classic, adaptive and mirror it is the delay
    from the cue to the press, and in rhythm the note is known in
    advance so it is a signed offset from the beat. One mean over both
    is a number with no referent. cue_rt_mean_ms therefore covers the
    cued modes and beat_abs_offset_ms covers rhythm, and nothing adds
    them together.

    Consistency does survive the difference. timing_sd_ms is the spread
    of whatever timing measure that mode recorded, and a spread answers
    "how repeatable was this person" either way.
    """
    if trials is None or trials.empty or "mode" not in trials.columns:
        return pd.DataFrame()
    rows = []
    for mode, g in trials.groupby("mode", sort=False):
        timing = g["time_difference_ms"].dropna() \
            if "time_difference_ms" in g.columns else pd.Series(dtype=float)
        if "early_late" in g.columns:
            # Scorable rows only: reaction's catch outcomes and free
            # retries are events, and (label != "Miss") reads them as
            # hits.
            scored = g[g["early_late"].notna() & (g["early_late"] != "")
                       & is_scorable(g)]
            hit = ((scored["early_late"] != "Miss").mean()
                   if len(scored) else np.nan)
        else:
            hit = np.nan
        rt = reaction_times(g)
        s = rt_stats(g)
        beat = rhythm_rows(g)["time_difference_ms"].abs() \
            if mode == "rhythm" else pd.Series(dtype=float)
        rows.append({
            "mode": mode,
            "games": g["game_label"].nunique(),
            "people": g["participant"].nunique(),
            "trials": len(g),
            "hit_rate": round(float(hit), 3) if pd.notna(hit) else np.nan,
            "timing_sd_ms": (round(float(timing.std()), 1)
                             if len(timing) > 1 else np.nan),
            "cue_rt_trials": s["n"],
            "cue_rt_mean_ms": s["mean_rt"],
            "cue_rt_cv": s["rt_cv"],
            "beat_abs_offset_ms": (round(float(beat.mean()), 1)
                                   if len(beat) else np.nan),
        })
    order = {"classic": 0, "adaptive": 1, "rhythm": 2, "mirror": 3}
    tbl = pd.DataFrame(rows)
    return (tbl.sort_values("mode", key=lambda c: c.map(
        lambda m: order.get(m, 9))).reset_index(drop=True))


def sec_mode_comparison(trials):
    """The three-way mode comparison the Abstract promises."""
    print("\n" + "=" * 62)
    print("COMPARING THE TRAINING MODES")
    print("=" * 62)
    if trials is None or trials.empty:
        _nothing("No trials are loaded, so there are no modes to compare.")
        return None
    tbl = mode_measures(trials)
    if tbl.empty:
        _nothing("No mode was recorded on these trials.")
        return None
    _show(tbl)
    print("games and people say how much is behind each row, so a mode")
    print("carried by one block of one person is not read as a condition.")
    print("cue_rt_mean_ms is the delay from the cue to the press and only")
    print("exists for classic, adaptive and mirror. beat_abs_offset_ms is")
    print("the average distance from the beat and only exists for rhythm.")
    print("They are different clocks and are never pooled. For mirror,")
    print("cue_rt_mean_ms is the LATER of the two simultaneous presses,")
    print("not a single-hand reaction time -- see BOTH HANDS for the")
    print("per-hand asynchrony mirror actually measures.")
    print("timing_sd_ms is the one speed-related measure that means the")
    print("same thing in every mode: how spread out the timing was.")
    caveat = censoring_caveat(trials)
    if caveat:
        print(caveat)
    if len(tbl) < 2:
        print(f"\nOnly the {tbl['mode'].iloc[0]} mode is in this selection,")
        print("so there is nothing to compare it against. Pick a person or")
        print("a day that covers more than one mode.")
        return tbl

    cols = [MODE_COLOUR.get(m, "#94a3b8") for m in tbl["mode"]]
    x = np.arange(len(tbl))
    fig, ax = plt.subplots(1, 3, figsize=(14, 3.6))
    ax[0].bar(x, tbl["hit_rate"], color=cols, width=.6)
    ax[0].axhspan(BAND_LO, BAND_HI, color="#16a34a", alpha=.15)
    ax[0].set_ylim(0, 1.02); ax[0].set_ylabel("hit rate")
    ax[0].set_title("Accuracy per mode")
    ax[1].bar(x, tbl["timing_sd_ms"], color=cols, width=.6)
    ax[1].set_ylabel("timing spread (ms)")
    ax[1].set_title("Consistency (lower is steadier)")
    speed = tbl["cue_rt_mean_ms"].fillna(tbl["beat_abs_offset_ms"])
    hatch = ["//" if m == "rhythm" else "" for m in tbl["mode"]]
    bars = ax[2].bar(x, speed, color=cols, width=.6)
    for b, h in zip(bars, hatch):
        if h:
            b.set_hatch(h)
    ax[2].set_ylabel("ms")
    ax[2].set_title("Speed (hatched = beat offset, not reaction time)")
    for a in ax:
        a.set_xticks(x); a.set_xticklabels(tbl["mode"])
    _save(fig, "mode_comparison"); plt.show()

    people = int(trials["participant"].nunique())
    if people < 2:
        print("\nOne participant in this selection, so any difference")
        print("between modes here is that person's, not a group result.")
    return tbl


# ============================================= Fixed cadence against RAS
# Classic as the fixed-cadence control against rhythm, on the measures
# that survive two different clocks.

# The progress report commits to this comparison three times: Section
# 2.3, "The final end of year thesis report will compare findings
# between fixed-cadence training and RAS-guided training"; Table 2,
# which lists "comparative study fixed-cadence vs RAS" as Semester 2
# scope; and Section 5.1, "Run the fixed-cadence versus RAS comparative
# study during patient trials (Weeks 9 to 10)".
#
# Classic mode is the fixed-cadence control and rhythm mode is the RAS
# condition. The analysis has to exist before the Week 9 to 10 window,
# not after, or the protocol will not be designed to feed it.

FIXED_CADENCE_MODE = "classic"
RAS_MODE = "rhythm"


def cadence_ras_rows(trials, metas=None):
    """One row for fixed cadence and one for RAS, on the measures that
    survive the difference between the two clocks.

    A classic trial's timing column is a reaction time to a cue. A
    rhythm trial's is a signed offset from a beat the patient could see
    coming. The mean levels are therefore not comparable and are not put
    next to each other. Hit rate, timing spread, the two error kinds and
    the repetition count are comparable, and they are what the
    comparison is built on.
    """
    if trials is None or trials.empty or "mode" not in trials.columns:
        return pd.DataFrame()
    rows = []
    for label, mode in (("fixed cadence (classic)", FIXED_CADENCE_MODE),
                        ("RAS (rhythm)", RAS_MODE)):
        g = trials[trials["mode"] == mode]
        if g.empty:
            continue
        # A no-press Miss row is not a press: rhythm writes a literal
        # 0.0 into time_difference_ms for it (classic writes an empty
        # cell instead), so a Miss must be dropped here too or every
        # missed note on the RAS side reads as a perfectly-on-beat
        # press and drags timing_sd_ms / timing_iqr_ms toward zero.
        timing = (g[g["early_late"] != "Miss"]["time_difference_ms"]
                  .dropna() if "early_late" in g.columns
                  else g["time_difference_ms"].dropna())
        scored = g[g["early_late"].notna() & (g["early_late"] != "")
                   & is_scorable(g)] \
            if "early_late" in g.columns else g
        if mode == RAS_MODE:
            # had_incorrect_press is hard-coded FALSE on every rhythm
            # row (rhythm's wrong presses are unmatched spurious
            # presses on a lane with no due note, not a wrong press
            # inside a scored trial, so this column can never be TRUE
            # for rhythm). Read the real count from each game's block
            # summary instead, or the RAS arm always shows zero
            # wrong-finger errors regardless of what was played.
            wrong = 0
            for name, meta in _meta_items(metas or {}):
                if name not in set(g["game"].unique()):
                    continue
                wrong += int((meta.get("block_summary", {}) or {})
                             .get("rhythm_spurious_presses", 0) or 0)
        else:
            wrong = int((g.get("had_incorrect_press") == True).sum())
        taps = []
        for name, meta in _meta_items(metas or {}):
            if name not in set(g["game"].unique()):
                continue
            v = (meta.get("block_summary", {}) or {}).get("tap_variability_cv")
            if v is not None:
                taps.append(float(v))
        rows.append({
            "condition": label,
            "games": g["game_label"].nunique(),
            "people": g["participant"].nunique(),
            "trials": len(g),
            "hit_rate": (round(float((scored["early_late"] != "Miss").mean()),
                               3) if len(scored) else np.nan),
            "misses": int((g.get("early_late") == "Miss").sum()),
            "wrong_finger": wrong,
            "timing_sd_ms": (round(float(timing.std()), 1)
                             if len(timing) > 1 else np.nan),
            "timing_iqr_ms": (round(float(timing.quantile(.75)
                                          - timing.quantile(.25)), 1)
                              if len(timing) > 3 else np.nan),
            "tap_cv": (round(float(np.mean(taps)), 3) if taps else np.nan),
        })
    return pd.DataFrame(rows)


def sec_cadence_vs_ras(trials, metas=None):
    """Fixed-cadence training against RAS-guided training."""
    print("\n" + "=" * 62)
    print("FIXED CADENCE AGAINST RAS")
    print("=" * 62)
    if trials is None or trials.empty:
        _nothing("No trials are loaded, so there is nothing to compare.")
        return None
    tbl = cadence_ras_rows(trials, metas)
    have = set(tbl["condition"]) if not tbl.empty else set()
    if len(have) < 2:
        missing = [n for n, m in (("classic, the fixed-cadence control",
                                   FIXED_CADENCE_MODE),
                                  ("rhythm, the RAS condition", RAS_MODE))
                   if trials[trials["mode"] == m].empty]
        _nothing("This comparison needs blocks of both conditions in one",
                 "selection, and this one is missing: " + ", ".join(missing)
                 + ".",
                 "Record a classic block and a rhythm block for the same",
                 "person, then pick that person or that day.")
        if not tbl.empty:
            _show(tbl)
        return tbl if not tbl.empty else None
    _show(tbl)
    print("hit_rate, the two error counts and the spread measures are")
    print("comparable between the conditions. The mean timing level is NOT:")
    print("classic records a reaction time to a cue and rhythm records a")
    print("signed offset from a beat the patient could see coming, so a")
    print("difference in the means is the task, not the person.")
    print("tap_cv is the inter-tap-interval CV the app stores, and it only")
    print("exists for rhythm blocks, so it is blank on the classic row.")
    print("CAVEAT: tap_cv also tracks the song's own interval unevenness,")
    print("not just the person's -- see stimulus_interval_cv in the rhythm")
    print("tap-variability section before reading a high tap_cv as poor")
    print("motor consistency rather than an uneven backing track.")

    x = np.arange(len(tbl))
    cols = [MODE_COLOUR["classic"], MODE_COLOUR["rhythm"]]
    fig, ax = plt.subplots(1, 3, figsize=(14, 3.6))
    ax[0].bar(x, tbl["hit_rate"], color=cols, width=.55)
    ax[0].axhspan(BAND_LO, BAND_HI, color="#16a34a", alpha=.15)
    ax[0].set_ylim(0, 1.02); ax[0].set_ylabel("hit rate")
    ax[0].set_title("Accuracy")
    ax[1].bar(x, tbl["timing_sd_ms"], color=cols, width=.55)
    ax[1].set_ylabel("timing spread (ms)")
    ax[1].set_title("Consistency (lower is steadier)")
    w = .38
    ax[2].bar(x - w / 2, tbl["misses"], w, label="missed", color="#dc2626")
    ax[2].bar(x + w / 2, tbl["wrong_finger"], w, label="wrong finger",
              color="#ea580c")
    ax[2].yaxis.set_major_locator(MaxNLocator(integer=True))
    ax[2].set_ylabel("trials"); ax[2].set_title("Errors")
    ax[2].legend(frameon=False, fontsize=8)
    for a in ax:
        a.set_xticks(x)
        a.set_xticklabels(tbl["condition"], rotation=12, ha="right",
                          fontsize=8)
    _save(fig, "cadence_vs_ras"); plt.show()

    d_hit = (tbl.loc[tbl["condition"].str.startswith("RAS"), "hit_rate"]
             .iloc[0]
             - tbl.loc[tbl["condition"].str.startswith("fixed"), "hit_rate"]
             .iloc[0])
    d_sd = (tbl.loc[tbl["condition"].str.startswith("RAS"), "timing_sd_ms"]
            .iloc[0]
            - tbl.loc[tbl["condition"].str.startswith("fixed"),
                      "timing_sd_ms"].iloc[0])
    print(f"\nRAS minus fixed cadence: hit rate {d_hit:+.3f}, "
          f"timing spread {d_sd:+.1f} ms")
    if tbl["people"].max() < 2 or tbl["games"].min() < 2:
        print("This is a bench comparison, not the study. The Week 9 to 10")
        print("version needs matched block lengths, a counterbalanced order")
        print("so the second condition is not always the practised one, and")
        print("the same cue settings on both. The statistics section at the")
        print("end puts an interval on this difference.")
    return tbl


# ========================================================= Reaction time
# How fast they reacted, per finger, over cued modes only with misses
# dropped.

def sec_reaction_time(trials):
    rt = reaction_times(trials, per="rows")
    if rt.empty:
        print("\nNo cued-mode reaction times here. This section covers")
        print("classic, adaptive and mirror blocks with a press logged")
        print("against the cue. Rhythm blocks are left out on purpose:")
        print("their timing column is an offset from the beat, not a")
        print("reaction time.")
        return rt
    print("\n" + "=" * 62)
    print("REACTION TIME")
    print("=" * 62)
    v = rt["time_difference_ms"]
    # CV through rt_stats so it can never come out negative. A negative
    # CV is impossible and only ever meant beat offsets had got in.
    s = rt_stats(rt)
    cv = f"{s['rt_cv']:.3f}" if pd.notna(s["rt_cv"]) else "not computable"
    print(f"n {len(v)}   mean {v.mean():.1f} ms   median {v.median():.1f} ms   "
          f"sd {v.std():.1f}   CV {cv}")
    print(f"fastest {v.min():.0f} ms   10th/90th "
          f"{v.quantile(.1):.0f}/{v.quantile(.9):.0f} ms")
    caveat = censoring_caveat(rt)
    if caveat:
        print(caveat)
    # Mirror's time_difference_ms is the LATER of two simultaneous
    # presses (systematically slower than a unimanual RT, since it is
    # a max of two draws), pooled in here alongside classic/adaptive's
    # single-press RTs. A block-composition shift (more mirror this
    # session than last) moves this mean/median/CV even with no change
    # in patient speed. See sec_bilateral's mirror asynchrony readout
    # for the per-hand view mirror actually measures.
    n_mirror = int((rt["mode"] == "mirror").sum()) if "mode" in rt else 0
    if n_mirror:
        print(f"CAVEAT: {n_mirror} of {len(rt)} trials here are mirror, "
              f"whose RT is the LATER")
        print("   of two simultaneous presses, not a single-hand "
              "reaction. Pooling it with")
        print("   classic/adaptive means this mean/median moves with "
              "mode mix, not just")
        print("   speed. See BOTH HANDS below for mirror's own "
              "asynchrony readout.")

    fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
    ax[0].hist(v, bins=_nbins(v), color="#2563eb", alpha=.85)
    ax[0].axvline(v.mean(), color="#dc2626", lw=2, label=f"mean {v.mean():.0f}")
    ax[0].axvline(v.median(), color="#16a34a", lw=2, ls="--",
                  label=f"median {v.median():.0f}")
    ax[0].set_xlabel("reaction time (ms)"); ax[0].set_ylabel("trials")
    ax[0].set_title("Distribution"); ax[0].legend(frameon=False)
    order = _order(rt)
    bp = ax[1].boxplot([rt[rt["finger"] == f]["time_difference_ms"]
                        for f in order], labels=order,
                       patch_artist=True, widths=.6)
    for p, f in zip(bp["boxes"], order):
        p.set_facecolor(FINGER_COLOUR[f]); p.set_alpha(.65)
    for m in bp["medians"]:
        m.set_color("white"); m.set_linewidth(2)
    ax[1].set_ylabel("reaction time (ms)"); ax[1].set_title("By finger")
    _save(fig, "reaction_time"); plt.show()

    per = (rt.groupby("finger")["time_difference_ms"]
             .agg(n="count", mean="mean", median="median", sd="std")
             .reindex(order).round(1))
    # Blank rather than negative where the mean is not positive.
    per["CV"] = (per["sd"] / per["mean"]).where(per["mean"] > 0).round(3)
    _show(per)

    if len(rt) > 8:
        fig, ax = plt.subplots(figsize=(9, 3.4))
        for g, sub in rt.groupby("game_label", sort=False):
            sub = sub.sort_values("trial")
            c = MODE_COLOUR.get(sub["mode"].iloc[0], "#2563eb")
            ax.plot(sub["trial"], sub["time_difference_ms"], "o", ms=3.5,
                    alpha=.3, color=c)
            ax.plot(sub["trial"],
                    sub["time_difference_ms"].rolling(5, min_periods=1).mean(),
                    lw=2, color=c)
        x = rt["trial"].astype(float).values
        y = rt["time_difference_ms"].astype(float).values
        slope, inter = np.polyfit(x, y, 1)
        xs = np.linspace(x.min(), x.max(), 40)
        ax.plot(xs, slope * xs + inter, "--", lw=2, color="#dc2626",
                label=f"trend {slope:+.2f} ms per trial")
        ax.set_xlabel("trial"); ax.set_ylabel("reaction time (ms)")
        ax.set_title("Across the block"); ax.legend(frameon=False, fontsize=8)
        _save(fig, "rt_learning"); plt.show()
        n_people = (int(rt["participant"].nunique())
                    if "participant" in rt.columns else 1)
        if n_people > 1:
            # A pooled slope over several people is a composition
            # effect, not anyone's learning, so it is not read as one.
            print(f"slope {slope:+.2f} ms per trial, pooled over "
                  f"{n_people} participants' blocks. That mixes people,")
            print("so it does not say any one participant got faster or")
            print("slower; the progress chapter reads change per person.")
        else:
            print(f"slope {slope:+.2f} ms per trial, so the participant "
                  f"{'got faster' if slope < 0 else 'got slower'}.")
    return rt


# ================================================= Fatigue across blocks
# Block to block rather than trial to trial, which is what separates
# tiring over a sitting from noise inside one block.

# MyChanges.md lists "Per-block fatigue slope (RT and peak force) |
# metrics.fatigue_slope | A, C" as a contribution of the data layer, and
# the app writes it into every block_summary as
# fatigue_slope_rt_ms_per_block and fatigue_slope_force_per_block.
# Nothing above reads either key.
#
# The trend the reaction-time section draws is a different quantity: it
# fits reaction time against trial index INSIDE the selection. That is
# within-block drift. Fatigue is block to block, which is what separates
# tiring over a sitting from noise inside one block.


def block_means(trials):
    """Per-block mean reaction time and mean peak force, oldest first.

    Ordered by the first timestamp in each block rather than by folder
    name, so the slope follows the order the blocks were actually played
    in.
    """
    if trials is None or trials.empty or "game" not in trials.columns:
        return pd.DataFrame()
    rows = []
    for game, g in trials.groupby("game", sort=False):
        stamp = (str(g["iso_ts"].min()) if "iso_ts" in g.columns
                 else str(game))
        s = rt_stats(g)
        force = g.get("peak_force_n", pd.Series(dtype=float)).dropna()
        sess = str(g["session"].iloc[0]) if "session" in g.columns else ""
        rows.append({
            "game": game,
            "label": g["game_label"].iloc[0],
            "who": g["participant"].iloc[0],
            # The session label is "day  who", so its first token is
            # the day: fatigue is within a sitting, and a slope must
            # not run across an overnight break.
            "day": sess.split()[0] if sess else stamp[:10],
            "mode": g["mode"].iloc[0],
            "started": stamp,
            "trials": len(g),
            "rt_trials": s["n"],
            "mean_rt": s["mean_rt"],
            "mean_force_raw": (round(float(force.mean()), 1)
                               if len(force) else np.nan),
        })
    out = pd.DataFrame(rows).sort_values("started").reset_index(drop=True)
    out["block_index"] = out.groupby("who").cumcount() + 1
    return out


def across_block_slope(values):
    """Least-squares slope of value against block index, or None when
    fewer than two blocks carry the measure. Same definition the app
    uses in metrics.fatigue_slope, recomputed here over the selection so
    it can be reproduced from the CSVs."""
    v = pd.Series(values, dtype="float64").dropna()
    if len(v) < 2:
        return None
    x = np.arange(len(v), dtype=float)
    return float(np.polyfit(x, v.values.astype(float), 1)[0])


def stored_fatigue(metas):
    """What each block recorded for itself.

    The app's value is cumulative over the blocks played in one app run,
    appended block by block, so the last block of a sitting carries the
    slope over that whole sitting and an earlier one carries less. It is
    not per block despite the key name, which is worth knowing before
    quoting a single number out of one metadata.json.
    """
    rows = []
    for name, meta in _meta_items(metas):
        bs = meta.get("block_summary", {}) or {}
        if not bs:
            continue
        rows.append({
            "game": name,
            "stored_rt_slope": bs.get("fatigue_slope_rt_ms_per_block"),
            "stored_force_slope": bs.get("fatigue_slope_force_per_block"),
        })
    return pd.DataFrame(rows)


def sec_fatigue(trials, metas):
    """Fatigue block to block, both what the app stored and what the
    trials say."""
    print("\n" + "=" * 62)
    print("FATIGUE ACROSS BLOCKS")
    print("=" * 62)
    if trials is None or trials.empty:
        _nothing("No trials are loaded, so there is no fatigue trend.")
        return None
    blocks = block_means(trials)
    stored = stored_fatigue(metas)
    if not stored.empty and (stored["stored_rt_slope"].notna().any()
                             or stored["stored_force_slope"].notna().any()):
        print("what the app stored on each block:")
        _show(stored)
        print("The app's slope is cumulative over the blocks played in one")
        print("run of the app, appended as each block finishes, so the last")
        print("block of a sitting carries the whole sitting and an earlier")
        print("one carries less. Read it that way, not as this block alone.")
    else:
        print("no stored fatigue slope on these blocks. The app only fills")
        print("it in once at least two blocks have finished in one run.")

    if len(blocks) < 2:
        print("\nOnly one block in this selection, so there is no across-")
        print("block slope to recompute. Pick a session or a person.")
        _show(blocks)
        return blocks
    _show(blocks[["label", "who", "mode", "block_index", "trials",
                  "rt_trials", "mean_rt", "mean_force_raw"]])
    caveat = censoring_caveat(trials)
    if caveat:
        print(caveat)

    # Fatigue is tiring over ONE sitting, so the fit never crosses a
    # day boundary: a slope over three separate days reads overnight
    # recovery and mode-mix changes as fatigue.
    print("\nrecomputed over this selection, per person and per day (a")
    print("slope never crosses a day boundary):")
    any_slope = False
    for (who, day), g in blocks.groupby(["who", "day"]):
        g = g.sort_values("block_index")
        rt_slope = across_block_slope(g["mean_rt"])
        f_slope = across_block_slope(g["mean_force_raw"])
        parts = []
        if rt_slope is not None:
            parts.append(f"reaction time {rt_slope:+.1f} ms per block")
        if f_slope is not None:
            parts.append(f"peak force {f_slope:+.1f} counts per block")
        if parts:
            any_slope = True
            print(f"   {who}  {day}: " + ", ".join(parts)
                  + f"   over {len(g)} blocks")
            modes_here = list(dict.fromkeys(g["mode"]))
            if len(modes_here) > 1:
                print(f"      mode mix that day: {', '.join(modes_here)}")
        else:
            print(f"   {who}  {day}: {len(g)} block(s), but no measure "
                  f"has a value in two of them")
    if not any_slope:
        print("   Nothing to fit. A slope needs the same measure present in")
        print("   at least two blocks for one person on one day.")
        return blocks

    fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
    for who, g in blocks.groupby("who"):
        g = g.sort_values("block_index")
        ax[0].plot(g["block_index"], g["mean_rt"], "o-", lw=2, label=who)
        ax[1].plot(g["block_index"], g["mean_force_raw"], "o-", lw=2,
                   label=who)
    ax[0].set_ylabel("block mean reaction time (ms)")
    ax[0].set_title("Slowing across blocks")
    ax[1].set_ylabel("block mean peak force (counts)")
    ax[1].set_title("Force fading across blocks")
    for a in ax:
        a.set_xlabel("block within the person's selection")
        a.xaxis.set_major_locator(MaxNLocator(integer=True))
        if blocks["who"].nunique() > 1:
            a.legend(frameon=False, fontsize=8)
    _save(fig, "fatigue_across_blocks"); plt.show()
    print("mean_force_raw is raw counts over whichever fingers came up, so")
    print("a change down the column can be the finger mix as well as the")
    print("person. Read the slope as a flag to look at, not a result.")
    print("The RT slope is a flag too, not a result: the mode mix changes")
    print("block to block, mirror logs the later of two presses and")
    print("adaptive censors at its cadence window, so a slope can be the")
    print("mix rather than tiring.")
    return blocks


# ====================================== Accuracy and the challenge point
# Hit rate against the 65 to 80 percent band the adaptive controller
# aims for.

def sec_accuracy(trials):
    """Hit rate against the challenge-point band.

    The band belongs to the adaptive controller, so when the selection
    holds adaptive blocks this narrows to them. That used to happen
    silently, and the printed hit rate then disagreed with the one the
    summary exported over every cued block. Both scopes are printed now,
    and the returned dict names which one the band was checked against
    so the summary can label them apart.
    """
    print("\n" + "=" * 62)
    print("ACCURACY AND THE CHALLENGE POINT")
    print("=" * 62)
    if trials.empty:
        _nothing("No trials are loaded, so there is no hit rate to show.")
        return None
    # Scorable rows only: reaction's catch outcomes and free retries
    # are events, and counting them as hits inflates the band read.
    cued = trials[is_cued(trials) & is_scorable(trials)]
    adaptive = trials[trials["mode"] == "adaptive"] if "mode" in trials \
        else trials.iloc[0:0]
    if cued.empty:
        _nothing("No classic, adaptive or mirror trials in this selection,",
                 "so there is no hit rate against the band to show. Rhythm",
                 "blocks are scored on beat timing instead, in the rhythm",
                 "section below.")
        return None
    target = adaptive if not adaptive.empty else cued
    scope = "adaptive blocks only" if not adaptive.empty \
        else "all cued blocks (classic, adaptive, mirror)"
    hit = (target["early_late"] != "Miss")
    hit_all = (cued["early_late"] != "Miss")
    print(f"SCOPE of this section: {scope}")
    print(f"   hit rate, {scope}, trial-share")
    print(f"      {hit.mean():.1%}  ({hit.sum()} of {len(hit)} trials)")
    print("   hit rate, all cued blocks (what the summary exports)")
    print(f"      {hit_all.mean():.1%}  ({hit_all.sum()} of {len(hit_all)} "
          f"trials)")
    if not adaptive.empty and len(hit) != len(hit_all):
        print("The band belongs to the adaptive controller, so it is")
        print("checked against the adaptive figure. The all-cued figure is")
        print("the one the summary exports. Two scopes, not a disagreement.")
    # The controller regulates the UNWEIGHTED MEAN of per-lane hit-rate
    # EMAs (finger_rehab/analytics/adaptive.py session_hit_rate), not the
    # trial-share rate above. weakness_bias oversamples struggling
    # fingers by design, so a lane doing badly pulls the trial-share
    # figure down harder than it pulls the per-lane mean the controller
    # actually watches; the two are not interchangeable readings of the
    # same thing.
    target_order = _order(target)
    lane_rates = [(target[target["finger"] == f]["early_late"] != "Miss").mean()
                  for f in target_order]
    lane_rates = [r for r in lane_rates if pd.notna(r)]
    lanemean = float(np.mean(lane_rates)) if lane_rates else float("nan")
    print(f"   hit rate, {scope}, per-lane mean (what the controller "
          "regulates)")
    if lane_rates:
        print(f"      {lanemean:.1%}  (mean of {len(lane_rates)} finger "
              "rates, unweighted by trial count)")
    else:
        print("      (no finger column to group by)")
    print(f"inside the {BAND_LO:.0%} to {BAND_HI:.0%} band, trial-share "
          f"({scope}): "
          f"{'yes' if BAND_LO <= hit.mean() <= BAND_HI else 'no'}")
    if lane_rates:
        print(f"inside the {BAND_LO:.0%} to {BAND_HI:.0%} band, per-lane "
              f"mean ({scope}): "
              f"{'yes' if BAND_LO <= lanemean <= BAND_HI else 'no'}")

    fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
    roll = hit.rolling(12, min_periods=3).mean()
    ax[0].axhspan(BAND_LO, BAND_HI, color="#16a34a", alpha=.15,
                  label="target 65 to 80%")
    ax[0].axhline(WILSON, color="#ca8a04", ls=":", lw=2, label="Wilson 85%")
    ax[0].plot(range(len(roll)), roll, lw=2, color="#2563eb")
    ax[0].set_ylim(0, 1.02); ax[0].set_xlabel("trial")
    ax[0].set_ylabel("hit rate (rolling 12)")
    ax[0].set_title(f"Did difficulty stay in the band ({scope})")
    ax[0].legend(frameon=False, fontsize=8)
    order = _order(target)
    # No band shading here: the controller regulates ONE shared BPM off
    # the overall rate above, not a per-finger target. This panel is
    # descriptive (how hard was each finger's own rate), and drawing
    # the band behind it would claim a per-finger guarantee the
    # controller does not make -- it reallocates practice toward weak
    # fingers instead (see the sampling-share table below).
    ax[1].bar(order, [(target[target["finger"] == f]["early_late"] != "Miss").mean()
                      for f in order],
              color=[FINGER_COLOUR[f] for f in order], width=.6)
    if lane_rates:
        ax[1].axhline(lanemean, color="#0f172a", ls="--", lw=1.5,
                      label=f"per-lane mean {lanemean:.0%}")
    ax[1].set_ylim(0, 1.02); ax[1].set_ylabel("hit rate")
    ax[1].set_title("Per finger (not individually regulated -- see caption)")
    if lane_rates:
        ax[1].legend(frameon=False, fontsize=8)
    _save(fig, "challenge_point"); plt.show()
    print(f"share of the block inside the band: "
          f"{roll.dropna().between(BAND_LO, BAND_HI).mean():.1%}")

    if not adaptive.empty and adaptive["bpm_at_trial"].notna().any():
        fig, ax = plt.subplots(figsize=(9, 3.2))
        for g, sub in adaptive.groupby("game_label", sort=False):
            sub = sub.sort_values("trial")
            ax.plot(sub["trial"], sub["bpm_at_trial"], lw=2)
            rec = sub[sub.get("in_recovery") == True]
            if not rec.empty:
                ax.scatter(rec["trial"], rec["bpm_at_trial"], s=40,
                           color="#dc2626", zorder=5, label="recovery mode")
        ax.set_xlabel("trial"); ax.set_ylabel("BPM")
        ax.set_title("Difficulty the controller chose")
        h, l = ax.get_legend_handles_labels()
        if l:
            ax.legend(dict(zip(l, h)).values(), dict(zip(l, h)).keys(),
                      frameon=False, fontsize=8)
        _save(fig, "adaptive_bpm"); plt.show()

    # A miss and a wrong-finger press are different failures.
    if not cued.empty:
        order = _order(cued)
        miss = [(cued[cued["finger"] == f]["early_late"] == "Miss").sum()
                for f in order]
        wrong = [(cued[cued["finger"] == f]["had_incorrect_press"] == True).sum()
                 for f in order]
        fig, ax = plt.subplots(figsize=(8, 3.2))
        x = np.arange(len(order)); w = .38
        ax.bar(x - w/2, miss, w, label="missed, no press in time", color="#dc2626")
        ax.bar(x + w/2, wrong, w, label="wrong finger pressed", color="#ea580c")
        ax.set_xticks(x); ax.set_xticklabels(order); ax.set_ylabel("trials")
        ax.yaxis.set_major_locator(MaxNLocator(integer=True))
        ax.set_title("Two kinds of error"); ax.legend(frameon=False)
        _save(fig, "errors"); plt.show()

    return {"scope": scope,
            "hit_rate_scoped": round(float(hit.mean()), 3),
            "hit_rate_lanemean_scoped": (round(lanemean, 3)
                                          if lane_rates else None),
            "hit_rate_all_cued": round(float(hit_all.mean()), 3),
            "trials_scoped": int(len(hit)),
            "trials_all_cued": int(len(hit_all))}


# ================================================ Timing judgement bands
# Perfect, Great, Good and Late split out, against the windows each
# block was actually scored under.

# Thread B objective 3, as the progress report words it: "Implement
# configurable Perfect, Great, Good, and Late timing windows through
# config/default.yaml". Section 4.2.1 adds that there are four
# difficulty levels and that Medium was tightened to a 125 ms tolerance
# after testing.
#
# Every section above collapses early_late to hit or miss with
# (early_late != "Miss"), so the four bands the objective is about never
# appear anywhere. The column already holds them and the windows they
# were scored under are in each config snapshot, so both sides of the
# claim are on disk.

# Best to worst, plus the two failure labels. Early is a press inside
# the early window before the cue, which the scoring treats separately
# from a miss.
JUDGEMENTS = ["Perfect", "Great", "Good", "Late", "Early", "Miss"]


JUDGEMENT_COLOUR = {"Perfect": "#16a34a", "Great": "#65a30d",
                    "Good": "#ca8a04", "Late": "#ea580c",
                    "Early": "#a855f7", "Miss": "#dc2626"}


def judgement_windows(metas):
    """Every distinct set of timing windows behind a selection.

    Cued modes and rhythm are scored under separate blocks of config, so
    they are listed separately rather than averaged into one row that no
    block ever ran under. Same reasoning as the press-threshold audit.
    """
    out = {}
    for name, meta in _meta_items(metas):
        snap = meta.get("config_snapshot") or {}
        mode = ((meta.get("block_summary", {}) or {}).get("block")
                or (snap.get("game") or {}).get("mode") or "unknown")
        if mode == "rhythm":
            cfg = snap.get("rhythm") or {}
            keys = ("perfect_ms", "great_ms", "good_ms", "miss_ms")
            source = f"rhythm, difficulty {cfg.get('difficulty', '?')}"
        else:
            cfg = snap.get("scoring") or {}
            keys = ("perfect_ms", "great_ms", "good_ms")
            source = "scoring (cued modes)"
        vals = tuple((k, cfg.get(k)) for k in keys)
        if all(v is None for _, v in vals):
            continue
        out.setdefault((source, vals), []).append(name)
    rows = []
    for (source, vals), games in sorted(out.items(), key=lambda kv: str(kv[0])):
        row = {"windows_from": source, "games": len(games)}
        row.update({k: v for k, v in vals})
        rows.append(row)
    return pd.DataFrame(rows)


def judgement_counts(trials, by=None):
    """Counts of each judgement, as a table with one column per band."""
    if trials is None or trials.empty or "early_late" not in trials.columns:
        return pd.DataFrame()
    df = trials[trials["early_late"].notna() & (trials["early_late"] != "")]
    if df.empty:
        return pd.DataFrame()
    if by is None:
        tab = df["early_late"].value_counts().to_frame().T
        tab.index = ["all trials"]
    else:
        tab = (df.groupby(by)["early_late"].value_counts().unstack()
               .fillna(0))
    for j in JUDGEMENTS:
        if j not in tab.columns:
            tab[j] = 0
    extra = [c for c in tab.columns if c not in JUDGEMENTS]
    return tab[JUDGEMENTS + extra].astype(int)


def sec_judgement_bands(trials, metas=None):
    """The Perfect, Great, Good and Late split the objective is about."""
    print("\n" + "=" * 62)
    print("TIMING JUDGEMENT BANDS")
    print("=" * 62)
    if trials is None or trials.empty:
        _nothing("No trials are loaded, so there are no judgements to split.")
        return None
    wins = judgement_windows(metas or {})
    if not wins.empty:
        print("windows these blocks were scored under:")
        _show(wins)
        print("Nothing is averaged here. A block was scored under its own")
        print("set, and two sets in one selection means the bands do not")
        print("mean the same thing across it.")
    else:
        print("no scoring windows in the config snapshots, so the bands")
        print("below cannot be tied to the milliseconds behind them.")

    tab = judgement_counts(trials)
    if tab.empty:
        _nothing("No judgement was recorded on these trials.")
        return None
    total = int(tab.iloc[0].sum())
    share = (tab.iloc[0] / total).round(3)
    _show(pd.DataFrame({"trials": tab.iloc[0], "share": share}))

    per_finger = judgement_counts(trials, "finger")
    order = [f for f in FINGERS if f in per_finger.index] \
        if not per_finger.empty else []
    if order:
        per_finger = per_finger.reindex(order)
        print("\nper finger:")
        _show(per_finger)
    per_mode = judgement_counts(trials, "mode")
    if not per_mode.empty and len(per_mode) > 1:
        print("\nper mode (the windows differ between rhythm and the rest):")
        _show(per_mode)

    bands = [j for j in JUDGEMENTS if int(tab.iloc[0][j]) > 0]
    fig, ax = plt.subplots(1, 2 if order else 1, figsize=(12, 3.6)
                           if order else (7, 3.4))
    axes = ax if order else [ax]
    axes[0].bar(bands, [int(tab.iloc[0][j]) for j in bands],
                color=[JUDGEMENT_COLOUR[j] for j in bands], width=.6)
    axes[0].set_ylabel("trials")
    axes[0].yaxis.set_major_locator(MaxNLocator(integer=True))
    axes[0].set_title("Judgements over the selection")
    if order:
        bottom = np.zeros(len(order))
        for j in bands:
            vals = per_finger[j].values.astype(float)
            axes[1].bar(order, vals, bottom=bottom, label=j,
                        color=JUDGEMENT_COLOUR[j], width=.6)
            bottom += vals
        axes[1].set_ylabel("trials")
        axes[1].yaxis.set_major_locator(MaxNLocator(integer=True))
        axes[1].set_title("Per finger")
        axes[1].legend(frameon=False, fontsize=7, ncol=2)
    _save(fig, "judgement_bands"); plt.show()

    good_or_better = sum(int(tab.iloc[0][j])
                         for j in ("Perfect", "Great", "Good"))
    print(f"\ninside a scoring window (Perfect, Great or Good): "
          f"{good_or_better} of {total}, {good_or_better/total:.1%}")
    print("Late is a press that arrived but after every window, so it")
    print("counts as a hit everywhere else in this notebook. Miss is no")
    print("press inside the response window at all.")
    return {"overall": tab, "per_finger": per_finger, "windows": wins}


# ============================================== Outcome rates per finger
# Timeout and misclick as rates, so a finger cued ten times and one
# cued forty can be compared.

# MyChanges.md lists two rollups this notebook never read: "Misclick /
# timeout rates % per lane | metrics.outcome_rates | A" and
# "Session-level outcome rates rollup (hit / timeout / misclick %) ...
# outcome_rates_overall".
#
# The accuracy section draws missed against wrong-finger as COUNTS per
# finger, which is the right distinction but not comparable between
# fingers: a finger cued ten times and a finger cued forty times sit on
# the same axis. Rates fix that, and the app already stores its own set
# to check them against.


def outcome_rates_per_finger(trials):
    """Hit, timeout and misclick rates per finger, recomputed from the
    trial rows.

    Denominator is the trials that were scored, matching the engine,
    which counts hits plus timeouts and normalises the wrong-finger
    events over the same total. A misclick is an event rather than a
    trial outcome, so its rate can in principle exceed the miss rate
    without anything being wrong.
    """
    if trials is None or trials.empty or "early_late" not in trials.columns:
        return pd.DataFrame()
    # Scorable rows only: reaction and buzz_hunt log catch
    # outcomes and free retries in the same CSV, and a rate
    # built over them reads an event as a hit.
    df = trials[trials["early_late"].notna()
                & (trials["early_late"] != "")
                & is_scorable(trials)]
    if df.empty:
        return pd.DataFrame()
    rows = []
    for side in ("right", "left"):
        sub_side = df[df["side"] == side] if "side" in df.columns else df
        if sub_side.empty:
            continue
        for f in FINGERS:
            g = sub_side[sub_side["finger"] == f]
            if g.empty:
                continue
            n = len(g)
            rows.append({
                "hand": side, "finger": f, "n_trials": n,
                "hit_rate": round(float((g["early_late"] != "Miss").mean()),
                                  3),
                "timeout_rate": round(float((g["early_late"] == "Miss")
                                            .mean()), 3),
                "misclick_rate": round(float((g.get("had_incorrect_press")
                                              == True).mean()), 3),
            })
        if "side" not in df.columns:
            break
    return pd.DataFrame(rows)


def stored_outcome_rates(metas):
    """The per-lane and overall rates the app wrote into each block."""
    per_lane, overall = [], []
    for name, meta in _meta_items(metas):
        bs = meta.get("block_summary", {}) or {}
        for lane, d in sorted((bs.get("per_lane") or {}).items(),
                              key=lambda kv: str(kv[0])):
            try:
                i = int(lane)
            except (TypeError, ValueError):
                continue
            per_lane.append({
                "game": name, "lane": i,
                "hand": "left" if i >= len(FINGERS) else "right",
                "finger": FINGERS[i % len(FINGERS)],
                "n_trials": d.get("n_trials"),
                "hit_rate": d.get("hit_rate"),
                "timeout_rate": d.get("timeout_rate"),
                "misclick_rate": d.get("misclick_rate"),
                "rt_cv": d.get("rt_cv"),
            })
        o = bs.get("outcome_rates_overall") or {}
        if o:
            overall.append({"game": name, **o})
    return pd.DataFrame(per_lane), pd.DataFrame(overall)


def sec_outcome_rates(trials, metas=None):
    """Timeout and misclick rates per finger, as rates rather than
    counts."""
    print("\n" + "=" * 62)
    print("OUTCOME RATES PER FINGER")
    print("=" * 62)
    if trials is None or trials.empty:
        _nothing("No trials are loaded, so there are no rates to report.")
        return None
    tbl = outcome_rates_per_finger(trials)
    if tbl.empty:
        _nothing("No scored trials, so there is nothing to rate.")
        return None
    _show(tbl)
    print("timeout_rate is no press inside the response window.")
    print("misclick_rate is the share of that finger's trials where some")
    print("other finger was pressed as well. It is an event rate, so it is")
    print("not one minus the hit rate and the three do not sum to 1.")

    lanes, overall = stored_outcome_rates(metas or {})
    if not overall.empty:
        print("\nwhat each block stored for itself (outcome_rates_overall):")
        _show(overall)
    if not lanes.empty:
        print("\nwhat each block stored per lane:")
        _show(lanes)
        print("The stored figures are per block and are not recomputed")
        print("here. Where a selection covers more than one block, the")
        print("recomputed table above pools them and the two will differ.")

    order = [f for f in FINGERS if f in set(tbl["finger"])]
    fig, ax = plt.subplots(figsize=(9, 3.4))
    hands = list(dict.fromkeys(tbl["hand"]))
    w = 0.8 / max(1, len(hands) * 2)
    x = np.arange(len(order))
    for i, hand in enumerate(hands):
        sub = tbl[tbl["hand"] == hand].set_index("finger").reindex(order)
        ax.bar(x + (2 * i - len(hands) + 0.5) * w, sub["timeout_rate"], w,
               label=f"{hand} timeout", color="#dc2626",
               alpha=1.0 if hand == "right" else .55)
        ax.bar(x + (2 * i - len(hands) + 1.5) * w, sub["misclick_rate"], w,
               label=f"{hand} misclick", color="#ea580c",
               alpha=1.0 if hand == "right" else .55)
    ax.set_xticks(x); ax.set_xticklabels(order)
    ax.set_ylabel("rate")
    ax.set_title("Timeout and misclick rate per finger")
    ax.legend(frameon=False, fontsize=8, ncol=2)
    _save(fig, "outcome_rates"); plt.show()

    worst = tbl.sort_values("timeout_rate", ascending=False).iloc[0]
    print(f"\nhighest timeout rate: {worst['finger']} ({worst['hand']} hand) "
          f"at {worst['timeout_rate']:.1%} over {int(worst['n_trials'])} "
          f"trials")
    return tbl


# ================================================================= Force
# The same press as raw counts, as newtons, and as a fraction of that
# finger's own calibration press, which is effort and not strength.

def sec_force(trials, unit="sensor counts", calset=None):
    """Peak force and impulse per finger, in three units.

    Raw counts are kept because they are what the device recorded, but
    they are NOT comparable between fingers: on this device the same
    light press reads about 49 counts on the index pad and 115 on the
    pinky. The newton column is that same reading times a datasheet
    constant, so it carries the same pad bias, and it exists so the
    numbers can be put next to Demouche's healthy fingertip forces.

    The calibrated column divides out the pad, but it divides out the
    finger with it, so it answers consistency and change over time
    rather than which finger is stronger. This is the section that
    states that in full, once, because it is where force is the subject.
    """
    trials = ensure_force_columns(trials, calset)
    force = trials[trials["peak_force_n"].notna()] if not trials.empty \
        else trials
    if force.empty:
        print("\n" + "=" * 62)
        print(f"FORCE   (logged unit: {unit})")
        print("=" * 62)
        _nothing("No force data here. The force columns only fill up",
                 "while the sensors are streaming, so they're empty for",
                 "keyboard blocks and for blocks where nothing crossed",
                 "a press threshold.")
        return force
    corrected = bool(force["force_calibrated"].any())
    print("\n" + "=" * 62)
    print(f"FORCE   (logged unit: {unit})")
    print("=" * 62)
    if calset is not None and calset.mixed_units:
        print("WARNING: these games did not all log force in the same unit.")
        print("Read the per-game tables rather than the pooled figures.\n")
    if corrected:
        n_missing = int((~force["force_calibrated"]).sum())
        print(normalisation_note())
        if n_missing:
            print(f"\n{n_missing} of {len(force)} force trials had no usable")
            print("calibration for their finger on their hand, and are blank")
            print("in the calibrated columns rather than borrowing another")
            print("hand's or another finger's reference.")
    else:
        print("NOT CORRECTED for per-sensor sensitivity: no calibration is")
        print("available for these games. Differences between fingers below")
        print("mix the patient with the pad and cannot be separated. The")
        print("newton figures rest on the SingleTact datasheet alone.")
    print()

    order = _order(force)
    fig, ax = plt.subplots(1, 3, figsize=(14, 3.6))
    if corrected:
        _boxes(ax[0], [force[force["finger"] == f]["peak_force_cal"].dropna()
                       for f in order], order,
               NORM_LABEL, "Effort against each finger's own reference")
        ax[0].axhline(1.0, color="#16a34a", ls="--", lw=1.5,
                      label="own reference press")
        ax[0].legend(frameon=False, fontsize=8)
    else:
        _boxes(ax[0], [force[force["finger"] == f]["peak_force_n"]
                       for f in order], order,
               f"peak force ({unit})", "Peak force, raw (not comparable)")
    _boxes(ax[1], [force[force["finger"] == f]["peak_force_n"]
                   for f in order], order,
           f"peak force ({unit})", "Peak force, raw counts")
    ycol = "peak_force_cal" if corrected else "peak_force_n"
    g = force.sort_values("trial")
    ax[2].plot(g["trial"], g[ycol], "o", ms=3.5, alpha=.35, color="#16a34a")
    ax[2].plot(g["trial"], g[ycol].rolling(5, min_periods=1).mean(),
               lw=2, color="#16a34a")
    ax[2].set_xlabel("trial")
    ax[2].set_ylabel(NORM_LABEL if corrected else f"peak force ({unit})")
    ax[2].set_title("Across the block (fatigue check)")
    _save(fig, "force"); plt.show()

    if force["impulse_n"].notna().any():
        icol = "impulse_cal" if corrected else "impulse_n"
        ilabel = (f"impulse ({NORM_UNIT} x s)" if corrected
                  else f"impulse ({unit} x s)")
        fig, ax = plt.subplots(figsize=(7, 3.4))
        _boxes(ax, [force[force["finger"] == f][icol].dropna()
                    for f in order], order, ilabel,
               "Effort held over the press")
        _save(fig, "impulse"); plt.show()

    cols = ["peak_force_n", "peak_force_N"]
    if corrected:
        cols.insert(0, "peak_force_cal")
    tbl = (force.groupby("finger")[cols]
                .agg(["count", "mean", "std", "max"])
                .reindex(order).round(3))
    _show(tbl)
    print("peak_force_n  raw counts above baseline, not comparable "
          "between fingers")
    if corrected:
        print("peak_force_cal  effort against that finger's own reference")
        print("              press, not a strength ranking")
    print("peak_force_N  newtons, absolute, for the Demouche comparison")

    # The between-finger question, answered on the only basis that can
    # answer it at all. Absolute newtons keep the real differences in
    # them, which the ratio above does not, at the price of keeping the
    # pad differences too. Printing it separately and saying so is more
    # use than printing nothing and letting the ratio be read as a
    # ranking.
    print("\nBETWEEN-FINGER STRENGTH, absolute newtons")
    strength = (force.groupby("finger")["peak_force_N"]
                     .agg(n="count", mean="mean", sd="std", max="max")
                     .reindex(order).round(2))
    _show(strength)
    print("Pad differences are NOT corrected here. Each pad reads a")
    print("different number of counts for the same real force, so part of")
    print("any difference down this column is where the sensor sits, not")
    print("the finger. It is still the right column for the question,")
    print("because the calibrated ratio has the strength divided out of")
    print("it. Treat a ranking from this table as provisional and say so")
    print("in the write-up.")

    # Absolute force against the only healthy data in this lineage.
    per_finger_n = force.groupby("finger")["peak_force_N"].mean()
    idx = per_finger_n.get("Index")
    pky = per_finger_n.get("Pinky")
    if pd.notna(idx) or pd.notna(pky):
        print(f"\nhealthy means (Demouche 2025): index "
              f"{DEMOUCHE_2025['index_mean']} N, "
              f"little {DEMOUCHE_2025['little_mean']} N")
        if pd.notna(idx):
            print(f"   this selection, index : {idx:.2f} N")
        if pd.notna(pky):
            print(f"   this selection, pinky : {pky:.2f} N")
        print("   These are game presses against a trigger, not maximal")
        print("   voluntary force, so a lower number here is expected and")
        print("   is not by itself evidence of weakness.")
        # The newton column is the datasheet constant applied to raw
        # counts, so it carries the whole per-pad bias the calibrated
        # column exists to remove. An over-reading pad reads as a finger
        # stronger than any healthy participant Demouche measured.
        print("   The newton figures are the datasheet constant applied to")
        print("   raw counts. They are NOT corrected for where each pad")
        print("   sits, so this comparison inherits the same per-sensor")
        print("   skew as the raw counts and cannot rank one finger")
        print("   against another.")
        if corrected:
            worst = None
            for finger in order:
                g = calset_gap_summary(force, finger)
                if g is not None and (worst is None or g > worst[1]):
                    worst = (finger, g)
            if worst and worst[1] >= 2.0:
                print(f"   Here the {worst[0]} pad reads {worst[1]:.1f}x the "
                      f"counts of the")
                print("   least sensitive pad for the same share of a")
                print("   reference press, so a newton difference of that")
                print("   size between fingers could be the pads alone.")
    return force


def calset_gap_summary(force, finger):
    """How many raw counts this finger logged per unit of its own
    calibration press, relative to the least sensitive finger present.

    Used only to say out loud how far apart the pads sit in the selection
    being reported, so the newton caveat carries a number rather than a
    warning nobody sizes.
    """
    sub = force[force["finger"] == finger]
    if sub.empty:
        return None
    ratios = []
    for f in force["finger"].dropna().unique():
        other = force[force["finger"] == f]
        raw = other["peak_force_n"].mean()
        cal = other["peak_force_cal"].mean()
        if pd.notna(raw) and pd.notna(cal) and cal:
            ratios.append(raw / cal)
    mine = sub["peak_force_n"].mean() / sub["peak_force_cal"].mean() \
        if sub["peak_force_cal"].notna().any() \
        and sub["peak_force_cal"].mean() else None
    if not ratios or mine is None or not min(ratios):
        return None
    return mine / min(ratios)


# ============================== canonical signal helpers (Rayan parity)
# Two functions copied VERBATIM from finger_rehab/analytics/signal.py,
# themselves exact ports of the reference scripts Rayan sent
# (docs/research/rayan/process_force_peaks.py and
# analyze_baseline_drift_modified_newtons.R). Welber's earlier sessions
# were processed with those scripts, so the point is number-for-number
# parity, not a reinterpretation. The notebook travels without the
# package, so it carries its own copy, and
# tests/test_notebook_matches_software.py pins this text to the package
# text so the copies cannot drift apart. Do not edit these two here:
# edit signal.py and re-copy.

def teasdale_onset(force, fs, min_rise=80.0, vmax_min=30.0,
                   first_slope_min=25.0, first_step_min=12.0,
                   slope_frac=0.18, step_s=0.05,
                   search_from=0, search_to=None, prefer_first=True):
    """Movement onset on one force segment, Teasdale-style.

    Exact port of the reference implementation Rayan sent
    (docs/research/rayan/process_force_peaks.py), the detector
    Welber's earlier sessions were processed with, so onsets from
    this function are directly comparable with that work. The
    technique is citable to Teasdale, Bard, Fleury, Young and
    Proteau (1993), "Determining movement onsets from temporal
    series", Journal of Motor Behavior 25(2), 97-106. An earlier
    version of this module carried a paraphrase (velocity over
    mean + k*sd of a baseline window); it and the notebook's own
    third variant are gone so there is exactly one onset detector.

    The same function text lives in analysis/session_analysis.ipynb,
    which travels without this package. Tests pin the two copies to
    each other and this one to Rayan's file.

    Steps, defaults as in the reference file:
      1. Low-pass the raw segment (2nd order Butterworth, 20 Hz,
         filtfilt so the filter delay does not shift the onset).
      2. Velocity = first difference times fs, then Savitzky-Golay
         (11/3) and a 10 Hz low-pass, because differentiating
         amplifies exactly the noise the first filter removed.
      3. Give up unless the segment rises at least `min_rise` counts
         over the mean of its first 20 samples: below that there is
         no press to time.
      4. First pass: the earliest sample where velocity clears
         max(first_slope_min, slope_frac * vmax) AND the force
         itself rises by at least `first_step_min` counts over the
         next `step_s` seconds. The step check is what stops one
         noisy velocity spike reading as an onset.
      5. Fallback (prefer_first False, or nothing passed step 4):
         walk back from the velocity peak to where velocity drops
         below 10 percent of that peak minus one sd, the backward
         search the 1993 paper describes.

    Returns (onset_idx, force_lp, dforce): the onset sample index
    within `force` (None when nothing convincing happened), plus the
    filtered force and velocity for callers that plot them.
    `search_from` and `search_to` are sample indices bounding the
    search; note dforce is one sample shorter than force_lp because
    the difference is not padded, exactly as in the reference.
    """
    from scipy.signal import butter, filtfilt, savgol_filter
    if force is None or len(force) < 20:
        return None, None, None
    x = np.asarray(force, dtype=float)
    baseline = np.mean(x[:min(20, len(x))])

    wn = min(max(20.0 / (fs / 2.0), 1e-6), 0.999999)
    b, a = butter(2, wn, btype="low")
    try:
        force_lp = filtfilt(b, a, x)
    except ValueError:
        return None, None, None

    dforce = np.diff(force_lp) * fs
    if len(dforce) > 11:
        dforce = savgol_filter(dforce, 11, 3)
    wn_d = min(max(10.0 / (fs / 2.0), 1e-6), 0.999999)
    bd, ad = butter(2, wn_d, btype="low")
    if len(dforce) > 20:
        dforce = filtfilt(bd, ad, dforce)

    if (np.max(force_lp) - baseline) < min_rise:
        return None, force_lp, dforce

    lo = max(0, int(search_from))
    hi = len(force_lp) if search_to is None else max(lo + 5, int(search_to))
    hi = min(hi, len(force_lp))
    if hi - lo < 8 or len(dforce) < 5:
        return None, force_lp, dforce

    if prefer_first:
        d_seg = dforce[lo:hi - 1]
        vmax = float(np.max(d_seg)) if len(d_seg) else 0.0
        thr = max(first_slope_min, slope_frac * vmax)
        step_k = max(1, int(step_s * fs))
        for j in range(len(d_seg) - step_k):
            i = lo + j
            if d_seg[j] >= thr:
                if (force_lp[i + step_k] - force_lp[i]) >= first_step_min:
                    return int(i), force_lp, dforce

    vmax_ind = lo + int(np.argmax(dforce[lo:max(lo + 1, hi - 1)]))
    vmax = float(dforce[vmax_ind])
    if vmax <= 0 or vmax < vmax_min:
        return None, force_lp, dforce

    d_int = dforce[:vmax_ind + 1]
    d_rev = d_int[::-1]
    s_level = vmax * 0.1
    below = np.where(d_rev < s_level)[0]
    s_ind = int(below[0]) if len(below) else 0
    if len(d_int) - s_ind > 0:
        sd = float(np.std(d_int[:len(d_int) - s_ind]))
        if not np.isfinite(sd) or sd == 0:
            sd = 1.0
    else:
        sd = 1.0
    candidates = np.where(d_rev[s_ind:] < (s_level - sd))[0]
    if len(candidates):
        onset = len(d_int) - int(s_ind + candidates[0])
    else:
        onset = vmax_ind
    onset_idx = int(max(0, min(onset, len(force_lp) - 1)))
    if not (lo <= onset_idx < hi):
        return None, force_lp, dforce
    return onset_idx, force_lp, dforce


def lookback_baseline(values, idx, window=50):
    """Mean of the `window` samples immediately before `values[idx]`.

    The 250 ms look-back zero from Rayan's baseline drift analysis
    (docs/research/rayan/analyze_baseline_drift_modified_newtons.R):
    at 200 Hz, 50 samples is the quarter second before a press, so
    subtracting this mean re-zeroes each press against the level the
    sensor was actually resting at, instead of against one session
    constant the baseline has long since drifted away from. The same
    function text lives in the analysis notebook; tests pin the two
    copies to each other.

    Truncates at the start of the array rather than failing, matching
    the reference (start_idx = max(1, row_idx - window)). NaN inside
    the window is skipped like the reference's na.rm = TRUE, and NaN
    comes back when nothing finite sits in the window, so a press at
    sample zero cannot be zeroed against thin air.
    """
    arr = np.asarray(values, dtype=float)
    stop = int(idx)
    start = max(0, stop - int(window))
    seg = arr[start:stop]
    seg = seg[np.isfinite(seg)]
    if seg.size == 0:
        return float("nan")
    return float(seg.mean())


def estimate_fs_span(t) -> float:
    """Sample rate as count over span, never the median gap.

    Rayan's estimate_fs takes the median sample gap, which is right on
    his files but wrong on some of ours: logs stamped at serial-receive
    time arrive in bursts of about four samples with microsecond gaps
    inside a burst and ~20 ms between bursts, so the median gap is a
    microsecond figure and the rate comes out around 500-fold high
    (and the old onset code here fell for the same thing through
    np.gradient on a median dt). Count over span reads ~200 Hz on
    bursty and clean logs alike. This is the one deliberate departure
    from his loader, and this comment is where it is declared.
    """
    t = pd.to_numeric(pd.Series(t), errors="coerce").dropna().to_numpy()
    if len(t) < 2:
        return 200.0
    span = float(np.max(t) - np.min(t))
    if span <= 0:
        return 200.0
    return (len(t) - 1) / span


def parse_trial_id(detail):
    """Trial number out of a stim row's detail cell ("trial_id=7").

    Rayan's parse_trial_id looks for "trial=", which never matches the
    "trial_id=" this logger writes, so his script numbers our trials
    by arrival order instead. Harmless in his standalone output, wrong
    the moment his rows are joined back onto trials.csv, so it is
    called out here and in the handover note.
    """
    m = re.search(r"trial_id\s*=\s*(\d+)", str(detail or ""))
    return int(m.group(1)) if m else None


# ======================================================== Baseline drift
# Rayan's raw-versus-zeroed comparison reproduced on this device's own
# logs: the evidence Welber asked for that a per-press look-back zero
# beats one session-wide offset. His R script reads each sensor at
# every stim and response moment, subtracts either a session constant
# (his rig idled near 255 counts, hence his flat -255) or the mean of
# the 250 ms immediately before the row, converts to newtons, and
# regresses each series on time: a raw trend that climbs while the
# zeroed trend stays flat IS the drift, measured.
#
# HOW THIS RELATES TO WHAT THE NOTEBOOK ALREADY DID. Zeroing was not
# missing here; it was scattered and never audited in one place. The
# game zeroes at capture (EMA baseline per pad, frozen at each press's
# rising edge), the continuous modes re-tare offline on a resting
# window, the press shapes tare on the pre-cue median. What was
# genuinely missing was his drift REGRESSION, the one-figure proof
# that the static alternative would have been wrong. This chapter adds
# it, prints which tare every downstream number rides, and measures
# how far the engine's EMA sits from his flat 50-sample mean instead
# of just asserting they agree.

RAYAN_COUNTS_PER_NEWTON = 51.2   # his rig's flat constant, printed
                                 # alongside ours so his numbers and
                                 # Welber's line up with these

DRIFT_LOOKBACK_SAMPLES = 50      # his baseline_window: 250 ms at 200 Hz


def drift_events(raw, col):
    """One sensor of one raw log, read at every stim and press moment
    two ways: minus the session-static offset, and minus the 250 ms
    look-back mean. Returns a frame with the static offset stashed in
    .attrs, or None when the log is too thin to say anything.

    The static offset is the pre-first-stim resting median, standing
    in for Rayan's flat -255 (his rig's idle level; each of our pads
    idles somewhere different, 255 to 315 on the sessions checked, so
    one hardware constant would be wrong per pad from the start). The
    look-back window ends at the event moment, exactly as in his
    script, so on press rows the last few rising samples leak into
    the baseline mean; the engine avoids that by freezing its EMA at
    the rising edge, and the agreement between the two is measured by
    the caller rather than argued about. Press rows are read at press
    DETECTION (the threshold crossing), again as in his script, so
    they sit below the trial's peak force on purpose.
    """
    if raw is None or col not in raw.columns or "event" not in raw.columns:
        return None
    ev = raw["event"].fillna("").astype(str)
    samp = raw[ev == ""].dropna(subset=["t_perf"]).sort_values("t_perf")
    ts = samp["t_perf"].to_numpy(dtype=float)
    vals = pd.to_numeric(samp[col], errors="coerce").to_numpy(dtype=float)
    if len(ts) < DRIFT_LOOKBACK_SAMPLES + 10:
        return None
    events = raw[ev.isin(("stim", "press"))].dropna(subset=["t_perf"])
    if events.empty:
        return None
    stim_t = events.loc[events["event"] == "stim", "t_perf"]
    t0 = float(stim_t.iloc[0]) if len(stim_t) else float(ts[0])
    pre = vals[ts < t0]
    pre = pre[np.isfinite(pre)]
    static = (float(np.median(pre)) if len(pre) >= 20
              else float(np.nanquantile(vals, 0.05)))
    rows = []
    for _, e in events.iterrows():
        t = float(e["t_perf"])
        # The event rows' fsr cells are zeros in these logs (the
        # logger only fills them on sample rows), so the level AT an
        # event is the nearest sample by time. Rayan's R reads the
        # event row itself because his logger did fill it: same
        # quantity, different plumbing.
        i = int(np.searchsorted(ts, t, side="left"))
        j = min(max(i, 0), len(vals) - 1)
        if j > 0 and abs(ts[j - 1] - t) <= abs(ts[j] - t):
            j -= 1
        v = vals[j]
        if not np.isfinite(v):
            continue
        base = lookback_baseline(vals, i, DRIFT_LOOKBACK_SAMPLES)
        rows.append({"t": t - t0, "kind": str(e["event"]),
                     "value": float(v), "baseline": base,
                     "static_counts": float(v) - static,
                     "zeroed_counts": (float(v) - base
                                       if np.isfinite(base) else np.nan)})
    if not rows:
        return None
    out = pd.DataFrame(rows)
    out.attrs["static_offset"] = static
    return out


def raw_n_per_count(calset, game):
    """Newtons per raw ADC count for one game's session, from its own
    calibration snapshot, falling back to the SingleTact manufacturer
    equation. This is the primary conversion; Rayan's flat 51.2
    counts per newton belongs to his rig and is only printed next to
    it for comparability."""
    if calset is None:
        return N_PER_COUNT
    per_unit_n = calset.newtons(1.0, game)
    per_unit_counts = calset.counts(game, 1.0)
    if not per_unit_counts or not np.isfinite(per_unit_n):
        return N_PER_COUNT
    return per_unit_n / per_unit_counts


def tare_inventory_note():
    """Which tare every downstream force number rides, stated once.
    The audit behind the fix-or-declare rule: nothing this notebook
    reports rides an un-zeroed raw level, so there is nothing to fix,
    and the claim is falsifiable line by line below."""
    print("WHICH TARE EACH DOWNSTREAM FORCE NUMBER RIDES")
    print("   peak_force_n, impulse_n, force_window_sum, force_window_peaks")
    print("      (so also: force chapter, individuation, cross-talk, chords,")
    print("      syllable peaks, bilateral force asymmetry, headline numbers)")
    print("      zeroed AT CAPTURE by the game. Each pad keeps an EMA")
    print("      baseline (fsr.baseline_alpha from the session's own")
    print("      config snapshot; the shipped default 0.0005 at 200 Hz is")
    print("      a ~10 s time constant) that only drifts while the finger")
    print("      is up and is frozen at each press's rising edge; every")
    print("      logged force is reading minus that baseline. That is a")
    print("      much slower zero than Rayan's 250 ms look-back, so the")
    print("      two are measured against each other below, per session,")
    print("      at the alpha the session actually ran; where they")
    print("      disagree, this notebook's offline recomputations (which")
    print("      use the flat look-back or the trial tare) are the")
    print("      numbers to quote.")
    print("   force tracking, precision hold, tactile re-scoring")
    print("      trial_tare: the median raw level over the announce-phase")
    print("      rest, 1 s back to 50 ms before the first scored segment.")
    print("   press shapes in the raw-stream chapter")
    print("      median of the 250 ms before each cue.")
    print("   onset reaction times (movement onset chapter)")
    print("      the detector's own baseline handling on each cut segment.")
    print("   inter-hand correlation")
    print("      raw streams on purpose: Pearson r subtracts the mean, so")
    print("      a static offset cancels; shared slow drift can only nudge")
    print("      r upward, the caveat its own chapter already prints.")


def sec_baseline_drift(folders, metas=None, calset=None):
    """Per sensor per game: force at every stim and press computed with
    a session-static offset and with the 250 ms look-back zero, both
    regressed on time, plotted in Rayan's style (faded raw, opaque
    zeroed, trend lines with their equations printed)."""
    print("\n" + "=" * 62)
    print("BASELINE DRIFT: STATIC OFFSET AGAINST THE 250 MS LOOK-BACK")
    print("=" * 62)
    tare_inventory_note()

    summary_rows = []
    ema_agreement = []
    plotted_any = False
    for folder in folders:
        raw = load_raw(folder)
        if raw is None:
            continue
        game = game_key(folder)
        hand_mode = read_meta(Path(folder)).get("hand", "right")
        n_sensors = 8 if hand_mode == "both" else 4
        cols = [f"fsr{i}" for i in range(1, n_sensors + 1)
                if f"fsr{i}" in raw.columns]
        per_n = raw_n_per_count(calset, game)
        per_sensor = {}
        for col in cols:
            de = drift_events(raw, col)
            if de is None or (de["kind"] == "press").sum() < 5:
                continue
            per_sensor[col] = de
        if not per_sensor:
            continue

        # The engine's EMA against his flat 50-sample mean, measured on
        # the same samples at the alpha THIS session ran (from its
        # config snapshot; shipped default 0.0005): run an ungated EMA
        # over the stream and read it just before each press. Ungated
        # is the honest approximation available offline (the live one pauses
        # while pressed), so the number below is an upper bound on the
        # disagreement, and it is already small.
        ev = raw["event"].fillna("").astype(str)
        samp = raw[ev == ""].dropna(subset=["t_perf"]).sort_values("t_perf")
        ts = samp["t_perf"].to_numpy(dtype=float)
        presses = raw[ev == "press"].dropna(subset=["t_perf"])
        for col, de in per_sensor.items():
            vals = pd.to_numeric(samp[col], errors="coerce")
            meta = (metas or {}).get(game_key(folder)) or (metas or {}).get(str(folder)) or {}
            snap = (meta.get("config_snapshot") or {}) if isinstance(meta, dict) else {}
            sess_alpha = float(((snap.get("fsr") or {}).get("baseline_alpha", 0.0005))
                               if isinstance(snap.get("fsr"), dict) else 0.0005)
            ema = vals.ewm(alpha=sess_alpha).mean().to_numpy(dtype=float)
            arr = vals.to_numpy(dtype=float)
            for t in presses["t_perf"].astype(float):
                i = int(np.searchsorted(ts, t, side="left"))
                if i < DRIFT_LOOKBACK_SAMPLES + 1 or i > len(arr):
                    continue
                flat = lookback_baseline(arr, i, DRIFT_LOOKBACK_SAMPLES)
                if np.isfinite(flat) and np.isfinite(ema[i - 1]):
                    ema_agreement.append(abs(ema[i - 1] - flat))

        ncols = 2 if len(per_sensor) > 1 else 1
        nrows = int(np.ceil(len(per_sensor) / ncols))
        fig, axes = plt.subplots(nrows, ncols,
                                 figsize=(6.8 * ncols, 3.1 * nrows),
                                 squeeze=False)
        flat_axes = axes.ravel()
        for ax in flat_axes[len(per_sensor):]:
            ax.set_visible(False)
        for ax, (col, de) in zip(flat_axes, per_sensor.items()):
            # His styling: stims blue triangles, responses red circles,
            # the static series faded, the zeroed series opaque, yellow
            # trend lines, equations printed on the panel.
            for kind, colour, marker in (("stim", "#1565C0", "^"),
                                         ("press", "#C62828", "o")):
                sub = de[de["kind"] == kind]
                if sub.empty:
                    continue
                ax.plot(sub["t"], sub["static_counts"] * per_n,
                        marker, ms=3, color=colour, alpha=0.25, lw=0)
                ax.plot(sub["t"], sub["zeroed_counts"] * per_n,
                        marker, ms=3, color=colour, alpha=0.95, lw=0)
            pr = de[(de["kind"] == "press")
                    & np.isfinite(de["zeroed_counts"])]
            eq_lines = []
            slopes = {}
            for label, series, style in (("raw", "static_counts", ":"),
                                         ("zeroed", "zeroed_counts", "-")):
                if len(pr) < 2 or pr["t"].nunique() < 2:
                    slopes[label] = np.nan
                    continue
                k, c = np.polyfit(pr["t"], pr[series] * per_n, 1)
                xs = np.array([float(de["t"].min()), float(de["t"].max())])
                ax.plot(xs, k * xs + c, style, color="#FFCC00", lw=2)
                eq_lines.append(f"{label}: y = {k:.5f}x + {c:.3f}")
                slopes[label] = k
            if eq_lines:
                ax.annotate("\n".join(eq_lines), xy=(0.02, 0.96),
                            xycoords="axes fraction", va="top",
                            fontsize=8, style="italic")
            ax.axhline(0, color="#94a3b8", lw=1, ls="--")
            ax.set_title(f"{game}  {col}", fontsize=9)
            ax.set_xlabel("time from first stim (s)")
            ax.set_ylabel("force (N, session cal)")

            mean_press_counts = float(pr["zeroed_counts"].mean()) if len(pr) else np.nan
            summary_rows.append({
                "game": game, "sensor": col, "presses": int(len(pr)),
                "static offset (counts)": round(de.attrs["static_offset"], 1),
                "baseline range (counts)": round(
                    float(np.nanmax(de["baseline"]) - np.nanmin(de["baseline"])), 1),
                "raw slope (N/min)": round(slopes.get("raw", np.nan) * 60, 4),
                "zeroed slope (N/min)": round(slopes.get("zeroed", np.nan) * 60, 4),
                "mean press (N, session cal)": round(mean_press_counts * per_n, 3),
                "mean press (N, flat 51.2)": round(
                    mean_press_counts / RAYAN_COUNTS_PER_NEWTON, 3),
            })
        fig.suptitle("Faded = session-static offset, opaque = 250 ms "
                     "look-back zero; yellow = press trends", fontsize=9)
        fig.tight_layout()
        # game_key carries the day folder (day/name), so flatten
        # the slash or savefig invents a directory.
        _save(fig, "baseline_drift_" + str(game).replace("/", "_"))
        plt.show()
        plotted_any = True

    if not plotted_any:
        _nothing("No selected game has a raw stream with enough presses",
                 "to measure drift on (5 per sensor is the floor), so",
                 "there is nothing to regress. Keyboard blocks and very",
                 "short blocks land here.")
        return None

    out = pd.DataFrame(summary_rows)
    print("\nPer sensor, presses regressed on time, both offsets:")
    _show(out)
    raw_slopes = out["raw slope (N/min)"].abs().dropna()
    zero_slopes = out["zeroed slope (N/min)"].abs().dropna()
    if len(raw_slopes) and len(zero_slopes):
        print(f"median |trend|: static offset {raw_slopes.median():.4f} "
              f"N/min, look-back zero {zero_slopes.median():.4f} N/min.")
        print("Where a pad's baseline wandered (see the baseline range")
        print("column), the static column carries that wander as if it")
        print("were force and the look-back column is what the")
        print("participant actually pressed; on pads whose baseline held")
        print("flat the two columns agree and neither is wrong. This is thee evidence for keeping every")
        print("force number on a dynamic zero, which the tare inventory")
        print("above shows they already are.")
    if ema_agreement:
        v = pd.Series(ema_agreement)
        n_per = [raw_n_per_count(calset, g) for g in out["game"].unique()]
        approx_n = v.median() * float(np.mean(n_per))
        print(f"\nEngine EMA against the flat 50-sample mean, read just")
        print(f"before each press: median |difference| "
              f"{v.median():.1f} counts (~{approx_n:.3f} N), "
              f"95th percentile {v.quantile(0.95):.1f} counts, "
              f"n {len(v)}.")
        print("That is the claimed equivalence measured: the capture-time")
        print("EMA tare and Rayan's 250 ms look-back are the same baseline")
        print("to within sensor noise on this data, so the logged force")
        print("columns already follow his convention and are not")
        print("re-zeroed a second time downstream.")
    print("\nCounts per newton used above, so his and Welber's numbers")
    print("can be read against ours:")
    for g in out["game"].unique():
        per_n = raw_n_per_count(calset, g)
        print(f"   {g}: session calibration {1.0 / per_n:.1f} counts/N "
              f"(primary), Rayan's flat {RAYAN_COUNTS_PER_NEWTON} "
              f"counts/N printed alongside in the table.")
    return out


# ================================================== Finger individuation
# Target-finger force over total force, worked two ways. Only the
# absolute one lines up with the published enslavement figures.

def individuation(trials: pd.DataFrame, calset=None) -> pd.DataFrame:
    """Target-finger force over total force across all fingers, per trial.
    1.0 means only the intended finger pressed; lower means the force
    spread onto its neighbours.

    Two indices come out, on two different bases, and they are not
    interchangeable.

    `individuation` is on absolute readings. That is the basis the
    enslavement figures in the literature use (13 percent unimpaired,
    25.1 percent after stroke), so it is the one that can be put next to
    them. It carries the per-pad sensitivity bias: a finger on an
    over-reading pad inflates the denominator on every trial and drags
    the index down for every other finger.

    `individuation_cal` divides each lane by its own reference press
    first. That removes the pad, but it also weights each finger's spill
    by the inverse of that finger's own press strength, so a weak finger
    with a small reference contributes more spill per newton than a
    strong one. The per-finger ranking it produces is not the ranking on
    absolute force, and the number is NOT comparable with the published
    enslavement figures. It answers a different question: how the force
    spread relative to what each finger can produce.

    `comparable` marks the trials where both indices exist, so a raw
    against corrected comparison can be made over the same trials. That
    matters because correctability is not random: a lane is dropped
    when its finger has no usable gap, which is exactly the lanes most
    likely to be carrying spill, so comparing the corrected mean over
    its own subset against the raw mean over all trials moves the
    corrected figure up for reasons that have nothing to do with the
    calibration.

    Restricted to clean single-target hits: a Miss or a trial where the
    wrong finger got pressed first is a response error, not enslaving,
    and pooling those in reads a deliberate wrong press as "force on
    the quiet fingers" (audit finding #101).
    """
    cols = ["row_id", "trial", "finger", "game", "game_label", "hand",
            "on_target", "spillover", "individuation", "on_target_cal",
            "spillover_cal", "individuation_cal", "corrected", "comparable",
            "why_not"]
    rows = []
    if trials.empty or "force_window_peaks" not in trials.columns:
        return pd.DataFrame(columns=cols)
    if "early_late" in trials.columns:
        trials = trials[trials["early_late"] != "Miss"]
    if "had_incorrect_press" in trials.columns:
        trials = trials[trials["had_incorrect_press"] != True]
    for idx, r in trials.iterrows():
        cell = r.get("force_window_peaks")
        peaks = parse_peaks(cell)
        lane0 = _lane0(r)
        if not peaks or lane0 is None:
            continue
        # Chords ask for several fingers at once, so force on another
        # TARGET is not spill. Individuation is a single-target
        # measure; multi-target rows are skipped here and read in the
        # chord section instead.
        keys = str(r.get("correct_keys") or "")
        if len([t for t in keys.split(",") if t.strip()]) > 1:
            continue
        # Same reasoning for syllables: the lane is a syllable
        # position, not an instructed finger, so spill is undefined.
        if r.get("mode") == "syllables":
            continue
        tgt = lane0
        hand_mode = r.get("hand_mode", "right")
        side = r.get("side") or lane_side(tgt, hand_mode)
        on_target = peaks.get(tgt, 0.0)
        spill = sum(v for k, v in peaks.items() if k != tgt)
        total = on_target + spill
        if total <= 0:
            continue
        # The trials row this came from. Joining on (game, trial) is not
        # safe: two folders can carry the same block name, and trial
        # numbers restart at 1 in every block.
        row = {"row_id": idx,
               "trial": r["trial"], "finger": r["finger"],
               "game": r.get("game"), "game_label": r["game_label"],
               "hand": side,
               "on_target": on_target, "spillover": spill,
               "individuation": on_target / total,
               "on_target_cal": np.nan, "spillover_cal": np.nan,
               "individuation_cal": np.nan, "corrected": False,
               "comparable": False, "why_not": ""}
        # Every lane in the trial needs a gap, including the target lane
        # even when it registered nothing: an on-target zero is a real
        # zero and has to stay in. Normalising some lanes and leaving
        # others raw would be worse still, because the mixture is
        # invisible in the result.
        norm = parse_peaks_normalised(cell, r.get("game"), calset, hand_mode)
        tgt_gap = (calset.gap(r.get("game"), tgt % len(FINGERS), side)
                   if calset is not None else None)
        if calset is None or not calset.usable:
            row["why_not"] = "no calibration for these games"
        elif not tgt_gap:
            row["why_not"] = (f"no usable {side}-hand gap for the target "
                              f"finger")
        elif len(norm) != len(peaks):
            missing = sorted(FINGERS[k % len(FINGERS)]
                             for k in peaks if k not in norm)
            row["why_not"] = ("no usable gap on a lane that registered: "
                              + ", ".join(missing))
        if not row["why_not"]:
            on_c = norm.get(tgt, 0.0)
            spill_c = sum(v for k, v in norm.items() if k != tgt)
            total_c = on_c + spill_c
            if total_c > 0:
                row.update({"on_target_cal": on_c, "spillover_cal": spill_c,
                            "individuation_cal": on_c / total_c,
                            "corrected": True, "comparable": True})
            else:
                row["why_not"] = "nothing registered once normalised"
        rows.append(row)
    return pd.DataFrame(rows, columns=cols)


def individuation_summary(ind) -> dict:
    """The individuation figures every caller should quote, on both
    bases and over a matched set of trials.

    Two things go wrong without this. First, the corrected index is not
    on the same basis as the published enslavement figures, so quoting it
    next to them compares two different measurements. Second, the
    corrected subset is not a random sample of trials: it drops trials
    where a lane had no usable gap, and a lane with a tiny gap is a lane
    on a pad that reads little, which is where spill hides. Comparing
    the corrected mean over its own subset against the raw mean over
    every trial therefore moves the number for reasons that are nothing
    to do with the calibration.
    """
    out = {"n_all": 0, "n_matched": 0, "raw_all": np.nan,
           "raw_matched": np.nan, "cal_matched": np.nan,
           "n_dropped": 0, "why": {}}
    if ind is None or ind.empty:
        return out
    matched = ind[ind["comparable"] == True]
    out["n_all"] = int(len(ind))
    out["n_matched"] = int(len(matched))
    out["n_dropped"] = int(len(ind) - len(matched))
    out["raw_all"] = round(float(ind["individuation"].mean()), 3)
    if not matched.empty:
        out["raw_matched"] = round(float(matched["individuation"].mean()), 3)
        out["cal_matched"] = round(
            float(matched["individuation_cal"].mean()), 3)
    dropped = ind[ind["comparable"] != True]
    if not dropped.empty:
        out["why"] = dropped["why_not"].value_counts().to_dict()
    return out


def sec_individuation(trials, calset=None):
    """Finger isolation, on both bases, with the limits of each stated.

    The index is target force over total force.

    On absolute readings it is the same basis as the enslavement figures
    in the literature, so it is the one that can be quoted against them.
    It is biased by the pads: an over-reading pad inflates the
    denominator on every trial and drags the index down for every other
    finger.

    Dividing each lane by its own reference press removes the pad, but
    it weights each finger's spill by the inverse of that finger's own
    press strength. A weak finger with a small reference then contributes
    more spill per newton than a strong one, which changes the ranking
    between fingers and moves the number away from the published
    figures. That version answers "how did the force spread relative to
    what each finger can produce", and it is not comparable with 13 and
    25.1 percent.
    """
    ind = individuation(trials, calset)
    print("\n" + "=" * 62)
    print("FINGER INDIVIDUATION")
    print("=" * 62)
    if ind.empty:
        _nothing("No individuation data. This one needs the force sensors",
                 "and a force_window_peaks column with more than one lane",
                 "reading, so it's empty for keyboard blocks.")
        return ind
    s = individuation_summary(ind)
    corrected = bool(ind["corrected"].any())

    print(f"{s['n_all']} trials with usable force spread")
    print("\nABSOLUTE BASIS (target force over total force, as recorded)")
    print(f"   mean index over all {s['n_all']} trials : {s['raw_all']:.3f}")
    spill = 1 - s["raw_all"]
    print(f"   enslavement, force on the other fingers: {spill:.3f}")
    print(f"   published: unimpaired {ENSLAVEMENT_REF['unimpaired']}, "
          f"stroke {ENSLAVEMENT_REF['stroke']} (Li via Lew)")
    print("   Those published figures are computed on ABSOLUTE force, so")
    print("   this is the line to put next to them. It carries the")
    print("   per-pad sensitivity bias: an over-reading pad inflates the")
    print("   total and pushes the index down.")

    if corrected:
        print(f"\nOWN-REFERENCE BASIS (each lane divided by its own "
              f"reference press)")
        print(f"   matched trials                : {s['n_matched']} of "
              f"{s['n_all']}")
        print(f"   absolute basis, same trials   : {s['raw_matched']:.3f}")
        print(f"   own-reference basis           : {s['cal_matched']:.3f}")
        print("   Both lines cover the SAME trials, so the difference")
        print("   between them is the correction and nothing else.")
        print("   This basis is NOT comparable with the published")
        print("   enslavement figures. It weights each finger's spill by")
        print("   the inverse of that finger's own press strength, so a")
        print("   weak finger's spill counts for more, and the ranking")
        print("   between fingers is not the ranking on absolute force.")
        if s["n_dropped"]:
            print(f"\n   {s['n_dropped']} trial(s) could not be corrected:")
            for why, n in s["why"].items():
                print(f"      {n}x  {why or 'unknown'}")
            print("   Those trials are excluded from BOTH matched lines, not")
            print("   just the corrected one. Dropping them from the")
            print("   corrected figure alone would raise it, because a lane")
            print("   with no usable gap is a lane on a pad that barely")
            print("   moves, which is where spill hides.")
    else:
        # Two different situations end up here and they need different
        # answers. Either nothing was calibrated, or a calibration was
        # measured and every trial still touched a lane whose gap is
        # unusable. Calling the second one "no calibration" hides the
        # only thing the user can act on, which is that one named pad is
        # the problem and recalibrating it brings the whole basis back.
        why = {w: n for w, n in (s["why"] or {}).items() if w}
        no_cal = all("no calibration" in w for w in why) if why else True
        if no_cal:
            print("\nNo usable calibration for these games, so there is no")
            print("own-reference version. Differences between fingers")
            print("above are a mix of the hand and the hardware.")
        else:
            print("\nA calibration was recorded, but NOT ONE trial could be")
            print("fully corrected, so there is no own-reference version:")
            for w, n in why.items():
                print(f"      {n}x  {w}")
            print("   Every trial touched a lane with no usable gap, so")
            print("   correcting any of them would mix normalised lanes")
            print("   with dropped ones. Recalibrate the pad named above")
            print("   and this basis comes back. Differences between")
            print("   fingers are a mix of the hand and the hardware")
            print("   until then.")

    col = "individuation_cal" if corrected else "individuation"
    shown = ind[ind["comparable"] == True] if corrected else ind
    order = _order(shown)
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
    label = ("individuation, own-reference basis" if corrected
             else "individuation, absolute basis (not corrected)")
    _boxes(ax[0], [shown[shown["finger"] == f][col] for f in order], order,
           label, "How isolated was each finger")
    ax[0].axhline(1.0, color="#16a34a", ls="--", lw=1.5,
                  label="perfect isolation")
    ax[0].set_ylim(0, 1.05)
    ax[0].legend(frameon=False, fontsize=8)
    g = shown.sort_values("trial")
    ax[1].plot(g["trial"], g[col], "o", ms=3.5, alpha=.35, color="#7c3aed")
    ax[1].plot(g["trial"], g[col].rolling(5, min_periods=1).mean(),
               lw=2, color="#7c3aed")
    ax[1].set_ylim(0, 1.05); ax[1].set_xlabel("trial")
    ax[1].set_ylabel(label); ax[1].set_title("Across the block")
    _save(fig, "individuation"); plt.show()

    cols = ["individuation_cal", "individuation"] if corrected \
        else ["individuation"]
    _show(shown.groupby("finger")[cols]
               .agg(["count", "mean", "std"]).reindex(order).round(3))
    print("individuation      absolute basis. Comparable with the")
    print("                   published enslavement figures.")
    if corrected:
        print("individuation_cal  own-reference basis. NOT comparable with")
        print("                   them, and its per-finger ranking is not")
        print("                   the ranking on absolute force.")
    ind.attrs["summary"] = s
    return ind


# ================================================================ Rhythm
# Beat offsets and whether the tempo was actually being tracked,
# rhythm blocks only.

def sec_rhythm(trials):
    rhy = rhythm_rows(trials)
    if rhy.empty:
        print("No rhythm blocks in this selection, so there's no\n"
              "timing offset to report. This section only applies to\n"
              "games played in rhythm mode.")
        return rhy
    print("\n" + "=" * 62)
    print("RHYTHM")
    print("=" * 62)
    off = rhy["time_difference_ms"]
    who_col = (rhy["participant"].astype(str)
               if "participant" in rhy.columns
               else pd.Series("all", index=rhy.index))
    n_people = int(who_col.nunique())
    print("These are SIGNED offsets from the beat, not reaction times. The")
    print("note is known in advance, so early is normal and the sign")
    print("matters. They are kept out of every reaction-time figure.")
    if n_people == 1:
        print(f"notes {len(off)}   accuracy {off.abs().mean():.1f} ms   "
              f"bias {off.mean():+.1f} ms "
              f"({'ahead of' if off.mean() < 0 else 'behind'} the beat)   "
              f"sd {off.std():.1f} ms")
    else:
        # A bias or sd pooled over different people is nobody's
        # timing, so the headline stays per participant below.
        print(f"{len(off)} notes from {n_people} participants. No pooled "
              "bias or sd is")
        print("printed: those are per-person quantities, read them per")
        print("participant below.")

    # Everything block-sensitive is computed per BLOCK. A concatenated
    # offset series correlates one block's last note with the next
    # block's first, and between-block bias differences push a pooled
    # lag-1 positive on their own, so bias and lag-1 never cross a
    # block boundary (sec_tap_variability's lag1_r_here shares this
    # convention).
    block_rows = []
    for game, gg in rhy.groupby("game", sort=False):
        o_b = gg["time_difference_ms"].astype(float).values  # play order
        row = {"who": (str(gg["participant"].iloc[0])
                       if "participant" in gg.columns else "all"),
               "session": (str(gg["session"].iloc[0])
                           if "session" in gg.columns else ""),
               "block": (gg["game_label"].iloc[0]
                         if "game_label" in gg.columns else str(game)),
               "notes": len(o_b),
               "accuracy_ms": round(float(np.abs(o_b).mean()), 1),
               "bias_ms": round(float(o_b.mean()), 1),
               "sd_ms": (round(float(o_b.std(ddof=1)), 1)
                         if len(o_b) > 1 else np.nan),
               "lag1_r": (round(float(np.corrcoef(o_b[1:], o_b[:-1])[0, 1]),
                                3)
                          if len(o_b) > 3 else np.nan)}
        block_rows.append(row)
    btab = pd.DataFrame(block_rows)
    print("\nper block (bias and lag-1 never cross a block boundary):")
    _show(btab)

    fig, ax = plt.subplots(1, 3, figsize=(14, 3.6))
    if n_people == 1:
        ax[0].hist(off, bins=_nbins(off), color="#0ea5e9", alpha=.85)
        ax[0].axvline(off.mean(), color="#dc2626", lw=2, ls="--",
                      label=f"bias {off.mean():+.0f}")
    else:
        for who in dict.fromkeys(who_col):
            vals = off[who_col == who]
            ax[0].hist(vals, bins=_nbins(off), alpha=.5, label=str(who))
    ax[0].axvline(0, color="#16a34a", lw=2, label="on the beat")
    ax[0].set_xlabel("offset (ms), negative is early")
    ax[0].set_ylabel("notes"); ax[0].set_title("Timing around the beat")
    ax[0].legend(frameon=False, fontsize=8)
    if rhy["song_time_s"].notna().any():
        # One trace per block: two blocks share song positions, so a
        # pooled rolling mean would mix them at every restart.
        for _game, gg in rhy.groupby("game", sort=False):
            g = gg.sort_values("song_time_s")
            ax[1].plot(g["song_time_s"], g["time_difference_ms"], "o",
                       ms=3.5, alpha=.4, color="#0ea5e9")
            ax[1].plot(g["song_time_s"],
                       g["time_difference_ms"]
                       .rolling(7, min_periods=1).mean(),
                       lw=2, alpha=.9, color="#0ea5e9")
        ax[1].axhline(0, color="#16a34a", lw=1.5)
        ax[1].set_xlabel("position in the song (s)")
        ax[1].set_ylabel("offset (ms)")
        ax[1].set_title("Did timing hold up (one trace per block)")
    # Persistence pairs stay inside their own block.
    prev, nxt = [], []
    for _game, gg in rhy.groupby("game", sort=False):
        o_b = gg["time_difference_ms"].astype(float).values
        if len(o_b) > 3:
            prev.extend(o_b[:-1]); nxt.extend(o_b[1:])
    lag1_blocks = btab["lag1_r"].dropna()
    if prev:
        r = float(lag1_blocks.mean())
        ax[2].scatter(prev, nxt, s=18, alpha=.5, color="#7c3aed")
        ax[2].axhline(0, color="#94a3b8", lw=1)
        ax[2].axvline(0, color="#94a3b8", lw=1)
        ax[2].set_xlabel("offset on note n")
        ax[2].set_ylabel("offset on note n+1")
        ax[2].set_title(f"Offset persistence, mean per-block r = {r:.2f}")
    _save(fig, "rhythm"); plt.show()

    def _lag1_reading(r):
        # In the standard Wing-Kristofferson / Vorberg-Wing phase-
        # correction account of sensorimotor synchronisation, active
        # error correction produces a NEGATIVE lag-1 r (an early press
        # tends to be followed by a compensating late one, and vice
        # versa) -- that is the actual signature of "tracking". A
        # POSITIVE r means consecutive offsets stay similar: slow
        # drift, not correction. This reading deliberately does not
        # call a positive r "tracking the tempo".
        return ("correcting after each note (negative lag-1, the "
                "textbook signature of active phase correction)"
                if r < -0.2
                else "drifting rather than correcting (positive "
                "lag-1: consecutive offsets stay similar)"
                if r > 0.2
                else "landing near beats with no clear "
                "correction or drift pattern")

    print("\nper participant (lag-1 averaged across blocks, never over a")
    print("concatenation of them):")
    for who, gb in btab.groupby("who"):
        n = int(gb["notes"].sum())
        acc = float(np.average(gb["accuracy_ms"], weights=gb["notes"]))
        bias = float(np.average(gb["bias_ms"], weights=gb["notes"]))
        print(f"   {who}: {n} notes   accuracy {acc:.1f} ms   "
              f"bias {bias:+.1f} ms "
              f"({'ahead of' if bias < 0 else 'behind'} the beat)")
        lag = gb["lag1_r"].dropna()
        if len(lag):
            r_w = float(lag.mean())
            print(f"      lag-1 {r_w:+.3f} over {len(lag)} block(s), "
                  f"which reads as {_lag1_reading(r_w)}.")
        else:
            print("      no block with more than 4 notes, so no lag-1 "
                  "reading.")

    # The trained ability is offset SD, so its across-session trend
    # lives here (and in the progress chapter's beat_sd_ms column)
    # rather than nowhere. Sessions are one person on one day.
    per_sess: dict = {}
    for (who, sess), g in (btab.dropna(subset=["sd_ms"])
                           .groupby(["who", "session"])):
        # The session label is "day  who"; the day alone reads better
        # on a line already headed by the participant's name.
        day = str(sess).split()[0] if str(sess) else str(sess)
        per_sess.setdefault(who, []).append(
            (day, float(np.average(g["sd_ms"], weights=g["notes"]))))
    trended = False
    for who, vals in per_sess.items():
        if len(vals) < 2:
            continue
        if not trended:
            print("\noffset SD per session (the trained beat-timing "
                  "ability; falling is improvement):")
            trended = True
        path = "   ->   ".join(f"{s}: {v:.1f} ms" for s, v in vals)
        print(f"   {who}:   {path}")
    if not trended and len(btab):
        print("\nNo participant has rhythm blocks on two days yet, so")
        print("there is no across-session beat-timing trend to read.")
        print("The progress chapter's beat_sd_ms column fills in as")
        print("repeat sessions appear.")
    return rhy


# ======================================== Tap variability and beat phase
# Inter-tap-interval CV and the two different entrainment measures,
# rhythm blocks only.

# Two rhythm metrics MyChanges.md claims and this notebook never read.
#
# "Tap variability CV (ITI consistency, distinct from RT CV) ...
# metrics.tap_variability_cv ... written as tap_variability_cv in
# session.json". Inter-tap-interval CV is the standard tapping measure
# in tremor and stroke work, and rhythm mode is the only mode where it
# means anything.
#
# "Tempo entrainment index (RT vs beat phase correlation) ...
# Distinguishes patients who track the beat (low RT variance across beat
# phase) from those who just land near it". The rhythm section above
# computes the lag-1 autocorrelation of the offsets, which is what
# Objective B2 in the progress report asks for word for word, but it is
# NOT the same quantity as the description above. Both are reported
# here, named apart, so the thesis can pick which claim it makes.


def tap_series(rhy):
    """Approximate press times and note times for one rhythm block.

    song_time_s is the audio position when the trial closed, so it lands
    on the note rather than on the press. press = note + offset
    recovers the press to within the trial-close latency. Good enough
    for an interval CV, not good enough to publish as an exact tap time,
    which is why the app's own tap_variability_cv is preferred below
    wherever it exists: that one is built from the real press times.
    """
    g = rhy.sort_values("song_time_s")
    note = g["song_time_s"].astype(float).values
    press = note + g["time_difference_ms"].astype(float).values / 1000.0
    return press, note


def interval_cv(times):
    """CV of the intervals between consecutive times, or None under
    three taps. Sample standard deviation, matching
    metrics.tap_variability_cv."""
    t = np.asarray(times, dtype=float)
    if len(t) < 3:
        return None
    itis = np.diff(t)
    mean = float(itis.mean())
    if mean <= 0:
        return None
    return float(itis.std(ddof=1) / mean)


def sec_tap_variability(trials, metas=None):
    """Tap variability and whether the tempo was actually tracked."""
    rhy = rhythm_rows(trials)
    print("\n" + "=" * 62)
    print("TAP VARIABILITY AND BEAT PHASE")
    print("=" * 62)
    if rhy.empty:
        _nothing("No rhythm blocks in this selection, so there are no taps",
                 "to measure. Inter-tap-interval CV only means something",
                 "when there is a beat to tap to, which is why the app only",
                 "records it for rhythm mode.")
        return None
    rows = []
    for game, g in rhy.groupby("game", sort=False):
        if g["song_time_s"].notna().sum() < 3:
            continue
        g = g[g["song_time_s"].notna()]
        press, note = tap_series(g)
        offsets = g.sort_values("song_time_s")["time_difference_ms"] \
            .astype(float).values
        tap_cv = interval_cv(press)
        stim_cv = interval_cv(note)
        # Does the tapping interval follow the stimulus interval. This
        # is the "tracks the beat" reading of the entrainment index:
        # a person who follows a tempo change moves their own interval
        # with it, a person who only lands near beats does not.
        iti, ioi = np.diff(press), np.diff(note)
        r_track = (float(np.corrcoef(iti, ioi)[0, 1])
                   if len(iti) > 2 and iti.std() > 0 and ioi.std() > 0
                   else np.nan)
        r_lag1 = (float(np.corrcoef(offsets[1:], offsets[:-1])[0, 1])
                  if len(offsets) > 3 and offsets.std() > 0 else np.nan)
        stored = ((metas or {}).get(game, {}) or {}).get("block_summary", {}) \
            or {}
        bo = stored.get("beat_offset_stats") or {}
        rows.append({
            "game": g["game_label"].iloc[0],
            "notes": len(g),
            "tap_cv_here": (round(tap_cv, 4) if tap_cv is not None
                            else np.nan),
            "tap_cv_stored": stored.get("tap_variability_cv"),
            "stimulus_interval_cv": (round(stim_cv, 4)
                                     if stim_cv is not None else np.nan),
            "iti_vs_ioi_r": (round(r_track, 3) if pd.notna(r_track)
                             else np.nan),
            "lag1_r_here": (round(r_lag1, 3) if pd.notna(r_lag1)
                            else np.nan),
            "lag1_r_stored": bo.get("entrainment_lag1_r"),
        })
    if not rows:
        _nothing("The rhythm blocks here have fewer than three notes with a",
                 "song position, so there are not enough intervals to get a",
                 "CV from. Two intervals is the minimum.")
        return None
    tbl = pd.DataFrame(rows)
    _show(tbl)
    print("tap_cv_here is recomputed from the CSV, where the press time is")
    print("the note position plus the signed offset. song_time_s is logged")
    print("when the trial closes, so that reconstruction carries the trial-")
    print("close latency. tap_cv_stored is the app's own, built from the")
    print("real press times: prefer it wherever it is filled in, and read")
    print("the recomputed one as a check rather than as the number.")
    print("stimulus_interval_cv is how uneven the notes themselves were, so")
    print("a high tap CV against a high stimulus CV is the song, not the")
    print("person.")
    print("\nTWO DIFFERENT ENTRAINMENT CLAIMS, and they are not the same:")
    print("   lag1_r      offset on one note against the offset on the")
    print("               next. This is what Objective B2 asks for word")
    print("               for word, and what the rhythm section above and")
    print("               the app both compute.")
    print("   iti_vs_ioi  the person's own tap interval against the note")
    print("               interval. This is the 'tracks the beat rather")
    print("               than landing near it' reading in MyChanges.md.")
    print("   Pick one for the thesis and say which. Reporting the first")
    print("   under the second's description is the gap worth closing.")
    print("CAVEAT: iti_vs_ioi_r here is built from RECONSTRUCTED press")
    print("times (press = note + offset), so the note's own interval sits")
    print("on both sides of the correlation by construction: it is inflated")
    print("toward 1.0 by the song's interval structure, not by the person")
    print("tracking anything. Read it as near-1 whenever the song's beats")
    print("are close to isochronous, whatever the patient did. Prefer")
    print("tap_cv_stored (built from the app's real press timestamps) for")
    print("any claim about the person, and treat iti_vs_ioi_r as informative")
    print("only when stimulus_interval_cv itself varies a lot within the")
    print("block (a genuinely uneven song), where tracking vs not tracking")
    print("can actually separate the two tap series.")

    fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
    x = np.arange(len(tbl))
    ax[0].bar(x - .2, tbl["tap_cv_here"], .4, label="taps",
              color="#0ea5e9")
    ax[0].bar(x + .2, tbl["stimulus_interval_cv"], .4, label="notes",
              color="#94a3b8")
    ax[0].set_ylabel("interval CV")
    ax[0].set_title("Tap variability against the song's own unevenness")
    ax[0].legend(frameon=False, fontsize=8)
    ax[1].bar(x - .2, tbl["lag1_r_here"], .4, label="lag-1 offset r",
              color="#7c3aed")
    ax[1].bar(x + .2, tbl["iti_vs_ioi_r"], .4, label="tap vs note interval r",
              color="#16a34a")
    ax[1].axhline(0, color="#94a3b8", lw=1)
    ax[1].set_ylim(-1.05, 1.05); ax[1].set_ylabel("r")
    ax[1].set_title("Two entrainment measures")
    ax[1].legend(frameon=False, fontsize=8)
    for a in ax:
        a.set_xticks(x)
        a.set_xticklabels(tbl["game"], rotation=20, ha="right", fontsize=8)
    _save(fig, "tap_variability"); plt.show()
    return tbl


# ============================================================ Both hands
# Left against right on bilateral blocks, cued trials with a press
# only.

def sec_bilateral(trials, unit="sensor counts", calset=None):
    """Left against right.

    Three ways this used to print a wrong number.

    The reaction-time line pooled every trial with a time_difference_ms,
    including rhythm beat offsets and misses. On the shipped data that
    gave "left -95 | right 158 -> asymmetry -8.027", where the -95 ms is
    a rhythm block's mean offset from the beat. Only cued-mode reaction
    times go in now.

    The one-hand caveat was suppressed whenever the SELECTION held both a
    left-calibrated and a right-calibrated game, which is exactly the
    case where each individual game still only covers one hand. The check
    is per game now.

    Every line here, reaction time and peak force alike, used to run on
    raw `trials`: an anticipation press (under 100 ms, not a plausible
    reaction) or a trial where the cue never reached the device rode
    straight through. On the shipped data that gave a left reaction time
    of 53.4 ms built entirely from eight sub-100 ms anticipations,
    exported under a summary table that claims "analysable trials only"
    (audit finding #100). Restricted to `analysable(trials)` now, the
    same kept set the headline table is built from.
    """
    print("\n" + "=" * 62)
    print("BOTH HANDS")
    print("=" * 62)
    if trials.empty:
        _nothing("No trials are loaded, so there is no left against right",
                 "to show.")
        return None
    trials, _flagged, _counts = analysable(trials)
    trials = ensure_force_columns(trials, calset)
    bil = trials[trials["hand_mode"] == "both"] if "hand_mode" in trials \
        else trials.iloc[0:0]
    if bil.empty:
        modes = sorted({str(h) for h in trials.get("hand_mode", [])
                        if str(h)})
        _nothing("Nothing to show: this section needs a block played with "
                 "both",
                 f"hands, and this selection is {', '.join(modes) or 'one'}"
                 f"-handed.",
                 "Play a bilateral or mirror block, or pick one from the",
                 "dropdown, to get a left against right comparison.")
        return None
    # Exclude mirror rows from every side-based split below: mirror
    # scores one row per trial on the LATER of its two same-finger
    # presses, and that row's lane/side is always the right-hand copy
    # by construction (log_trial keys the per-finger histogram on it),
    # so a mirror row is never actually a right-hand-only measurement.
    # Left out here; the dedicated mirror asynchrony section further
    # down is the correct per-hand view for mirror.
    bil_lr = bil[bil["mode"] != "mirror"] if "mode" in bil else bil
    L, R = bil_lr[bil_lr["side"] == "left"], bil_lr[bil_lr["side"] == "right"]

    def asym(l, r):
        if pd.isna(l) or pd.isna(r) or (l + r) == 0:
            return float("nan")
        return (l - r) / ((l + r) / 2)

    # Reaction times only: cued modes, misses out, mirror out. A
    # rhythm block's signed beat offset in this line is what produced
    # the -8.027. mirror rows are excluded here (see bil_lr above) even
    # though they are technically cued and non-Miss, because their
    # side is always right by construction; they get their own
    # asynchrony readout below instead.
    lrt, rrt = reaction_times(L), reaction_times(R)
    n_mirror = int((bil["mode"] == "mirror").sum()) if "mode" in bil else 0
    rt_rows_bil_lr = reaction_times(bil_lr, per="rows")
    excluded_bil = bil_lr.loc[bil_lr.index.difference(rt_rows_bil_lr.index)]
    n_other = len(excluded_bil)
    if len(lrt) and len(rrt):
        lr, rr = lrt.mean(), rrt.mean()
        print(f"reaction time  left {lr:.0f} | right {rr:.0f} ms  "
              f"-> asymmetry {asym(lr, rr):+.3f}")
        print(f"   from {len(lrt)} left and {len(rrt)} right cued trials "
              f"with a press")
    else:
        print("reaction time  not available: this selection has no cued")
        print("   trials with a press on both hands.")
    if n_other:
        # Name what actually got excluded instead of assuming rhythm
        # (audit finding #106): the old text hard-coded "are rhythm
        # offsets or misses", so a selection with reaction, pattern or
        # chords blocks (none of which is in CUED_MODES) printed that
        # sentence even with zero rhythm trials present.
        by_mode = (excluded_bil["mode"].value_counts()
                  if "mode" in excluded_bil.columns else pd.Series(dtype=int))
        bits = []
        for m, n in by_mode.items():
            label = (f"{n} {m} (offset/non-RT block, not in CUED_MODES)"
                     if m not in CUED_MODES
                     else f"{n} {m} (missed or no RT logged)")
            bits.append(label)
        print(f"   {n_other} of {len(bil_lr)} bilateral trials (non-mirror) "
              "are left out of")
        print("   the line above: " + ("; ".join(bits) if bits
              else "no reaction time logged") + ".")
    if n_mirror:
        print(f"   {n_mirror} bilateral trials are mirror and are also "
              f"left out of")
        print("   the line above; see the mirror asynchrony readout "
              "below instead.")

    # Mirror-specific per-hand split. Mirror fires both hands' copies
    # of the same finger inside a SINGLE trial and scores that trial on
    # the LATER press, so the left/right split above can't see mirror
    # at all: `side` comes from `lane`, and a mirror row's lane is
    # always the right-hand finger by construction (log_trial keys the
    # per-finger histogram on it), so every mirror trial counted as
    # "right" and never "left". mirror_right_rt_ms / mirror_left_rt_ms
    # hold each hand's OWN press latency for exactly this reason: they
    # are per-trial columns, not per-row sides, so they survive the
    # lane collapse above.
    mir = bil[bil["mode"] == "mirror"] if "mode" in bil else bil.iloc[0:0]
    if not mir.empty and "mirror_right_rt_ms" in mir.columns:
        mr = pd.to_numeric(mir["mirror_right_rt_ms"], errors="coerce")
        ml = pd.to_numeric(mir["mirror_left_rt_ms"], errors="coerce")
        both_in = mr.notna() & ml.notna()
        # A wrong-finger-then-correct trial downgrades to Miss but both
        # hands still eventually pressed, so both_in on its own would
        # count the ERROR-RECOVERY time (how long the fumble took to
        # correct) as if it were a coordination gap. Filter those out
        # so the gap only reflects genuinely clean synchronised pairs,
        # matching the mirror-mode fix that downgrades an async pair
        # to Miss for the same reason.
        clean = (mir["had_incorrect_press"].astype(str).str.upper() != "TRUE"
                 if "had_incorrect_press" in mir.columns
                 else pd.Series(True, index=mir.index))
        both_in_clean = both_in & clean
        n_fumbled = int((both_in & ~clean).sum())
        print("\nmirror asynchrony (each hand's own press latency on the "
              "same trial)")
        if mr.notna().any() and ml.notna().any():
            print(f"   right hand mean {mr.mean():.0f} ms over "
                  f"{int(mr.notna().sum())} presses")
            print(f"   left hand mean  {ml.mean():.0f} ms over "
                  f"{int(ml.notna().sum())} presses")
        if both_in_clean.any():
            gap = (ml[both_in_clean] - mr[both_in_clean]).abs()
            print(f"   mean |left - right| gap {gap.mean():.0f} ms over "
                  f"{int(both_in_clean.sum())} clean trials where both "
                  f"hands pressed")
            print("   0 ms is perfectly synchronised. time_difference_ms")
            print("   above is scored on the LATER of the two presses, not")
            print("   this gap, so the two numbers answer different")
            print("   questions: one is speed, this is coordination.")
            if n_fumbled:
                print(f"   ({n_fumbled} wrong-finger-then-correct trial(s) "
                      f"excluded: that gap is")
                print("   error-recovery time, not coordination.)")
        else:
            print("   no trial in this selection has both hands' own press")
            print("   time recorded on a clean pair (every trial here was "
                  "a Miss on at")
            print("   least one side, or a rejected wrong-finger trial)")
    elif not mir.empty:
        print("\nmirror asynchrony: this save predates the per-hand RT")
        print("   columns, so left/right press timing inside a mirror")
        print("   trial cannot be split apart. Re-record to get it.")

    corrected = bool(bil["force_calibrated"].any())
    lf, rf = L["peak_force_n"].mean(), R["peak_force_n"].mean()
    if pd.notna(lf) and pd.notna(rf):
        print(f"peak force     left {lf:.0f} | right {rf:.0f} {unit}  "
              f"-> asymmetry {asym(lf, rf):+.3f}   NOT comparable")
    if corrected:
        lc, rc = L["peak_force_cal"].mean(), R["peak_force_cal"].mean()
        # Per game, not per selection. A left-calibrated game and a
        # right-calibrated game in one selection do not add up to a game
        # with both hands calibrated.
        gaps_missing = {}
        for game, sub in bil.groupby("game"):
            needed = {s for s in sub["side"].dropna().unique()}
            miss = (calset.missing_hands(game, needed) if calset
                    else sorted(needed))
            if miss:
                gaps_missing[game] = miss
        both_hands_ok = not gaps_missing
        if pd.notna(lc) and pd.notna(rc):
            print(f"own reference  left {lc:.2f} | right {rc:.2f} "
                  f"{NORM_UNIT}  -> asymmetry {asym(lc, rc):+.3f}")
            print_norm_short()
            if both_hands_ok:
                print("   Both hands of every game here were calibrated")
                print("   separately, so this asymmetry is not carrying one")
                print("   hand's pads. It still compares effort against each")
                print("   hand's own reference, so a hand that was weak at")
                print("   calibration is weak in the reference too.")
            else:
                print("   Do NOT report this as a patient asymmetry.")
        if gaps_missing:
            print("   CAVEAT: these games have trials on a hand with no")
            print("   usable calibration, so those rows are left")
            print("   uncorrected rather than normalised with the other")
            print("   hand's pads:")
            for game, miss in gaps_missing.items():
                print(f"      {game}: no profile for the "
                      f"{', '.join(miss)} hand")
            wanted = sorted({h for miss in gaps_missing.values()
                             for h in miss})
            on_disk = hand_profiles_on_disk()
            for hand in wanted:
                if hand in on_disk:
                    print(f"   config/calibration/current_{hand}.json "
                          f"exists, taken {on_disk[hand]['created_at']},")
                    print("      but it is not what these blocks ran under "
                          "so it is not applied.")
                else:
                    print(f"   config/calibration/current_{hand}.json does "
                          f"not exist.")
            print("   Run Calibrate once per hand before the next session.")
    elif pd.notna(lf) and pd.notna(rf):
        print("   no calibration, so the force asymmetry above is a mix of")
        print("   the two hands and the eight pads and should not be")
        print("   reported as a patient asymmetry.")
    order = _order(bil)
    rt_rows = reaction_times(bil_lr, per="rows")
    fig, ax = plt.subplots(figsize=(8, 3.4))
    x = np.arange(len(order)); w = .38
    for i, side in enumerate(("right", "left")):
        ax.bar(x + (i - .5) * w,
               [rt_rows[(rt_rows["side"] == side) & (rt_rows["finger"] == f)]
                ["time_difference_ms"].mean() if not rt_rows.empty else np.nan
                for f in order],
               w, label=side, color=HAND_COLOUR[side])
    ax.set_xticks(x); ax.set_xticklabels(order)
    ax.set_ylabel("mean reaction time (ms)")
    ax.set_title("Reaction time, both hands (cued trials only)")
    ax.legend(frameon=False)
    _save(fig, "bilateral"); plt.show()
    return {"n_left": int(len(L)), "n_right": int(len(R)),
            "rt_left": round(float(lrt.mean()), 1) if len(lrt) else np.nan,
            "rt_right": round(float(rrt.mean()), 1) if len(rrt) else np.nan}


# ================================================ Inter-hand correlation
# The real Pearson r off the raw force streams, which the app can only
# store as a placeholder.

# Thread C objective 3 asks for three bilateral measures: "Calculate
# bilateral performance measures, including peak-force asymmetry,
# reaction-time asymmetry, and inter-hand correlation, and store the
# results in session.json". The section above delivers the first two.
# The third is a placeholder in the app: metadata.json carries
# inter_hand_correlation: null on every block, because the engine keeps
# per-press peaks in memory and not the block-long force streams.
# Section 5.1 puts the fix in Semester 2 Weeks 3 to 5, "Build the
# bilateral force-stream resampling that replaces the inter-hand
# correlation placeholder with a real Pearson r".
#
# Offline analysis does not have the engine's memory problem. raw.csv
# already logs fsr1 to fsr8 on one row per sample, so the two hands are
# on a shared time grid with no resampling needed, and the Pearson r can
# be computed here today.

RIGHT_CHANNELS = ["fsr1", "fsr2", "fsr3", "fsr4"]


LEFT_CHANNELS = ["fsr5", "fsr6", "fsr7", "fsr8"]


# How far either side of zero lag to look for a better match. Mirror
# therapy is about interlimb coupling, so a hand that follows the other
# by a fixed delay is a different finding from one that moves with it.
MAX_LAG_S = 0.25


def hand_streams(raw):
    """Total force per hand, sample by sample, from one raw log.

    Returns (t, left, right) with the event rows dropped. The two hands
    are already on one grid because the merger writes both boards into
    the same row, so nothing is resampled and no interpolation error
    gets into the correlation.
    """
    if raw is None or raw.empty:
        return None
    s = raw[raw["event"].isna() | (raw["event"] == "")]
    have_r = [c for c in RIGHT_CHANNELS if c in s.columns]
    have_l = [c for c in LEFT_CHANNELS if c in s.columns]
    if len(s) < 50 or not have_r or not have_l:
        return None
    t = pd.to_numeric(s["t_perf"], errors="coerce").values.astype(float)
    right = s[have_r].apply(pd.to_numeric, errors="coerce").sum(axis=1).values
    left = s[have_l].apply(pd.to_numeric, errors="coerce").sum(axis=1).values
    return t, left.astype(float), right.astype(float)


def zero_lag_r(left, right):
    """Pearson r at zero lag, or None when either side never moves.

    Same contract as metrics.inter_hand_correlation, which returns None
    on a constant series because the correlation is undefined there. A
    hand that was never touched is exactly that case, and returning 0.0
    would read as "no coupling" rather than "no data".
    """
    if len(left) < 2 or len(left) != len(right):
        return None
    if float(np.std(left)) == 0 or float(np.std(right)) == 0:
        return None
    return float(np.corrcoef(left, right)[0, 1])


def best_lag_r(t, left, right, max_lag_s=MAX_LAG_S):
    """(r, lag in seconds) at the lag that fits best inside the window.

    Secondary to the zero-lag figure, which is what the objective asks
    for. This one says whether one hand is following the other, which is
    the interlimb-coupling question mirror mode exists to ask.
    """
    if len(t) < 10:
        return None, None
    dt = float(np.median(np.diff(t)))
    if not np.isfinite(dt) or dt <= 0:
        return None, None
    steps = int(max_lag_s / dt)
    best = (None, None)
    for k in range(-steps, steps + 1):
        if k < 0:
            a, b = left[-k:], right[:len(right) + k]
        elif k > 0:
            a, b = left[:len(left) - k], right[k:]
        else:
            a, b = left, right
        r = zero_lag_r(a, b)
        if r is None:
            continue
        if best[0] is None or abs(r) > abs(best[0]):
            best = (r, k * dt)
    return best


def sec_inter_hand(folders, metas=None, trials=None):
    """The real Pearson r the placeholder in session.json stands in
    for."""
    print("\n" + "=" * 62)
    print("INTER-HAND CORRELATION")
    print("=" * 62)
    both = []
    for f in folders:
        meta = (metas or {}).get(game_key(f), {}) or {}
        if str(meta.get("hand", "")).strip().lower() == "both":
            both.append(f)
    if not both:
        _nothing("No block in this selection was played with both hands, so",
                 "there are no two streams to correlate. This needs a",
                 "bilateral or mirror block recorded with two boards.")
        return None
    rows = []
    for f in both:
        name = game_key(f)
        stored = ((metas or {}).get(name, {}) or {}) \
            .get("block_summary", {}) or {}
        streams = hand_streams(load_raw(f))
        if streams is None:
            rows.append({"game": name, "samples": 0, "r_zero_lag": np.nan,
                         "r_best": np.nan, "best_lag_ms": np.nan,
                         "stored_r": stored.get("inter_hand_correlation"),
                         "why": "no usable raw sample stream"})
            continue
        t, left, right = streams
        r0 = zero_lag_r(left, right)
        rb, lag = best_lag_r(t, left, right)
        why = ""
        if r0 is None:
            why = ("one hand never moved, so the correlation is undefined"
                   if float(np.std(left)) == 0 or float(np.std(right)) == 0
                   else "not enough samples")
        rows.append({
            "game": name, "samples": len(t),
            "r_zero_lag": round(r0, 3) if r0 is not None else np.nan,
            "r_best": round(rb, 3) if rb is not None else np.nan,
            "best_lag_ms": (round(lag * 1000, 1) if lag is not None
                            else np.nan),
            "stored_r": stored.get("inter_hand_correlation"),
            "why": why})
    tbl = pd.DataFrame(rows)
    _show(tbl)
    print("r_zero_lag is the measure the objective asks for. stored_r is")
    print("what the app wrote, and it is None on every block by design:")
    print("the engine keeps per-press peaks rather than the block-long")
    print("streams, so it cannot compute this in the moment. Offline it")
    print("can, because raw.csv puts both hands on one row per sample.")
    print("r_best and best_lag_ms are secondary: a hand that follows the")
    print("other by a fixed delay is a different finding from one that")
    print("moves with it, which is the coupling question mirror mode is")
    print("there to ask.")

    usable = tbl[tbl["r_zero_lag"].notna()]
    if usable.empty:
        print("\nNothing to correlate in this selection. Reasons, per block:")
        for w in sorted({w for w in tbl["why"] if w}):
            print(f"   {w}")
        print("A keyboard session logs a flat stream and a short block logs")
        print("too few samples, so both come out empty here for a reason")
        print("rather than as a zero correlation.")
        return tbl

    fig, ax = plt.subplots(1, 2, figsize=(12, 3.6))
    x = np.arange(len(usable))
    ax[0].bar(x - .2, usable["r_zero_lag"], .4, label="zero lag",
              color="#2563eb")
    ax[0].bar(x + .2, usable["r_best"], .4, label="best lag",
              color="#a855f7")
    ax[0].axhline(0, color="#94a3b8", lw=1)
    ax[0].set_ylim(-1.05, 1.05); ax[0].set_ylabel("Pearson r")
    ax[0].set_title("Left against right force, per block")
    ax[0].set_xticks(x)
    ax[0].set_xticklabels(usable["game"], rotation=20, ha="right",
                          fontsize=8)
    ax[0].legend(frameon=False, fontsize=8)
    # The block behind the second panel has to be one that actually
    # correlated, not merely one with a stream: a flat trace under a
    # heading about coupling reads as a result.
    shown = None
    for f in both:
        streams = hand_streams(load_raw(f))
        if streams and zero_lag_r(streams[1], streams[2]) is not None:
            shown = (f, streams)
            break
    first, (t, left, right) = shown
    t0 = t - t.min()
    ax[1].plot(t0, right, lw=1.2, color=HAND_COLOUR["right"], label="right")
    ax[1].plot(t0, left, lw=1.2, color=HAND_COLOUR["left"], label="left")
    ax[1].set_xlabel("seconds into the block")
    ax[1].set_ylabel("total force (counts)")
    ax[1].set_title(f"The streams behind it ({game_key(first)})")
    ax[1].legend(frameon=False, fontsize=8)
    _save(fig, "inter_hand_correlation"); plt.show()
    print("\nTotal force per hand is the sum of that hand's four pads, so")
    print("the r is about the two hands moving together, not about which")
    print("finger moved. The pads are not equally sensitive, so a hand with")
    print("a hot pad carries more weight in its own sum.")
    return tbl


# ====================================== Affected against unaffected hand
# Relabels the two hands clinically, which is the only version of the
# asymmetry that survives pooling a cohort.

# The Both hands section reports asymmetry as (left - right) / mean.
# For a stroke cohort that sign is close to meaningless once more than
# one person is pooled: a left-affected and a right-affected patient
# with identical impairment produce asymmetries of opposite sign and
# cancel.
#
# The clinical framing the report sets out needs the affected side, not
# the geometric one. Section 2.4 wants "a way to record and analyse
# bilateral practice by measuring differences in performance between the
# two hands over time", and Section 3.1 wants "objective, per-trial
# measures of hand and finger movement". metadata.json already carries
# affected_side, dominant_hand and impairment_score per session, so the
# relabelling costs nothing and has to be in place before the Week 9 to
# 10 data arrives, or that data needs reprocessing.


def clinical_fields(metas):
    """affected_side, dominant_hand and impairment_score per game."""
    rows = []
    for name, meta in _meta_items(metas):
        rows.append({
            "game": name,
            "participant": meta.get("participant", "NA"),
            "hand_mode": meta.get("hand", "?"),
            "affected_side": normalise_hand(meta.get("affected_side")),
            "dominant_hand": normalise_hand(meta.get("dominant_hand")),
            "impairment_score": (meta.get("impairment_score") or None),
        })
    return pd.DataFrame(rows)


def label_affected(trials, metas):
    """Copy of `trials` with a role column: affected or unaffected.

    Rows from a game with no affected_side recorded get None rather than
    a guess. Nothing downstream fills that in.
    """
    df = trials.copy()
    if df.empty:
        df["affected_side"] = pd.Series(dtype="object")
        df["role"] = pd.Series(dtype="object")
        return df
    sides = {name: normalise_hand((meta or {}).get("affected_side"))
             for name, meta in _meta_items(metas)}
    df["affected_side"] = df["game"].map(sides)
    df["role"] = [
        None if not a or not s else ("affected" if s == a else "unaffected")
        for a, s in zip(df["affected_side"], df.get("side", []))]
    return df


def sec_affected_side(trials, metas, unit="sensor counts", calset=None):
    """Affected against unaffected, which is the comparison a cohort can
    be pooled on."""
    print("\n" + "=" * 62)
    print("AFFECTED AGAINST UNAFFECTED HAND")
    print("=" * 62)
    fields = clinical_fields(metas)
    recorded = fields[fields["affected_side"].notna()] \
        if not fields.empty else fields
    if fields.empty or recorded.empty:
        _nothing("No block in this selection recorded an affected side, so",
                 "left against right is all there is. Every session on disk",
                 "has affected_side, dominant_hand and impairment_score",
                 "blank, which is fine for development data and not fine",
                 "for patient data.",
                 "",
                 "Why it matters: the Both hands section signs its",
                 "asymmetry as (left - right). Pool a left-affected and a",
                 "right-affected patient on that and the two cancel, so a",
                 "real group deficit reads as a symmetric group. Fill the",
                 "field in on the session screen and this section takes",
                 "over.")
        if not fields.empty:
            _show(fields)
        return None
    _show(fields)
    df = label_affected(ensure_force_columns(trials, calset), metas)
    df = df[df["role"].notna()]
    if df.empty:
        _nothing("An affected side is recorded, but no trial in this",
                 "selection sits on a hand that can be labelled with it.")
        return None
    rows = []
    for who, g in df.groupby("participant"):
        aff, un = g[g["role"] == "affected"], g[g["role"] == "unaffected"]
        art, urt = reaction_times(aff), reaction_times(un)
        af = aff["peak_force_n"].dropna()
        uf = un["peak_force_n"].dropna()

        def sign_asym(a, u):
            if pd.isna(a) or pd.isna(u) or (a + u) == 0:
                return np.nan
            return round(float((a - u) / ((a + u) / 2)), 3)
        rows.append({
            "participant": who,
            "affected": g["affected_side"].dropna().iloc[0]
            if g["affected_side"].notna().any() else "?",
            "trials_affected": int(len(aff)),
            "trials_unaffected": int(len(un)),
            # Scorable rows only: catch outcomes and free retries are
            # events, not hits, on either arm of this comparison.
            "hit_affected": (round(float(
                (aff[is_scorable(aff)]["early_late"] != "Miss").mean()), 3)
                if len(aff[is_scorable(aff)]) else np.nan),
            "hit_unaffected": (round(float(
                (un[is_scorable(un)]["early_late"] != "Miss").mean()), 3)
                if len(un[is_scorable(un)]) else np.nan),
            "rt_affected_ms": (round(float(art.mean()), 1) if len(art)
                               else np.nan),
            "rt_unaffected_ms": (round(float(urt.mean()), 1) if len(urt)
                                 else np.nan),
            "rt_asymmetry": sign_asym(art.mean() if len(art) else np.nan,
                                      urt.mean() if len(urt) else np.nan),
            "force_asymmetry_raw": sign_asym(af.mean() if len(af) else np.nan,
                                             uf.mean() if len(uf) else np.nan),
        })
    tbl = pd.DataFrame(rows)
    _show(tbl)
    print("Sign convention here is (affected - unaffected) / mean, so a")
    print("positive rt_asymmetry is a slower affected hand and a negative")
    print("force_asymmetry_raw is a weaker one. That sign survives pooling")
    print("across a cohort, which (left - right) does not.")
    print("force_asymmetry_raw is raw counts on two sets of pads and is NOT")
    print("corrected for pad sensitivity, so part of it is the hardware.")

    fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
    x = np.arange(len(tbl)); w = .38
    ax[0].bar(x - w / 2, tbl["rt_affected_ms"], w, label="affected",
              color="#dc2626")
    ax[0].bar(x + w / 2, tbl["rt_unaffected_ms"], w, label="unaffected",
              color="#2563eb")
    ax[0].set_ylabel("mean reaction time (ms)")
    ax[0].set_title("Reaction time by role")
    ax[0].legend(frameon=False, fontsize=8)
    ax[1].bar(x, tbl["rt_asymmetry"], .5, color="#0f172a")
    ax[1].axhline(0, color="#94a3b8", lw=1)
    ax[1].set_ylabel("(affected - unaffected) / mean")
    ax[1].set_title("Reaction-time asymmetry")
    for a in ax:
        a.set_xticks(x); a.set_xticklabels(tbl["participant"], fontsize=8)
    _save(fig, "affected_side"); plt.show()
    return tbl


# ===================================================== Raw sample stream
# The 200 Hz log behind the first selected game that has one.

def sec_raw(folders, unit="sensor counts", calset=None):
    """A look at the sample stream behind one game.

    The average press shape at the end puts all four fingers on one axis,
    so it is a cross-finger force comparison and gets the same
    correction as the rest.
    """
    print("\n" + "=" * 62)
    print("RAW STREAM")
    print("=" * 62)
    raw, game, folder = None, None, None
    for f in folders:
        candidate = load_raw(f)
        if candidate is not None and len(candidate) > 50:
            raw, game, folder = candidate, game_key(f), Path(f)
            break
    if raw is None:
        # Silence here read as a section that crashed. Say which games
        # were looked at and why none of them had a stream.
        missing = [game_key(f) for f in folders
                   if not (Path(f) / "raw.csv").exists()]
        _nothing(f"None of the {len(folders)} selected game(s) carry a "
                 f"usable raw.csv,",
                 "so there is no sample stream, no press duration and no",
                 "press shape to show here.")
        if missing:
            print(f"no raw.csv at all: {', '.join(missing)}")
        thin = [game_key(f) for f in folders
                if (Path(f) / "raw.csv").exists()
                and game_key(f) not in missing]
        if thin:
            print(f"raw.csv present but under 50 rows: {', '.join(thin)}")
        print("The 200 Hz stream is only written when the force sensors are")
        print("streaming, so keyboard blocks and very short blocks have none.")
        return None
    hand_mode = read_meta(folder).get("hand", "right") if folder else "right"
    samples = raw[raw["event"].isna() | (raw["event"] == "")]
    events = raw[raw["event"].notna() & (raw["event"] != "")]
    if len(samples) > 1:
        dur = samples["t_perf"].max() - samples["t_perf"].min()
        print(f"{len(samples)} samples over {dur:.1f} s "
              f"({len(samples)/max(dur,1e-9):.0f} Hz), {len(events)} events")
    else:
        # A raw.csv holding only event rows still passes the length check
        # above, so without this the section prints its heading and
        # nothing else and reads as a section that failed.
        print(f"{game} logged {len(events)} events but no sensor samples,")
        print("so there's no press shape or press duration to show here.")
        print("The sample stream only gets written while the force sensors")
        print("are streaming, so this comes up empty for keyboard sessions.")
        return None
    presses = events[events["event"] == "press"]
    releases = events[events["event"] == "release"]
    durs = []
    for _, p in presses.iterrows():
        after = releases[(releases["lane"] == p["lane"])
                         & (releases["t_perf"] > p["t_perf"])]
        if len(after):
            durs.append((after["t_perf"].iloc[0] - p["t_perf"]) * 1000)
    if durs:
        durs = pd.Series(durs)
        print(f"press duration: mean {durs.mean():.0f} ms, "
              f"median {durs.median():.0f} ms")
        fig, ax = plt.subplots(figsize=(7, 3.0))
        ax.hist(durs, bins=_nbins(durs, 24), color="#0f172a", alpha=.8)
        ax.set_xlabel("press duration (ms)"); ax.set_ylabel("presses")
        ax.set_title("How long each press was held")
        _save(fig, "press_duration"); plt.show()

    stims = raw[raw["event"] == "stim"]
    if len(stims) and len(samples) > 50:
        BEFORE, AFTER = 0.25, 1.25
        stacked = {i: [] for i in range(4)}
        for _, s in stims.iterrows():
            lane = s.get("lane")
            if pd.isna(lane) or int(lane) > 3:
                continue
            lane = int(lane); t0 = s["t_perf"]; col = f"fsr{lane+1}"
            w = samples[(samples["t_perf"] >= t0 - BEFORE)
                        & (samples["t_perf"] <= t0 + AFTER)]
            if len(w) > 20 and col in w.columns:
                base = w[w["t_perf"] < t0][col].median()
                if pd.notna(base):
                    stacked[lane].append((w["t_perf"].values - t0,
                                          w[col].values - base))
        if any(stacked.values()):
            # Lanes 0 to 3, so the right hand in a bilateral block and
            # whichever hand was played in a one-handed one.
            gaps = ([calset.lane_gap(game, i, hand_mode) for i in range(4)]
                    if calset is not None else [None] * 4)
            # Only correct when every lane that has traces has a gap,
            # so the plot never mixes normalised and raw curves.
            corrected = all(gaps[l] for l, tr in stacked.items() if tr)
            fig, ax = plt.subplots(figsize=(9, 3.6))
            grid = np.linspace(-BEFORE, AFTER, 150)
            for lane, traces in stacked.items():
                if not traces:
                    continue
                mean = np.mean([np.interp(grid, t, v) for t, v in traces], axis=0)
                if corrected:
                    mean = mean / gaps[lane]
                ax.plot(grid * 1000, mean, lw=2,
                        color=FINGER_COLOUR[FINGERS[lane]],
                        label=f"{FINGERS[lane]} (n={len(traces)})")
            ax.axvline(0, color="#dc2626", lw=1.5, ls="--", label="cue")
            ax.set_xlabel("time from cue (ms)")
            ax.set_ylabel(NORM_LABEL if corrected
                          else f"force above baseline ({unit})")
            ax.set_title("Average shape of a press"
                         + ("" if corrected else ", raw counts"))
            ax.legend(frameon=False, fontsize=8)
            _save(fig, "force_waveform"); plt.show()
            if corrected:
                print("Press shapes are divided by each finger's own")
                print("reference press, so the pads are out of the four")
                print("heights.")
                print_norm_short("")
                print("A taller trace here means more effort against that")
                print("finger's own reference, not a stronger finger.")
            else:
                print("Press shapes are in raw counts. The four traces sit")
                print("on four differently sensitive pads, so their heights")
                print("cannot be compared with each other. Shape and timing")
                print("still can.")
    return raw


# ========================== Movement onset and rate of force development
# Onset taken off the force trace instead of a threshold crossing, and
# the gap between the two estimates.
#
# HOW THREE ONSET DETECTORS BECAME ONE. This project had grown three:
# the viewer algorithm as Rayan's reference file carries it
# (docs/research/rayan/process_force_peaks.py), a paraphrase of it that
# used to live in this chapter (its own sigma bands, its own
# thresholds, a moving-average smoother, and an np.gradient on the
# median timestamp gap that silently broke on burst-stamped logs), and
# a third variant in finger_rehab/analytics/signal.py (velocity over
# mean + k*sd of a baseline window). Three detectors give three
# reaction times for the same press, and only Rayan's has already been
# run over Welber's earlier data. So there is now exactly one: the
# exact port of his file, living in signal.py, copied verbatim into
# the canonical-helpers block above, pinned by tests on both sides and
# against his file itself. Parameters are his throughout: 20 Hz
# Butterworth on force, Savitzky-Golay 11/3 then a 10 Hz low-pass on
# the derivative, minimum rise 80 counts, first crossing of
# max(25, 0.18 x vmax) backed by a 12-count rise over the next 50 ms,
# search window 50 ms before the cue to 1.2 s after it.
#
# The onset RT and the game's logged RT are different measurements of
# the same press and both are kept: the comparison below puts them on
# one scatter and prints when each is the right number.

ONSET_SEARCH_MIN_S = -0.05   # his SEARCH_MIN_S: 50 ms before the cue
ONSET_SEARCH_MAX_S = 1.20    # his WIN_POST_DEFAULT and SEARCH_MAX_S


def onset_from_trace(t, force, t_stim, fs):
    """Teasdale onset for one cue, his windowing exactly: the caller
    hands in the segment cut from 50 ms before the cue to 1.2 s after,
    the search endpoint is int(1.2 * fs) samples from the segment
    start (his search_to_idx, which therefore lands 1.15 s after the
    cue), and the winning index converts back through the timestamp
    array. Returns (rt_ms, peak_dforce) or (None, None). rt can be
    slightly negative, as in his script: an onset in the 50 ms before
    the cue is kept, and reads as anticipation rather than error.
    peak_dforce is the velocity peak over the segment, in counts per
    second, kept for the rate-of-force-development panels."""
    t = np.asarray(t, dtype=float)
    # His loader turns a non-finite sample into 0.0 rather than
    # dropping it, so the port does too; a dropped sample would shift
    # every later index and the onset with it.
    f = np.nan_to_num(np.asarray(force, dtype=float), nan=0.0)
    onset_idx, _force_lp, dforce = teasdale_onset(
        f, fs=fs, search_from=0,
        search_to=int(ONSET_SEARCH_MAX_S * fs))
    if onset_idx is None or dforce is None or not len(dforce):
        return None, None
    if onset_idx >= len(t):
        return None, None
    vmax = float(np.max(dforce))
    return (t[onset_idx] - t_stim) * 1000.0, vmax


def onset_table(folders, unit="sensor counts", calset=None) -> pd.DataFrame:
    """The canonical detector over every cue in the raw streams.

    peak_dforce is counts per second, so it carries the same per-pad
    bias as any other raw force number. peak_dforce_cal divides by
    that finger's own calibration press, which is the column to
    compare between fingers. `trial` comes off the stim row's
    trial_id detail so each onset can be matched back to the trial
    the game logged.
    """
    rows = []
    for folder in folders:
        raw = load_raw(folder)
        if raw is None:
            continue
        hand_mode = read_meta(Path(folder)).get("hand", "right")
        ev = raw["event"].fillna("").astype(str)
        samples = raw[ev == ""].dropna(subset=["t_perf"]).sort_values("t_perf")
        stims = raw[ev == "stim"].dropna(subset=["t_perf"])
        if not len(stims) or len(samples) < 50:
            continue
        ts = samples["t_perf"].to_numpy(dtype=float)
        fs = estimate_fs_span(ts)
        game = game_key(folder)
        for _, s in stims.iterrows():
            lane = s.get("lane")
            if pd.isna(lane) or int(lane) > 7:
                continue
            lane = int(lane)
            col = f"fsr{lane + 1}"
            if col not in samples.columns:
                continue
            t0 = float(s["t_perf"])
            i_start = int(np.searchsorted(ts, t0 + ONSET_SEARCH_MIN_S,
                                          side="left"))
            i_end = int(np.searchsorted(ts, t0 + ONSET_SEARCH_MAX_S,
                                        side="right"))
            if i_end - i_start < 20:
                continue
            seg = pd.to_numeric(samples[col].iloc[i_start:i_end],
                                errors="coerce").to_numpy(dtype=float)
            rt, vmax = onset_from_trace(ts[i_start:i_end], seg, t0, fs)
            if rt is None:
                continue
            # Lanes 4 to 7 are the left hand, so they need the left
            # hand's profile. Dividing them by the right hand's gaps
            # produced a corrected-looking number off the wrong pads.
            side = lane_side(lane, hand_mode)
            gap = (calset.lane_gap(game, lane, hand_mode)
                   if calset is not None else None)
            rows.append({"game": game,
                         "trial": parse_trial_id(s.get("detail")),
                         "finger": FINGERS[lane % 4], "lane": lane,
                         "hand": side,
                         "onset_rt_ms": rt, "peak_dforce": vmax,
                         "peak_dforce_cal": vmax / gap if gap else np.nan})
    return pd.DataFrame(
        rows, columns=["game", "trial", "finger", "lane", "hand",
                       "onset_rt_ms", "peak_dforce", "peak_dforce_cal"])


def four_finger_peaks(folders):
    """Per cue: the post-cue peak on ALL four sensors of the
    stimulated hand, in counts above the 250 ms look-back baseline
    taken at the cue, under both window conventions at once: the
    game's cue-anchored window and Rayan's onset-anchored one (onset
    to 1.2 s after the cue). One frame, one row per (cue, sensor),
    so sec_onset can measure whether the two conventions disagree
    instead of arguing from first principles."""
    rows = []
    for folder in folders:
        raw = load_raw(folder)
        if raw is None:
            continue
        game = game_key(folder)
        ev = raw["event"].fillna("").astype(str)
        samples = raw[ev == ""].dropna(subset=["t_perf"]).sort_values("t_perf")
        stims = raw[ev == "stim"].dropna(subset=["t_perf"])
        if len(samples) < 60 or not len(stims):
            continue
        ts = samples["t_perf"].to_numpy(dtype=float)
        fs = estimate_fs_span(ts)
        streams = {c: pd.to_numeric(samples[c], errors="coerce")
                   .to_numpy(dtype=float)
                   for c in samples.columns if c.startswith("fsr")}
        for _, s in stims.iterrows():
            lane = s.get("lane")
            if pd.isna(lane) or int(lane) > 7:
                continue
            lane = int(lane)
            board = lane // 4
            col = f"fsr{lane + 1}"
            if col not in streams:
                continue
            t0 = float(s["t_perf"])
            i_start = int(np.searchsorted(ts, t0 + ONSET_SEARCH_MIN_S,
                                          side="left"))
            i_end = int(np.searchsorted(ts, t0 + ONSET_SEARCH_MAX_S,
                                        side="right"))
            i_stim = int(np.searchsorted(ts, t0, side="left"))
            if i_end - i_start < 20:
                continue
            seg = np.nan_to_num(streams[col][i_start:i_end], nan=0.0)
            onset_idx, _f, _d = teasdale_onset(
                seg, fs=fs, search_from=0,
                search_to=int(ONSET_SEARCH_MAX_S * fs))
            i_onset = i_start + onset_idx if onset_idx is not None else None
            for k in range(4):
                sensor_lane = board * 4 + k
                sc = f"fsr{sensor_lane + 1}"
                if sc not in streams:
                    continue
                v = streams[sc]
                base = lookback_baseline(v, i_stim, DRIFT_LOOKBACK_SAMPLES)
                if not np.isfinite(base):
                    continue
                win = v[i_stim:i_end]
                peak_stim = (float(np.nanmax(win) - base)
                             if len(win) else np.nan)
                peak_onset = np.nan
                if i_onset is not None and i_onset < i_end:
                    win_on = v[i_onset:i_end]
                    if len(win_on):
                        peak_onset = float(np.nanmax(win_on) - base)
                rows.append({"game": game,
                             "trial": parse_trial_id(s.get("detail")),
                             "stim_lane": lane,
                             "sensor_lane": sensor_lane,
                             "peak_stim_counts": peak_stim,
                             "peak_onset_counts": peak_onset,
                             "onset_found": i_onset is not None})
    return pd.DataFrame(
        rows, columns=["game", "trial", "stim_lane", "sensor_lane",
                       "peak_stim_counts", "peak_onset_counts",
                       "onset_found"])


def sec_onset(folders, trials, unit="sensor counts", calset=None):
    """Onset-based reaction time and rate of force development, the
    per-trial comparison against the RT the game logged, and the
    window check on the four-finger peak extraction."""
    ons = onset_table(folders, unit, calset=calset)
    if ons.empty:
        print("Couldn't measure movement onset. This needs the raw sample\n"
              "stream (raw.csv) with force rising after a cue, so it's\n"
              "empty for keyboard sessions and for blocks where no press\n"
              "crossed the 80-count minimum rise.")
        return ons
    corrected = bool(ons["peak_dforce_cal"].notna().any())
    print("\n" + "=" * 62)
    print("MOVEMENT ONSET AND RATE OF FORCE DEVELOPMENT")
    print("=" * 62)
    v = ons["onset_rt_ms"]
    print(f"\nonset reaction time : n {len(v)}   mean {v.mean():.1f} ms   "
          f"median {v.median():.1f} ms   sd {v.std():.1f} ms")
    cv = v.std() / v.mean() if len(v) > 1 and v.mean() > 0 else np.nan
    if pd.notna(cv):
        print(f"response stability  : CV {cv:.3f}  "
              f"(sd over mean, lower is steadier)")
    else:
        print("response stability  : CV not computable from this sample")
    d = ons["peak_dforce"]
    print(f"rate of force dev.  : mean {d.mean():.0f} {unit} per second "
          f"(raw, not comparable between fingers)")
    if corrected:
        print(f"rate, own reference : mean "
              f"{ons['peak_dforce_cal'].mean():.2f} {NORM_UNIT} per second")
        print_norm_short()
    else:
        print("no calibration available, so the per-finger rates below are")
        print("not corrected for the pads and should not be ranked.")

    # Threshold-crossing RT from the game, for comparison. Cued modes
    # only: a rhythm beat offset is not a reaction time and would drag
    # the comparison below in either direction.
    thr = reaction_times(trials)

    fig, ax = plt.subplots(1, 3, figsize=(14, 3.6))
    ax[0].hist(v, bins=_nbins(v), color="#0ea5e9", alpha=.85, label="onset")
    if len(thr):
        ax[0].hist(thr, bins=_nbins(thr), color="#94a3b8", alpha=.5,
                   label="threshold crossing")
        ax[0].legend(frameon=False, fontsize=8)
    ax[0].set_xlabel("reaction time (ms)"); ax[0].set_ylabel("trials")
    ax[0].set_title("Onset vs threshold crossing")

    order = [f for f in FINGERS if f in ons["finger"].unique()]
    bp = ax[1].boxplot([ons[ons["finger"] == f]["onset_rt_ms"] for f in order],
                       labels=order, patch_artist=True, widths=.6)
    for p, f in zip(bp["boxes"], order):
        p.set_facecolor(FINGER_COLOUR[f]); p.set_alpha(.65)
    for m in bp["medians"]:
        m.set_color("white"); m.set_linewidth(2)
    ax[1].set_ylabel("onset reaction time (ms)")
    ax[1].set_title("Onset by finger")

    dcol = "peak_dforce_cal" if corrected else "peak_dforce"
    dlabel = (f"peak dForce ({NORM_UNIT} per s)" if corrected
              else f"peak dForce ({unit} per s, raw)")
    _boxes(ax[2], [ons[ons["finger"] == f][dcol].dropna() for f in order],
           order, dlabel, "How fast force was built")
    _save(fig, "onset_rfd"); plt.show()

    cols = ["onset_rt_ms", "peak_dforce"]
    if corrected:
        cols.append("peak_dforce_cal")
    _show(ons.groupby("finger")[cols]
             .agg(["count", "mean", "std"]).reindex(order).round(2))

    # ---- the same press on two clocks, matched trial by trial
    merged = pd.DataFrame()
    thr_rows = reaction_times(trials, per="rows")
    if len(thr_rows) and "game" in thr_rows.columns:
        have = ons.dropna(subset=["trial"]).copy()
        if len(have):
            have["trial"] = have["trial"].astype(int)
            t = thr_rows[["game", "trial", "time_difference_ms"]].copy()
            t["trial"] = pd.to_numeric(t["trial"], errors="coerce")
            t = t.dropna(subset=["trial"])
            t["trial"] = t["trial"].astype(int)
            t = t.drop_duplicates(subset=["game", "trial"])
            merged = have.merge(t, on=["game", "trial"], how="inner")
    if len(merged) >= 3:
        gap = merged["time_difference_ms"] - merged["onset_rt_ms"]
        fig, ax = plt.subplots(figsize=(4.6, 4.2))
        ax.scatter(merged["onset_rt_ms"], merged["time_difference_ms"],
                   s=16, alpha=.6, color="#0ea5e9")
        lo = float(min(merged["onset_rt_ms"].min(),
                       merged["time_difference_ms"].min()))
        hi = float(max(merged["onset_rt_ms"].max(),
                       merged["time_difference_ms"].max()))
        ax.plot([lo, hi], [lo, hi], ls="--", color="#94a3b8", lw=1)
        ax.set_xlabel("onset RT (ms)")
        ax.set_ylabel("game's logged RT (ms)")
        ax.set_title("Same trials, two clocks")
        _save(fig, "onset_vs_logged_rt"); plt.show()
        r = np.nan
        if merged["onset_rt_ms"].std() > 0 and \
                merged["time_difference_ms"].std() > 0:
            r = float(np.corrcoef(merged["onset_rt_ms"],
                                  merged["time_difference_ms"])[0, 1])
        print(f"\n{len(merged)} trials matched by (game, trial).")
        print(f"bias: the logged RT sits {gap.mean():+.0f} ms after onset "
              f"on average (sd {gap.std():.0f} ms"
              + (f", r {r:.2f})." if pd.notna(r) else ")."))
        print("Which number when: the logged RT is a threshold crossing,")
        print("so it includes however long the rise to threshold took and")
        print("it moves with press vigour; it is the right number for how")
        print("the game scored and paced itself. The onset RT is when the")
        print("force first left its own baseline, so it is the right")
        print("number for motor initiation: comparisons with Rayan's")
        print("processed sessions, with Nakayama's figures, and with")
        print("anything EEG-aligned. Report both, never pool them.")
    elif len(thr):
        gap = thr.mean() - v.mean()
        print(f"\nNo per-trial match was possible (no shared trial ids), "
              f"so only the pooled means can be compared:")
        print(f"threshold crossing sits {gap:+.0f} ms after onset on average.")

    # ---- four-finger peaks and the window convention
    print("\nFOUR-FINGER PEAKS AND THE WINDOW CONVENTION")
    print("The game already logs what Rayan's script extracts: on every")
    print("trial, force_window_peaks carries the post-cue peak of EVERY")
    print("lane in play (reading minus that pad's frozen EMA baseline),")
    print("and the individuation and cross-talk chapters are built on")
    print("it. His script anchors the window at the detected onset")
    print("instead of at the cue; the check below measures whether that")
    print("choice changes the answer on this data.")
    peaks = four_finger_peaks(folders)
    both = (peaks.dropna(subset=["peak_stim_counts", "peak_onset_counts"])
            if not peaks.empty else peaks)
    if peaks.empty or len(both) < 4:
        print("Not enough onset-resolved presses here to run the window")
        print("comparison, so the cue-anchored convention stands by")
        print("default.")
    else:
        diff = (both["peak_stim_counts"] - both["peak_onset_counts"]).abs()
        n_big = int((diff > 2.0).sum())
        print(f"{len(both)} (cue, sensor) cells have a peak under both")
        print(f"windows: median |cue-anchored minus onset-anchored| "
              f"{diff.median():.2f} counts, {n_big} cell(s) above 2 counts.")
        n_lost = int(peaks["peak_onset_counts"].isna().sum())
        if n_lost:
            print(f"{n_lost} cell(s) have no onset-anchored peak at all")
            print("because no onset resolved on that trial; the cue-")
            print("anchored peak exists for every one of them.")
        print("Verdict: the cue-anchored window stays. It never loses a")
        print("trial to a failed onset fit, and it keeps leak that starts")
        print("BEFORE the target finger moves, which onset-anchoring cuts")
        print("off by construction; where both windows exist they agree")
        print("to within a few counts, so nothing was traded away.")
    return ons


# ====================================== Objective 1, per-finger hit rate
# Each finger against the band over its own rolling window. Dashed
# where that window is shorter than the objective assumes.

def sec_objective_one(trials, window=32, calset=None):
    """Objective 1 as the progress report words it: a per-finger hit rate
    between 65 and 80 percent over a 32-trial block.

    The session-level rolling figure elsewhere can sit inside the band
    while individual fingers sit well outside it, so this checks each
    finger against its own trials, which is what the objective claims.

    A finger's hit rate depends on how hard its own trigger is to reach,
    and the triggers are set per pad. `calset` adds each finger's trigger
    as a share of its own calibration press, which is what separates a
    finger that is genuinely weak from a finger sitting behind a harder
    threshold than the others.
    """
    # Scorable rows only: a survived catch passes (label != "Miss") and
    # would count as a hit inside the rolling window.
    cued = (trials[is_cued(trials) & is_scorable(trials)]
            if not trials.empty else trials)
    n_cued = len(cued)
    if "stim_delivered" in cued.columns:
        cued = cued[cued["stim_delivered"] != False]
    if cued.empty:
        if n_cued:
            print(f"All {n_cued} cued trials in this selection recorded the")
            print("cue as never delivered, so there is no trial the")
            print("participant can be scored on. Objective 1 needs blocks")
            print("where the cue actually reached the device.")
        else:
            print("No cued-mode trials in this selection, so there is no")
            print("per-finger hit rate to check against the band. Objective")
            print("1 applies to classic, adaptive and mirror blocks.")
        return None
    # A partial window is not a window. The objective is worded as a hit
    # rate over a 32-trial block, so a finger with 8 trials has not been
    # measured against it yet, and printing an 8-trial rolling mean under
    # a "32-trial window" heading claims a measurement that was not made.
    min_n = max(5, window // 4)
    print("\n" + "=" * 62)
    print(f"OBJECTIVE 1: PER-FINGER HIT RATE, ROLLING WINDOW UP TO "
          f"{window} TRIALS")
    print("=" * 62)
    print(f"The objective is worded over a full {window}-trial block. A "
          f"finger needs")
    print(f"{window} trials of its own for one full window. Fingers with "
          f"fewer are")
    print(f"shown over a partial window of at least {min_n} trials, drawn "
          f"dashed,")
    print("and in_band_share is left blank for them rather than computed")
    print("from a window the objective does not describe.")
    order = [f for f in FINGERS if f in cued["finger"].unique()]
    fig, axes = plt.subplots(1, len(order), figsize=(3.4 * len(order), 3.2),
                             sharey=True)
    if len(order) == 1:
        axes = [axes]
    rows = []
    for ax, f in zip(axes, order):
        g = cued[cued["finger"] == f].sort_values("trial")
        hit = (g["early_late"] != "Miss").astype(float)
        roll = hit.rolling(window, min_periods=min_n).mean()
        full = hit.rolling(window).mean().dropna()
        n_full = int(len(full))
        inband = full.between(BAND_LO, BAND_HI)
        first = None
        for k, (idx, val) in enumerate(full.items()):
            if BAND_LO <= val <= BAND_HI:
                first = k
                break
        rows.append({"finger": f, "trials": len(g),
                     "hit_rate": round(hit.mean(), 3),
                     "full_windows": n_full,
                     "in_band_share": (round(inband.mean(), 3)
                                       if n_full else np.nan),
                     "windows_to_settle": first})
        ax.axhspan(BAND_LO, BAND_HI, color="#16a34a", alpha=.15)
        ax.axhline(WILSON, color="#ca8a04", ls=":", lw=1.5)
        drawn = roll.reset_index(drop=True)
        n_partial = max(0, min(len(drawn), window - 1))
        ax.plot(range(len(drawn)), drawn, lw=2, ls="--", alpha=.55,
                color=FINGER_COLOUR[f])
        if n_full:
            ax.plot(range(n_partial, len(drawn)), drawn.iloc[n_partial:],
                    lw=2, color=FINGER_COLOUR[f])
        if first is not None:
            ax.axvline(n_partial + first, color="#0f172a", ls="--", lw=1,
                       label="first full window in band")
            ax.legend(frameon=False, fontsize=7)
        title = f if n_full else f"{f} (no full window)"
        ax.set_ylim(0, 1.02); ax.set_title(title); ax.set_xlabel("trial")
    axes[0].set_ylabel(f"hit rate (rolling, up to {window})")
    fig.suptitle(f"Objective 1 per finger, band 65 to 80 percent, "
                 f"solid = full {window}-trial window",
                 fontsize=11, fontweight="bold", x=0.02, ha="left")
    _save(fig, "objective_one"); plt.show()
    tbl = pd.DataFrame(rows)
    short = tbl[tbl["full_windows"] == 0]["finger"].tolist()
    if short:
        print(f"\nNOT ENOUGH DATA for a full {window}-trial window on: "
              f"{', '.join(short)}.")
        print("Their hit_rate is over every trial they have, which is fewer")
        print(f"than the {window} the objective asks for, so it is not yet a")
        print("test of the objective.")

    # How hard each finger's own trigger was, as a share of the press
    # that finger produced at calibration. Comparable between fingers,
    # unlike the trigger in counts.
    if calset is not None and calset.usable:
        shares = []
        for f in tbl["finger"]:
            vals = []
            # Per hand, so a left-hand trigger is compared against the
            # left hand's own reference press and not the right's.
            for game, hand, cal in calset.all_cals:
                gap = calset.gap(game, f, hand)
                on = (cal or {}).get("on_delta") or []
                i = _finger_index(f)
                if gap and i is not None and i < len(on):
                    try:
                        vals.append(float(on[i]) / gap)
                    except (TypeError, ValueError):
                        pass
            shares.append(round(float(np.mean(vals)), 2) if vals else np.nan)
        tbl["trigger_share_of_press"] = shares
    _show(tbl)
    if "trigger_share_of_press" in tbl.columns:
        print("trigger_share_of_press: how far into that finger's own")
        print("reference press the trigger sits. Fingers differ here only")
        print("because of the pads, so a finger with a higher share and a")
        print("lower hit rate is behind a harder threshold, not necessarily")
        print("weaker.")
    met = tbl["hit_rate"].between(BAND_LO, BAND_HI)
    print(f"\nfingers whose overall hit rate met the band: "
          f"{int(met.sum())} of {len(tbl)}")
    if not met.all():
        miss = ", ".join(tbl.loc[~met, "finger"])
        print(f"outside the band: {miss}")
    return tbl


# ====================================================== Trial exclusions
# Trials where no cue was delivered and presses faster than 100 ms,
# which the summary is built without.

def sec_exclusions(trials):
    """Flag trials that should not count, and show how much the headline
    numbers move once they are removed.

    Nakayama's search window let very fast presses count as genuine
    reactions, which matters most in exactly the predictable condition
    where anticipation is the confound. Anything under about 100 ms is
    faster than a real cued reaction and is almost certainly a guess.
    """
    print("\n" + "=" * 62)
    print("TRIAL EXCLUSIONS")
    print("=" * 62)
    if trials.empty:
        _nothing("No trials are loaded, so there is nothing to flag.")
        return None
    kept, df, counts = analysable(trials)
    print(f"recorded trials            : {counts['recorded']}")
    print(f"cue never delivered        : {counts['no_cue']}")
    print(f"faster than {ANTICIPATION_MS:.0f} ms         : "
          f"{counts['anticipation']}   (anticipation, not a reaction)")
    print(f"analysed                   : {counts['analysed']}")
    if counts["anticipation"]:
        # The app's top reward tier (Perfect, rt <= 100 ms, pinned to
        # Satoru's 2025 scoring schema for historical comparability)
        # sits at the same boundary, so the screen can celebrate a
        # press this analysis refuses to count. Named here so the two
        # readings cannot be mistaken for a contradiction.
        print(f"Note: the app's Perfect tier covers presses at or under")
        print(f"{ANTICIPATION_MS:.0f} ms, so in cadence modes nearly every "
              "on-screen Perfect")
        print("lands under the anticipation floor and is excluded here.")
        print("The two readings disagree on purpose: the screen rewards")
        print("effort, the analysis counts reactions.")

    def headline(d):
        c = d[is_cued(d) & is_scorable(d)]
        s = rt_stats(d)
        return {"hit_rate": round(float((c["early_late"] != "Miss").mean()), 3)
                             if len(c) else np.nan,
                "mean_rt": s["mean_rt"], "rt_cv": s["rt_cv"]}
    before, after = headline(df), headline(kept)
    cmp_tbl = pd.DataFrame([{"": "with everything", **before},
                            {"": "after exclusions", **after}])
    _show(cmp_tbl)
    print("The summary at the end of the notebook uses the second row.")
    print("Sections above print over every recorded trial unless they say")
    print("otherwise, so a figure there can differ from the summary.")
    if counts["analysed"] == 0:
        print("\nNOTHING IN THIS SELECTION CAN BE ANALYSED. Every trial is")
        print("flagged, so there is no hit rate and no reaction time to")
        print("report from it. Fix the cue path and record again.")
    return df


# ================================================== Pretest to aftertest
# Stays quiet until a protocol with phases has actually been run.

def sec_phase(trials):
    """Pretest, main and aftertest comparison.

    Nakayama and Lee's headline claim is that the gain is specific to the
    trained sequence rather than general warm-up, and it rests entirely
    on comparing the aftertest against the last trained block. The phase
    column already exists in the CSV, so this is free once a protocol
    has actually been run.
    """
    if "phase" not in trials.columns:
        print("No protocol phase recorded, so there is no pretest to\n"
              "aftertest change to show. Phase is only set when a block\n"
              "is run as part of a protocol.")
        return None
    ph = trials[trials["phase"].notna() & (trials["phase"] != "")]
    if ph.empty or ph["phase"].nunique() < 2:
        print("Only one protocol phase in this selection. A pretest to\n"
              "aftertest comparison needs at least two.")
        return None
    print("\n" + "=" * 62)
    print("PRETEST TO AFTERTEST")
    print("=" * 62)
    rows = []
    for (who, phase), g in ph.groupby(["participant", "phase"]):
        s = rt_stats(g)
        scored = g[is_scorable(g)]
        rows.append({"participant": who, "phase": phase, "trials": len(g),
                     "rt_trials": s["n"],
                     "mean_rt": s["mean_rt"], "rt_cv": s["rt_cv"],
                     "hit_rate": (round(float((scored["early_late"] != "Miss")
                                              .mean()), 3)
                                  if len(scored) else np.nan)})
    tbl = pd.DataFrame(rows)
    _show(tbl)

    fig, ax = plt.subplots(figsize=(8, 3.4))
    order = [p for p in ("pretest", "main", "aftertest")
             if p in tbl["phase"].unique()]
    for who, g in tbl.groupby("participant"):
        g = g.set_index("phase").reindex(order)
        ax.plot(order, g["mean_rt"], "o-", lw=2, label=who)
    ax.set_ylabel("mean reaction time (ms)")
    ax.set_title("Reaction time by protocol phase")
    ax.legend(frameon=False, fontsize=8)
    _save(fig, "phase"); plt.show()

    if {"pretest", "aftertest"} <= set(tbl["phase"]):
        pre = tbl[tbl["phase"] == "pretest"]["mean_rt"].mean()
        post = tbl[tbl["phase"] == "aftertest"]["mean_rt"].mean()
        print(f"\npretest {pre:.0f} ms, aftertest {post:.0f} ms, "
              f"change {post - pre:+.0f} ms")
    return tbl


# =========================================== Press thresholds in newtons
# What force each finger needed before it registered a press, against
# the healthy fingertip forces Demouche measured.

def threshold_sets(metas) -> list:
    """Every distinct press-threshold set behind a selection, with the
    games that ran under each.

    Taking the first entry of metas and printing it as the report's
    thresholds is wrong twice over: metas is insertion ordered, so the
    first entry is the OLDEST game, and a selection can easily span more
    than one calibration. Returns a list of dicts instead, so nothing has
    to be collapsed.
    """
    out = {}
    for name, meta in _meta_items(metas):
        # One entry per hand: the two hands have their own pads and
        # their own thresholds, and averaging or overwriting one with
        # the other prints a trigger no finger ever ran under.
        cals = read_calibrations(meta)
        found = False
        for hand, cal in sorted(cals.items()):
            on = cal.get("on_delta")
            if not on:
                continue
            found = True
            source = f"calibration, {hand} hand"
            stamp = cal.get("created_at") or "unknown"
            key = (source, stamp, tuple(on))
            out.setdefault(key, {"source": source, "stamp": stamp,
                                 "on_delta": list(on), "hand": hand,
                                 "games": []})["games"].append(name)
        if found:
            continue
        snap = ((meta.get("config_snapshot") or {}).get("fsr") or {})
        on = snap.get("on_delta")
        source, stamp = "config snapshot", "no calibration"
        if not on:
            source, stamp, on = "unrecorded", "unknown", None
        key = (source, stamp, tuple(on) if on else None)
        out.setdefault(key, {"source": source, "stamp": stamp,
                             "on_delta": list(on) if on else None,
                             "hand": "?", "games": []})["games"].append(name)
    return [v for _, v in sorted(out.items(), key=lambda kv: str(kv[0]))]


def sec_threshold_audit(cfg_on_delta=None, metas=None, calset=None):
    """Put the press thresholds into newtons and check them against the
    only healthy force data in this project's lineage.

    Demouche measured healthy peak fingertip force on the 2025 button
    device. If a trigger sits above what a healthy little finger can
    produce, a weak finger cannot reach it either, and the game will
    score a genuine attempt as a miss. That reads as a patient deficit
    when it is really a threshold problem, so it is worth checking
    before any participant session.

    One table per distinct threshold set. Nothing is averaged and no set
    stands in for another, because a game only ever ran under its own.
    """
    print("\n" + "=" * 62)
    print("PRESS THRESHOLDS IN NEWTONS")
    print("=" * 62)
    print(f"SingleTact {SENSOR_RATING_N:.0f} N part, "
          f"{N_PER_COUNT:.4f} N per count")

    sets = []
    if cfg_on_delta is not None:
        sets = [{"source": "supplied", "stamp": "supplied by the caller",
                 "on_delta": list(cfg_on_delta), "hand": "?", "games": []}]
    elif metas:
        found = threshold_sets(metas)
        sets = [s for s in found if s["on_delta"]]
        for s in (s for s in found if not s["on_delta"]):
            print(f"\n{len(s['games'])} game(s) recorded no thresholds at "
                  f"all: {', '.join(s['games'])}")
    if not sets:
        try:
            # Only when the game's own package is actually sitting
            # there. Putting the parent directory on the path blind
            # makes this notebook import whatever happens to be next
            # to wherever it was copied.
            import sys as _s
            # Walk up the way the sessions folder is found. Looking only
            # at the parent tied this branch to a kernel started inside
            # analysis/: start Jupyter at the repo root instead and
            # finger_rehab/config.py is sitting right there, but this still
            # said it could not read the config and returned nothing.
            _here = Path.cwd().resolve()
            _repo = next((b for b in (_here, *list(_here.parents)[:4])
                          if (b / "rehab" / "config.py").exists()), None)
            if _repo is None:
                raise ImportError("no rehab package near this notebook")
            _s.path.insert(0, str(_repo))
            from finger_rehab.config import Config
            sets = [{"source": "current config", "on_delta":
                     list(Config.load().get("fsr.on_delta")),
                     "stamp": "as the config reads today", "hand": "?",
                     "games": []}]
            print("\nNo thresholds in the session metadata, so these are the")
            print("CURRENT config values. They are not necessarily what this")
            print("data ran under.")
        except Exception:
            print("Could not read fsr.on_delta from the config.")
            return None

    tables = []
    for s in sets:
        # One row per finger whatever the list holds, so a short or
        # partly unreadable list leaves a blank in the right place
        # instead of shifting the fingers along.
        rows = []
        for i, finger in enumerate(FINGERS):
            d = np.nan
            if i < len(s["on_delta"]):
                try:
                    d = float(s["on_delta"][i])
                except (TypeError, ValueError):
                    d = np.nan
            rows.append({"finger": finger, "on_delta_counts": d,
                         "trigger_N": (round(counts_to_newtons(d), 2)
                                       if pd.notna(d) else np.nan)})
        if all(pd.isna(r["on_delta_counts"]) for r in rows):
            continue
        tbl = pd.DataFrame(rows)
        tbl.attrs["label"] = f"{s['source']}, {s['stamp']}"
        tbl.attrs["games"] = s["games"]
        tables.append(tbl)
        print(f"\nfrom the {s['source']} ({s['stamp']})")
        if s["games"]:
            print(f"   {len(s['games'])} game(s): {', '.join(s['games'])}")
        _show(tbl)

    if not tables:
        return None

    fig, ax = plt.subplots(figsize=(9, 3.6))
    width = 0.8 / len(tables)
    for i, tbl in enumerate(tables):
        x = np.arange(len(tbl)) + (i - (len(tables) - 1) / 2) * width
        ax.bar(x, tbl["trigger_N"], width,
               color=[FINGER_COLOUR[f] for f in tbl["finger"]],
               edgecolor="white",
               label=tbl.attrs["label"] if len(tables) > 1 else None)
    ax.set_xticks(np.arange(len(tables[0])))
    ax.set_xticklabels(tables[0]["finger"])
    ax.axhline(DEMOUCHE_2025["little_mean"], color="#dc2626", ls="--", lw=2,
               label=f"healthy little finger mean "
                     f"{DEMOUCHE_2025['little_mean']} N")
    ax.axhline(DEMOUCHE_2025["little_max"], color="#dc2626", ls=":", lw=2,
               label=f"healthy little finger max "
                     f"{DEMOUCHE_2025['little_max']} N")
    ax.set_ylabel("force needed to register a press (N)")
    ax.set_title("Trigger thresholds against healthy force (Demouche 2025)")
    ax.legend(frameon=False, fontsize=8)
    _save(fig, "threshold_audit"); plt.show()

    for tbl in tables:
        pinky = tbl[tbl["finger"] == "Pinky"]
        if pinky.empty:
            continue
        little = float(pinky.iloc[0]["trigger_N"])
        if little > DEMOUCHE_2025["little_max"]:
            print(f"\n   WARNING ({tbl.attrs['label']}): the pinky trigger")
            print(f"   is {little:.2f} N, above the highest little-finger")
            print(f"   force Demouche recorded in healthy participants "
                  f"({DEMOUCHE_2025['little_max']} N).")
            print("   A weak little finger may be physically unable to reach")
            print("   it, and every attempt would be logged as a miss. Run")
            print("   the Calibrate step from the title screen with the")
            print("   participant's own hand before the next session, and")
            print("   look at why that pad carries so much load at rest.")
    return tables[0] if len(tables) == 1 else tables


# ========================================================== Cue modality
# Visual against vibration against both, which needs blocks recorded
# under at least two cue settings.

def cue_condition(trials):
    """Give every trial a readable cue label, whichever build recorded it.

    Sessions from v3.2 on carry cue_flags ("BS/BS": before then after,
    buzzer slot then sound slot, a dash for off) plus cue_target_shown.
    Older sessions carry cue_mode with only "both", "visual" or
    "vibration". Both are turned into one cue_mode column here so a
    comparison can pool them.

    buzz_before, sound_before and show_target are three independent
    switches on the Settings screen, so with the screen off there are
    three different non-visual conditions, not one: buzzer alone
    ("vibration", the tactile-only condition), tone alone ("auditory"),
    or both together ("both-nonvisual"). Folding buzz and sound into a
    single "vibration" bucket would silently report an auditory-only
    trial as if the finger had been found by touch, so the three stay
    separate here even though only "vibration" isolates touch.
    """
    if trials is None or trials.empty:
        return trials
    if "cue_flags" not in trials.columns:
        return trials
    out = trials.copy()

    def truthy(v):
        # as_bool works on a Series; this is one cell at a time. Anything
        # missing counts as shown, because that is what every build
        # before the switch existed actually did.
        if v is None or (isinstance(v, float) and v != v):
            return True
        if isinstance(v, str):
            return v.strip().lower() not in ("false", "0", "no", "")
        return bool(v)

    def label(row):
        flags = str(row.get("cue_flags") or "")
        shown = truthy(row.get("cue_target_shown"))
        before = flags.split("/")[0] if "/" in flags else flags
        buzz = "B" in before
        sound = "S" in before
        if not shown:
            if buzz and sound:
                return "both-nonvisual"
            if buzz:
                return "vibration"
            if sound:
                return "auditory"
            return "none"
        if buzz or sound:
            return "both"
        return "visual"

    out["cue_mode"] = out.apply(label, axis=1)
    # The exact four-way state, for anyone who wants the finer split
    # rather than the three names the earlier work reported.
    out["cue_detail"] = (out["cue_flags"].astype(str) + " screen="
                         + out["cue_target_shown"].map(
                             lambda v: "on" if truthy(v) else "off"))
    return out


def sec_cue_modality(trials, calset=None):
    """Compare visual only, vibration only, auditory only and both.

    Palmer (2024) found reaction time differed between an LED-only cue
    and all cues together, and the 2023 device existed to test exactly
    that. Vibration-only is the condition worth reporting: the screen
    does not say which finger, so it has to be found by touch, which
    isolates the tactile channel. Expect it to be slower and less
    accurate, and that difference is the result. Auditory-only (a tone,
    no buzz, no screen) and both-nonvisual (buzz and tone together, no
    screen) are reported alongside it but are not the tactile-isolation
    condition: a session found by ear is not a session found by touch.
    """
    trials = cue_condition(trials)
    if "cue_mode" not in trials.columns:
        print("\n" + "=" * 62)
        print("CUE MODALITY")
        print("=" * 62)
        print("No cue setting was recorded on these trials, so there is")
        print("nothing to split by.")
        return None
    trials = ensure_force_columns(trials, calset)
    cm = trials[trials["cue_mode"].notna() & (trials["cue_mode"] != "")]
    # Palmer's comparison is the SAME task under different cue
    # settings, so only the modes that consume the cue toggles belong
    # here. buzz_hunt runs screen-off BY DESIGN (the buzz is its
    # stimulus, not its cue), so its rows would land in a "none" cue
    # condition and a task difference would read as a cue effect; the
    # continuous modes fly their own corridor, and pattern, chords and
    # syllables carry trial structures with chapters of their own.
    if "mode" in cm.columns:
        toggled = cm["mode"].isin(CUED_MODES + ("rhythm",))
        n_other = int((~toggled).sum())
        cm = cm[toggled]
    else:
        n_other = 0
    if cm.empty or cm["cue_mode"].nunique() < 2:
        if not cm.empty or n_other:
            print("\n" + "=" * 62)
            print("CUE MODALITY")
            print("=" * 62)
        if not cm.empty:
            only = cm["cue_mode"].iloc[0]
            print(f"Every trial here used the '{only}' cue, so there is")
            print("nothing to compare. Run blocks under at least two")
            print("settings (Settings screen, CUE pill) to get the")
            print("comparison Palmer's result rests on.")
        elif n_other:
            print("Every trial here comes from a mode that does not")
            print("consume the cue settings (buzz_hunt's buzz IS its")
            print("stimulus; the continuous modes fly their own")
            print("corridor), so there is no cue comparison to make.")
        return None

    print("\n" + "=" * 62)
    print("CUE MODALITY: VISUAL vs VIBRATION vs BOTH")
    print("=" * 62)
    if n_other:
        print(f"{n_other} trial(s) from modes that do not consume the cue")
        print("settings are left out: their task differs, so their rows")
        print("would read a task difference as a cue effect.")
    rows = []
    for mode, g in cm.groupby("cue_mode"):
        # Cued modes only, so a rhythm block under the same cue setting
        # cannot drag the mean toward zero with its beat offsets.
        v = reaction_times(g)
        s = rt_stats(g)
        scored = g[is_scorable(g)]
        rows.append({
            "cue": mode, "trials": len(g),
            "hit_rate": (round(float((scored["early_late"] != "Miss")
                                     .mean()), 3)
                         if len(scored) else np.nan),
            "rt_trials": s["n"],
            "mean_rt": s["mean_rt"],
            "median_rt": round(float(v.median()), 1) if len(v) else np.nan,
            "rt_cv": s["rt_cv"],
            "wrong_finger": int((g["had_incorrect_press"] == True).sum()),
            "mean_force_raw": (round(g["peak_force_n"].mean(), 1)
                               if g["peak_force_n"].notna().any() else np.nan),
            # Each cue condition pools all four fingers, so a raw mean
            # force is weighted by which pads happened to come up. The
            # calibrated mean is not.
            "mean_force_cal": (round(g["peak_force_cal"].mean(), 3)
                               if g["peak_force_cal"].notna().any()
                               else np.nan),
        })
    tbl = pd.DataFrame(rows).sort_values("cue").reset_index(drop=True)
    _show(tbl)
    print("mean_rt is over cued modes only. rt_trials says how many trials")
    print("are behind it, and it is blank for a cue setting that only ever")
    print("ran in rhythm mode.")
    if tbl["mean_force_cal"].notna().any():
        print(f"mean_force_cal is in {NORM_UNIT}: relative {NORM_SHORT}.")
        print("It compares cues within a finger, not fingers with each")
        print("other.")
    elif tbl["mean_force_raw"].notna().any():
        print("mean_force_raw is raw counts pooled over four differently")
        print("sensitive pads, so a difference between cues here can come")
        print("from which fingers were cued rather than from the cue.")

    colours = {"both": "#2563eb", "visual": "#ca8a04",
               "vibration": "#a855f7", "auditory": "#0d9488",
               "both-nonvisual": "#dc2626"}
    cols = [colours.get(c, "#94a3b8") for c in tbl["cue"]]
    fig, ax = plt.subplots(1, 3, figsize=(14, 3.6))
    x = np.arange(len(tbl))
    ax[0].bar(x, tbl["mean_rt"], color=cols, width=.6)
    ax[0].set_ylabel("mean reaction time (ms)")
    ax[0].set_title("Speed by cue")
    ax[1].bar(x, tbl["hit_rate"], color=cols, width=.6)
    ax[1].axhspan(BAND_LO, BAND_HI, color="#16a34a", alpha=.15)
    ax[1].set_ylim(0, 1.02); ax[1].set_ylabel("hit rate")
    ax[1].set_title("Accuracy by cue")
    ax[2].bar(x, tbl["wrong_finger"], color=cols, width=.6)
    ax[2].set_ylabel("trials with a wrong finger")
    ax[2].set_title("Wrong-finger errors by cue")
    for a in ax:
        a.set_xticks(x); a.set_xticklabels(tbl["cue"])
    _save(fig, "cue_modality"); plt.show()

    # Per-finger RT by cue, since a tactile-only cue may hurt the weaker
    # fingers more than the strong ones. Built from reaction_times, the
    # same cued-modes-only pool the table above uses (audit finding
    # #105): this chart used to average raw time_difference_ms over
    # every mode, so a rhythm block under a cue setting dragged a
    # finger's bar toward or below zero with its signed beat offsets,
    # the exact pooling the table's "mean_rt is over cued modes only"
    # line exists to prevent.
    order = [f for f in FINGERS if f in cm["finger"].unique()]
    if len(order) > 1:
        fig, ax = plt.subplots(figsize=(9, 3.4))
        w = 0.8 / max(1, len(tbl))
        for i, mode in enumerate(tbl["cue"]):
            rt_rows = reaction_times(cm[cm["cue_mode"] == mode], per="rows")
            vals = [rt_rows[rt_rows["finger"] == f]["time_difference_ms"]
                    .mean() if not rt_rows.empty else np.nan for f in order]
            ax.bar(np.arange(len(order)) + (i - (len(tbl)-1)/2) * w, vals, w,
                   label=mode, color=colours.get(mode, "#94a3b8"))
        ax.set_xticks(np.arange(len(order))); ax.set_xticklabels(order)
        ax.set_ylabel("mean reaction time (ms)")
        ax.set_title("Reaction time per finger, by cue")
        ax.legend(frameon=False, fontsize=8)
        _save(fig, "cue_modality_per_finger"); plt.show()

    if {"visual", "vibration"} <= set(tbl["cue"]):
        vis = tbl[tbl["cue"] == "visual"].iloc[0]
        vib = tbl[tbl["cue"] == "vibration"].iloc[0]
        d_rt = vib["mean_rt"] - vis["mean_rt"]
        d_hit = vib["hit_rate"] - vis["hit_rate"]
        print(f"\nvibration minus visual: {d_rt:+.0f} ms, "
              f"hit rate {d_hit:+.3f}")
    if "auditory" in set(tbl["cue"]):
        print("auditory is a tone with no buzz and no screen: the finger")
        print("is found by ear, not by touch, so it is not the")
        print("tactile-isolation condition even though the screen is off.")
    return tbl


# ================================================================== Dose
# Repetitions against Lang's clinical benchmark, using `on_task` from
# the overview section.

def sec_dose(trials, on_task_min=0.0):
    """Repetitions against the clinical benchmark.

    Lang's figure of about 32 repetitions in a typical therapy session is
    the number the whole dose argument rests on, so it is worth plotting
    rather than only citing.

    `on_task_min` is what sec_overview returns, divided by 60. Passing 0
    skips the per-minute lines rather than guessing at them.

    Counted from analysable trials only (audit finding #100): this used
    to be len(trials), so a trial where the cue never reached the device
    (nothing was presented) or that scored as an anticipation counted as
    a repetition, under a headline table that separately claims
    "analysable trials only". CatchOk rows -- a correctly WITHHELD
    press, reaction and buzz_hunt's catch-trial control -- are excluded
    too, because the correct response there is not pressing, so it is
    not a repetition of the movement Lang's benchmark counts. So are
    trials with no press at all (timeouts, missed notes): a trial that
    contains zero movement is not a repetition under Lang's construct,
    and on real data those were 14 percent of the count.
    """
    kept, _flagged, _counts = analysable(trials)
    if "early_late" in kept.columns:
        kept = kept[kept["early_late"] != "CatchOk"]
    # Rows that predate the num_presses column count as pressed,
    # matching the old behaviour rather than zeroing an old session's
    # dose.
    presses = (pd.to_numeric(kept["num_presses"], errors="coerce")
               if "num_presses" in kept.columns
               else pd.Series(np.nan, index=kept.index))
    no_press = int((presses.fillna(1) <= 0).sum())
    reps = len(kept) - no_press
    print("\n" + "=" * 62)
    print("DOSE")
    print("=" * 62)
    print(f"repetitions this selection : {reps}  (analysable movement "
          "trials with at least one press)")
    if no_press:
        print(f"no-press trials left out   : {no_press}  (timeouts and "
              "missed notes: no movement, so no repetition)")
    print(f"typical clinical session   : {LANG_REPS_PER_SESSION} (Lang)")
    if reps:
        print(f"ratio                      : {reps/LANG_REPS_PER_SESSION:.1f}x")
    if on_task_min > 0:
        print(f"rate                       : {reps/on_task_min:.1f} per minute")
        print(f"projected over 30 min      : "
              f"{reps/on_task_min*30:.0f} repetitions")
    fig, ax = plt.subplots(figsize=(7, 2.6))
    ax.barh(["this selection", "typical clinical session"],
            [reps, LANG_REPS_PER_SESSION],
            color=["#16a34a", "#94a3b8"], height=.55)
    ax.set_xlabel("repetitions")
    ax.set_title("Repetitions against the clinical benchmark")
    _save(fig, "dose"); plt.show()
    # Handed back so the headline table carries the dose argument as
    # well as printing it. The ratio is the number the thesis quotes.
    out = {"reps": reps, "lang_reps": LANG_REPS_PER_SESSION,
           "ratio": round(reps / LANG_REPS_PER_SESSION, 2) if reps else 0.0}
    if on_task_min > 0:
        out["reps_per_min"] = round(reps / on_task_min, 1)
        out["projected_30_min"] = int(round(reps / on_task_min * 30))
    return out


# ============================================================== Sampling
# How many logged samples actually carry new sensor data, which is the
# real resolution behind every onset figure.

def sec_sampling_note(folders):
    """How many logged samples actually carry new sensor data.

    The SingleTact interface board updates its output register at about
    50 to 120 Hz whatever rate it is polled at, so a 200 Hz log contains
    repeated frames. That sets the real resolution of any onset time or
    rate-of-force figure, and it belongs in the limitations section.

    Checked against all eight channels, right hand and left (audit
    finding #108): this used to check fsr1-4 only, so on a bilateral
    block a frame where only the left hand changed still counted as a
    duplicate, and this section's "effective new-data rate" could
    disagree with sample_rate_rows's per-game figure for the identical
    log. Same channel list and the same numeric coercion as
    sample_rate_rows now, so the two cannot drift apart again.
    """
    for folder in folders:
        raw = load_raw(folder)
        if raw is None:
            continue
        s = raw[raw["event"].isna() | (raw["event"] == "")]
        if len(s) < 50:
            continue
        cols = [c for c in [f"fsr{i}" for i in range(1, 9)]
                if c in s.columns]
        if not cols:
            continue
        same = (s[cols].apply(pd.to_numeric, errors="coerce")
                .diff().abs().sum(axis=1) == 0)
        dup = float(same.mean())
        dur = s["t_perf"].max() - s["t_perf"].min()
        logged = len(s) / max(dur, 1e-9)
        print("\n" + "=" * 62)
        print("SAMPLING")
        print("=" * 62)
        print(f"logged rate            : {logged:.0f} Hz")
        print(f"frames identical to the one before : {dup:.0%}")
        print(f"effective new-data rate: {logged * (1 - dup):.0f} Hz")
        return {"logged_hz": logged, "duplicate_fraction": dup,
                "effective_hz": logged * (1 - dup)}
    print("Not enough raw samples to estimate the sampling rate.\n"
          "This needs at least 50 rows in raw.csv, so it's empty for\n"
          "keyboard sessions and very short blocks.")
    return None


# ======================================================= Startup latency
# Connection to first sample against the 2 second objective, per
# board.

# Thread D's first two objectives, word for word from the progress
# report: "Reduce setup time from device connection to first data sample
# to less than 2 seconds through automatic USB device detection", and
# "Record and store startup latency for each connected device in
# session.json at the beginning of every session".
#
# The app records it. Nothing in this notebook read it, so an objective
# with a hard numeric target had no figure to be judged against, and the
# data already on disk says the target is being missed by more than
# double.
#
# Read the number with the boot wait in mind. The README says the
# firmware buzzes all four motors as a self-test on connect, about 1.6
# s, and the host then waits 3 s after opening the port so the boot
# finishes before it starts reading. So most of a 4.2 s measurement is a
# deliberate wait, not discovery. That is an argument about where the
# clock should start, and it has to be made in the thesis rather than
# assumed: either the objective is measured from the end of the boot
# wait and says so, or it is measured as it is now and reported as not
# met.

STARTUP_TARGET_MS = 2000.0


FIRMWARE_BOOT_WAIT_MS = 3000.0


def startup_rows(folders, metas):
    """One row per board per block, with the latency the app stored."""
    rows = []
    for f in folders:
        name = game_key(f)
        meta = (metas or {}).get(name, {}) or {}
        bs = meta.get("block_summary", {}) or {}
        lat = bs.get("startup_latency_ms")
        if not bs:
            rows.append({"game": name, "port": "", "latency_ms": np.nan,
                         "recorded": False, "note": "block never finished"})
            continue
        if lat is None:
            rows.append({"game": name, "port": "", "latency_ms": np.nan,
                         "recorded": False,
                         "note": "no serial source, so nothing to time"})
            continue
        if not isinstance(lat, dict):
            lat = {"?": lat}
        for port, ms in sorted(lat.items()):
            try:
                v = float(ms)
            except (TypeError, ValueError):
                v = np.nan
            rows.append({
                "game": name, "port": str(port),
                "latency_ms": round(v, 1) if pd.notna(v) else np.nan,
                "recorded": bool(pd.notna(v)),
                "note": ("" if pd.notna(v)
                         else "port listed but never timed")})
    return pd.DataFrame(rows)


def sec_startup_latency(folders, metas):
    """Startup latency against the 2 second objective."""
    print("\n" + "=" * 62)
    print("STARTUP LATENCY")
    print("=" * 62)
    if not folders:
        _nothing("Nothing is selected, so there is no startup time to read.")
        return None
    tbl = startup_rows(folders, metas)
    if tbl.empty:
        _nothing("No block summary in this selection, so no startup time",
                 "was stored.")
        return None
    _show(tbl)
    timed = tbl[tbl["latency_ms"].notna()]
    n_blocks = tbl["game"].nunique()
    n_timed = timed["game"].nunique()
    print(f"\nOBJECTIVE D2, stored for every session: {n_timed} of "
          f"{n_blocks} block(s) carry a measured value.")
    if n_timed < n_blocks:
        print("   The rest ran on the keyboard source or never finished a")
        print("   block, so there was no port to time. That is the reason,")
        print("   not an excuse: the objective says every session, so a")
        print("   session with no board should say so explicitly rather")
        print("   than leave the key null.")
    if timed.empty:
        print("\nNo block in this selection timed a board, so Objective D1")
        print("cannot be checked here. Pick a session recorded over serial.")
        return tbl

    met = timed["latency_ms"] < STARTUP_TARGET_MS
    print(f"\nOBJECTIVE D1, under {STARTUP_TARGET_MS:.0f} ms: "
          f"{int(met.sum())} of {len(timed)} board start-ups met it.")
    print(f"   median {timed['latency_ms'].median():.0f} ms, "
          f"range {timed['latency_ms'].min():.0f} to "
          f"{timed['latency_ms'].max():.0f} ms")
    if not met.all():
        print(f"   NOT MET on {int((~met).sum())} of {len(timed)}. The host")
        print(f"   waits {FIRMWARE_BOOT_WAIT_MS:.0f} ms after opening the")
        print("   port so the firmware self-test finishes, and that wait is")
        print("   inside this measurement, which accounts for most of it.")
        print("   Two honest ways to write this up: report it as not met as")
        print("   measured, or redefine the objective as time from the end")
        print("   of the boot wait to the first sample and say so in the")
        print("   method. Do not quietly change the clock.")

    fig, ax = plt.subplots(figsize=(9, 3.4))
    x = np.arange(len(timed))
    colours = ["#16a34a" if v < STARTUP_TARGET_MS else "#dc2626"
               for v in timed["latency_ms"]]
    ax.bar(x, timed["latency_ms"], color=colours, width=.6)
    ax.axhline(STARTUP_TARGET_MS, color="#0f172a", ls="--", lw=2,
               label=f"objective {STARTUP_TARGET_MS:.0f} ms")
    ax.axhline(FIRMWARE_BOOT_WAIT_MS, color="#94a3b8", ls=":", lw=2,
               label=f"host boot wait {FIRMWARE_BOOT_WAIT_MS:.0f} ms")
    ax.set_xticks(x)
    ax.set_xticklabels(timed["game"], rotation=25, ha="right", fontsize=8)
    ax.set_ylabel("connection to first sample (ms)")
    ax.set_title("Startup latency against the 2 second objective")
    ax.legend(frameon=False, fontsize=8)
    _save(fig, "startup_latency"); plt.show()
    return tbl


# ======================================= Constraint budget from the logs
# Achieved sample rate against the target, and which constraints these
# logs cannot check at all.

# Table 3 of the progress report is the cross-subsystem constraint
# budget, and three of its six rows are software ones this data could
# speak to: "Sustained serial sample rate per Arduino: 200 Hz",
# "Game-loop frame budget: 120 FPS (8.3 ms per frame)", marked "verified
# visually", and "Sensor settling time after a press rising edge: below
# 5 ms".
#
# The sampling section above gives the effective new-data rate, which is
# the right caveat on every onset figure, but it never puts an achieved
# number next to the 200 Hz target. This does, per block, and says
# plainly which rows of that table the logs cannot answer at all. A
# constraint marked achieved with no measurement behind it is worth
# less in the write-up than one marked not measured with a reason.


def sample_rate_rows(folders, metas=None):
    """Achieved logging rate per block against the configured target."""
    rows = []
    for f in folders:
        name = game_key(f)
        meta = (metas or {}).get(name, {}) or {}
        target = ((meta.get("config_snapshot") or {}).get("fsr") or {}) \
            .get("sample_rate_hz")
        raw = load_raw(f)
        if raw is None:
            rows.append({"game": name, "target_hz": target, "samples": 0,
                         "logged_hz": np.nan, "duplicate_share": np.nan,
                         "effective_hz": np.nan,
                         "boards": "", "note": "no raw stream"})
            continue
        s = raw[raw["event"].isna() | (raw["event"] == "")]
        if len(s) < 50:
            rows.append({"game": name, "target_hz": target,
                         "samples": int(len(s)), "logged_hz": np.nan,
                         "duplicate_share": np.nan, "effective_hz": np.nan,
                         "boards": "", "note": "under 50 samples"})
            continue
        t = pd.to_numeric(s["t_perf"], errors="coerce")
        dur = float(t.max() - t.min())
        logged = len(s) / max(dur, 1e-9)
        cols = [c for c in [f"fsr{i}" for i in range(1, 9)]
                if c in s.columns]
        same = (s[cols].apply(pd.to_numeric, errors="coerce")
                .diff().abs().sum(axis=1) == 0)
        dup = float(same.mean())
        hands = sorted({str(h) for h in s["hand"].dropna().unique()})
        rows.append({
            "game": name, "target_hz": target, "samples": int(len(s)),
            "logged_hz": round(logged, 1),
            "duplicate_share": round(dup, 3),
            "effective_hz": round(logged * (1 - dup), 1),
            "boards": "two" if "both" in hands else "one",
            "note": ""})
    return pd.DataFrame(rows)


def sec_constraints(folders, metas=None):
    """What the logs can and cannot say about the constraint budget."""
    print("\n" + "=" * 62)
    print("CONSTRAINT BUDGET FROM THE LOGS")
    print("=" * 62)
    if not folders:
        _nothing("Nothing is selected, so there is no log to measure.")
        return None
    tbl = sample_rate_rows(folders, metas)
    _show(tbl)
    ok = tbl[tbl["logged_hz"].notna()]
    if ok.empty:
        print("No block here logged enough samples to measure a rate.")
    else:
        for _, r in ok.iterrows():
            if not r["target_hz"]:
                continue
            share = r["logged_hz"] / float(r["target_hz"])
            verdict = "holds" if share >= 0.95 else "SHORT of target"
            print(f"{r['game']}: {r['logged_hz']:.0f} Hz logged against a "
                  f"{float(r['target_hz']):.0f} Hz target, {share:.0%}, "
                  f"{verdict}")
        print("\nOn a one-board block the logged row rate is that board's")
        print("rate. On a two-board block the merger writes one row per")
        print("merged sample, so the figure is the merged rate and not the")
        print("rate of either board on its own.")
        print("effective_hz is the rate of rows that actually carry NEW")
        print("sensor values. The SingleTact interface board refreshes its")
        print("output register at roughly 50 to 120 Hz whatever it is")
        print("polled at, so the gap between logged and effective is the")
        print("sensor, not a dropped sample.")

    print("\nWHAT THESE LOGS CANNOT ANSWER, and why:")
    print("   Game-loop frame budget (120 FPS, 8.3 ms). Nothing in")
    print("   raw.csv or trials.csv is a frame timestamp, so the claim")
    print("   stays verified visually until the engine logs frame times.")
    print("   Sensor settling time (below 5 ms). One sample at 200 Hz is")
    print("   5 ms, so this log cannot resolve it even in principle. It")
    print("   needs a bench measurement against a scope or a faster")
    print("   capture.")
    print("   Press to CSV row latency (below 50 ms). The trial row is")
    print("   stamped at trial close, not at the threshold crossing, so")
    print("   the two timestamps in the logs do not bracket it.")
    return tbl


# ============================================== Progress per participant
# Every session a person has done, in order, where a session is one
# person on one day.

def _progress_game(folder, mode) -> pd.DataFrame:
    """One game's trials, with the columns the progress table needs.

    Read here rather than from `trials` because this section covers
    everyone on disk, not just the current selection.
    """
    folder = Path(folder)
    try:
        df = pd.read_csv(folder / "trials.csv")
    except OSError:
        return pd.DataFrame()
    if df.empty:
        return df
    for c in ("time_difference_ms", "peak_force_n", "lane"):
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    if "stim_delivered" in df.columns:
        df["stim_delivered"] = as_bool(df["stim_delivered"])
    meta = read_meta(folder)
    hand_mode = meta.get("hand", "right")
    df["mode"] = mode
    df["game"] = game_key(folder)
    df["hand_mode"] = hand_mode
    df["side"] = [lane_side((int(l) - 1) if pd.notna(l) else None, hand_mode)
                  for l in df.get("lane", pd.Series(dtype=float))]
    return df


def sec_participant_progress(root=None, cat=None):
    """Every session a participant has done, in order, so progress
    across the whole programme is visible rather than one block at a
    time.

    A session is one person on one day. Counting games as sessions turned
    two blocks in one sitting into a two-session training trend, which is
    the opposite of what this table is for.

    Reaction time is from cued modes only. Pooling rhythm beat offsets
    into it produced a negative coefficient of variation, which cannot
    exist, and a first-to-latest change that was really the difference
    between a rhythm block and a classic one.

    Force is the number most exposed to the calibration problem, because
    a trend over sessions can cross a recalibration. Each game is
    normalised against its own calibration and its own hand.
    """
    cat = build_catalogue(root) if cat is None else cat
    print("\n" + "=" * 62)
    print("PROGRESS PER PARTICIPANT")
    print("=" * 62)
    if cat.empty:
        _nothing("Nothing is recorded yet, so there is no progress to show.")
        return None
    people = [p for p in cat["who"].unique() if str(p) not in ("NA", "")]
    if not people:
        _nothing("Every recording on disk is under the placeholder name NA,",
                 "so there is no participant to follow across sessions.",
                 "Enter a participant name on the session screen.")
        return None

    rows = []
    # Per-session per-mode mean RTs, so the first-to-latest change can
    # be computed within the modes both ends share: mirror logs the
    # later of two presses and adaptive censors at its cadence window,
    # so a change in the mode MIX moves the pooled column on its own.
    rt_by_mode: dict = {}
    for who in people:
        mine = cat[cat["who"] == who]
        # One row per DAY, not per game. Two blocks in one sitting are
        # one session, and reading them as two points on a training
        # curve invents a trend.
        for n, (day, day_games) in enumerate(
                sorted(mine.groupby("day")), start=1):
            frames = [_progress_game(g["folder"], g["mode"])
                      for _, g in day_games.iterrows()]
            frames = [f for f in frames if not f.empty]
            if not frames:
                continue
            df = pd.concat(frames, ignore_index=True)
            kept, _, counts = analysable(df)
            hit_rows = kept[is_cued(kept) & is_scorable(kept)]
            s = rt_stats(kept)
            modes_rt = {}
            for m, sub_m in kept.groupby("mode"):
                v = reaction_times(sub_m)
                if len(v):
                    modes_rt[str(m)] = float(v.mean())
            rt_by_mode[(who, n)] = modes_rt
            beat = rhythm_rows(kept)["time_difference_ms"].astype(float)
            force = kept.get("peak_force_n", pd.Series(dtype=float))
            cal_force = []
            for game, sub in kept.groupby("game", sort=False):
                folder = next((Path(g["folder"])
                               for _, g in day_games.iterrows()
                               if game_key(g["folder"]) == game), None)
                if folder is None:
                    continue
                cs = calibration_factors({game: read_meta(folder)})
                hand_mode = sub["hand_mode"].iloc[0]
                for v, l in zip(sub.get("peak_force_n", []),
                                sub.get("lane", [])):
                    gap = (cs.lane_gap(game, int(l) - 1, hand_mode)
                           if pd.notna(l) else None)
                    cal_force.append(cs.counts(game, v) / gap
                                     if (pd.notna(v) and gap) else np.nan)
            cal_force = pd.Series(cal_force, dtype="float64")
            rows.append({
                "who": who, "session": n, "day": day,
                "games": len(day_games),
                "modes": ", ".join(dict.fromkeys(day_games["mode"])),
                "trials": counts["recorded"],
                "analysed": counts["analysed"],
                "hit_rate": (round(float((hit_rows["early_late"] != "Miss")
                                         .mean()), 3)
                             if len(hit_rows) else np.nan),
                "rt_trials": s["n"],
                "mean_rt": s["mean_rt"],
                "rt_cv": s["rt_cv"],
                "beat_sd_ms": (round(float(beat.std(ddof=1)), 1)
                               if len(beat) > 2 else np.nan),
                "mean_force_raw": (round(float(force.mean()), 1)
                                   if len(force) and force.notna().any()
                                   else np.nan),
                "mean_force_cal": (round(float(cal_force.mean()), 3)
                                   if cal_force.notna().any() else np.nan),
            })
    prog = pd.DataFrame(rows)
    if prog.empty:
        _nothing("No participant has a readable trials.csv, so there is no",
                 "progress table to build.")
        return None
    _show(prog)
    print("session  one person on one day. Two blocks in one sitting are")
    print("         one session, not two.")
    print("trials   recorded. analysed leaves out trials with no cue")
    print("         delivered and presses under "
          f"{ANTICIPATION_MS:.0f} ms.")
    print("mean_rt  cued modes only, misses out. Rhythm blocks contribute")
    print("         no reaction time: their timing column is a signed")
    print("         offset from the beat. rt_trials says how many trials")
    print("         are behind it, and it is blank when there are none.")
    print("         Two cued modes measure differently (mirror logs the")
    print("         later of a pair, adaptive censors at its cadence")
    print("         window), so the modes column matters: a change down")
    print("         this column can be the mode mix, and the change")
    print("         report below fits within shared modes only.")
    print("hit_rate cued modes only (classic, adaptive, mirror,")
    print("         reaction), scorable rows: catch outcomes and free")
    print("         retries are neither hits nor misses.")
    print("beat_sd_ms  rhythm blocks only: spread of the signed beat")
    print("         offsets, the trained beat-timing ability. Falling")
    print("         is improvement. Blank when a day has under 3 notes.")

    fig, ax = plt.subplots(1, 3, figsize=(14, 3.6))
    for who, g in prog.groupby("who"):
        g = g.sort_values("session")
        ax[0].plot(g["session"], g["mean_rt"], "o-", lw=2, label=who)
        ax[1].plot(g["session"], g["hit_rate"], "o-", lw=2, label=who)
        ax[2].plot(g["session"], g["rt_cv"], "o-", lw=2, label=who)
    ax[0].set_ylabel("mean reaction time (ms)"); ax[0].set_title("Speed")
    ax[1].axhspan(BAND_LO, BAND_HI, color="#16a34a", alpha=.15)
    ax[1].set_ylim(0, 1.02); ax[1].set_ylabel("hit rate")
    ax[1].set_title("Accuracy")
    ax[2].set_ylabel("reaction time CV")
    ax[2].set_title("Consistency (lower is steadier)")
    for a in ax:
        a.set_xlabel("session number")
        a.xaxis.set_major_locator(MaxNLocator(integer=True))
        if len(people) > 1:
            a.legend(frameon=False, fontsize=8)
    _save(fig, "participant_progress"); plt.show()

    print("\nchange from first to latest session:")
    for who, g in prog.groupby("who"):
        g = g.sort_values("session")
        if len(g) < 2:
            print(f"   {who}: only one session so far, so there is no "
                  f"change to report")
            continue
        parts = []
        # Reaction time first, computed WITHIN the modes the two end
        # sessions share: session 1 mixing reaction and mirror against
        # a session 3 of adaptive alone would otherwise report the
        # mode mix as improvement.
        first_modes = rt_by_mode.get((who, int(g["session"].iloc[0])), {})
        last_modes = rt_by_mode.get((who, int(g["session"].iloc[-1])), {})
        common = sorted(set(first_modes) & set(last_modes))
        rt_note = None
        if common:
            d_rt = (float(np.mean([last_modes[m] for m in common]))
                    - float(np.mean([first_modes[m] for m in common])))
            parts.append(f"reaction time {d_rt:+.0f} ms "
                         f"({', '.join(common)} only)")
        elif first_modes and last_modes:
            rt_note = ("first and latest sessions share no cued mode, "
                       "so no reaction-time change is reported")
        # Each other measure only when both ends of it exist. A blank
        # at either end used to come out as nan formatted into a
        # change, which reads as a measured result.
        def change(col, fmt, name):
            v = g[col].dropna()
            if len(v) < 2:
                return None
            return f"{name} {fmt.format(v.iloc[-1] - v.iloc[0])}"
        for col, fmt, name in (("hit_rate", "{:+.3f}", "hit rate"),
                               ("rt_cv", "{:+.3f}", "consistency"),
                               ("beat_sd_ms", "{:+.1f} ms", "beat sd"),
                               ("mean_force_cal", "{:+.3f}",
                                f"effort ({NORM_UNIT})")):
            got = change(col, fmt, name)
            if got:
                parts.append(got)
        if not parts:
            print(f"   {who}: {len(g)} sessions, but no measure has a value "
                  f"at both ends")
            if rt_note:
                print(f"      {rt_note}")
            continue
        print(f"   {who}: " + ", ".join(parts) + f" over {len(g)} sessions")
        if rt_note:
            print(f"      {rt_note}")
        blank = [c for c in ("hit_rate", "rt_cv", "beat_sd_ms",
                             "mean_force_cal")
                 if len(g[c].dropna()) < 2]
        if blank:
            print(f"      no change reported for: {', '.join(blank)} "
                  f"(missing at one end)")
    if prog["mean_force_cal"].notna().any():
        print(f"\nmean_force_cal is {NORM_LABEL}. Each game is against its")
        print("own calibration and its own hand, so a recalibration between")
        print("sessions does not move it.")
        print_norm_short()
        print("   A rise down this column is more effort against the same")
        print("   reference, not a stronger finger in newtons.")
    if prog["mean_force_raw"].notna().any():
        print("mean_force_raw is raw counts and cannot be read down the")
        print("column: it mixes the fingers that came up with the pads they")
        print("sat on, and a recalibration between sessions moves it on its")
        print("own.")
    return prog


# =========================================== Convergence across sessions
# Asymmetry and inter-hand correlation session by session, the pair
# the convergence study is defined on.

# Section 5.1, Thread C: "Run the convergence study in the trial window,
# tracking asymmetry index and inter-hand correlation across sessions
# (Weeks 9 to 10)".
#
# The progress table above is a real cross-session view, but the two
# quantities the convergence study is defined on are not among its
# columns. The Both hands section computes an asymmetry for the current
# selection only and hands back four numbers, none of which carry
# forward, and the inter-hand r did not exist anywhere until the section
# above. This puts both on one row per session so a trend can be read.


def bilateral_session_row(who, day, day_games):
    """One session's bilateral measures, or None when that day had no
    both-hands block."""
    frames, folders = [], []
    for _, g in day_games.iterrows():
        meta = read_meta(Path(g["folder"]))
        if str(meta.get("hand", "")).strip().lower() != "both":
            continue
        df = _progress_game(g["folder"], g["mode"])
        if df.empty:
            continue
        frames.append(df)
        folders.append(Path(g["folder"]))
    if not frames:
        return None
    df = pd.concat(frames, ignore_index=True)
    kept, _flagged, counts = analysable(df)
    # mirror rows are excluded from this left/right split: mirror's
    # side is always right by construction (see sec_bilateral's
    # bil_lr note), so leaving them in would misattribute mirror's
    # later-press RT as right-hand progress.
    kept_lr = kept[kept["mode"] != "mirror"] if "mode" in kept else kept
    L = kept_lr[kept_lr["side"] == "left"]
    R = kept_lr[kept_lr["side"] == "right"]
    lrt, rrt = reaction_times(L), reaction_times(R)
    lf = L.get("peak_force_n", pd.Series(dtype=float)).dropna()
    rf = R.get("peak_force_n", pd.Series(dtype=float)).dropna()

    def asym(l, r):
        if pd.isna(l) or pd.isna(r) or (l + r) == 0:
            return np.nan
        return round(float((l - r) / ((l + r) / 2)), 3)
    rs = []
    for f in folders:
        streams = hand_streams(load_raw(f))
        if streams is None:
            continue
        r = zero_lag_r(streams[1], streams[2])
        if r is not None:
            rs.append(r)
    return {
        "who": who, "day": day, "games": len(frames),
        "trials": counts["recorded"], "analysed": counts["analysed"],
        "rt_left": round(float(lrt.mean()), 1) if len(lrt) else np.nan,
        "rt_right": round(float(rrt.mean()), 1) if len(rrt) else np.nan,
        "rt_asymmetry": asym(lrt.mean() if len(lrt) else np.nan,
                             rrt.mean() if len(rrt) else np.nan),
        "force_asymmetry_raw": asym(lf.mean() if len(lf) else np.nan,
                                    rf.mean() if len(rf) else np.nan),
        "inter_hand_r": round(float(np.mean(rs)), 3) if rs else np.nan,
        "blocks_with_streams": len(rs),
    }


def sec_convergence(cat=None, root=None):
    """Asymmetry and inter-hand correlation, session by session."""
    cat = build_catalogue(root) if cat is None else cat
    print("\n" + "=" * 62)
    print("CONVERGENCE ACROSS SESSIONS")
    print("=" * 62)
    if cat.empty:
        _nothing("Nothing is recorded yet, so there is no trend to follow.")
        return None
    rows = []
    for who in [p for p in cat["who"].unique() if str(p) not in ("NA", "")]:
        mine = cat[cat["who"] == who]
        for n, (day, day_games) in enumerate(sorted(mine.groupby("day")),
                                             start=1):
            row = bilateral_session_row(who, day, day_games)
            if row:
                row["session"] = n
                rows.append(row)
    if not rows:
        _nothing("No participant has a both-hands block on disk, so there",
                 "is no asymmetry or inter-hand correlation to track. The",
                 "convergence study needs bilateral or mirror blocks",
                 "recorded under a participant name, across more than one",
                 "day.")
        return None
    prog = pd.DataFrame(rows)[
        ["who", "session", "day", "games", "trials", "analysed",
         "rt_left", "rt_right", "rt_asymmetry", "force_asymmetry_raw",
         "inter_hand_r", "blocks_with_streams"]]
    _show(prog)
    print("rt_asymmetry and force_asymmetry_raw are signed (left - right)")
    print("over the mean. Once patients are recorded, flip these to")
    print("affected against unaffected using the affected-side section,")
    print("otherwise two patients affected on opposite sides cancel.")
    print("inter_hand_r is the mean zero-lag Pearson r over that day's")
    print("blocks that logged a usable force stream. blocks_with_streams")
    print("says how many did: a zero there means the day was played on the")
    print("keyboard and the column is empty for a reason.")

    if prog["session"].nunique() < 2:
        print("\nOne session per person so far, so there is no convergence")
        print("to plot yet. This fills in over the Week 9 to 10 window.")
        return prog
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
    for who, g in prog.groupby("who"):
        g = g.sort_values("session")
        ax[0].plot(g["session"], g["rt_asymmetry"], "o-", lw=2, label=who)
        ax[1].plot(g["session"], g["inter_hand_r"], "o-", lw=2, label=who)
    ax[0].axhline(0, color="#94a3b8", lw=1)
    ax[0].set_ylabel("(left - right) / mean")
    ax[0].set_title("Reaction-time asymmetry")
    ax[1].axhline(0, color="#94a3b8", lw=1)
    ax[1].set_ylim(-1.05, 1.05); ax[1].set_ylabel("Pearson r")
    ax[1].set_title("Inter-hand correlation")
    for a in ax:
        a.set_xlabel("session number")
        a.xaxis.set_major_locator(MaxNLocator(integer=True))
        if prog["who"].nunique() > 1:
            a.legend(frameon=False, fontsize=8)
    _save(fig, "convergence"); plt.show()
    return prog


# ==================================== Hand size against the sizing range
# What each participant's hand measured, against the range the chassis
# claims to cover.

# Thread A objective 3: "Design and validate an adjustable chassis
# capable of accommodating hand sizes across the 5th to 95th percentile
# range". Section 4.1.2 turns that into a claim: "The available travel
# is sufficient to cover the 5th to 95th percentile hand-length range
# identified in the ANSUR II dataset".
#
# That is an assertion with nothing measured behind it. metadata.json
# already carries hand_length_mm and hand_breadth_mm per session, so
# once participants are measured the claim becomes evidence at almost no
# cost. Most of this objective is goniometer and ruler work on the
# bench; this section only covers the part the logs can carry.

# From Section 2.1 of the progress report: "Even within a single sex,
# the difference between the 5th and 95th percentile hand lengths is
# around 30 mm, and the overall range becomes larger when both male and
# female measurements are considered."
#
# The absolute 5th and 95th percentile hand lengths are deliberately NOT
# hard-coded here. The report cites the spread, not the endpoints, and
# inventing endpoints to draw two tidy lines against would be a made-up
# number sitting in a thesis figure. Put the real ANSUR II values in
# below once they have been read off the dataset, and the plot picks
# them up.
ANSUR_II_SPREAD_MM = 30.0


ANSUR_II_HAND_LENGTH_MM = {"p5": None, "p95": None}


def hand_size_rows(cat=None, root=None):
    """Recorded hand measurements, one row per participant."""
    cat = build_catalogue(root) if cat is None else cat
    if cat.empty:
        return pd.DataFrame()
    rows = {}
    for _, g in cat.iterrows():
        meta = read_meta(Path(g["folder"]))
        who = meta.get("participant") or g["who"]

        def num(key):
            try:
                v = float(str(meta.get(key, "")).strip())
            except (TypeError, ValueError):
                return np.nan
            return v if v > 0 else np.nan
        length, breadth = num("hand_length_mm"), num("hand_breadth_mm")
        cur = rows.setdefault(str(who), {"participant": str(who),
                                         "sessions": 0,
                                         "hand_length_mm": np.nan,
                                         "hand_breadth_mm": np.nan})
        cur["sessions"] += 1
        if pd.isna(cur["hand_length_mm"]):
            cur["hand_length_mm"] = length
        if pd.isna(cur["hand_breadth_mm"]):
            cur["hand_breadth_mm"] = breadth
    return pd.DataFrame(list(rows.values()))


def sec_hand_size(cat=None, root=None):
    """Recorded hand sizes against the range the chassis claims to
    cover."""
    print("\n" + "=" * 62)
    print("HAND SIZE AGAINST THE SIZING RANGE")
    print("=" * 62)
    tbl = hand_size_rows(cat, root)
    if tbl.empty:
        _nothing("Nothing is recorded yet, so there are no hands to plot.")
        return None
    measured = tbl[tbl["hand_length_mm"].notna()]
    _show(tbl)
    if measured.empty:
        _nothing("",
                 "No session on disk has hand_length_mm filled in, so the",
                 "5th to 95th percentile claim has no measurement behind it",
                 "yet. The field is on the session screen and is written to",
                 "every metadata.json, so measuring each participant once",
                 "with a ruler turns the assertion in Section 4.1.2 into",
                 "evidence.",
                 "",
                 "Record for each participant: hand length, hand breadth,",
                 "and the actuator position that fitted them. The third one",
                 "has no field yet and is the piece that actually proves",
                 "the travel covers the range.")
        return tbl
    spread = float(measured["hand_length_mm"].max()
                   - measured["hand_length_mm"].min())
    print(f"\nparticipants measured : {len(measured)}")
    print(f"hand length           : "
          f"{measured['hand_length_mm'].min():.0f} to "
          f"{measured['hand_length_mm'].max():.0f} mm, spread {spread:.0f} mm")
    print(f"the report cites about {ANSUR_II_SPREAD_MM:.0f} mm between the "
          f"5th and 95th")
    print("   percentile within one sex, and more across both.")
    if spread < ANSUR_II_SPREAD_MM:
        print("   This cohort spans less than that, so it does not on its")
        print("   own demonstrate the chassis covers the range. Say that,")
        print("   rather than reading a comfortable fit across three")
        print("   similar hands as coverage of the population.")

    fig, ax = plt.subplots(figsize=(8, 3.2))
    y = np.arange(len(measured))
    ax.barh(y, measured["hand_length_mm"], height=.55, color="#2563eb")
    ax.set_yticks(y); ax.set_yticklabels(measured["participant"])
    ax.set_xlabel("hand length (mm)")
    ax.set_title("Recorded hand length per participant")
    for key, style, label in (("p5", "--", "ANSUR II 5th percentile"),
                              ("p95", ":", "ANSUR II 95th percentile")):
        v = ANSUR_II_HAND_LENGTH_MM.get(key)
        if v:
            ax.axvline(float(v), color="#dc2626", ls=style, lw=2,
                       label=label)
    if any(ANSUR_II_HAND_LENGTH_MM.values()):
        ax.legend(frameon=False, fontsize=8)
    else:
        print("\nThe percentile lines are not drawn. The report cites the")
        print("30 mm spread and not the endpoints, so there is no sourced")
        print("number to draw them at. Put the ANSUR II values into")
        print("ANSUR_II_HAND_LENGTH_MM in this cell and they appear.")
    _save(fig, "hand_size"); plt.show()
    return tbl


# =================================================== Intervals and tests
# A bootstrap interval and a permutation test on every difference the
# report commits to.

# Every number this notebook produces above is descriptive: a mean, a
# median, a CV, a count, a proportion, or a least-squares slope with
# nothing on it. Section 5.4 of the progress report lists scipy in the
# software stack and Section 5.1 commits to comparisons that are claims
# about difference, not description: the fixed-cadence versus RAS study
# and the convergence study. The pretest and aftertest design comes from
# Nakayama and Lee, and Demouche's Section 6.4.4 claims "a clear and
# statistically reliable decrease in reaction time across trials".
#
# A bare "-40 ms" in a thesis will be asked whether it is distinguishable
# from noise, so every contrast here comes with an interval and a test.
#
# Two deliberate choices.
#
# Resampling rather than t-tests. Reaction times are skewed and the
# group sizes are small, so a bootstrap interval and a permutation test
# assume less than a t-test does and need no distributional story. They
# are also written out here in numpy, so this stays a notebook that runs
# on pandas, numpy and matplotlib alone.
#
# Trial level, not patient level. With three to five participants there
# is no population inference to be had. These intervals describe the
# trials in the selection, and trials from one person are not
# independent of each other, so a narrow interval here is NOT evidence
# about patients in general. That caveat is printed with the table
# rather than left in a comment.

BOOT_N = 4000


BOOT_SEED = 20260805      # fixed so the same selection gives the same
                          # interval every run, which a thesis needs


def _rng(seed=BOOT_SEED):
    return np.random.default_rng(seed)


def boot_ci(values, stat=np.mean, n=BOOT_N, alpha=0.05, seed=BOOT_SEED):
    """Percentile bootstrap interval for any statistic of one sample."""
    v = np.asarray(pd.Series(values, dtype="float64").dropna(), dtype=float)
    if len(v) < 3:
        return (np.nan, np.nan)
    rng = _rng(seed)
    idx = rng.integers(0, len(v), size=(n, len(v)))
    draws = stat(v[idx], axis=1)
    return (float(np.quantile(draws, alpha / 2)),
            float(np.quantile(draws, 1 - alpha / 2)))


def diff_ci(a, b, n=BOOT_N, alpha=0.05, seed=BOOT_SEED):
    """Bootstrap interval for mean(a) - mean(b), resampling each group
    on its own."""
    x = np.asarray(pd.Series(a, dtype="float64").dropna(), dtype=float)
    y = np.asarray(pd.Series(b, dtype="float64").dropna(), dtype=float)
    if len(x) < 3 or len(y) < 3:
        return (np.nan, np.nan)
    rng = _rng(seed)
    dx = x[rng.integers(0, len(x), size=(n, len(x)))].mean(axis=1)
    dy = y[rng.integers(0, len(y), size=(n, len(y)))].mean(axis=1)
    d = dx - dy
    return (float(np.quantile(d, alpha / 2)),
            float(np.quantile(d, 1 - alpha / 2)))


def perm_p(a, b, n=BOOT_N, seed=BOOT_SEED):
    """Two-sided permutation p for a difference in means.

    Labels are shuffled, which is the null that the two groups came from
    one pool. Reported with a floor of 1/(n+1) because a permutation
    test cannot resolve past its own resampling.
    """
    x = np.asarray(pd.Series(a, dtype="float64").dropna(), dtype=float)
    y = np.asarray(pd.Series(b, dtype="float64").dropna(), dtype=float)
    if len(x) < 3 or len(y) < 3:
        return np.nan
    obs = abs(x.mean() - y.mean())
    pool = np.concatenate([x, y])
    rng = _rng(seed)
    hits = 0
    for _ in range(n):
        rng.shuffle(pool)
        if abs(pool[:len(x)].mean() - pool[len(x):].mean()) >= obs:
            hits += 1
    return float((hits + 1) / (n + 1))


def hedges_g(a, b):
    """Standardised mean difference with the small-sample correction.

    Reported because a p value at these sample sizes says more about how
    many trials were recorded than about how big the difference is.
    """
    x = np.asarray(pd.Series(a, dtype="float64").dropna(), dtype=float)
    y = np.asarray(pd.Series(b, dtype="float64").dropna(), dtype=float)
    if len(x) < 2 or len(y) < 2:
        return np.nan
    nx, ny = len(x), len(y)
    sp = np.sqrt(((nx - 1) * x.var(ddof=1) + (ny - 1) * y.var(ddof=1))
                 / (nx + ny - 2))
    if sp == 0:
        return np.nan
    d = (x.mean() - y.mean()) / sp
    j = 1 - 3 / (4 * (nx + ny) - 9)
    return float(d * j)


def slope_ci(x, y, n=BOOT_N, alpha=0.05, seed=BOOT_SEED):
    """(slope, lo, hi) for a straight line through (x, y), the interval
    from resampling the pairs."""
    df = pd.DataFrame({"x": pd.to_numeric(pd.Series(x), errors="coerce"),
                       "y": pd.to_numeric(pd.Series(y), errors="coerce")}
                      ).dropna()
    if len(df) < 5:
        return (np.nan, np.nan, np.nan)
    xs = df["x"].values.astype(float)
    ys = df["y"].values.astype(float)
    slope = float(np.polyfit(xs, ys, 1)[0])
    rng = _rng(seed)
    draws = []
    for _ in range(min(n, 1500)):
        i = rng.integers(0, len(xs), size=len(xs))
        if np.std(xs[i]) == 0:
            continue
        draws.append(np.polyfit(xs[i], ys[i], 1)[0])
    if len(draws) < 20:
        return (slope, np.nan, np.nan)
    return (slope, float(np.quantile(draws, alpha / 2)),
            float(np.quantile(draws, 1 - alpha / 2)))


def contrast_row(name, unit, a, b, label_a, label_b):
    """One two-group comparison, with an interval, a test and an effect
    size."""
    x = pd.Series(a, dtype="float64").dropna()
    y = pd.Series(b, dtype="float64").dropna()
    lo, hi = diff_ci(x, y)
    return {
        "contrast": name, "unit": unit,
        "group_a": label_a, "n_a": int(len(x)),
        "group_b": label_b, "n_b": int(len(y)),
        "mean_a": round(float(x.mean()), 3) if len(x) else np.nan,
        "mean_b": round(float(y.mean()), 3) if len(y) else np.nan,
        "difference": (round(float(x.mean() - y.mean()), 3)
                       if len(x) and len(y) else np.nan),
        "ci_lo": round(lo, 3) if pd.notna(lo) else np.nan,
        "ci_hi": round(hi, 3) if pd.notna(hi) else np.nan,
        "perm_p": (round(perm_p(x, y), 4)
                   if len(x) >= 3 and len(y) >= 3 else np.nan),
        "hedges_g": (round(hedges_g(x, y), 3)
                     if len(x) >= 2 and len(y) >= 2 else np.nan),
    }


def sec_statistics(trials):
    """Intervals and tests for the contrasts the report commits to."""
    print("\n" + "=" * 62)
    print("INTERVALS AND TESTS")
    print("=" * 62)
    if trials is None or trials.empty:
        _nothing("No trials are loaded, so there is nothing to test.")
        return None
    kept, _flagged, _counts = analysable(trials)
    print("Built from the analysable trials only, the same basis as the")
    print("headline table: trials whose cue never reached the device and")
    print("presses faster than "
          f"{ANTICIPATION_MS:.0f} ms are out. The section tables above")
    print("print over every recorded trial, so the counts here are smaller.")
    rows, skipped = [], []

    # Pretest against aftertest. Nakayama and Lee's headline claim rests
    # on this comparison, and the phase section above prints the change
    # with nothing on it.
    if "phase" in kept.columns:
        ph = kept[kept["phase"].notna() & (kept["phase"] != "")]
        pre = reaction_times(ph[ph["phase"] == "pretest"])
        post = reaction_times(ph[ph["phase"] == "aftertest"])
        if len(pre) >= 3 and len(post) >= 3:
            rows.append(contrast_row("aftertest minus pretest", "ms",
                                     post, pre, "aftertest", "pretest"))
        else:
            skipped.append("pretest against aftertest: needs at least three "
                           "reaction times in each phase")
    else:
        skipped.append("pretest against aftertest: no phase column")

    # Fixed cadence against RAS. The comparison Sections 2.3, 3.3 and
    # 5.1 all commit to. Hit is 1 or 0 per trial, so a difference in
    # means is a difference in hit rate.
    cls = kept[kept["mode"] == FIXED_CADENCE_MODE]
    ras = kept[kept["mode"] == RAS_MODE]
    if len(cls) >= 3 and len(ras) >= 3:
        rows.append(contrast_row(
            "RAS minus fixed cadence, hit rate", "proportion",
            (ras["early_late"] != "Miss").astype(float),
            (cls["early_late"] != "Miss").astype(float),
            "rhythm", "classic"))
    else:
        skipped.append("fixed cadence against RAS: needs classic and rhythm "
                       "blocks in one selection")

    # Cue modality. Palmer's result is the reason the vibration-only
    # condition exists, and the table above prints the gap with nothing
    # on it.
    cm = cue_condition(kept)
    if "cue_mode" in cm.columns:
        vis = reaction_times(cm[cm["cue_mode"] == "visual"])
        vib = reaction_times(cm[cm["cue_mode"] == "vibration"])
        if len(vis) >= 3 and len(vib) >= 3:
            rows.append(contrast_row("vibration minus visual", "ms",
                                     vib, vis, "vibration", "visual"))
        else:
            skipped.append("vibration against visual: needs at least three "
                           "reaction times under each cue setting")
    else:
        skipped.append("vibration against visual: no cue setting recorded")

    # Left against right, on bilateral blocks only. mirror excluded:
    # its side is always right by construction (see sec_bilateral),
    # so pooling it in here would misattribute its later-press RT as
    # a right-hand-only reading.
    bil = kept[kept["hand_mode"] == "both"] if "hand_mode" in kept else \
        kept.iloc[0:0]
    bil = bil[bil["mode"] != "mirror"] if "mode" in bil else bil
    if not bil.empty:
        lrt = reaction_times(bil[bil["side"] == "left"])
        rrt = reaction_times(bil[bil["side"] == "right"])
        if len(lrt) >= 3 and len(rrt) >= 3:
            rows.append(contrast_row("left minus right", "ms", lrt, rrt,
                                     "left", "right"))
        else:
            skipped.append("left against right: needs at least three cued "
                           "reaction times on each hand")
    else:
        skipped.append("left against right: no both-hands block")

    if rows:
        tbl = pd.DataFrame(rows)
        _show(tbl)
        print("difference is mean(group_a) minus mean(group_b).")
        print("ci_lo and ci_hi are a 95 percent percentile bootstrap over")
        print(f"{BOOT_N} resamples. An interval that spans zero means the")
        print("data does not separate the two groups.")
        print(f"perm_p is a two-sided permutation test on the same")
        print(f"difference, floored at 1 in {BOOT_N + 1} because the test")
        print("cannot resolve past its own resampling.")
        print("hedges_g is the standardised difference with the small-")
        print("sample correction, because a p value at these sizes says as")
        print("much about how many trials were recorded as about the gap.")
    else:
        print("No contrast in this selection has enough data to test.")

    for s in skipped:
        print(f"   not run: {s}")

    # The learning slope the reaction-time section prints bare.
    rt_rows = reaction_times(kept, per="rows")
    slope, lo, hi = slope_ci(rt_rows.get("trial", pd.Series(dtype=float)),
                             rt_rows.get("time_difference_ms",
                                         pd.Series(dtype=float)))
    if pd.notna(slope):
        print(f"\nreaction time against trial index: {slope:+.2f} ms per "
              f"trial")
        if pd.notna(lo):
            print(f"   95 percent interval {lo:+.2f} to {hi:+.2f} ms per "
                  f"trial")
            if lo <= 0 <= hi:
                print("   The interval spans zero, so this selection does")
                print("   not show a within-block trend.")
            else:
                print("   The interval sits on one side of zero, so the")
                print("   direction holds within this selection.")
        print("   This is trials pooled inside the selection, not a")
        print("   session-to-session learning curve. The progress and")
        print("   fatigue sections cover across-session and across-block.")

    print("\nWHAT THESE NUMBERS ARE, AND ARE NOT")
    people = int(kept["participant"].nunique()) if not kept.empty else 0
    print(f"   Trials, not patients. This selection holds {people} "
          f"participant(s).")
    print("   Trials from one person are not independent of each other, so")
    print("   an interval here describes the trials in front of it and NOT")
    print("   patients in general. With three to five participants there is")
    print("   no population inference available, and a narrow interval is")
    print("   not a substitute for one.")
    print("   For the Week 9 to 10 write-up, report per participant first,")
    print("   then the group as a set of individual results, rather than")
    print("   pooling every trial into one test.")
    return pd.DataFrame(rows) if rows else None


# ====================================================== Headline numbers
# Built from the trials that survived exclusion. Refuses to run unless
# every section above ran on the save picked now.

def build_summary(trials, unit="sensor counts", calset=None, on_task=0.0,
                  folders=None, onset=None, accuracy=None, bilateral=None,
                  dose=None, cues=None, phase=None,
                  objective_one=None) -> dict:
    """The headline numbers, built in one place so the printed summary
    and the exported CSV cannot drift apart.

    Built from the trials that CAN be analysed. A trial whose cue command
    never reached the device was never presented, and a press under
    100 ms is anticipation rather than a response. The old summary
    counted both, so on the shipped default it published a hit rate and a
    mean reaction time computed from 48 trials the exclusions section
    said could not be analysed. Every count that went into the figures is
    in the table, so the basis is visible rather than assumed.
    """
    kept, flagged, counts = analysable(trials)
    calset = calset if calset is not None else CalibrationSet()
    s = {
        "games": len(folders) if folders is not None else np.nan,
        "summary_basis": "analysable trials only",
        "trials_recorded": counts["recorded"],
        "trials_no_cue": counts["no_cue"],
        "trials_anticipation": counts["anticipation"],
        "trials_analysed": counts["analysed"],
        "time_on_task_min": round(on_task / 60, 1) if on_task else 0.0,
        "calibration": calset.status,
    }
    if counts["analysed"] == 0:
        s["warning"] = ("no analysable trials, every figure below is blank "
                        "on purpose")
        return s

    rts = rt_stats(kept)
    s["rt_trials"] = rts["n"]
    s["rt_mean_ms"] = rts["mean_rt"]
    s["rt_cv"] = rts["rt_cv"]
    s["rt_basis"] = "cued modes, misses and rhythm beat offsets excluded"

    cued = kept[is_cued(kept) & is_scorable(kept)]
    if not cued.empty:
        s["hit_rate_all_cued"] = round(
            float((cued["early_late"] != "Miss").mean()), 3)
        # The scope names what the code does: CUED_MODES includes
        # reaction, and event rows (catch outcomes, free retries) are
        # dropped rather than counted as hits.
        s["hit_rate_scope"] = ("classic, adaptive, mirror and reaction "
                               "blocks, scorable rows only")
    if accuracy:
        # The accuracy section narrows to adaptive blocks when it can.
        # Both scopes go in the table, named, rather than one number that
        # disagrees with the printed one.
        s["hit_rate_section_scope"] = accuracy.get("scope")
        s["hit_rate_section"] = accuracy.get("hit_rate_scoped")
        s["hit_rate_section_basis"] = "all recorded trials, as printed above"

    force = ensure_force_columns(kept, calset)
    force = force[force["peak_force_n"].notna()]
    if not force.empty:
        s["force_unit"] = unit
        s["peak_force_mean_raw"] = round(
            float(force["peak_force_n"].mean()), 1)
        s["peak_force_mean_N"] = round(float(force["peak_force_N"].mean()), 2)
        s["peak_force_N_meaning"] = ("absolute, pad sensitivity NOT "
                                     "corrected, use for strength")
        if force["peak_force_cal"].notna().any():
            s["peak_force_mean_cal"] = round(
                float(force["peak_force_cal"].mean()), 3)
            s["peak_force_cal_meaning"] = NORM_SHORT

    ind = individuation(kept, calset)
    isum = individuation_summary(ind)
    if isum["n_all"]:
        s["individuation_absolute"] = isum["raw_all"]
        s["individuation_absolute_n"] = isum["n_all"]
        s["individuation_absolute_meaning"] = (
            "target over total on absolute force, the basis the published "
            "enslavement figures use")
        if isum["n_matched"]:
            s["individuation_own_reference"] = isum["cal_matched"]
            s["individuation_absolute_matched"] = isum["raw_matched"]
            s["individuation_matched_n"] = isum["n_matched"]
            s["individuation_own_reference_meaning"] = (
                "each lane over its own reference press, NOT comparable "
                "with the published enslavement figures")
            s["individuation_matched_note"] = (
                "the two matched figures cover the same trials, so the "
                "difference is the correction alone")

    rhy = rhythm_rows(kept)
    if not rhy.empty:
        s["beat_accuracy_ms"] = round(
            float(rhy["time_difference_ms"].abs().mean()), 1)
        s["beat_bias_ms"] = round(float(rhy["time_difference_ms"].mean()), 1)
        s["beat_note"] = "signed offset from the beat, not a reaction time"

    if onset is not None and not onset.empty:
        v = onset["onset_rt_ms"]
        s["onset_rt_mean_ms"] = round(float(v.mean()), 1)
        s["onset_rt_cv"] = (round(float(v.std() / v.mean()), 3)
                            if len(v) > 1 and v.mean() > 0 else np.nan)
        s["onset_rfd_mean_raw"] = round(float(onset["peak_dforce"].mean()), 1)
        if onset["peak_dforce_cal"].notna().any():
            s["onset_rfd_mean_cal"] = round(
                float(onset["peak_dforce_cal"].mean()), 3)
        s["onset_basis"] = ("raw sample stream, trial exclusions do not "
                            "apply to it")

    # Everything below is printed by a section above and used to stop
    # here, which left session_summary.csv narrower than the notebook it
    # came from. The exported table is the one that reaches the thesis,
    # so anything the report is judged on belongs in it.
    if bilateral:
        s["bilateral_trials_left"] = bilateral.get("n_left")
        s["bilateral_trials_right"] = bilateral.get("n_right")
        s["bilateral_rt_left_ms"] = bilateral.get("rt_left")
        s["bilateral_rt_right_ms"] = bilateral.get("rt_right")
        lv, rv = bilateral.get("rt_left"), bilateral.get("rt_right")
        if pd.notna(lv) and pd.notna(rv) and (lv + rv):
            s["bilateral_rt_asymmetry"] = round(
                float((lv - rv) / ((lv + rv) / 2)), 3)
            s["bilateral_asymmetry_sign"] = "(left - right) over the mean"

    if dose:
        s["dose_reps"] = dose.get("reps")
        s["dose_vs_lang_ratio"] = dose.get("ratio")
        if dose.get("projected_30_min") is not None:
            s["dose_projected_30_min"] = dose.get("projected_30_min")

    if cues is not None and not getattr(cues, "empty", True):
        for _, cue_row in cues.iterrows():
            tag = str(cue_row["cue"]).replace(" ", "_")
            s[f"cue_{tag}_trials"] = int(cue_row["trials"])
            s[f"cue_{tag}_hit_rate"] = cue_row["hit_rate"]
            s[f"cue_{tag}_mean_rt_ms"] = cue_row["mean_rt"]

    if phase is not None and not getattr(phase, "empty", True):
        if {"pretest", "aftertest"} <= set(phase["phase"]):
            pre = phase[phase["phase"] == "pretest"]["mean_rt"].mean()
            post = phase[phase["phase"] == "aftertest"]["mean_rt"].mean()
            if pd.notna(pre) and pd.notna(post):
                s["pretest_rt_ms"] = round(float(pre), 1)
                s["aftertest_rt_ms"] = round(float(post), 1)
                s["pretest_to_aftertest_ms"] = round(float(post - pre), 1)

    if objective_one is not None and not getattr(objective_one, "empty", True):
        met = objective_one["hit_rate"].between(BAND_LO, BAND_HI)
        s["objective_one_band"] = f"{BAND_LO:.0%} to {BAND_HI:.0%}"
        s["objective_one_fingers_in_band"] = f"{int(met.sum())} of {len(met)}"
        s["objective_one_met"] = bool(met.all())
        if "full_windows" in objective_one.columns:
            s["objective_one_full_windows"] = int(
                pd.to_numeric(objective_one["full_windows"],
                              errors="coerce").fillna(0).sum())
    return s


def sec_summary(trials, unit="sensor counts", calset=None, on_task=0.0,
                folders=None, onset=None, accuracy=None, bilateral=None,
                dose=None, cues=None, phase=None,
                objective_one=None) -> dict:
    """Print the headline numbers and say plainly what they were built
    from."""
    s = build_summary(trials, unit, calset, on_task, folders, onset,
                      accuracy, bilateral, dose, cues, phase, objective_one)
    print("\n" + "=" * 62)
    print("SUMMARY")
    print("=" * 62)
    excluded = s["trials_no_cue"] + s["trials_anticipation"]
    if excluded:
        print(f"BUILT FROM {s['trials_analysed']} OF "
              f"{s['trials_recorded']} RECORDED TRIALS.")
        print(f"   {s['trials_no_cue']} had no cue delivered, so nothing was "
              f"presented.")
        print(f"   {s['trials_anticipation']} were faster than "
              f"{ANTICIPATION_MS:.0f} ms, so they are anticipation.")
        print("   Those are excluded here. Sections above print over every")
        print("   recorded trial unless they say otherwise, so a figure")
        print("   there can differ from the one below.")
        if s["trials_analysed"] == 0:
            print("   NOTHING IS ANALYSABLE IN THIS SELECTION. There is no")
            print("   hit rate and no reaction time to report from it.")
        print()
    else:
        print(f"Built from all {s['trials_recorded']} recorded trials: none "
              f"were flagged.\n")
    _show(pd.DataFrame([s]).T.rename(columns={0: "value"}))
    return s


# ======================================================= Save the tables
# Writes the CSVs next to this notebook, with `excluded` on every
# trial row so filtering on it reproduces the headline numbers.

IND_EXPORT = "individuation_per_trial.csv"


def write_exports(summary, trials, calset=None, ind=None):
    """Write the CSVs, with the exclusion flags on every trial row.

    selected_trials.csv keeps every recorded trial so nothing disappears,
    but each row now carries whether it could be analysed and why not, so
    anyone recomputing from the CSV lands on the same figures as the
    summary rather than on the ones that include flagged trials.

    Only the files actually written this run are described. A selection
    with no individuation leaves any earlier individuation_per_trial.csv
    untouched, and that stale file gets called out rather than listed as
    though it went with the summary.
    """
    OUTDIR.mkdir(parents=True, exist_ok=True)
    pd.DataFrame([summary]).T.rename(columns={0: "value"}).to_csv(
        OUTDIR / "session_summary.csv")
    flags = exclusion_flags(trials)
    flags.to_csv(OUTDIR / "selected_trials.csv", index=False)
    written = ["session_summary.csv", "selected_trials.csv"]
    if ind is None:
        ind = individuation(trials, calset)
    if ind is not None and not ind.empty:
        # Carry the same flag onto the per-trial individuation rows, so
        # the file the summary was built from can be reconstructed from
        # either CSV without guessing which trials went in.
        out = ind.copy()
        if not flags.empty and "row_id" in out.columns:
            excluded = flags["excluded"]
            out["excluded"] = [bool(excluded.get(i, False))
                               for i in out["row_id"]]
        out.to_csv(OUTDIR / IND_EXPORT, index=False)
        written.append(IND_EXPORT)
    print("\nwritten: " + ", ".join(written))
    if IND_EXPORT in written:
        print(f"selected_trials.csv and {IND_EXPORT} both carry an")
        print("excluded column, and selected_trials.csv also carries")
        print("exclusion_reason.")
    else:
        print("selected_trials.csv carries an excluded column and an")
        print("exclusion_reason column.")
    print("The summary is built from the rows where excluded is False, so")
    print("filter on it to reproduce the headline numbers.")
    # A selection with no individuation writes no individuation file,
    # but one from an earlier run can still be sitting in the SAME
    # OUTDIR under the same name. Saying nothing points the reader at
    # a file of another selection's trials as though it matched the
    # summary above it. The check looks in OUTDIR, where these exports
    # actually go: a cwd-relative check could never fire, because
    # _point_outputs_at moves OUTDIR into the analysed session folder.
    if IND_EXPORT not in written and (OUTDIR / IND_EXPORT).exists():
        print(f"\nWARNING: {IND_EXPORT} on disk is left over from an")
        print("earlier run. This selection has no individuation data, so")
        print("that file was NOT rewritten and its rows belong to a")
        print("different selection. Delete it, or re-run the export on the")
        print("selection it came from, before reading anything off it.")
    print("figures are in figures/ ready for the report")
    return written


# ================================================== the 2026 mode parsers
# The four newer modes cannot widen the CSV, so they pack their per-trial
# detail into the stimulus column. Each packing is documented in the mode
# file that writes it (reaction.py, pattern.py, chords.py, syllables.py);
# the parsers here read those formats and nothing else.

def stimulus_parts(cell):
    """A packed stimulus split into its leading token and its key=value
    pairs. Bare tokens like 'catch' come back as flags set True.
    Reaction writes 'choice;fp=2.314', pattern 'seq;b=3;soc=trained;pos=7',
    syllables 'word;lvl=3;...;taps=...;asyn=...'."""
    if cell is None or (isinstance(cell, float) and np.isnan(cell)):
        return "", {}
    text = str(cell).strip()
    if not text:
        return "", {}
    parts = text.split(";")
    kv = {}
    for p in parts[1:]:
        if "=" in p:
            k, v = p.split("=", 1)
            kv[k] = v
        elif p:
            kv[p] = True
    return parts[0], kv


def mode_rows(trials, mode):
    """Copy of the trials for one mode, empty when there are none."""
    if trials is None or trials.empty or "mode" not in trials.columns:
        return pd.DataFrame()
    return trials[trials["mode"] == mode].copy()


def stored_mode_stats(metas, key):
    """block_summary.<key> for every game that carries one. The modes
    fold their own aggregates into metadata.json at block end, so these
    are the numbers the app computed on the day."""
    out = {}
    for name, meta in _meta_items(metas):
        bs = meta.get("block_summary", {}) or {}
        sub = bs.get(key)
        if isinstance(sub, dict):
            out[name] = sub
    return out


def row_targets(row):
    """Every 0-based lane a trial asked for. correct_keys carries the
    full set ('1,3,4' for a chord); rows without it fall back to the
    single lane the row is keyed on."""
    out = []
    for t in str(row.get("correct_keys") or "").split(","):
        t = t.strip()
        if t:
            try:
                out.append(int(float(t)) - 1)
            except ValueError:
                return []
    if out:
        return sorted(set(out))
    lane0 = _lane0(row)
    return [lane0] if lane0 is not None else []


# ========================================================= Reaction mode
# The 2026 baseline block: press as fast as possible after a randomised
# wait. Scorable trials look like classic rows; events that never became
# one (false starts, catch outcomes) share the CSV with error_type set,
# and the scheduled foreperiod rides the stimulus column so the
# anticipation diagnostic can run from trials.csv alone.

# The 2026 baseline block: press as fast as possible after a randomised
# wait. Scorable trials look like classic rows; events that never became
# one (false starts, catch outcomes, and the SIMPLE sub-mode's free
# wrong-finger retry) share the CSV with error_type set, and the
# scheduled foreperiod rides the stimulus column so the anticipation
# diagnostic can run from trials.csv alone.
#
# CHOICE sub-mode's wrong-finger press is different: reaction.py closes
# it as a scorable Miss (early_late=="Miss"), the same convention
# Classic uses for a wrong press, because the attempt slot IS consumed
# and accuracy needs it in the denominator. error_type=="wrong_finger"
# therefore means two different things depending on early_late, and
# only the "Wrong" one (the simple sub-mode's free retry, no slot
# consumed) is a never-scorable event.

REACTION_NEVER_SCORABLE = ("false_start", "anticipation",
                           "catch_false_start")


# The PVT lapse convention (Basner and Dinges 2011). reaction.py flags
# the same boundary, so the two counts should agree.
LAPSE_MS = 500.0


def reaction_frame(trials):
    """Reaction rows with the stimulus unpacked: sub-mode, scheduled
    foreperiod, catch flag, and whether the row is a scorable trial or
    one of the mode's logged events.

    is_event follows reaction.py's own split: false_start,
    anticipation and catch_false_start never touch a scorable slot, a
    survived catch (label CatchOk) never touches one either, and the
    simple sub-mode's wrong-finger retry (error_type=="wrong_finger",
    early_late=="Wrong") is a free retry, not a trial. Choice mode's
    wrong-finger row (same error_type, early_late=="Miss") IS a
    scorable trial and belongs in the scored denominator, not here.
    """
    rows = mode_rows(trials, "reaction")
    if rows.empty or "stimulus" not in rows.columns:
        return pd.DataFrame()
    heads, fps, catches = [], [], []
    for cell in rows["stimulus"]:
        head, kv = stimulus_parts(cell)
        heads.append(head or "unknown")
        catches.append(bool(kv.get("catch")))
        try:
            fps.append(float(kv.get("fp")))
        except (TypeError, ValueError):
            fps.append(np.nan)
    rows["sub_mode"] = heads
    rows["fp_s"] = fps
    rows["is_catch"] = catches
    err = (rows["error_type"].fillna("").astype(str)
           if "error_type" in rows.columns
           else pd.Series("", index=rows.index))
    label = (rows["early_late"].fillna("").astype(str)
             if "early_late" in rows.columns
             else pd.Series("", index=rows.index))
    simple_retry = (err == "wrong_finger") & (label == "Wrong")
    rows["is_event"] = (err.isin(REACTION_NEVER_SCORABLE)
                        | simple_retry
                        | (label == "CatchOk"))
    return rows


def reaction_floor_note():
    """What the hardware can and cannot resolve, stated before any
    figure so no number below gets read past it."""
    print("MEASUREMENT FLOOR, read before the figures")
    print("   The cue lands on a 60 Hz display, so stimulus onset is")
    print("   quantised to 16.7 ms frames plus an unmeasured panel")
    print("   delay, and the 200 Hz sensors add 0 to 5 ms before a")
    print("   press is seen. Single-trial differences under about")
    print("   20 ms are noise. Block medians over 25 trials resolve")
    print("   about 20 ms and a session pooling 50 or more valid")
    print("   trials about 10 to 15 ms, because quantisation error")
    print("   averages out (Ulrich and Giray 1989).")
    print("   Absolute values are NOT comparable with published norms:")
    print("   the clock stops at a force-threshold crossing, and labs")
    print("   differ by over 100 ms on hardware alone (Woods 2015).")
    print("   Within-device change is the valid comparison.")


def _reaction_accuracy_warning(accuracy, n_correct, n_wrong):
    """Flag choice-RT accuracy against the brief's two lines: under
    80 percent reads as guessing (a favourite-finger mash beats the
    clock on a quarter of trials and posts a fast, meaningless RT),
    under 90 percent is still worth a caution. reaction.py's own
    docstring names accuracy 'a headline metric for choice RT'."""
    if accuracy is None or pd.isna(accuracy):
        return
    print(f"\nchoice accuracy {accuracy:.0%} ({n_correct} correct, "
          f"{n_wrong} wrong-finger) over the rows behind this "
          "distribution.")
    if accuracy < 0.80:
        print("WARNING: under 80% reads as guessing, not responding")
        print("(the brief's threshold). The RT distribution above is")
        print("not interpretable for this group.")
    elif accuracy < 0.90:
        print("Under the 90% 'good' line; read the RT with some")
        print("caution.")


def _reaction_exgaussian_fit(v):
    """Method-of-moments ex-Gaussian fit (Hohle 1965): closed-form,
    no optimiser needed, matching the rest of this notebook's
    no-scipy fits. mu is the Gaussian centre, sigma its spread, tau
    the exponential tail that carries the right skew RT distributions
    show. Gated by the caller at n >= 40 (the brief's suppress-below
    threshold) because skewness is a third moment and noisy below
    that. Returns None when the sample skew is non-positive, since a
    non-positive skew has no real tau under this estimator."""
    n = len(v)
    mean = float(np.mean(v))
    sd = float(np.std(v, ddof=1))
    if sd <= 0:
        return None
    skew = float(np.mean(((v - mean) / sd) ** 3))
    if skew <= 0:
        return None
    tau = sd * (skew / 2.0) ** (1.0 / 3.0)
    var_g = sd ** 2 - tau ** 2
    if var_g <= 0:
        return None
    sigma = var_g ** 0.5
    mu = mean - tau
    return {"n": n, "mu": round(mu, 1), "sigma": round(sigma, 1),
            "tau": round(tau, 1)}


def _reaction_time_on_task(valid, stored_slope, group_label=""):
    """RT against trial number within each block (spec subsection 4),
    the figure the mode's own stored slope_rt_ms_per_trial has never
    had anywhere to show up until now. One panel per game so a
    multi-session selection does not fold several blocks' trial
    counters onto the one axis; a rolling median rides over the raw
    points and lapses (>= LAPSE_MS) are marked so a fatigue trend
    reads at a glance."""
    games = [g for g in valid["game"].unique()]
    if not games:
        return
    usable = [g for g in games if (valid["game"] == g).sum() >= 5]
    if not usable:
        return
    ncols = min(3, len(usable))
    nrows = -(-len(usable) // ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.2 * ncols, 3.2 * nrows),
                             squeeze=False)
    for i, g in enumerate(usable):
        ax = axes[i // ncols][i % ncols]
        gv = valid[valid["game"] == g].copy()
        gv["trial_n"] = pd.to_numeric(gv.get("trial"), errors="coerce")
        gv = gv.dropna(subset=["trial_n"]).sort_values("trial_n")
        if gv.empty:
            ax.axis("off")
            continue
        lapse = gv["rt"] >= LAPSE_MS
        ax.plot(gv["trial_n"], gv["rt"], "o", ms=4, alpha=.4,
                color="#0ea5e9")
        ax.plot(gv.loc[lapse, "trial_n"], gv.loc[lapse, "rt"], "x",
                ms=8, color="#b45309", label="lapse")
        roll = gv["rt"].rolling(7, center=True, min_periods=3).median()
        ax.plot(gv["trial_n"], roll, lw=2, color="#0f172a",
               label="rolling median")
        slope_here = stored_slope.get(g)
        title = g
        if slope_here is not None:
            title += f"\nstored slope {slope_here:+.2f} ms/trial"
        ax.set_title(title, fontsize=8)
        ax.set_xlabel("trial")
        if i % ncols == 0:
            ax.set_ylabel("reaction time (ms)")
        ax.legend(frameon=False, fontsize=7)
    for j in range(len(usable), nrows * ncols):
        axes[j // ncols][j % ncols].axis("off")
    fig.suptitle(f"Time on task{': ' + group_label if group_label else ''}",
                 fontsize=10)
    fig.tight_layout()
    _save(fig, "reaction_time_on_task")
    plt.show()


def sec_reaction_mode(trials, metas=None):
    """The reaction block: the floor statement, the RT distribution
    with anticipations excluded and counted, consistency across
    sessions, and the foreperiod diagnostic the exponential wait
    exists for.

    reaction.py's docstring names cue_flags as how the analysis is
    meant to separate blocks run under different cue mixes rather
    than pooling them blindly, and simple/choice measure different
    things (Der and Deary 2006). So this pools nothing across
    sub_mode: each sub_mode present gets its own distribution, median,
    trend and time-on-task figure below, and a mismatched cue_flags
    set inside a sub_mode is flagged rather than silently blended.
    """
    rx = reaction_frame(trials)
    print("\n" + "=" * 62)
    print("REACTION MODE")
    print("=" * 62)
    if rx.empty:
        _nothing("No reaction blocks in this selection. This section",
                 "reads the newer reaction mode, which packs its",
                 "foreperiod into the stimulus column; classic and",
                 "adaptive blocks are covered by the sections above.")
        return None
    reaction_floor_note()

    present = list(dict.fromkeys(rx["sub_mode"]))
    sub_modes = [m for m in ("simple", "choice") if m in present]
    sub_modes += [m for m in present if m not in sub_modes]
    multi = len(sub_modes) > 1

    if "cue_flags" in rx.columns:
        for sm in sub_modes:
            flags = sorted(v for v in
                           rx.loc[rx["sub_mode"] == sm, "cue_flags"]
                           .dropna().unique())
            if len(flags) > 1:
                print(f"\nWARNING: cue_flags is not constant within "
                      f"sub_mode={sm} ({', '.join(flags)}). "
                      "reaction.py's docstring names cue_flags as how "
                      "blocks under different cue mixes are told "
                      "apart; the group below pools audio-tactile-"
                      "visual RT with a leaner mix.")

    results = {}
    for i, sm in enumerate(sub_modes, start=1):
        rx_sm = rx[rx["sub_mode"] == sm]
        group_label = (f"sub_mode={sm} ({i} of {len(sub_modes)})"
                       if multi else "")
        results[sm] = _reaction_mode_group(rx_sm, metas, group_label)

    if multi and {"simple", "choice"} <= results.keys():
        ms, mc = results.get("simple"), results.get("choice")
        if (ms and mc and ms.get("median_ms") is not None
                and mc.get("median_ms") is not None):
            print(f"\nchoice overhead {mc['median_ms'] - ms['median_ms']:+.1f}"
                  f" ms, the cost of a 2-bit selection (Hick 1952).")

    return results if multi else next(iter(results.values()), None)


def _reaction_mode_group(rx, metas=None, group_label=""):
    """One sub_mode's worth of the reaction chapter: distribution,
    accuracy check, foreperiod diagnostic, per-session and per-hand
    splits, time-on-task, and what the block stored about itself.
    Split out of sec_reaction_mode so a selection spanning both
    sub_modes runs this once per sub_mode instead of pooling simple
    and choice RTs into one number."""
    if group_label:
        print(f"\n--- {group_label} ---")
    events = rx[rx["is_event"]]
    scored = rx[~rx["is_event"]]
    label = scored["early_late"].fillna("").astype(str)
    rt = pd.to_numeric(scored["time_difference_ms"], errors="coerce")
    hit = (label != "Miss") & rt.notna()
    # Belt and braces on the sub-100 ms cut: the mode routes these to
    # event rows already, so any that land here are a logging bug worth
    # seeing, not silently pooling.
    n_sub_cut = int((hit & (rt < ANTICIPATION_MS)).sum())
    valid = scored[hit & (rt >= ANTICIPATION_MS)].copy()
    valid["rt"] = rt[hit & (rt >= ANTICIPATION_MS)]

    kind = events["error_type"].fillna("").astype(str)
    kind = kind.where(kind != "", "catch_ok")
    ev_counts = kind.value_counts().to_dict()
    n_anticip = (ev_counts.get("false_start", 0)
                + ev_counts.get("anticipation", 0)
                + ev_counts.get("catch_false_start", 0) + n_sub_cut)

    print(f"\n{len(scored)} scorable trials: {len(valid)} valid RTs, "
          f"{int((label == 'Miss').sum())} misses")
    print(f"{len(events)} event rows that never became scorable trials:")
    for k, n in sorted(ev_counts.items()):
        print(f"   {n}x  {k}")
    if n_sub_cut:
        print(f"   {n_sub_cut}x  scorable rows under "
              f"{ANTICIPATION_MS:.0f} ms, excluded here. The mode should")
        print("   have routed these to event rows, so check the logs.")

    # Choice accuracy: rows[error_type=="wrong_finger", Miss] are the
    # scorable wrong-choice trials this group now correctly counts as
    # scored rather than as events; the timeouts (Miss with no
    # incorrect press) are excluded from the denominator, the same
    # convention reaction.py's own block_stats uses.
    scored_err = (scored["error_type"].fillna("").astype(str)
                 if "error_type" in scored.columns
                 else pd.Series("", index=scored.index))
    n_wrong_choice_rows = int(((label == "Miss")
                               & (scored_err == "wrong_finger")).sum())
    n_correct_rows = int((label != "Miss").sum())
    if n_wrong_choice_rows or n_correct_rows:
        denom = n_correct_rows + n_wrong_choice_rows
        accuracy = (n_correct_rows / denom) if denom > 0 else np.nan
    else:
        accuracy = np.nan
    if n_wrong_choice_rows:
        _reaction_accuracy_warning(accuracy, n_correct_rows,
                                   n_wrong_choice_rows)

    if valid.empty:
        _nothing("\nNo valid reaction times survived the exclusions, so",
                 "there is no distribution to draw.")
        return {"n_valid": 0}

    v = valid["rt"]
    lapses = int((v >= LAPSE_MS).sum())
    print(f"\nmedian {v.median():.1f} ms   p10 {v.quantile(.1):.1f} ms   "
          f"mean {v.mean():.1f} ms   sd {v.std():.1f}")
    print(f"lapses at or over {LAPSE_MS:.0f} ms: {lapses} "
          f"({lapses / len(v):.0%}), the PVT convention")

    if len(v) >= 40:
        fit = _reaction_exgaussian_fit(v.to_numpy())
        if fit:
            print(f"ex-Gaussian (method of moments, n={fit['n']}): "
                  f"mu {fit['mu']:.1f} ms   sigma {fit['sigma']:.1f} ms"
                  f"   tau {fit['tau']:.1f} ms (the right-tail component)")
        else:
            print("ex-Gaussian fit: sample skew too low or non-positive "
                  "for this estimator to resolve tau; skipped.")
    else:
        print(f"ex-Gaussian fit needs n >= 40 (have {len(v)}); skipped, "
              "per the brief's suppress-below-40 caution.")

    fp_rows = valid[valid["fp_s"].notna()]
    rho = np.nan
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
    ax[0].axvspan(0, ANTICIPATION_MS, color="#dc2626", alpha=.12)
    ax[0].hist(v, bins=_nbins(v), color="#0ea5e9", alpha=.85)
    ax[0].axvline(v.median(), color="#16a34a", lw=2,
                  label=f"median {v.median():.0f} ms")
    ax[0].axvline(LAPSE_MS, color="#b45309", lw=1.5, ls="--",
                  label=f"lapse {LAPSE_MS:.0f} ms ({lapses})")
    ax[0].set_xlabel("reaction time (ms)")
    ax[0].set_ylabel("trials")
    ax[0].set_title(f"Distribution, {n_anticip} anticipation(s) excluded")
    ax[0].legend(frameon=False, fontsize=8)
    if len(fp_rows) >= 5:
        s = fp_rows[["fp_s", "rt"]].sort_values("fp_s")
        ax[1].plot(s["fp_s"], s["rt"], "o", ms=4, alpha=.45,
                   color="#0ea5e9")
        ax[1].plot(s["fp_s"],
                   s["rt"].rolling(9, center=True, min_periods=3).median(),
                   lw=2, color="#0f172a", label="rolling median")
        rho = float(s["fp_s"].rank().corr(s["rt"].rank()))
        ax[1].set_xlabel("scheduled foreperiod (s)")
        ax[1].set_ylabel("reaction time (ms)")
        ax[1].set_title(f"Anticipation diagnostic, rho {rho:+.2f}")
        ax[1].legend(frameon=False, fontsize=8)
    else:
        ax[1].axis("off")
        ax[1].set_title("Too few foreperiods logged for the diagnostic")
    _save(fig, "reaction_mode")
    plt.show()

    if pd.notna(rho):
        if rho <= -0.2:
            print(f"rho {rho:+.2f}: RT falls as the wait grows, so the")
            print("stimulus is being timed rather than reacted to. Raise")
            print("reaction.catch_rate or check fp_mode before trusting")
            print("these numbers (Niemi and Naatanen 1981).")
        elif rho >= 0.2:
            # A strongly positive rho is not the healthy flat-expectancy
            # result either: RT growing with the wait reads as slowing
            # over the longer foreperiods (time on task, flagging
            # attention), so it gets its own sentence rather than the
            # "near zero" one printed beside a number that is not near
            # zero.
            print(f"rho {rho:+.2f}: RT GROWS as the wait grows. Not")
            print("anticipation, but not flat expectancy either: it")
            print("reads as slowing across the longer waits, so check")
            print("the time-on-task view before reading these RTs as")
            print("stable.")
        else:
            print(f"rho {rho:+.2f}: near zero is the healthy result, the")
            print("exponential foreperiod kept expectancy flat.")

    stored = stored_mode_stats(metas, "reaction")
    stored_slope = {g: d.get("slope_rt_ms_per_trial")
                    for g, d in (stored or {}).items()
                    if d.get("slope_rt_ms_per_trial") is not None}
    _reaction_time_on_task(valid, stored_slope, group_label)

    per = (valid.groupby("session")["rt"]
           .agg(n="count", median="median",
                p10=lambda x: x.quantile(.10))
           .round(1))
    print("\nmedian and p10 per session (p10 is the best consistent "
          "speed):")
    _show(per)
    if len(per) >= 2:
        fig, ax = plt.subplots(figsize=(8, 3.2))
        xs = range(1, len(per) + 1)
        ax.plot(xs, per["median"], "o-", lw=2, color="#0ea5e9",
                label="median")
        ax.plot(xs, per["p10"], "s--", lw=2, color="#16a34a", label="p10")
        ax.set_xticks(list(xs))
        ax.set_xticklabels(list(per.index), rotation=20, fontsize=8)
        ax.set_ylabel("reaction time (ms)")
        ax.set_title("Across sessions")
        ax.legend(frameon=False, fontsize=8)
        _save(fig, "reaction_sessions")
        plt.show()
        print("Early sessions dropping fast is practice, not recovery;")
        print("transfer beyond the trained task is unproven (Owen 2010).")

    if "side" in valid.columns and valid["side"].nunique() > 1:
        # Each trial's hand comes from the lane the mode actually cued
        # (finger_rehab.game.engine.log_trial's per-trial hand override), not
        # the block's hand_mode, so this splits a genuinely bilateral
        # block even though every row shares one reaction.py config.
        print("\nbilateral block: median and p10 by hand (each trial's")
        print("hand follows its cued lane, never the block's hand_mode):")
        per_hand = (valid.groupby("side")["rt"]
                    .agg(n="count", median="median",
                         p10=lambda x: x.quantile(.10))
                    .round(1))
        _show(per_hand)
        sides = [s for s in ("right", "left") if s in per_hand.index]
        if len(sides) == 2:
            fig, ax = plt.subplots(1, 2, figsize=(11, 3.6), sharey=True)
            for i, hand in enumerate(sides):
                hv = valid.loc[valid["side"] == hand, "rt"]
                ax[i].axvspan(0, ANTICIPATION_MS, color="#dc2626", alpha=.12)
                ax[i].hist(hv, bins=_nbins(hv), color="#0ea5e9", alpha=.85)
                ax[i].axvline(hv.median(), color="#16a34a", lw=2,
                              label=f"median {hv.median():.0f} ms")
                ax[i].set_xlabel("reaction time (ms)")
                ax[i].set_title(f"{hand.title()} hand, n={len(hv)}")
                ax[i].legend(frameon=False, fontsize=8)
            ax[0].set_ylabel("trials")
            _save(fig, "reaction_mode_by_hand")
            plt.show()
            fp_h = valid[valid["fp_s"].notna()]
            if len(fp_h) >= 10:
                fig, ax = plt.subplots(figsize=(8, 3.4))
                colours = {"right": "#0ea5e9", "left": "#f97316"}
                for hand in sides:
                    s = fp_h[fp_h["side"] == hand].sort_values("fp_s")
                    if len(s) < 5:
                        continue
                    ax.plot(s["fp_s"],
                            s["rt"].rolling(9, center=True,
                                             min_periods=3).median(),
                            lw=2, color=colours[hand],
                            label=f"{hand} rolling median")
                ax.set_xlabel("scheduled foreperiod (s)")
                ax.set_ylabel("reaction time (ms)")
                ax.set_title("Anticipation diagnostic by hand")
                ax.legend(frameon=False, fontsize=8)
                _save(fig, "reaction_fp_trend_by_hand")
                plt.show()

    per_game = []
    for game, g in scored.groupby("game", sort=False):
        n_ev = int((events["game"] == game).sum())
        n_fs = int(kind[events["game"] == game]
                   .isin(("false_start", "anticipation",
                          "catch_false_start")).sum())
        per_game.append({"game": game, "scorable": len(g),
                         "false_starts": n_fs, "events": n_ev,
                         "fs_rate": round(n_fs / max(1, len(g)), 3)})
    pg = pd.DataFrame(per_game)
    high = pg[pg["fs_rate"] > 0.10]
    if not high.empty:
        print("\nfalse-start rate over 10 percent, worth flagging:")
        _show(high)

    if stored:
        tbl = pd.DataFrame([{
            "game": g,
            "stored_median_ms": d.get("median_rt_ms"),
            "stored_p10_ms": d.get("p10_rt_ms"),
            "stored_slope_ms_per_trial": d.get("slope_rt_ms_per_trial"),
            "stored_rho_rt_vs_fp": d.get("spearman_rho_rt_vs_fp"),
            "level": d.get("level"),
            "stored_accuracy": d.get("accuracy"),
        } for g, d in stored.items()])
        print("\nwhat each block stored about itself (block_summary."
              "reaction):")
        _show(tbl)

    return {"n_valid": int(len(v)),
            "median_ms": round(float(v.median()), 1),
            "p10_ms": round(float(v.quantile(.1)), 1),
            "accuracy": (round(float(accuracy), 3)
                        if pd.notna(accuracy) else np.nan),
            "rho_rt_vs_fp": (round(rho, 3) if pd.notna(rho) else np.nan),
            "per_session": per}


# ============================================ Muscle memory (patterns)
# The SRTT block: a hidden 12-item sequence repeats under most takes,
# and learning is the RT jump when a probe take swaps in fresh material.
# pattern.py packs the take label and material id into stimulus and
# marks trained trials pattern_trial TRUE; the probe-minus-flankers
# subtraction below is the measurement the mode was designed around.

def pattern_take_table(trials, side=None, start_trim_by_game=None):
    """One row per take, with the mode's own RT hygiene: misses out,
    block-start trials out where the block says so, under-100 ms out,
    then a mean + 2.5 SD trim inside the take. Accuracy keeps every
    trial. The per-take RT lists ride along for the bootstrap
    intervals.

    side="left" or "right" keeps only that hand's trials, which is how
    the bimanual chapter scores each hand of an eight-finger take on
    its own. The trim then runs within the hand, so one slow hand
    cannot eat the other hand's outliers.

    start_trim_by_game maps game -> how many trials at the START of
    every take leave the RT pool (pattern.py's block-start exclusion:
    the first presses after a rest carry a recovery transient that is
    not learning; Das 2025, Gupta and Rickard 2022). The caller reads
    it from what each block stored about itself, so blocks saved
    before the trim existed stay un-trimmed and keep matching their
    own stored take means. Position within the take is computed
    before any side filter, so a bimanual take's first cycle is its
    first 24 trials, not the first 24 rows of one hand."""
    rows = mode_rows(trials, "pattern")
    if rows.empty or "stimulus" not in rows.columns:
        return pd.DataFrame()
    rows = rows.copy()
    kinds, takes, socs = [], [], []
    for cell in rows["stimulus"]:
        head, kv = stimulus_parts(cell)
        kinds.append(head or "unknown")
        takes.append(str(kv.get("b", "?")))
        socs.append(str(kv.get("soc", "")))
    rows["kind"] = kinds
    rows["take"] = takes
    rows["soc"] = socs
    rows["_ord"] = pd.to_numeric(rows["trial"], errors="coerce")
    rows = rows.sort_values(["game", "_ord"], kind="stable")
    rows["_pos_in_take"] = rows.groupby(["game", "take"]).cumcount()
    if side is not None:
        if "side" not in rows.columns:
            return pd.DataFrame()
        rows = rows[rows["side"] == side].copy()
        if rows.empty:
            return pd.DataFrame()
    rows["rt"] = pd.to_numeric(rows["time_difference_ms"], errors="coerce")
    rows["correct"] = rows["early_late"].fillna("").astype(str) != "Miss"
    # Contract check on the dedicated column: pattern_trial TRUE must
    # mean exactly the trained-sequence takes. A mismatch says the CSV
    # and the stimulus packing have drifted apart.
    if "pattern_trial" in rows.columns:
        pt = as_bool(rows["pattern_trial"]).fillna(False)
        mismatch = int((pt != (rows["kind"] == "seq")).sum())
        if mismatch:
            print(f"WARNING: pattern_trial disagrees with the stimulus "
                  f"packing on {mismatch} row(s).")
    trims = start_trim_by_game or {}
    out = []
    for (game, take), g in rows.groupby(["game", "take"], sort=False):
        k = int(trims.get(game, 0) or 0)
        ok = g["correct"] & g["rt"].notna()
        rts = g.loc[ok & (g["_pos_in_take"] >= k), "rt"]
        n_start = int((ok & (g["_pos_in_take"] < k)).sum())
        rts = rts[rts >= ANTICIPATION_MS]
        n_trim = 0
        if len(rts) >= 3:
            cut = rts.mean() + 2.5 * rts.std()
            n_trim = int((rts > cut).sum())
            rts = rts[rts <= cut]
        out.append({
            "game": game,
            "session": g["session"].iloc[0],
            "take": take,
            "kind": g["kind"].iloc[0],
            "soc": g["soc"].iloc[0],
            "order": float(pd.to_numeric(g["trial"],
                                         errors="coerce").min()),
            "n": int(len(g)),
            "accuracy": round(float(g["correct"].mean()), 3),
            "rt_ms": (round(float(rts.mean()), 1) if len(rts) else np.nan),
            "n_rt": int(len(rts)),
            "n_start_excluded": n_start,
            "n_trimmed": n_trim,
            "rts": rts,
        })
    df = pd.DataFrame(out)
    if df.empty:
        return df
    return (df.sort_values(["session", "game", "order"])
            .reset_index(drop=True))


def pattern_learning_scores(takes):
    """Probe RT minus the mean of its flanking trained takes, one row
    per probe, with a bootstrap interval over the trial RTs. Positive
    means the fresh material really was slower, which is the learning.

    The point score averages the two flankers' TAKE MEANS (mean of
    means), matching pattern.py's own block_stats convention (the
    classic block-mean rebound), NOT a trial-count-weighted pool of
    both flankers' raw RTs. The two differ whenever a flanker take was
    cut short, so this must stay aligned with the stored
    learning_score_ms or the notebook and the mode disagree about a
    number both call the same name (audit finding #15). The bootstrap
    CI still resamples from the pooled trial-level RTs, which is a
    different (and standard) use of the raw values."""
    out = []
    if takes is None or takes.empty:
        return pd.DataFrame(out)
    for game, g in takes.groupby("game", sort=False):
        g = g.sort_values("order").reset_index(drop=True)
        seq_idx = list(g.index[g["kind"] == "seq"])
        for i in g.index[g["kind"] == "probe"]:
            before = [j for j in seq_idx if j < i]
            after = [j for j in seq_idx if j > i]
            fl = ([before[-1]] if before else []) \
                + ([after[0]] if after else [])
            fl_rts = (pd.concat([g.loc[j, "rts"] for j in fl])
                      if fl else pd.Series(dtype="float64"))
            fl_means = [g.loc[j, "rts"].mean() for j in fl
                       if len(g.loc[j, "rts"])]
            probe_rts = g.loc[i, "rts"]
            score = np.nan
            if len(probe_rts) and fl_means:
                score = float(probe_rts.mean()
                             - sum(fl_means) / len(fl_means))
            lo, hi = diff_ci(probe_rts, fl_rts)
            out.append({
                "game": game,
                "session": g.loc[i, "session"],
                "probe_take": g.loc[i, "take"],
                "soc": g.loc[i, "soc"],
                "n_flankers": len(fl),
                "learning_score_ms": (round(score, 1)
                                      if pd.notna(score) else np.nan),
                "ci_lo": round(lo, 1) if pd.notna(lo) else np.nan,
                "ci_hi": round(hi, 1) if pd.notna(hi) else np.nan,
            })
    return pd.DataFrame(out)


def pattern_config_signature(rows_for_game):
    """The distinct cue_flags value(s) inside one game's pattern rows.

    pattern.py logs cue_flags per trial precisely so a run under
    different cue settings can be told apart later; a single block
    should therefore carry exactly one value. Returns (value, mixed):
    value is that one code, or None when the rows disagree or there
    are none; mixed is TRUE only for the disagreement case, which
    should never happen from one block and is worth its own warning."""
    cue_vals = []
    if "cue_flags" in rows_for_game.columns:
        cue_vals = sorted(v for v in rows_for_game["cue_flags"].dropna()
                          .unique() if str(v) != "")
    if len(cue_vals) == 1:
        return cue_vals[0], False
    return None, len(cue_vals) > 1


def pattern_consistency_groups(trials, metas):
    """Selected pattern games grouped by (rsi_ms, timeout_ms, cue_flags).

    pattern.py's own docstring is explicit that the trial mix and the
    RSI/timeout/cue settings must stay fixed for a participant across
    every session, and that cue_flags on each row is how the analysis
    is meant to verify it did. rsi_ms and timeout_ms are not per-trial
    columns, so they come from the block summary the mode wrote about
    itself; cue_flags comes straight off the trial rows.

    Returns (groups, signatures): groups maps a signature tuple to the
    list of games sharing it, in first-seen order; signatures maps
    every game to its own {rsi_ms, timeout_ms, cue_flags, cue_mixed}.
    A selection that is fully consistent comes back as one group, so
    callers that only care about that case can just check len == 1.
    """
    rows = mode_rows(trials, "pattern")
    games = list(dict.fromkeys(rows["game"])) if not rows.empty else []
    sigs = {}
    for g in games:
        bs = ((metas or {}).get(g, {}) or {}).get("block_summary", {})
        bs = (bs or {}).get("pattern", {}) or {}
        cue, mixed = pattern_config_signature(rows[rows["game"] == g])
        # demo=True marks the Test Mode two-take miniature (pattern.py
        # __init__, demo_trials branch): built to show pattern_trial
        # TRUE and FALSE at least once for a supervisor, not to measure
        # anything. Carried through so callers can keep it out of the
        # learning curve and probe scores (audit finding #16).
        sigs[g] = {"rsi_ms": bs.get("rsi_ms"), "timeout_ms": bs.get("timeout_ms"),
                   "cue_flags": cue, "cue_mixed": mixed,
                   "demo": bool(bs.get("demo", False))}
    groups: dict = {}
    for g in games:
        s = sigs[g]
        key = (s["rsi_ms"], s["timeout_ms"], s["cue_flags"])
        groups.setdefault(key, []).append(g)
    return groups, sigs


def sec_pattern_srtt(trials, metas=None):
    """The SRTT learning curve and the probe-minus-flankers score.

    Warm-up takes are dropped (they exist so the patient settles), the
    random take is the general-speed baseline, and probes are scored
    against their flanking trained takes so a slow day cannot fake a
    learning score.

    Before any of that, the selected sessions are checked against
    pattern.py's own consistency promise (rsi_ms, timeout_ms and
    cue_flags fixed for a participant). Sessions that disagree are
    never pooled into one curve or one learning score: each distinct
    setting gets scored as its own group, so a therapist-changed
    setting shows up as a labelled split instead of a blended, wrong
    number.
    """
    print("\n" + "=" * 62)
    print("MUSCLE MEMORY (PATTERNS, SRTT)")
    print("=" * 62)
    groups, sigs = pattern_consistency_groups(trials, metas)
    if not groups:
        _nothing("No pattern blocks in this selection. This section",
                 "needs the pattern mode's takes, which pack their",
                 "labels into the stimulus column and mark trained",
                 "trials with pattern_trial TRUE.")
        return None

    # Test Mode demo blocks are a supervisor-facing miniature, not a
    # measurement: dropped from the curve and the probe scores rather
    # than silently pooling in alongside real sessions (audit finding
    # #16). Consistency groups are rebuilt without them so a selection
    # that is all demo still reports "nothing" rather than a curve
    # built from a session nobody meant to analyse.
    demo_games = [g for g, s in sigs.items() if s.get("demo")]
    if demo_games:
        print("\nEXCLUDED: " + ", ".join(demo_games) + " "
              + ("is a" if len(demo_games) == 1 else "are") + " Test "
              "Mode demo block" + ("" if len(demo_games) == 1 else "s")
              + " (block_summary.pattern.demo=True), not a real "
              "session; left out of the curve and probe scores below.")
        groups = {key: [g for g in gms if g not in demo_games]
                  for key, gms in groups.items()}
        groups = {key: gms for key, gms in groups.items() if gms}
        if not groups:
            _nothing("Every pattern block in this selection is a demo.",
                     "Nothing left to analyse once those are excluded.")
            return None

    mixed_games = [g for g, s in sigs.items() if s["cue_mixed"]]
    if mixed_games:
        print("\nWARNING: cue_flags is not constant WITHIN "
              f"{', '.join(mixed_games)}. pattern.py logs one cue_flags "
              "value per block, so a mix inside one game means the CSV "
              "and the mode's cue settings have drifted apart; RTs from "
              "it are not trustworthy for any cue comparison.")

    if len(groups) > 1:
        print("\nSPLIT: rsi_ms / timeout_ms / cue_flags are not the same "
              "across every selected session. pattern.py's docstring "
              "says these must stay fixed for a participant, so the RT "
              "contrast is only valid within a matching group; the "
              "sessions below are scored as separate groups rather than "
              "pooled into one curve:")
        for key, gms in groups.items():
            rsi, timeout, cue = key
            print(f"   rsi={rsi} timeout={timeout} cue={cue}: "
                  + ", ".join(gms))

    all_takes = []
    for i, (key, gms) in enumerate(groups.items(), start=1):
        trials_g = trials[trials["game"].isin(gms)]
        metas_g = ({g: m for g, m in metas.items() if g in gms}
                  if metas else metas)
        label = None
        if len(groups) > 1:
            rsi, timeout, cue = key
            label = f"group {i} of {len(groups)} (rsi={rsi} " \
                    f"timeout={timeout} cue={cue}): {', '.join(gms)}"
        t = _pattern_srtt_group(trials_g, metas_g, label)
        if t is not None and not t.empty:
            all_takes.append(t)
    if not all_takes:
        return None
    # Each group carries a DataFrame in .attrs (the scores table), and
    # pandas concat compares .attrs across inputs for the combined
    # frame's own attrs, which is ambiguous once that value is itself
    # a DataFrame. The per-group scores already printed above, so the
    # combined table drops them rather than fighting pandas over it.
    for t in all_takes:
        t.attrs = {}
    return pd.concat(all_takes, ignore_index=True)


def _pattern_srtt_group(trials, metas, label=None):
    """One consistency group's worth of the SRTT chapter: the curve,
    the probe-minus-flankers score, the per-hand split for bimanual
    play, and what the block stored about itself. Split out of
    sec_pattern_srtt so a selection spanning more than one rsi_ms /
    timeout_ms / cue_flags setting can run this once per setting
    instead of pooling RTs across a timing change."""
    if label:
        print(f"\n--- {label} ---")
    # Start-of-take exclusion, aligned with the mode: a block that
    # stored start_trim gets exactly that trim here, so recomputed
    # take means keep matching the stored ones (finding #15). Blocks
    # saved before the trim existed stored un-trimmed means and are
    # recomputed un-trimmed rather than silently re-scored.
    stored = stored_mode_stats(metas, "pattern") if metas else {}
    start_trims = {g: int(d.get("start_trim") or 0)
                   for g, d in (stored or {}).items()}
    if any(start_trims.values()):
        print("\nRT hygiene: the first cycle of every take leaves the")
        print("take means (block-start transient after a rest, not")
        print("learning), matching what each block stored.")
        untrimmed = sorted(g for g, k in start_trims.items() if not k)
        if untrimmed:
            print("Saved before the trim existed, kept un-trimmed: "
                  + ", ".join(untrimmed))
    takes = pattern_take_table(trials, start_trim_by_game=start_trims)
    if takes is None or takes.empty:
        _nothing("No pattern blocks in this group.")
        return None

    plot = takes[takes["kind"] != "warmup"].reset_index(drop=True)
    plot = plot[plot["rt_ms"].notna()].reset_index(drop=True)
    if plot.empty:
        _nothing("Every take is warm-up or has no usable correct RTs,",
                 "so there is no curve to draw.")
        return takes

    plot["x"] = range(1, len(plot) + 1)
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(plot["x"], plot["rt_ms"], "-", lw=1, color="#cbd5e1",
            zorder=1)
    styles = {
        "seq": dict(marker="o", color="#2563eb", label="trained"),
        "random": dict(marker="s", color="#94a3b8", label="random"),
    }
    for k, st in styles.items():
        sub = plot[plot["kind"] == k]
        if not sub.empty:
            ax.plot(sub["x"], sub["rt_ms"], st["marker"], ms=7,
                    color=st["color"], label=st["label"], zorder=3)
    probes = plot[plot["kind"] == "probe"]
    if not probes.empty:
        ax.plot(probes["x"], probes["rt_ms"], "o", ms=9, mfc="none",
                mec="#dc2626", mew=2, label="probe (fresh material)",
                zorder=4)
    # Vertical dividers where a new game starts, so several sessions on
    # one axis stay readable.
    prev = None
    for _, r in plot.iterrows():
        if prev is not None and r["game"] != prev:
            ax.axvline(r["x"] - 0.5, color="#e2e8f0", lw=1.5, ls="--")
        prev = r["game"]
    ax.set_xticks(list(plot["x"]))
    ax.set_xticklabels(list(plot["take"]), fontsize=8)
    ax.set_xlabel("take")
    ax.set_ylabel("mean correct RT (ms), trimmed")
    ax.set_title("SRTT learning curve")
    ax.legend(frameon=False, fontsize=8)
    _save(fig, "pattern_learning_curve")
    plt.show()

    print("READING THE CURVE")
    print("   Learning shows as RT falling across the trained takes and")
    print("   SPIKING at each open red probe: the probe is fresh")
    print("   material the trained sequence cannot help with. Flat")
    print("   probes mean no sequence-specific learning; a uniform")
    print("   fall with no spikes is general speed-up only.")
    print("ANCHORS")
    print("   Nissen and Bullemer (1987): 327 to 163 ms over training,")
    print("   with a clear rebound on the swapped-in random block.")
    print("   Kal et al (2016), stroke meta-analysis: 69 ms mean")
    print("   rebound with the unaffected side; the affected-side")
    print("   pooled effect was null, so affected-hand blocks are")
    print("   measurement, not proven therapy.")
    print("   Healthy young adults on 12-item SOCs: roughly 50-100 ms.")

    scores = pattern_learning_scores(takes)
    scores = scores[scores["learning_score_ms"].notna()]
    if scores.empty:
        print("\nNo probe could be scored: a score needs a probe take")
        print("with a trained take on at least one side.")
    else:
        print("\nlearning score per probe (probe minus flankers, "
              "positive = learning):")
        _show(scores.drop(columns=["game"]))
        for sess, g in scores.groupby("session"):
            m = g["learning_score_ms"].mean()
            solid = (g["ci_lo"] > 0).all() and g["ci_lo"].notna().all()
            note = ("interval excludes zero"
                    if solid else "interval includes zero")
            print(f"   {sess}: session score {m:+.1f} ms   ({note})")
        fig, ax = plt.subplots(figsize=(7, 3.2))
        xs = range(1, len(scores) + 1)
        ax.bar(xs, scores["learning_score_ms"], color="#2563eb",
               alpha=.8)
        yerr_lo = (scores["learning_score_ms"] - scores["ci_lo"]).values
        yerr_hi = (scores["ci_hi"] - scores["learning_score_ms"]).values
        ax.errorbar(list(xs), scores["learning_score_ms"],
                    yerr=[yerr_lo, yerr_hi], fmt="none",
                    ecolor="#0f172a", capsize=3, lw=1.2)
        ax.axhline(0, color="#dc2626", lw=1.5)
        ax.set_xticks(list(xs))
        ax.set_xticklabels([f"{r.session}\ntake {r.probe_take}"
                            for r in scores.itertuples()], fontsize=7)
        ax.set_ylabel("learning score (ms)")
        ax.set_title("Probe minus flankers, with bootstrap 95% CI")
        _save(fig, "pattern_learning_score")
        plt.show()
        print("One session alone cannot claim memory: a single-session")
        print("rebound can fade within minutes (Trofimova 2020). The")
        print("claim needs the score to hold up across days.")

    # ---- each hand of a bimanual sequence -----------------------------
    # With Both hands selected the trained sequence spans all eight
    # fingers, so the curve above mixes the hands. Score each hand on
    # its own here: same take structure, same hygiene, RTs filtered to
    # the hand before the trim.
    rows_p = mode_rows(trials, "pattern")
    if (not rows_p.empty and "hand_mode" in rows_p.columns
            and "side" in rows_p.columns):
        both_p = rows_p[rows_p["hand_mode"] == "both"]
    else:
        both_p = rows_p.iloc[0:0]
    if not both_p.empty and {"left", "right"} <= set(both_p["side"]):
        print("\nBOTH HANDS: the trained sequence spans all eight")
        print("fingers, so each hand's takes are scored on their own")
        print("below. Sequence knowledge is substantially effector-")
        print("independent (Japikse et al 2003), so the hands should")
        print("learn together; one hand spiking at probes while the")
        print("other stays flat is a hand-specific effect worth a")
        print("second look.")
        per_side = {}
        for s in ("right", "left"):
            t = pattern_take_table(both_p, side=s,
                                   start_trim_by_game=start_trims)
            if t is not None and not t.empty:
                plot_t = (t[(t["kind"] != "warmup")
                            & t["rt_ms"].notna()]
                          .reset_index(drop=True))
                per_side[s] = (t, plot_t)
        if any(len(pt) for _t, pt in per_side.values()):
            fig, ax = plt.subplots(figsize=(10, 3.6))
            colours = {"right": "#2563eb", "left": "#0d9488"}
            for s, (_t, pt) in per_side.items():
                if pt.empty:
                    continue
                pt = pt.copy()
                pt["x"] = range(1, len(pt) + 1)
                ax.plot(pt["x"], pt["rt_ms"], "o-", ms=5, lw=1.2,
                        color=colours.get(s, "#64748b"),
                        label=f"{s} hand")
                pr = pt[pt["kind"] == "probe"]
                if not pr.empty:
                    ax.plot(pr["x"], pr["rt_ms"], "o", ms=9,
                            mfc="none", mec="#dc2626", mew=2)
                ax.set_xticks(list(pt["x"]))
                ax.set_xticklabels(list(pt["take"]), fontsize=8)
            ax.set_xlabel("take (bimanual games)")
            ax.set_ylabel("mean correct RT (ms), trimmed within hand")
            ax.set_title("SRTT curve per hand, probes circled")
            ax.legend(frameon=False, fontsize=8)
            _save(fig, "pattern_learning_curve_by_hand")
            plt.show()
        hand_scores = []
        for s, (t, _pt) in per_side.items():
            sc = pattern_learning_scores(t)
            if sc is not None and not sc.empty:
                sc = sc.copy()
                sc.insert(0, "hand", s)
                hand_scores.append(sc)
        hs_all = (pd.concat(hand_scores, ignore_index=True)
                  if hand_scores else pd.DataFrame())
        if not hs_all.empty:
            hs_all = hs_all[hs_all["learning_score_ms"].notna()]
        if not hs_all.empty:
            print("learning score per probe, per hand:")
            _show(hs_all.drop(columns=["game"]))
            takes.attrs["scores_by_hand"] = hs_all
        else:
            print("No per-hand score yet: a hand needs usable RTs in")
            print("a probe take AND a flanking trained take.")

    if stored:
        tbl = pd.DataFrame([{
            "game": g,
            "stored_session_score_ms": d.get("session_learning_score_ms"),
            "start_trim": d.get("start_trim"),
            "best_3star_run": d.get("three_star_streak_best"),
            "end_reason": d.get("end_reason"),
            "short_session": d.get("short_session"),
        } for g, d in stored.items()])
        print("\nwhat each block stored about itself (block_summary."
              "pattern):")
        _show(tbl)

    show_cols = ["session", "take", "kind", "soc", "n", "accuracy",
                 "rt_ms", "n_rt", "n_start_excluded", "n_trimmed"]
    _show(takes[show_cols])
    takes.attrs["scores"] = scores
    return takes


def chord_difficulty(fingers):
    """Predicted hardness D: for each QUIET finger, its enslavability
    weight times how many active neighbours pull on it, plus 1.5 per
    finger above two. Same sum chords.py builds its tiers from."""
    active = set(int(f) for f in fingers)
    d = 0.0
    for f in range(4):
        if f in active:
            continue
        d += CHORD_ENSLAVABILITY[f] * sum(1 for a in CHORD_ADJACENT[f]
                                          if a in active)
    d += CHORD_SIZE_PENALTY * max(0, len(active) - 2)
    return d


def chord_fingers(cell):
    """The chord's fingers (0..3) out of a stimulus like '1+3+4'."""
    try:
        return tuple(sorted({(int(t) - 1) % 4
                             for t in str(cell).split("+") if t.strip()}))
    except (TypeError, ValueError):
        return ()


def chord_frame(trials, calset=None):
    """WITHIN-hand chord rows with the chord decoded and the enslaving
    ratio ER recomputed the way chords.py computes it: mean
    quiet-finger force over mean target force, each lane divided by
    its own calibrated reference press when every lane of the hand has
    one, raw counts otherwise (er_basis says which).

    kind "probe" (a single-finger stimulus) only appears in sessions
    recorded before 2026-09, when the mode opened and closed with
    single-finger probe trials. Newer sessions are chords only: the
    single-press reference moved to the quick-cal light-press capture
    (the same calibrated reference this frame already divides by), so
    nothing here changes shape, older data just carries extra rows.

    Cross-hand chords (stimulus "x:1+5") are deliberately NOT parsed
    here: chord_fingers maps lanes mod 4, under which "1+5" would read
    as a single-finger probe, so the mode marks the descriptor and
    this frame stays scope-pure. The bimanual subsection reads them
    from the block summary instead."""
    rows = mode_rows(trials, "chords")
    if rows.empty or "stimulus" not in rows.columns:
        return pd.DataFrame()
    out = []
    for idx, r in rows.iterrows():
        fingers = chord_fingers(r.get("stimulus"))
        if not fingers:
            continue
        targets = row_targets(r)
        if not targets:
            continue
        board = targets[0] // 4
        hand_lanes = [board * 4 + i for i in range(4)]
        peaks = parse_peaks(r.get("force_window_peaks"))
        er, basis, press_level = np.nan, "", np.nan
        leaks = {}
        if peaks:
            hand_mode = r.get("hand_mode", "right")
            gaps = {l: (calset.lane_gap(r.get("game"), l, hand_mode)
                        if calset is not None else None)
                    for l in hand_lanes}
            use_cal = all(gaps[l] for l in hand_lanes)

            def level(lane):
                v = max(0.0, float(peaks.get(lane, 0.0)))
                return v / gaps[lane] if use_cal else v

            press = [level(l) for l in targets]
            quiet = {l: level(l) for l in hand_lanes if l not in targets}
            mp = sum(press) / len(press) if press else 0.0
            if mp > 0:
                basis = "calibrated" if use_cal else "raw"
                press_level = mp
                leaks = {FINGERS[l % 4]: v for l, v in quiet.items()}
                if quiet:
                    er = sum(quiet.values()) / len(quiet) / mp
        side = r.get("side")
        if side not in ("left", "right"):
            # Fall back to the board the targets sit on. The loader's
            # side column already covers unilateral left sessions,
            # where lanes 0..3 belong to the left hand.
            side = "left" if board == 1 else "right"
        out.append({
            "row_id": idx,
            "game": r.get("game"),
            "session": r.get("session"),
            "side": side,
            "chord": "".join(CHORD_LETTERS[f] for f in fingers),
            "kind": "probe" if len(fingers) == 1 else "chord",
            "d": chord_difficulty(fingers),
            "clean": str(r.get("early_late"))
                     in ("Perfect", "Great", "Good"),
            "er": er,
            "er_basis": basis,
            "press_level": press_level,
            "leaks": leaks,
        })
    return pd.DataFrame(out)


def _draw_matrix(ax, m, vmax, title, fmt="{:.1f}"):
    """One 4x4 finger matrix as an annotated heatmap. Diagonal cells
    are masked: a finger cannot leak onto itself."""
    ax.imshow(np.array(m, dtype=float), cmap="Reds", vmin=0, vmax=vmax)
    ax.set_xticks(range(4))
    ax.set_xticklabels(FINGERS, fontsize=8)
    ax.set_yticks(range(4))
    ax.set_yticklabels(FINGERS, fontsize=8)
    ax.set_title(title, fontsize=9)
    ax.grid(False)
    for i in range(4):
        for j in range(4):
            v = m[i][j]
            if i == j or v is None or (isinstance(v, float)
                                       and np.isnan(v)):
                ax.text(j, i, "-", ha="center", va="center",
                        color="#94a3b8", fontsize=9)
            else:
                dark = vmax and v > 0.55 * vmax
                ax.text(j, i, fmt.format(v), ha="center", va="center",
                        color="white" if dark else "#0f172a", fontsize=9)


def sec_chords(trials, metas=None, calset=None):
    """Chord mode: does the predicted difficulty ladder hold for this
    hand, how loud were the quiet fingers, and the probe enslaving
    matrices the session's edges exist for."""
    ch = chord_frame(trials, calset)
    print("\n" + "=" * 62)
    print("CHORD MODE")
    print("=" * 62)
    if ch is None or ch.empty:
        _nothing("No chord blocks in this selection. This section reads",
                 "the chords mode, which writes the chord into the",
                 "stimulus column ('1+3+4') and the full target set",
                 "into correct_keys.")
        return None

    chords_only = ch[ch["kind"] == "chord"]
    probes = ch[ch["kind"] == "probe"]
    rows_all = mode_rows(trials, "chords")
    n_cross_rows = (rows_all["stimulus"].astype(str)
                    .str.startswith("x:").sum()
                    if "stimulus" in rows_all.columns else 0)
    print(f"{len(chords_only)} within-hand chord trials"
          + (f", {len(probes)} single-finger probes (recorded before "
             "the 2026-09 probe removal)" if len(probes) else "")
          + (f", {int(n_cross_rows)} cross-hand chord rows "
             "(bimanual subsection below)" if n_cross_rows else ""))
    if not chords_only.empty and chords_only["side"].nunique() > 1:
        parts = ", ".join(
            f"{h}: {int((chords_only['side'] == h).sum())} chords / "
            f"{int((probes['side'] == h).sum())} probes"
            for h in ("right", "left"))
        print("both hands played (" + parts + "); everything below is")
        print("split per hand, because cross-talk is a within-hand")
        print("quantity and pooling the hands would average away the")
        print("hand difference a bilateral session exists to show.")

    # ---- difficulty validation ----------------------------------------
    stored = stored_mode_stats(metas, "chords")
    # Keyed by (hand, chord, w_ms): the level ladder interleaves tier
    # and window (level = window*4 + tier), so an easy chord is met at
    # wide windows and a hard one first met at the tightest, and
    # pooling their hit rates lets the window artefact masquerade as
    # the enslaving pattern the rank test below is meant to read off.
    per = {}
    for g, d in stored.items():
        for row in d.get("per_chord") or []:
            name = row.get("chord")
            if not name:
                continue
            # Bilateral summaries name the hand on every per_chord
            # row; older unilateral ones name it once on the block.
            hand = str(row.get("hand") or d.get("hand") or "right")
            w_ms = row.get("w_ms")
            c = per.setdefault((hand, name, w_ms),
                               {"d": row.get("d"), "n": 0,
                                "hits": 0.0, "ers": []})
            n = int(row.get("n") or 0)
            c["n"] += n
            if row.get("hit_rate") is not None:
                c["hits"] += float(row["hit_rate"]) * n
            if row.get("median_er") is not None:
                c["ers"].append(float(row["median_er"]))
    if per:
        hit_basis = ("the mode's own outcome classes "
                     "(block_summary.chords)")
        tbl = pd.DataFrame([{
            "hand": hand, "chord": name, "w_ms": w_ms, "d": v["d"],
            "n": v["n"],
            "hit_rate": (round(v["hits"] / v["n"], 3) if v["n"] else
                         np.nan),
            "median_er": (round(float(np.median(v["ers"])), 3)
                          if v["ers"] else np.nan),
        } for (hand, name, w_ms), v in per.items()])
    elif not chords_only.empty:
        hit_basis = ("CSV outcome labels, which cannot see leak fails, "
                     "so this over-counts clean hits")
        tbl = (chords_only.groupby(["side", "chord"])
               .agg(d=("d", "first"), n=("chord", "count"),
                    hit_rate=("clean", "mean"),
                    median_er=("er", "median"))
               .round(3).reset_index()
               .rename(columns={"side": "hand"}))
        # trials.csv carries no per-trial synchrony window, so this
        # fallback path cannot split by window the way the stored
        # per_chord path above does; the rank test below prints a
        # pooled-window caveat whenever it runs on this branch.
        tbl["w_ms"] = np.nan
    else:
        tbl = pd.DataFrame()
    if not tbl.empty:
        tbl["d"] = pd.to_numeric(tbl["d"], errors="coerce")
        tbl = tbl.sort_values(["hand", "d"]).reset_index(drop=True)
        hands_in = [h for h in ("right", "left")
                    if (tbl["hand"] == h).any()]
        hands_in += [h for h in tbl["hand"].unique()
                     if h not in hands_in]
        print(f"\nhit rate per chord, basis: {hit_basis}")
        fig, axes = plt.subplots(1, len(hands_in),
                                 figsize=(9 if len(hands_in) == 1
                                          else 6 * len(hands_in), 3.4),
                                 sharey=True, squeeze=False)
        for ax, hand in zip(axes[0], hands_in):
            sub = tbl[tbl["hand"] == hand].reset_index(drop=True)
            xs = range(len(sub))
            ax.bar(xs, sub["hit_rate"], color="#dc2626", alpha=.75)
            ax.set_xticks(list(xs))
            ax.set_xticklabels([f"{r.chord}\nD={r.d:g}"
                                for r in sub.itertuples()], fontsize=8)
            ax.set_ylabel("hit rate")
            ax.set_ylim(0, 1.05)
            ax.set_title("Hit rate ordered by predicted difficulty D"
                         if len(hands_in) == 1
                         else f"{hand} hand, ordered by predicted D",
                         fontsize=10)
            if sub["median_er"].notna().any():
                ax2 = ax.twinx()
                ax2.plot(list(xs), sub["median_er"], "o-", lw=2,
                         color="#0f172a", label="median ER")
                ax2.set_ylabel("median enslaving ratio")
                ax2.grid(False)
                ax2.legend(frameon=False, fontsize=8, loc="upper left")
        _save(fig, "chord_difficulty")
        plt.show()

        # Rank agreement rather than an exact order match: ties and a
        # couple of swapped neighbours are noise, a positive or flat
        # correlation is a real disagreement with the healthy ladder.
        # Tested per hand AND per synchrony window: stroke can break
        # one hand's ordering while sparing the other (Lang and
        # Schieber 2004), and the level ladder interleaves tier and
        # window (easy chords met at wide windows, hard ones first met
        # at the tightest), so a rank test pooled across windows can
        # mistake the window ladder for the enslaving pattern.
        for hand in hands_in:
            windows_here = sorted(
                (w for w in tbl.loc[tbl["hand"] == hand, "w_ms"]
                 .dropna().unique()), reverse=True)
            groups = ([(w, tbl[(tbl["hand"] == hand)
                                & (tbl["w_ms"] == w)])
                      for w in windows_here] if windows_here
                      else [(None, tbl[tbl["hand"] == hand])])
            pooled_caveat = windows_here == [] or len(windows_here) == 0
            for w, sub in groups:
                if w is None:
                    label = (f"the {hand} hand" if len(hands_in) > 1
                             else "this hand")
                    label += " (pooled across windows, no per-window data)"
                else:
                    label = (f"the {hand} hand at W={w:g}ms"
                             if len(hands_in) > 1 else f"W={w:g}ms")
                rho_d = float(sub["d"].rank().corr(sub["hit_rate"].rank()))
                if pd.isna(rho_d) or len(sub) < 3:
                    print(f"Too few chords or every hit rate tied for "
                          f"{label}, so the")
                    print("predicted-vs-actual ordering cannot be tested "
                          "yet.")
                elif rho_d <= -0.6:
                    print(f"The predicted ordering held for {label} (rank")
                    print(f"correlation {rho_d:+.2f}): hit rate falls as D")
                    print("rises, the adjacency story (Zatsiorsky 2000;")
                    print("Hager-Ross and Schieber 2000).")
                else:
                    print(f"DISAGREEMENT for {label} with the healthy")
                    print(f"ordering (rank correlation {rho_d:+.2f}). That")
                    print("is plausible after stroke (Lang and Schieber")
                    print("2004), so here is the empirical ladder, easiest")
                    print("first, for personalising the tiers:")
                    emp = sub.sort_values("hit_rate", ascending=False)
                    for r in emp.itertuples():
                        print(f"   {r.chord:5} hit {r.hit_rate:.0%}   "
                              f"predicted D {r.d:g}")

    # ---- cross-talk per chord -----------------------------------------
    ers = chords_only[chords_only["er"].notna()]
    basis = ""
    if not ers.empty:
        basis = ("calibrated" if (ers["er_basis"] == "calibrated").any()
                 else "raw")
        ers = ers[ers["er_basis"] == basis]
    if ers.empty:
        print("\nNo per-trial enslaving ratio could be recomputed: that")
        print("needs force_window_peaks rows, which keyboard blocks do")
        print("not log.")
    else:
        er_hands = [h for h in ("right", "left")
                    if (ers["side"] == h).any()]
        print(f"\nenslaving ratio per chord, {len(ers)} trial(s), "
              f"basis: {basis}")
        if basis == "raw":
            print("   raw counts: no usable calibration, so pad")
            print("   sensitivity is in these numbers. See the")
            print("   individuation section for what that does.")
        # One panel per hand. Leak is a within-hand quantity, and in
        # bilateral play pooling the hands would average away exactly
        # the hand difference the session exists to see.
        fig, axes = plt.subplots(1, max(1, len(er_hands)),
                                 figsize=(9 if len(er_hands) < 2
                                          else 6 * len(er_hands), 3.6),
                                 sharey=True, squeeze=False)
        for ax, hand in zip(axes[0], er_hands):
            eh = ers[ers["side"] == hand]
            order = (eh.groupby("chord")["d"].first()
                     .sort_values().index.tolist())
            ax.axhspan(HEALTHY_ER_BAND[0], HEALTHY_ER_BAND[1],
                       color="#16a34a", alpha=.15,
                       label="healthy 8-15% (light force)")
            data = [eh.loc[eh["chord"] == c, "er"] for c in order]
            bp = ax.boxplot(data, labels=order, patch_artist=True,
                            widths=.6)
            for p in bp["boxes"]:
                p.set_facecolor("#dc2626")
                p.set_alpha(.6)
            for mline in bp["medians"]:
                mline.set_color("white")
                mline.set_linewidth(2)
            ax.axhline(0.25, color="#b45309", lw=1.5, ls="--",
                       label="leak-fail threshold 0.25")
            ax.set_ylabel("ER (quiet over target force)")
            ax.set_title("Cross-talk per chord, ordered by predicted D"
                         if len(er_hands) < 2
                         else f"{hand} hand, ordered by predicted D",
                         fontsize=10)
            ax.legend(frameon=False, fontsize=8)
        _save(fig, "chord_crosstalk")
        plt.show()
        if len(er_hands) > 1:
            meds = "   ".join(
                f"{h}: "
                f"{float(ers.loc[ers['side'] == h, 'er'].median()):.3f}"
                for h in er_hands)
            print("median ER per hand:   " + meds)
        else:
            print(f"median ER {float(ers['er'].median()):.3f}.")
        print("Healthy light-force enslaving sits around 0.08 to 0.15")
        print("(Abolins et al. 2020; Zatsiorsky 2000). Device- and")
        print("posture-specific: order-of-magnitude context, never")
        print("validation.")

        # Chord by quiet finger: where does the leak actually go. Per
        # hand, because a column only means one thing when every row
        # in it belongs to the same four fingers.
        panels = []
        vmax_all = 0.0
        for hand in er_hands:
            eh = ers[ers["side"] == hand]
            order = (eh.groupby("chord")["d"].first()
                     .sort_values().index.tolist())
            grid = np.full((len(order), 4), np.nan)
            for i, c in enumerate(order):
                sub = eh[eh["chord"] == c]
                for j, f in enumerate(FINGERS):
                    vals = [lk[f] for lk in sub["leaks"] if f in lk]
                    if vals:
                        grid[i, j] = float(np.median(vals))
            panels.append((hand, order, grid))
            if np.isfinite(grid).any():
                vmax_all = max(vmax_all, float(np.nanmax(grid)))
        vmax = vmax_all or 1.0
        n_rows_max = max((len(o) for _h, o, _g in panels), default=1)
        fig, axes = plt.subplots(1, max(1, len(panels)),
                                 figsize=(6 * max(1, len(panels)),
                                          0.6 * n_rows_max + 1.6),
                                 squeeze=False)
        for ax, (hand, order, grid) in zip(axes[0], panels):
            ax.imshow(grid, cmap="Reds", vmin=0, vmax=vmax)
            ax.set_xticks(range(4))
            ax.set_xticklabels(FINGERS, fontsize=8)
            ax.set_yticks(range(len(order)))
            ax.set_yticklabels(order, fontsize=8)
            ax.grid(False)
            if len(panels) > 1:
                ax.set_title(f"{hand} hand", fontsize=9)
            for i in range(len(order)):
                for j in range(4):
                    v = grid[i, j]
                    if np.isnan(v):
                        ax.text(j, i, "-", ha="center", va="center",
                                color="#94a3b8", fontsize=8)
                    else:
                        dark = vmax and v > 0.55 * vmax
                        ax.text(j, i, f"{v:.2f}", ha="center",
                                va="center",
                                color="white" if dark else "#0f172a",
                                fontsize=8)
        unit = ("x own reference press" if basis == "calibrated"
                else "raw counts")
        fig.suptitle(f"Median quiet-finger leak per chord ({unit})",
                     fontsize=10)
        _save(fig, "chord_leak_heatmap")
        plt.show()
        print("A dash is a target finger in that chord. The ring column")
        print("running hottest is the expected picture; one finger hot")
        print("regardless of chord says check the hardware before")
        print("reading it as neurology.")

    # ---- probe enslaving matrices -------------------------------------
    def _mat_has_data(mat):
        return any(v is not None for rrow in (mat or []) for v in rrow)

    def _game_matrices(d):
        """{hand: {"start": m, "end": m}} for one block summary.
        Bilateral blocks store per-hand matrices under
        enslaving_matrices; unilateral and older blocks store the
        flat legacy keys, filed here under the block's hand."""
        mats = d.get("enslaving_matrices")
        if isinstance(mats, dict) and mats:
            return {h: {"start": (v or {}).get("start"),
                        "end": (v or {}).get("end")}
                    for h, v in mats.items() if isinstance(v, dict)}
        hand = str(d.get("hand") or "right")
        return {hand: {"start": d.get("enslaving_matrix_start"),
                       "end": d.get("enslaving_matrix_end")}}

    def off_mean(mat):
        vals2 = [v for i, rrow in enumerate(mat or [])
                for j, v in enumerate(rrow)
                if i != j and v is not None]
        return (sum(vals2) / len(vals2)) if vals2 else None

    with_mat = []
    for g, d in stored.items():
        mats = {h: m for h, m in _game_matrices(d).items()
                if _mat_has_data(m.get("start"))
                or _mat_has_data(m.get("end"))}
        if mats:
            with_mat.append((g, mats))
    if with_mat:
        g, mats = with_mat[-1]
        mat_hands = [h for h in ("right", "left") if h in mats]
        mat_hands += [h for h in sorted(mats) if h not in mat_hands]
        fig, axes = plt.subplots(len(mat_hands), 2,
                                 figsize=(9, 4 * len(mat_hands)),
                                 squeeze=False)
        vals = [v for h in mat_hands
                for mat in (mats[h].get("start"), mats[h].get("end"))
                if mat for rrow in mat for v in rrow if v is not None]
        vmax = max(vals) if vals else 1.0
        for k, h in enumerate(mat_hands):
            for j, name in enumerate(("session start", "session end")):
                mat = mats[h].get("start" if j == 0 else "end")
                ax = axes[k][j]
                title = (name if len(mat_hands) == 1
                         else f"{h} hand, {name}")
                if _mat_has_data(mat):
                    _draw_matrix(ax, mat, vmax, title)
                else:
                    ax.axis("off")
                    ax.set_title(f"{title}: not recorded")
        fig.suptitle(f"Enslaving matrices, {g} "
                     f"(normalised leak per normalised press, "
                     f"own-reference units)", fontsize=10)
        _save(fig, "chord_enslaving_matrix")
        plt.show()

        for h in mat_hands:
            s_m = off_mean(mats[h].get("start"))
            e_m = off_mean(mats[h].get("end"))
            if s_m is None:
                continue
            lead = f"{h} hand: " if len(mat_hands) > 1 else ""
            print(f"{lead}mean off-diagonal at start {s_m:.1f}%",
                  end="")
            if e_m is not None:
                print(f", at end {e_m:.1f}%.", end=" ")
                if e_m > s_m:
                    print("Worse by the end of the session is the")
                    print("within-session fatigue picture (Danion "
                          "2000): consider a lower dose.")
                else:
                    print("Not worse by the end, so the dose did not")
                    print("visibly fatigue the hand.")
            else:
                print(".")
        print("Anchor: healthy hands leak under about 10% of the")
        print("instructed force at submaximal effort (Xu 2017 healthy")
        print("slope 0.087); stroke hands sit above and lose the clean")
        print("adjacency structure. Caveat: the cells above are a")
        print("doubly-normalised ratio (each finger's peak divided by")
        print("its OWN calibrated reference before the leak/press ratio")
        print("is taken), not raw percent of instructed force, so they")
        print("are comparable within a finger and over time but NOT")
        print("with Xu's raw-count figure directly (per-finger")
        print("references differ by up to 3.5x here).")
        print("What changed in 2026-09: play has no single-finger")
        print("probes any more, so newer blocks build these matrices")
        print("from the CHORDS of the first and last sub-block, cell")
        print("(i, j) averaged over every chord where i was active and")
        print("j quiet. The single-press reference under both numbers")
        print("is the quick-cal light-press capture, the same divisor")
        print("ER always used, so the units did not move; what did is")
        print("the condition: i presses in company, so under the")
        print("additive connection-matrix model a cell upper-bounds")
        print("the single-finger value. Start-to-end still reads as")
        print("the within-session fatigue contrast, now first versus")
        print("last round. Compare like with like across the change:")
        print("probe matrices to probe matrices, chord-conditioned to")
        print("chord-conditioned.")
    else:
        print("\nNo enslaving matrices in these block summaries.")
        print("The mode records them for the first and last chord")
        print("sub-blocks (older sessions: the probe sets); a demo")
        print("block only has the start set, and keyboard blocks log")
        print("no force at all, so their matrices come back empty.")

    # ---- data quality ---------------------------------------------------
    # Brief section C1: is this session's data trustworthy before any of
    # the numbers above get read as neurology.
    print("\ndata quality")
    over_force_total = sum(int(d.get("over_force_trials") or 0)
                           for d in stored.values())
    settle_meds = [d.get("median_settle_ms") for d in stored.values()
                  if d.get("median_settle_ms") is not None]
    classes_total: dict = {}
    for d in stored.values():
        for k, v in (d.get("outcome_classes") or {}).items():
            classes_total[k] = classes_total.get(k, 0) + int(v)
    if classes_total:
        n_all = sum(classes_total.values())
        print(f"outcome classes across {len(stored)} block(s), {n_all} "
              f"trials:")
        for k in sorted(classes_total, key=lambda k: -classes_total[k]):
            v = classes_total[k]
            print(f"   {k:12} {v:4}  ({v / n_all:.0%})")
    else:
        print("no outcome-class data in these block summaries (older")
        print("sessions, or trials.csv only).")
    print(f"over-force trials (press lighter): {over_force_total}")
    if settle_meds:
        print(f"median baseline-settle time: "
              f"{float(np.median(settle_meds)):.0f} ms, "
              f"{len(settle_meds)} block(s)")
    else:
        print("no baseline-settle time recorded.")

    # ---- across-session learning curve -----------------------------------
    # Brief section C5: does ER and the matrix off-diagonal actually trend
    # down over repeat sessions, the thesis-level question C4's
    # single-block matrices above cannot answer.
    #
    # A "session" here is one person on one day, the same convention
    # the progress chapter uses. stored is keyed per BLOCK, and the
    # Results screen's Retry makes two chords blocks in one sitting
    # routine, so counting blocks as sessions would print a training
    # trend over one sitting's retries. Blocks from different people
    # never share a curve or a trend either: pooling participants on
    # one line makes the between-person difference read as learning.
    er_hands_avail = [h for h in ("right", "left")
                      if any(h in (d.get("per_hand") or {})
                            for d in stored.values())]
    who_day = {}
    if trials is not None and not trials.empty and "game" in trials.columns:
        for g, sub in trials.groupby("game", sort=False):
            day = (str(sub["session"].iloc[0]).split()[0]
                   if "session" in sub.columns else "")
            who = (str(sub["participant"].iloc[0])
                   if "participant" in sub.columns else "")
            who_day[g] = (who, day)

    def _block_who_day(g):
        if g in who_day:
            return who_day[g]
        # A block with no readable trial rows falls back to the folder
        # naming convention, sessions/<day>/<who>_<HHMMSS>_<mode>.
        day, _, name = str(g).rpartition("/")
        m = re.match(r"^(.*)_\d{6}", name)
        return (m.group(1) if m else name, day)

    sittings: dict = {}
    for g, d in stored.items():
        sittings.setdefault(_block_who_day(g), []).append(d)
    per_person: dict = {}
    for (who, day), blocks in sorted(sittings.items(),
                                     key=lambda kv: (kv[0][0], kv[0][1])):
        per_person.setdefault(who, []).append((day, blocks))
    n_sit_max = max((len(v) for v in per_person.values()), default=0)

    def _sit_er(blocks, hand):
        """One sitting's ER for one hand: repeat blocks in the sitting
        averaged into one point, so a Retry cannot mint a trend."""
        ers = [(b.get("per_hand") or {}).get(hand, {}).get("median_er")
               for b in blocks]
        ers = [float(v) for v in ers if v is not None]
        return float(np.mean(ers)) if ers else None

    if n_sit_max >= 2 and er_hands_avail:
        if any(len(blocks) > 1 for sits in per_person.values()
               for _day, blocks in sits):
            print("\nRepeat blocks inside one sitting are averaged into")
            print("one session point: a Retry is practice, not a new")
            print("session.")
        fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
        for who, sits in per_person.items():
            for hand in er_hands_avail:
                er_pts, off_pts = [], []
                for i, (_day, blocks) in enumerate(sits, start=1):
                    er = _sit_er(blocks, hand)
                    if er is not None:
                        er_pts.append((i, er))
                    offs = []
                    for b in blocks:
                        mats = _game_matrices(b).get(hand) or {}
                        off = off_mean(mats.get("end") or mats.get("start"))
                        if off is not None:
                            offs.append(off)
                    if offs:
                        off_pts.append((i, float(np.mean(offs))))
                tag = (f"{hand} hand" if len(per_person) == 1
                       else f"{who}, {hand}")
                if er_pts:
                    xs, ys = zip(*er_pts)
                    axes[0].plot(xs, ys, "o-", label=tag)
                if off_pts:
                    xs2, ys2 = zip(*off_pts)
                    axes[1].plot(xs2, ys2, "o-", label=tag)
        axes[0].axhspan(HEALTHY_ER_BAND[0], HEALTHY_ER_BAND[1],
                        color="#16a34a", alpha=.12,
                        label="healthy 8-15%")
        axes[0].set_title("Median ER across sessions (one person on one "
                          "day)", fontsize=10)
        axes[0].set_xlabel("session # within participant")
        axes[0].set_ylabel("median ER")
        axes[0].legend(frameon=False, fontsize=8)
        axes[1].set_title("Matrix off-diagonal leak across sessions",
                          fontsize=10)
        axes[1].set_xlabel("session # within participant")
        axes[1].set_ylabel("mean off-diagonal (%)")
        axes[1].legend(frameon=False, fontsize=8)
        _save(fig, "chord_learning_curve")
        plt.show()

        # The trend is fitted per participant, never across two
        # people's points, so it cannot be printed over a mix.
        for who, sits in per_person.items():
            for hand in er_hands_avail:
                pts = []
                for i, (_day, blocks) in enumerate(sits, start=1):
                    er = _sit_er(blocks, hand)
                    if er is not None:
                        pts.append((i, er))
                if not pts:
                    continue
                label = (f"{hand} hand" if len(per_person) == 1
                         else f"{who} {hand} hand")
                if len(pts) >= 3:
                    xs, ys = zip(*pts)
                    slope, intercept = np.polyfit(xs, ys, 1)
                    resid = np.array(ys) - (slope * np.array(xs)
                                            + intercept)
                    se = float(np.std(resid, ddof=2)) if len(xs) > 2 else 0.0
                    print(f"{label} ER trend: {slope:+.4f} per session "
                          f"(residual SE {se:.3f}, {len(xs)} sessions).")
                    if slope < 0 and abs(slope) > 2 * se:
                        print("Negative and outside the noise band: the")
                        print("direction C5 defines as improvement. Confirm")
                        print("against the matrix slope, not this alone.")
                else:
                    print(f"{label}: fewer than 3 sessions with an ER "
                          f"value, too early for a trend.")
    elif n_sit_max < 2:
        print("\nOnly one session with chords in this selection: the")
        print("across-session ER and matrix-leak learning curves (brief")
        print("section C5) need repeat sessions to appear.")
        if len(stored) >= 2:
            print("Two or more blocks in one sitting are still one")
            print("session: a Retry is practice, not a new day.")
    else:
        print("\nNo per_hand median ER stored on these blocks (older")
        print("sessions predate that field): the across-session ER and")
        print("matrix-leak learning curves (brief section C5) cannot be")
        print("drawn from this selection.")

    # ---- timing and fatigue ------------------------------------------
    # Brief section C6: the span distribution against the W ladder, and
    # how hit rate and ER moved sub-block by sub-block within a session.
    all_block_trials = [r for d in stored.values()
                        for r in (d.get("trials") or [])]
    # Scope-pure: cross-hand trials (scope "cross") carry a two-hand
    # span that measures coupling, not within-hand synchrony, so they
    # stay out of this histogram and out of the trajectory below.
    spans = [r["span_ms"] for r in all_block_trials
            if r.get("kind") == "chord"
            and r.get("scope", "within") != "cross"
            and r.get("span_ms") is not None]
    if spans:
        fig, ax = plt.subplots(figsize=(8, 3.2))
        ax.hist(spans, bins=30, color="#0ea5e9", alpha=.8)
        for w in sorted({r["w_ms"] for r in all_block_trials
                         if r.get("w_ms")}, reverse=True):
            ax.axvline(w, color="#b45309", ls="--", lw=1,
                      label=f"W={w:g}ms")
        ax.set_xlabel("chord span (ms)")
        ax.set_ylabel("trials")
        ax.set_title("Chord span distribution against the synchrony "
                     "windows", fontsize=10)
        ax.legend(frameon=False, fontsize=8)
        _save(fig, "chord_span_histogram")
        plt.show()
    else:
        print("\nNo per-trial span data in these block summaries.")

    if with_mat or all_block_trials:
        # Sub-block trajectory from the most recent session that has
        # one: hit rate and ER should not be sliding down within a
        # single sitting (Danion 2000/2001 fatigue).
        last_g, last_d = list(stored.items())[-1] if stored else (None, None)
        sub_trials = []
        if last_d:
            # Within-hand sub-blocks only: cross sub-blocks are
            # legitimately harder (their own ladder), so mixing them
            # in here would fake a fatigue dip at sub-blocks 3 and 5.
            sub_trials = [r for r in (last_d.get("trials") or [])
                         if r.get("subblock") is not None
                         and r.get("scope", "within") != "cross"]
        if sub_trials:
            by_sub: dict = {}
            for r in sub_trials:
                s = by_sub.setdefault(r["subblock"], {"n": 0, "hits": 0,
                                                       "ers": []})
                s["n"] += 1
                s["hits"] += 1 if r.get("class") == "hit" else 0
                if r.get("er") is not None:
                    s["ers"].append(r["er"])
            subs = sorted(by_sub)
            hit_rates = [by_sub[s]["hits"] / by_sub[s]["n"]
                        for s in subs]
            med_ers = [float(np.median(by_sub[s]["ers"]))
                      if by_sub[s]["ers"] else np.nan for s in subs]
            fig, ax1 = plt.subplots(figsize=(8, 3.2))
            ax1.bar(subs, hit_rates, color="#dc2626", alpha=.7,
                   label="hit rate")
            ax1.set_xlabel("sub-block")
            ax1.set_ylabel("hit rate")
            ax1.set_ylim(0, 1.05)
            ax2 = ax1.twinx()
            ax2.plot(subs, med_ers, "o-", color="#0f172a",
                    label="median ER")
            ax2.set_ylabel("median ER")
            fig.suptitle(f"Within-session trajectory, {last_g}",
                        fontsize=10)
            fig.legend(frameon=False, fontsize=8, loc="upper right")
            _save(fig, "chord_subblock_trajectory")
            plt.show()
        else:
            print("\nNo sub-block breakdown in the most recent block "
                 "summary.")

    # ---- cross-hand (bimanual) chords --------------------------------
    # A chord spanning the hands measures bimanual coordination, not
    # individuation: the two hands yoke into one functional unit
    # (Kelso 1979; Swinnen 2002), mirror-symmetric patterns are the
    # stable default (Kelso 1984; Mechsner 2001), and non-mirror
    # patterns demand suppression of that default coupling, disturbed
    # after stroke (Murase 2004). The mode logs these trials under
    # their own scope with per-hand ER, so nothing here pools with the
    # within-hand numbers above.
    print("\ncross-hand (bimanual) chords")
    cross_secs = {g: d.get("cross") for g, d in stored.items()
                  if isinstance(d.get("cross"), dict)
                  and int((d.get("cross") or {}).get("n_chords") or 0)}
    if not cross_secs:
        print("none in this selection. Bilateral sessions deal them in")
        print("the third and fifth sub-blocks; unilateral sessions have")
        print("none, and blocks recorded before the bimanual upgrade")
        print("predate the measure.")
    else:
        n_cross = sum(int(c.get("n_chords") or 0)
                      for c in cross_secs.values())
        print(f"{n_cross} cross-hand chords across "
              f"{len(cross_secs)} block(s)")

        def _pool(key):
            vals3 = [c.get(key) for c in cross_secs.values()
                     if c.get(key) is not None]
            return float(np.median(vals3)) if vals3 else None

        hr_m = _pool("hit_rate_mirror")
        hr_n = _pool("hit_rate_nonmirror")
        sp_m = _pool("median_span_mirror_ms")
        sp_n = _pool("median_span_nonmirror_ms")
        if hr_m is not None or hr_n is not None:
            fig, axes = plt.subplots(1, 2, figsize=(9, 3.2))
            labels = ["mirror", "non-mirror"]
            axes[0].bar(labels, [hr_m or 0, hr_n or 0],
                        color=["#16a34a", "#b45309"], alpha=.8)
            axes[0].set_ylim(0, 1.05)
            axes[0].set_ylabel("hit rate")
            axes[0].set_title("Cross-hand hit rate: the symmetry "
                              "advantage", fontsize=10)
            axes[1].bar(labels, [sp_m or 0, sp_n or 0],
                        color=["#16a34a", "#b45309"], alpha=.8)
            axes[1].set_ylabel("median span (ms)")
            axes[1].set_title("Cross-hand span (first to last onset)",
                              fontsize=10)
            _save(fig, "chord_cross_mirror_cost")
            plt.show()
            if hr_m is not None and hr_n is not None:
                if hr_n <= hr_m:
                    print("mirror chords land more often than")
                    print("non-mirror, the symmetry advantage the tier")
                    print("ladder predicts (Kelso 1984; Mechsner 2001;")
                    print("Aramaki 2006).")
                else:
                    print("NON-MIRROR chords land more often than")
                    print("mirror here, against the healthy prediction:")
                    print("check the per-tier counts before reading it")
                    print("as neurology (low n per tier early on).")
        lag = _pool("median_lag_ms")
        if lag is not None:
            print(f"median between-hand lag {lag:.0f} ms "
                  "(first onset to first onset).")
        for g, c in cross_secs.items():
            leads = c.get("lead_hand_counts") or {}
            if any(leads.values()):
                lead_txt = ", ".join(f"{h} led {n2}x"
                                     for h, n2 in leads.items() if n2)
                print(f"   {g}: {lead_txt}")
            deficit = c.get("bilateral_deficit") or {}
            for h, per_f in deficit.items():
                cells4 = ", ".join(f"{f} {v:.2f}"
                                   for f, v in per_f.items())
                print(f"   {g}: bilateral deficit ratio {h} "
                      f"({cells4})")
        if any((c.get("bilateral_deficit") or {})
               for c in cross_secs.values()):
            print("Bilateral deficit: a finger's press in a mirror")
            print("chord per unit of the same finger's unimanual")
            print("reference press (the in-session probe before")
            print("2026-09, the quick-cal light press after); below")
            print("1.0 is the two-hand force drop of Li 2000 and")
            print("2001. Device-specific, order-of-magnitude context.")
        er_l = _pool("median_er_left")
        er_r = _pool("median_er_right")
        if er_l is not None or er_r is not None:
            l_txt = "n/a" if er_l is None else f"{er_l:.3f}"
            r_txt = "n/a" if er_r is None else f"{er_r:.3f}"
            print(f"per-hand ER on cross chords: left {l_txt}, "
                  f"right {r_txt}. Never pooled with within-hand ER:")
            print("the other hand moving is a coupling confound")
            print("(Li 2001).")

    ch.attrs["difficulty_table"] = tbl
    return ch


# ========================================== Cross-talk between fingers
# The one-figure answer to "show me the cross-talk": force on every
# finger that was NOT the target while a target was pressed, built from
# force_window_peaks across every mode, not just chords. Rows are the
# instructed finger, columns where the force leaked.

def crosstalk_cells(trials, calset=None):
    """Per-trial leak fractions for every single-target trial that
    logged a force spread. Chords and other multi-target rows are left
    out: with two instructed fingers down, leak cannot be pinned on
    one of them. Each lane is divided by its own calibrated reference
    press when the game has a usable one for all four lanes of that
    hand, raw counts otherwise; the basis rides every row.

    Restricted to clean single-target hits: a Miss or a trial where the
    wrong finger got pressed first is a response error, not enslaving.
    Pooling those in used to put a deliberate wrong press on a neighbour
    into "force on the quiet fingers", which is how a 103.6 percent
    off-diagonal cell -- more force on the quiet finger than on the
    target -- made it into the matrix (audit finding #101)."""
    if (trials is None or trials.empty
            or "force_window_peaks" not in trials.columns):
        return pd.DataFrame(), 0
    if "early_late" in trials.columns:
        trials = trials[trials["early_late"] != "Miss"]
    if "had_incorrect_press" in trials.columns:
        trials = trials[trials["had_incorrect_press"] != True]
    recs = []
    n_multi = 0
    for idx, r in trials.iterrows():
        peaks = parse_peaks(r.get("force_window_peaks"))
        if not peaks:
            continue
        # Syllable rows are keyed on syllable position, not on an
        # instructed finger (a level-1 child may tap ANY finger), so
        # a target/quiet split there reads free choice as leak.
        if r.get("mode") == "syllables":
            continue
        targets = row_targets(r)
        if len(targets) != 1:
            if len(targets) > 1:
                n_multi += 1
            continue
        tgt = targets[0]
        board = tgt // 4
        hand_lanes = [board * 4 + i for i in range(4)]
        hand_mode = r.get("hand_mode", "right")
        gaps = {l: (calset.lane_gap(r.get("game"), l, hand_mode)
                    if calset is not None else None)
                for l in hand_lanes}
        use_cal = all(gaps[l] for l in hand_lanes)

        def level(lane):
            v = max(0.0, float(peaks.get(lane, 0.0)))
            return v / gaps[lane] if use_cal else v

        p = level(tgt)
        if p <= 0:
            continue
        for l in hand_lanes:
            if l == tgt:
                continue
            recs.append({
                "row_id": idx,
                "session": r.get("session"),
                "who": r.get("participant"),
                "mode": r.get("mode"),
                "game": r.get("game"),
                "target": FINGERS[tgt % 4],
                "quiet": FINGERS[l % 4],
                "leak_frac": level(l) / p,
                "basis": "calibrated" if use_cal else "raw",
            })
    return pd.DataFrame(recs), n_multi


def crosstalk_matrix(cells):
    """(matrix, counts): mean leak as a percentage of the target press
    per (target, quiet) pair, NaN where never measured."""
    m = np.full((4, 4), np.nan)
    n = np.zeros((4, 4), dtype=int)
    if cells is None or cells.empty:
        return m, n
    for (t, q), g in cells.groupby(["target", "quiet"]):
        i, j = FINGERS.index(t), FINGERS.index(q)
        m[i, j] = 100.0 * float(g["leak_frac"].mean())
        n[i, j] = len(g)
    return m, n


def sec_crosstalk(trials, calset=None):
    """The 4x4 enslavement picture over every mode's single-target
    trials, and the same matrix per session so change over time is
    visible. Clean hits only: crosstalk_cells drops a Miss or a
    wrong-finger press before this runs, so a deliberate wrong press
    cannot read as spill onto the "quiet" finger (audit finding #101)."""
    cells, n_multi = crosstalk_cells(trials, calset)
    print("\n" + "=" * 62)
    print("CROSS-TALK BETWEEN FINGERS")
    print("=" * 62)
    if cells is None or cells.empty:
        _nothing("No trial here logged a per-finger force spread with a",
                 "single target finger, so there is no matrix to build.",
                 "Keyboard blocks log no force, and chords are read in",
                 "their own section because leak cannot be pinned on",
                 "one of two instructed fingers.")
        return None

    n_cal = int((cells["basis"] == "calibrated").sum())
    basis = "calibrated" if n_cal else "raw"
    used = cells[cells["basis"] == basis]
    n_dropped = cells.loc[cells["basis"] != basis, "row_id"].nunique()
    trials_used = used["row_id"].nunique()
    print(f"{trials_used} single-target trials across "
          f"{used['mode'].nunique()} mode(s); "
          f"{n_multi} multi-target trial(s) left out.")
    if basis == "calibrated":
        print("Each lane is divided by its own reference press first, so")
        print("the pads cancel out. Comparable within a finger and over")
        print("time, NOT with the published absolute enslavement figures")
        print("(see the individuation section for why).")
        if n_dropped:
            print(f"{n_dropped} trial(s) from uncalibrated games are "
                  f"left out rather than mixed in raw.")
    else:
        print("No usable calibration, so this is RAW counts: the same")
        print("absolute basis as the published figures (unimpaired 0.13,")
        print(f"stroke {ENSLAVEMENT_REF['stroke']}, Li via Lew), but pad "
              f"sensitivity is in every cell.")

    m, n = crosstalk_matrix(used)
    vmax = float(max(15.0, np.nanmax(m))) if np.isfinite(m).any() else 15.0
    fig, ax = plt.subplots(figsize=(5.6, 5))
    _draw_matrix(ax, [[None if np.isnan(v) else v for v in rrow]
                      for rrow in m], vmax,
                 "Force on each quiet finger, % of the target press")
    ax.set_ylabel("target finger (pressed)")
    ax.set_xlabel("quiet finger (should stay still)")
    _save(fig, "crosstalk_matrix")
    plt.show()

    off = m[~np.eye(4, dtype=bool)]
    off = off[np.isfinite(off)]
    if len(off):
        print(f"mean off-diagonal {off.mean():.1f}%. Neighbours running")
        print("hottest is the healthy adjacency structure (Zatsiorsky")
        print("2000); a flat or scrambled pattern after stroke is the")
        print("loss of individuation this device trains against.")
    counts = pd.DataFrame(n, index=FINGERS, columns=FINGERS)
    print("trials behind each cell:")
    _show(counts)

    sessions = list(dict.fromkeys(used["session"]))
    if len(sessions) >= 2:
        fig, axes = plt.subplots(1, len(sessions),
                                 figsize=(3.1 * len(sessions), 3.4))
        per_means = []
        for ax, sess in zip(np.atleast_1d(axes), sessions):
            sm, _sn = crosstalk_matrix(used[used["session"] == sess])
            _draw_matrix(ax, [[None if np.isnan(v) else v for v in rrow]
                              for rrow in sm], vmax, str(sess))
            soff = sm[~np.eye(4, dtype=bool)]
            soff = soff[np.isfinite(soff)]
            per_means.append(float(soff.mean()) if len(soff) else np.nan)
        fig.suptitle("Same matrix per session, shared colour scale",
                     fontsize=10)
        _save(fig, "crosstalk_by_session")
        plt.show()
        line = "   ".join(f"{s}: {v:.1f}%" if pd.notna(v) else f"{s}: -"
                          for s, v in zip(sessions, per_means))
        print("mean off-diagonal per session:   " + line)
        # First-to-last is a WITHIN-PERSON comparison: a session is one
        # person on one day, so a mixed selection's session list walks
        # across participants and "fell from first to last" would
        # compare one person's hand with another's.
        sess_who = (dict(zip(used["session"], used["who"]))
                    if "who" in used.columns else {})
        compared = False
        for who in dict.fromkeys(sess_who.get(s) for s in sessions):
            vals = [v for s, v in zip(sessions, per_means)
                    if sess_who.get(s) == who and pd.notna(v)]
            if len(vals) < 2:
                continue
            word = "fell" if vals[-1] < vals[0] else "did not fall"
            print(f"{who}: cross-talk {word} from their first to their "
                  f"last session here.")
            compared = True
        if compared:
            print("Improvement should show here before it shows anywhere")
            print("else.")
        elif len(set(sess_who.get(s) for s in sessions)) > 1:
            print("No participant has two sessions in this selection, so")
            print("there is no within-person change to read; the")
            print("per-session matrices above compare different people.")

    out = pd.DataFrame(m, index=FINGERS, columns=FINGERS)
    out.attrs["basis"] = basis
    out.attrs["cells"] = cells
    return out


# ======================================================== Syllable beats
# The children's phonological mode: tap the beats inside words. A
# different population from the rest of the file, and its packing is the
# densest: word, level, band, syllable count, pacing, per-tap times and
# peaks, and per-tap signed asynchronies all ride the stimulus column.

def syllable_frame(trials):
    """Syllable rows with the stimulus fully unpacked. taps become
    (lane, t_ms, peak) tuples with peak None in keyboard blocks; asyn
    is the per-tap signed asynchrony list, negative = early, paced
    trials only; model_lanes are the 1-indexed lanes the model phase
    buzzed, packed only by bilateral blocks; off is the sliding
    window's start slot in the desk row (map=off<k> in the stimulus,
    None for rows logged before the window), from which
    syllable_expected_lanes rebuilds exactly which lane carried which
    unit; row marks cross-hand trials, where the word runs off one
    hand onto the other and each position owns exactly one lane: a
    windowed trial whose window straddles the midline, a legacy
    map=row spanning word, or a legacy row inferred from nsyll >= 5;
    ease marks the reward layer's biased draws (ease=1, after two
    Misses the draw favours the child's best unit count, so these
    rows are held out of the accuracy-by-count chart) and streak is
    the consecutive-Great count after the word. Both keys are
    optional: rows from before the reward layer read ease False and
    streak None and every chart below still runs."""
    rows = mode_rows(trials, "syllables")
    if rows.empty or "stimulus" not in rows.columns:
        return pd.DataFrame()
    recs = []
    for idx, r in rows.iterrows():
        head, kv = stimulus_parts(r.get("stimulus"))
        if not head:
            continue

        def num(key, cast=float):
            try:
                return cast(kv.get(key))
            except (TypeError, ValueError):
                return None

        taps = []
        for t in str(kv.get("taps") or "").split(","):
            bits = t.split(":")
            if len(bits) < 2:
                continue
            try:
                lane = int(bits[0])
                t_ms = float(bits[1])
            except ValueError:
                continue
            peak = None
            if len(bits) >= 3 and bits[2]:
                try:
                    peak = float(bits[2])
                except ValueError:
                    peak = None
            taps.append((lane, t_ms, peak))
        asyn = []
        for a in str(kv.get("asyn") or "").split(","):
            try:
                asyn.append(float(a))
            except ValueError:
                pass
        model_lanes = []
        for t in str(kv.get("model") or "").split(","):
            t = t.strip()
            if t:
                try:
                    model_lanes.append(int(t))
                except ValueError:
                    pass
        nsyll_v = num("nsyll", int)
        map_s = str(kv.get("map") or "")
        off = None
        if map_s.startswith("off"):
            try:
                off = int(map_s[3:])
            except ValueError:
                off = None
        crosses = (off is not None and nsyll_v is not None
                   and str(r.get("hand_mode", "right")) == "both"
                   and off < 4 < off + nsyll_v)
        recs.append({
            "row_id": idx,
            "session": r.get("session"),
            "game": r.get("game"),
            "hand_mode": r.get("hand_mode", "right"),
            "model_lanes": model_lanes,
            "word": head,
            "level": num("lvl", int),
            "band": kv.get("band"),
            "nsyll": nsyll_v,
            "stress": num("stress", int),
            "off": off,
            "row": (map_s == "row" or crosses
                    or (off is None and nsyll_v is not None
                        and nsyll_v >= 5)),
            "paced": str(kv.get("paced")) == "1",
            "ioi_ms": num("ioi"),
            "err": str(kv.get("err") or ""),
            "correct": str(kv.get("err")) == "ok",
            "ease": str(kv.get("ease")) == "1",
            "streak": num("streak", int),
            "taps": taps,
            "asyn": asyn,
        })
    return pd.DataFrame(recs)


def syllable_desk_row(hand_mode):
    """The desk row in the CSV's 1-indexed lanes: every playing
    finger in physical left-to-right order. A right hand runs index
    to little (1-4); a single left hand plays lanes 1-4 of its own
    board, so the desk order is little to index (4-1); bilateral play
    is the left board (5-8, little leftmost) then the right (1-4).
    Mirrors SyllablesMode.desk_row."""
    if hand_mode == "both":
        return [8, 7, 6, 5, 1, 2, 3, 4]
    if hand_mode == "left":
        return [4, 3, 2, 1]
    return [1, 2, 3, 4]


def syllable_expected_lanes(r):
    """The 1-indexed lanes positions 0..n-1 sat on for one syllables
    trial, or None when the mapping cannot be rebuilt. Windowed rows
    (map=off<k>) are desk_row[k:k+n]; legacy spanning rows are the
    old centred row; legacy short rows let either hand carry a
    position, so no single lane list exists and None comes back."""
    n = r.get("nsyll")
    if not n:
        return None
    n = int(n)
    off = r.get("off")
    if off is not None:
        desk = syllable_desk_row(str(r.get("hand_mode", "right")))
        off = int(off)
        if off + n <= len(desk):
            return desk[off:off + n]
        return None
    if r.get("row"):
        left_n = min(4, (n + 1) // 2)
        right_n = min(4, max(0, n - left_n))
        return ([4 + left_n - k for k in range(left_n)]
                + [1 + k for k in range(right_n)])
    return None


def sec_syllables(trials, metas=None, calset=None):
    """Syllable beats: segmentation accuracy on the Liberman curve, beat
    synchronisation at the 2 Hz rate the dyslexia literature marks,
    stress marking once level 4 data exists, per-finger window
    participation (the sliding window's coverage promise), and the
    cross-hand windows split out from the one-hand windows they are
    not comparable to. The reward layer (2026-09) adds two panels
    kept deliberately separate: a within-block learning curve
    (rolling accuracy over word index, the chart the rewards exist
    to serve) and an engagement table (streaks, stickers, ease-in
    draws, skipped breaks), which is engagement bookkeeping and
    never a skill measure."""
    sy = syllable_frame(trials)
    print("\n" + "=" * 62)
    print("SYLLABLE BEATS")
    print("=" * 62)
    if sy is None or sy.empty:
        _nothing("No syllable blocks in this selection. This mode is",
                 "for children with reading difficulty, a different",
                 "population from the stroke modes, and none has been",
                 "recorded yet.")
        return None

    # ---- segmentation accuracy ----------------------------------------
    # nsyll= is the packed UNIT count (syllables at levels 1-4, onset-
    # rime pairs at 5, graphemes at 6 -- syllables.py's own docstring
    # calls it "target taps"), so grouping the whole frame by nsyll
    # would plot a level-6 word's 2-4 phoneme graphemes as though they
    # were 2-4 "syllables" against the Liberman anchor below, which is
    # defined on actual syllable tapping. Restricted to levels 1-4,
    # where nsyll really does count syllables.
    lvl14 = sy[sy["level"].isin([1, 2, 3, 4])]
    n_higher = len(sy) - len(lvl14)
    if n_higher:
        print(f"({n_higher} level 5/6 word(s) excluded from the syllable")
        print(" count chart below: nsyll there counts onset-rime pairs")
        print(" or graphemes, not syllables, so they are not comparable")
        print(" to the Liberman syllable-tapping anchor.)")
    seg_sy = lvl14[~lvl14["row"].astype(bool)]
    n_row14 = len(lvl14) - len(seg_sy)
    if n_row14:
        print(f"({n_row14} cross-hand word(s) also held out of the")
        print(" chart: a word running off one hand onto the other adds")
        print(" a spatial mapping demand Liberman's 1-3 syllable dowel")
        print(" tapping did not have. They are charted in the")
        print(" cross-hand section below.)")
    if "ease" in seg_sy.columns:
        n_ease = int(seg_sy["ease"].astype(bool).sum())
        if n_ease:
            print(f"({n_ease} ease-in word(s) held out of the chart:")
            print(" after two Misses the draw is biased to the child's")
            print(" best count, so pooling those words would flatter")
            print(" the curve. They still count everywhere else, band")
            print(" gate included.)")
            seg_sy = seg_sy[~seg_sy["ease"].astype(bool)]
    acc = (seg_sy[seg_sy["nsyll"].notna()]
           .groupby(["session", "nsyll"])["correct"]
           .agg(n="count", accuracy="mean").reset_index())
    acc["accuracy"] = acc["accuracy"].round(3)
    print(f"{len(sy)} words over {sy['session'].nunique()} session(s)")
    if not acc.empty:
        sessions = list(dict.fromkeys(acc["session"]))
        counts = sorted(acc["nsyll"].unique())
        fig, ax = plt.subplots(figsize=(8, 3.4))
        width = 0.8 / max(1, len(sessions))
        for k, sess in enumerate(sessions):
            sub = acc[acc["session"] == sess].set_index("nsyll")
            xs = [c + (k - (len(sessions) - 1) / 2) * width
                  for c in counts]
            ys = [sub["accuracy"].get(c, np.nan) for c in counts]
            ax.bar(xs, ys, width=width * .95, label=str(sess),
                   alpha=.85)
        ax.set_xticks(counts)
        ax.set_xticklabels([f"{int(c)}" for c in counts])
        ax.set_xlabel("syllables in the word")
        ax.set_ylabel("share fully correct")
        ax.set_ylim(0, 1.05)
        ax.set_title("Segmentation accuracy by syllable count "
                     "(one-hand windows; cross-hand words held out)")
        if len(sessions) > 1:
            ax.legend(frameon=False, fontsize=8)
        _save(fig, "syllable_accuracy")
        plt.show()
        print("ANCHOR   Liberman et al (1974): typically developing six")
        print("   year olds tap syllables at about 90 percent (and")
        print("   phonemes at about 70); only 46 percent of four year")
        print("   olds manage syllables at all. Accuracy easing down as")
        print("   the count rises is normal; high 1-2 syllable scores")
        print("   with chance-level 3-4 is counting, not segmenting.")
    by_level = (sy[sy["level"].notna()]
                .groupby("level")["correct"]
                .agg(n="count", accuracy="mean").round(3))
    if not by_level.empty:
        print("\naccuracy by level:")
        _show(by_level)
    errs = sy.loc[~sy["correct"], "err"].value_counts()
    if not errs.empty:
        print("error types: " + ", ".join(f"{k} x{v}"
                                          for k, v in errs.items()))

    # ---- within-block learning curve ----------------------------------
    # Does the child improve INSIDE a session? Rolling accuracy over
    # word index (window of 10, the band gate's own window), one line
    # per session, with a dashed marker wherever the band changed
    # (read from each trial's own band= value, so sessions from
    # before the reward layer plot identically). This is the chart
    # the reward layer exists to serve: in-session engagement is
    # only worth buying if this curve points up.
    curve_sessions = [(sess, g) for sess, g
                      in sy.groupby("session", sort=False)
                      if len(g) >= 10]
    if curve_sessions:
        fig, ax = plt.subplots(figsize=(8, 3.2))
        for sess, g in curve_sessions:
            roll = (g["correct"].astype(float)
                    .rolling(10, min_periods=10).mean())
            ax.plot(range(1, len(g) + 1), list(roll), lw=2,
                    label=str(sess))
            bands = list(g["band"])
            for i in range(1, len(bands)):
                if bands[i] != bands[i - 1]:
                    ax.axvline(i + 1, color="#94a3b8", lw=1, ls="--")
                    ax.text(i + 1, 1.03, str(bands[i]), fontsize=7,
                            ha="center", color="#64748b")
        ax.set_xlabel("word index in the block")
        ax.set_ylabel("rolling accuracy (last 10)")
        ax.set_ylim(0, 1.1)
        ax.set_title("Within-block learning curve "
                     "(dashed: band changes)")
        if len(curve_sessions) > 1:
            ax.legend(frameon=False, fontsize=8)
        _save(fig, "syllable_learning_curve")
        plt.show()
    else:
        print("\n(no session has the 10 or more words a rolling")
        print(" learning curve needs yet)")

    # ---- beat synchronisation -----------------------------------------
    paced = sy[sy["paced"] & sy["asyn"].map(len).astype(bool)]
    if paced.empty:
        print("\nBeat pacing starts at level 3 and every word here was")
        print("free-paced, so there is no asynchrony to draw yet.")
    else:
        pooled = [a for row in paced["asyn"] for a in row]
        pooled = pd.Series(pooled, dtype="float64")
        ioi = paced["ioi_ms"].dropna()
        ioi_ms = float(ioi.iloc[0]) if len(ioi) else 500.0
        window = 150.0
        for d in (stored_mode_stats(metas, "syllables") or {}).values():
            if d.get("on_beat_window_ms"):
                window = float(d["on_beat_window_ms"])
                break
        fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
        ax[0].axvspan(-window, window, color="#16a34a", alpha=.12,
                      label=f"on-beat window +-{window:.0f} ms")
        ax[0].hist(pooled, bins=_nbins(pooled), color="#14b8a6",
                   alpha=.85)
        ax[0].axvline(pooled.mean(), color="#0f172a", lw=2,
                      label=f"mean {pooled.mean():+.0f} ms")
        ax[0].axvline(0, color="#94a3b8", lw=1)
        ax[0].set_xlabel("signed asynchrony (ms), negative = early")
        ax[0].set_ylabel("taps")
        ax[0].set_title(f"Beat asynchrony at {ioi_ms:.0f} ms IOI")
        ax[0].legend(frameon=False, fontsize=8)
        per_sess = []
        for sess, g in paced.groupby("session"):
            vals = pd.Series([a for row in g["asyn"] for a in row],
                             dtype="float64")
            per_sess.append({"session": sess, "n_taps": len(vals),
                             "mean_ms": round(float(vals.mean()), 1),
                             "sd_ms": (round(float(vals.std()), 1)
                                       if len(vals) > 1 else np.nan)})
        ps = pd.DataFrame(per_sess)
        xs = range(1, len(ps) + 1)
        ax[1].plot(list(xs), ps["sd_ms"], "o-", lw=2, color="#14b8a6")
        ax[1].set_xticks(list(xs))
        ax[1].set_xticklabels(list(ps["session"]), rotation=20,
                              fontsize=8)
        ax[1].set_ylabel("asynchrony SD (ms)")
        ax[1].set_title("Tap variability across sessions")
        _save(fig, "syllable_asynchrony")
        plt.show()
        _show(ps)
        print("The SD at the 500 ms rate (2 Hz) is the headline number:")
        print("that is the rate where dyslexic children tap noisiest")
        print("(Thomson and Goswami 2008), and a shrinking SD is the")
        print("training signal. A negative mean (tapping slightly early)")
        print("is the normal pattern at every age (Repp and Su 2013).")
        print("CAVEAT: the mean is not corrected for unmeasured audio")
        print("output latency (no measurement is logged anywhere in this")
        print("build), so it carries an unknown constant positive bias --")
        print("a child synchronising perfectly to the delayed audible")
        print("tick reads as reactive rather than anticipatory. The SD is")
        print("unaffected and stays the reading to trust.")

    # ---- warm-up synchronisation probe ---------------------------------
    # The warm-up (10 free taps to the metronome before the first word)
    # runs and is logged every session, including levels 1-2 where every
    # word is free-paced and the block above prints "no asynchrony to
    # draw yet". Reading it from the stored block stats (block_summary.
    # syllables.warmup_asyn_mean_ms/sd) gives those sessions a
    # synchronisation reading too, which is what the probe exists for.
    stored_warmup = stored_mode_stats(metas, "syllables") or {}
    warm_rows = [{"session": g, "n_taps": d.get("warmup_taps"),
                  "mean_ms": d.get("warmup_asyn_mean_ms"),
                  "sd_ms": d.get("warmup_asyn_sd_ms")}
                 for g, d in stored_warmup.items()
                 if d.get("warmup_asyn_sd_ms") is not None]
    if warm_rows:
        wdf = pd.DataFrame(warm_rows)
        print("\nwarm-up synchronisation probe (10 free taps to the")
        print("metronome before the first word, logged every session")
        print("regardless of level):")
        _show(wdf)
        fig, ax = plt.subplots(figsize=(6, 3))
        xs = range(1, len(wdf) + 1)
        ax.plot(list(xs), wdf["sd_ms"], "o-", lw=2, color="#f97316")
        ax.set_xticks(list(xs))
        ax.set_xticklabels(list(wdf["session"]), rotation=20, fontsize=8)
        ax.set_ylabel("warm-up asynchrony SD (ms)")
        ax.set_title("Warm-up tap steadiness across sessions")
        _save(fig, "syllable_warmup_sd")
        plt.show()
    else:
        print("\nNo stored warm-up asynchrony yet (older session, or the")
        print("warm-up was skipped in Test Mode).")

    # ---- stress marking -----------------------------------------------
    lvl4 = sy[sy["level"] == 4]
    if lvl4.empty:
        print("\nStress marking is level 4 and no level 4 words are here")
        print("yet, so there is no stress ratio to score.")
    else:
        # Ratio is the stressed tap against the median of the OTHER
        # taps only (never a median that includes the tap being judged
        # -- for a 2-syllable word that median is always the louder of
        # the two taps, so a correctly accented word could never clear
        # any criterion above 1.0), on each finger's own calibrated
        # light-press gap (raw ADC counts are not comparable across
        # fingers). Mirrors finger_rehab/game/modes/syllables.py _score_stress.
        ratios = []
        n_unnorm = 0
        for _, r in lvl4.iterrows():
            s_idx = r["stress"]
            raw_taps = r["taps"]
            if (s_idx is None or len(raw_taps) < 2
                    or s_idx >= len(raw_taps)):
                continue
            if any(p is None for _, _, p in raw_taps):
                continue  # keyboard block, no force data to normalise
            game, hand_mode = r["game"], r["hand_mode"]
            norm, ok = [], True
            for lane, _t_ms, peak in raw_taps:
                g = (calset.lane_gap(game, lane - 1, hand_mode)
                     if calset is not None else None)
                if not g:
                    ok = False
                    break
                norm.append(peak / g)
            if not ok:
                n_unnorm += 1
                continue
            others = sorted(v for i, v in enumerate(norm) if i != s_idx)
            m = len(others)
            ref = (others[m // 2] if m % 2 else
                   (others[m // 2 - 1] + others[m // 2]) / 2.0)
            if ref <= 0:
                continue
            ratios.append(norm[int(s_idx)] / ref)
        n_wrong = int((lvl4["err"] == "wrong_stress").sum())
        print(f"\nstress marking, {len(lvl4)} level-4 words:")
        print(f"   wrong_stress errors: {n_wrong}")
        if n_unnorm:
            print(f"   {n_unnorm} words dropped: no calibration to "
                  f"normalise their fingers against")
        if ratios:
            rs = pd.Series(ratios, dtype="float64")
            print(f"   stressed-tap ratio median {rs.median():.2f} "
                  f"(criterion: at least 2.0 over the median of the "
                  f"other taps, own-finger calibrated)")
            fig, ax = plt.subplots(figsize=(6, 3))
            ax.hist(rs, bins=_nbins(rs), color="#14b8a6", alpha=.85)
            ax.axvline(2.0, color="#dc2626", lw=2, label="criterion 2.0")
            ax.axvline(1.0, color="#94a3b8", lw=1, label="no accent")
            ax.set_xlabel("stressed peak over median of the other taps "
                          "(own-finger calibrated)")
            ax.set_ylabel("words")
            ax.set_title("Stress production")
            ax.legend(frameon=False, fontsize=8)
            _save(fig, "syllable_stress")
            plt.show()
            print("   Ratios near 1 mean no accent was produced. This is")
            print("   the production mirror of the DeeDee perception")
            print("   deficit (Goswami 2010, 2013), not the same measure.")
        else:
            print("   no calibrated force peaks in the taps (keyboard")
            print("   block, or no calibration recorded), so the ratio")
            print("   cannot be computed; placement errors above are")
            print("   still real.")

    # ---- the read-across row (5-8 units, both hands) --------------------
    # Words of 5-8 units span both hands as one left-to-right row,
    # each position owning exactly one lane (left walk then right
    # walk, centred on the midline). Row trials add a spatial-mapping
    # demand short words do not have, so the regimes are split here
    # and never pooled anywhere above.
    row_sy = sy[sy["row"].astype(bool)]
    if not row_sy.empty:
        print("\nCROSS-HAND WINDOWS: the word's window runs off the")
        print("left hand onto the right (every 5-8 unit word, and any")
        print("short word whose offset straddles the midline). Not")
        print("comparable to one-hand windows: the crossing adds a")
        print("spatial mapping demand, so the regimes are reported")
        print("apart.")
        short_sy = sy[~sy["row"].astype(bool)]
        reg = pd.DataFrame([
            {"regime": "one-hand window", "n": len(short_sy),
             "accuracy": (round(float(short_sy["correct"].mean()), 3)
                          if len(short_sy) else np.nan)},
            {"regime": "cross-hand window", "n": len(row_sy),
             "accuracy": round(float(row_sy["correct"].mean()), 3)},
        ])
        _show(reg)
        by_units = (row_sy[row_sy["nsyll"].notna()]
                    .groupby("nsyll")["correct"]
                    .agg(n="count", accuracy="mean").round(3))
        if not by_units.empty:
            print("row accuracy by unit count:")
            _show(by_units)

        # The one NEW motor event the crossing introduces is the
        # hand hop: the position after the left hand's last finger
        # starts the right hand. The expected lanes rebuild from the
        # packed offset (or the legacy centred row), so each trial's
        # first wrong position can be located relative to the hop.
        at_hop, elsewhere = 0, 0
        for _, r in row_sy.iterrows():
            expect = syllable_expected_lanes(r)
            if not expect:
                continue
            left_n = sum(1 for l in expect if int(l) >= 5)
            lanes = [l for l, _t, _p in r["taps"]]
            mism = [k for k in range(min(len(expect), len(lanes)))
                    if lanes[k] != expect[k]]
            if not mism:
                continue
            if mism[0] == left_n:
                at_hop += 1
            else:
                elsewhere += 1
        if at_hop or elsewhere:
            print(f"first wrong position: at the hand transition "
                  f"{at_hop}, elsewhere {elsewhere}")
            if at_hop > elsewhere:
                print("errors cluster AT the left-to-right hand")
                print("transition: report that as a motor artefact of")
                print("the row, not a phonological finding.")
        else:
            print("no positional mismatches on row trials.")

        tap_h = {"left": 0, "right": 0}
        for taps_list in row_sy["taps"]:
            for lane1, _t, _p in taps_list:
                tap_h["left" if int(lane1) >= 5 else "right"] += 1
        print(f"row taps by hand: left {tap_h['left']}, "
              f"right {tap_h['right']} (the window prescribes the "
              f"hand,")
        print("so an imbalance here is material and offset mix, not")
        print("preference).")
        recs_r = []
        for _, r in row_sy[row_sy["paced"]].iterrows():
            for (lane1, _t, _p), a in zip(r["taps"], r["asyn"]):
                recs_r.append({"hand": ("left" if int(lane1) >= 5
                                        else "right"), "asyn_ms": a})
        if recs_r:
            hr = pd.DataFrame(recs_r)
            print("row paced asynchrony per hand:")
            _show(hr.groupby("hand")["asyn_ms"]
                  .agg(n_taps="count", mean_ms="mean", sd_ms="std")
                  .round(1))

    # ---- which hand carried the beats (one-hand windows) ----------------
    # Cross-hand trials sit in their own section above. What remains
    # splits by era: windowed rows (map=off) PRESCRIBE the hand, so
    # their split is delivered coverage; legacy rows let either hand
    # carry a position, so their split was child preference.
    both_sy = (sy[(sy["hand_mode"] == "both")
                  & ~sy["row"].astype(bool)]
               if "hand_mode" in sy.columns else sy.iloc[0:0])
    if not both_sy.empty:
        def _tap_side(lane1):
            # Taps and model lanes are 1-indexed; 5..8 is the left
            # board in bilateral play.
            return "left" if int(lane1) >= 5 else "right"

        def _tap_split(frame):
            n = {"left": 0, "right": 0}
            for taps in frame["taps"]:
                for lane1, _t, _p in taps:
                    n[_tap_side(lane1)] += 1
            return n

        win_b = both_sy[both_sy["off"].notna()]
        leg_b = both_sy[both_sy["off"].isna()]
        if not win_b.empty:
            tap_n = _tap_split(win_b)
            n_taps = sum(tap_n.values())
            print("\nBOTH HANDS (windowed): the window prescribes the")
            print("hand, so the split below is delivered coverage, the")
            print("thing the sliding window exists to guarantee, never")
            print("child preference.")
            if n_taps:
                print(f"   taps: right {tap_n['right']} "
                      f"({100 * tap_n['right'] / n_taps:.0f}%), left "
                      f"{tap_n['left']} "
                      f"({100 * tap_n['left'] / n_taps:.0f}%). A very")
                print("   lopsided split over a full block means the")
                print("   offset bags were starved (a software fault),")
                print("   not that the child chose a hand.")
        if not leg_b.empty:
            tap_n = _tap_split(leg_b)
            model_n = {"left": 0, "right": 0}
            for lanes in leg_b["model_lanes"]:
                for lane1 in lanes:
                    model_n[_tap_side(lane1)] += 1
            print("\nBOTH HANDS (legacy, before the window): either")
            print("hand's finger counted for its beat, so which hand")
            print("the child used is data, never an error.")
            n_taps = sum(tap_n.values())
            if n_taps:
                print(f"   taps: right {tap_n['right']} "
                      f"({100 * tap_n['right'] / n_taps:.0f}%), left "
                      f"{tap_n['left']} "
                      f"({100 * tap_n['left'] / n_taps:.0f}%). A strong")
                print("   one-hand habit is worth knowing, not")
                print("   correcting: the levels test which beat and")
                print("   when, never which hand.")
            n_model = sum(model_n.values())
            if n_model:
                print(f"   model buzzes: right {model_n['right']}, "
                      f"left {model_n['left']}. Legacy blocks dealt")
                print("   the buzzing hand off a shuffle bag, so a")
                print("   lopsided split here is a software fault, not")
                print("   a child's choice.")
        # Tap k pairs with beat k, so each signed asynchrony belongs
        # to the hand that tapped it; a per-hand SD gap on the same
        # beats is a genuine hand difference.
        recs2 = []
        for _, r in both_sy[both_sy["paced"]].iterrows():
            for (lane1, _t, _p), a in zip(r["taps"], r["asyn"]):
                recs2.append({"hand": _tap_side(lane1), "asyn_ms": a})
        if recs2:
            ha = pd.DataFrame(recs2)
            tblh = (ha.groupby("hand")["asyn_ms"]
                    .agg(n_taps="count", mean_ms="mean", sd_ms="std")
                    .round(1))
            print("   paced asynchrony per hand:")
            _show(tblh)

    # ---- per-finger window participation --------------------------------
    # The sliding window exists so every finger of the selected
    # hand(s) takes part; this table shows the spread. cued counts
    # how often the finger sat inside a word's window, tapped counts
    # its taps, and pos_correct is the share of its cued positions
    # whose tap landed on it. Rows with a rebuildable mapping only
    # (windowed rows and legacy spanning rows): legacy short rows had
    # no single lane per position.
    FING_NAMES = ("index", "middle", "ring", "little")

    def _lane_finger_label(lane1, hand_mode):
        if hand_mode == "both":
            if lane1 >= 5:
                return f"left {FING_NAMES[lane1 - 5]}"
            return f"right {FING_NAMES[lane1 - 1]}"
        return f"{hand_mode} {FING_NAMES[lane1 - 1]}"

    part = {}
    n_mapped = 0
    for _, r in sy.iterrows():
        expect = syllable_expected_lanes(r)
        if not expect:
            continue
        n_mapped += 1
        hm = str(r.get("hand_mode", "right"))
        lanes = [l for l, _t, _p in r["taps"]]
        for k, lane1 in enumerate(expect):
            d = part.setdefault((hm, int(lane1)),
                                {"cued": 0, "tapped": 0, "hit": 0})
            d["cued"] += 1
            if k < len(lanes) and int(lanes[k]) == int(lane1):
                d["hit"] += 1
        for lane1 in lanes:
            part.setdefault((hm, int(lane1)),
                            {"cued": 0, "tapped": 0,
                             "hit": 0})["tapped"] += 1
    if part:
        tbl = pd.DataFrame([
            {"finger": _lane_finger_label(lane1, hm), "lane": lane1,
             "cued": d["cued"], "tapped": d["tapped"],
             "pos_correct": (round(d["hit"] / d["cued"], 3)
                             if d["cued"] else np.nan)}
            for (hm, lane1), d in sorted(part.items())])
        print(f"\nPER-FINGER PARTICIPATION ({n_mapped} mappable "
              f"word(s)):")
        print("cued = sat inside a word's window. The window's job is")
        print("a cued column with no zero rows and a bounded spread;")
        print("edge slots sit inside fewer windows than middle slots,")
        print("so a mild taper is normal, a zero or a wild skew is")
        print("not. pos_correct is per-finger tap accuracy AT its")
        print("cued positions, real data across all fingers now that")
        print("the window visits them all.")
        _show(tbl)

    stored = stored_mode_stats(metas, "syllables")
    if stored:
        tbl = pd.DataFrame([{
            "game": g,
            "level": d.get("level"),
            "accuracy": d.get("accuracy"),
            "asyn_sd_ms": d.get("asyn_sd_ms"),
            "band_final": d.get("band_final"),
        } for g, d in stored.items()])
        print("\nwhat each block stored about itself (block_summary."
              "syllables):")
        _show(tbl)
        trace_rows = [{"session": g,
                       "band_trace": "->".join(d.get("band_trace") or [])}
                      for g, d in stored.items() if d.get("band_trace")]
        if trace_rows:
            print("\nband trace per session (within-block promotion off")
            print("the 8/10 -- 5/10 rule; not a difficulty measure across")
            print("sessions -- LEVEL only moves when the researcher edits")
            print("config, and above level 4 the band never moves inside")
            print("a block: level 6's bilateral stretch pool at band C")
            print("is a researcher start setting, so a trace there")
            print("stays flat):")
            _show(pd.DataFrame(trace_rows))

    # ---- engagement (the reward layer's own numbers) --------------------
    stored_all = stored_mode_stats(metas, "syllables") or {}
    eng_rows = []
    for g, d in stored_all.items():
        skips = d.get("skipped_rest_kinds") or {}
        eng_rows.append({
            "session": g,
            "max_streak": d.get("max_streak"),
            "stickers": d.get("stickers"),
            "n_ease_in": d.get("n_ease_in"),
            "skipped_breaks": skips.get("break", 0),
            "end_reason": d.get("end_reason"),
        })
    if eng_rows:
        print("\nENGAGEMENT (reward layer): stickers count completed")
        print("rounds (earned by showing up, not by scoring),")
        print("max_streak is the longest run of fully correct words,")
        print("n_ease_in counts the biased draws that broke a failure")
        print("spiral, and skipped_breaks with end_reason say how the")
        print("child moved through the session. None values mean the")
        print("session predates the reward layer.")
        _show(pd.DataFrame(eng_rows))
        if ("streak" in sy.columns and sy["streak"].notna().any()
                and sy["session"].nunique() > 1):
            run_ends = []
            for _sess, g in sy.groupby("session", sort=False):
                vals = [int(v) for v in g["streak"] if pd.notna(v)]
                run_ends.extend(
                    vals[i] for i in range(len(vals))
                    if vals[i] and (i + 1 == len(vals)
                                    or not vals[i + 1]))
            if run_ends:
                print("streak run lengths pooled across sessions (a")
                print("run's length is its final streak= value):")
                _show(pd.Series(run_ends, dtype="int64").value_counts()
                      .sort_index().rename("runs").to_frame())
        print("CAUTION: streak length correlates with band and word")
        print("length by construction (harder material breaks runs),")
        print("so it is an engagement number, never a skill measure,")
        print("and neither streaks nor stickers may be reported as an")
        print("active ingredient: the direct experiment in this game")
        print("family (Ronimus 2014) found reward systems sharpen the")
        print("session, not the outcome.")
    else:
        print("\n(no stored block summary for the engagement panel)")

    # ---- claim limits ---------------------------------------------------
    print("\nCLAIM LIMITS. Everything in this chapter is a WITHIN-TASK")
    print("learning curve: segmentation accuracy, beat synchronisation")
    print("and stress marking on this mode's OWN taps, rising or falling")
    print("across this mode's own sessions. It is not evidence that")
    print("reading has improved (that needs standardised pre/post reading")
    print("measures and a control group, per the mode's own docstring),")
    print("and this mode is not a diagnostic instrument: a session here")
    print("must never be used to label a child dyslexic. The sliding")
    print("window and two-hand play are scaffolding and engagement (plus")
    print("per-finger measurement coverage): neither has a demonstrated")
    print("additive PA effect, and cross-hand windows carry a")
    print("spatial-mapping demand that keeps them split from one-hand")
    print("windows in every comparison above.")
    return sy


# ============================================================== wrappers
# Three sections had a call that ran to more than three lines. Those
# lines live here instead, so every section cell below is a heading and
# a call.

CTX_NAMES = ("cat", "sel", "folders", "metas", "sessions", "trials",
             "unit", "calset")


def load_selection(pick, root=None) -> dict:
    """prepare(), and what came back, printed."""
    ctx = prepare(pick, root)
    sel, trials = ctx["sel"], ctx["trials"]
    print(f"loaded: {ctx['selection']}")
    if trials.empty:
        print("\nNothing to analyse here. Every cell below says why rather")
        print("than falling over, so you can keep running them.")
    else:
        print(f"{len(ctx['folders'])} game(s), {len(trials)} trials, "
              f"{sel['session'].nunique()} session(s)")
        print(f"force logged in {ctx['unit']}, "
              f"calibration: {ctx['calset'].status}")
    if sel.empty:
        print("nothing selected")
    else:
        _show(sel[["day", "time", "who", "mode", "hand", "trials"]]
              .rename_axis("id"))
    return ctx


def unpack(ctx):
    """The context under the names the section cells use, in CTX_NAMES
    order, then on_task, which the overview section fills in."""
    return tuple(ctx[n] for n in CTX_NAMES) + (0.0,)


def run_summary(ctx, trials, unit, calset, folders):
    """The headline numbers, from what the sections above stored."""
    ran = need(ctx, "on_task", "rt", "accuracy", "force", "ind", "rhythm",
               "onset", "flagged", "bilateral", "dose", "cues", "phase",
               "objective_one")
    summary = sec_summary(trials, unit, calset, ran["on_task"],
                          folders=folders, onset=ran["onset"],
                          accuracy=ran["accuracy"],
                          bilateral=ran["bilateral"], dose=ran["dose"],
                          cues=ran["cues"], phase=ran["phase"],
                          objective_one=ran["objective_one"])
    return keep(ctx, "summary", summary)


def run_exports(ctx, trials, calset):
    """The CSVs, from the summary and the individuation kept above."""
    ran = need(ctx, "summary", "ind")
    return write_exports(ran["summary"], trials, calset, ran["ind"])


# ---------------------------------------------------------------- run it
# One cell, one dropdown. Picking a save sets `pick`, which is what every
# section below analyses. Nothing in VS Code lets Python start a run, so
# the run itself is Run All from the notebook toolbar.

use_style()
check()

cat = build_catalogue()

# Run All re-runs this cell, so a plain `pick = "latest"` here would throw
# the chosen save away at the moment of running and analyse the newest
# game instead. A pick already made survives its own re-run.
#
# pick takes more than the dropdown offers, if you set it here by hand:
#
#   "latest"              the most recent game
#   "all"                 every game on disk
#   3  or  [0, 2, 5]      game ids, from catalogue()
#   "P01"                 every game for one person
#   "2026-08-05"          every game on one day
#   "adaptive"            every game in one mode
#   "2026-08-05  P01"     one session, a person on one day
#   ["Basil", "rhythm"]   filters stack, so this narrows twice
pick = globals().get("pick", "latest")   # what every cell below analyses

if not HAVE_WIDGETS:
    # No dropdown, so print the same list it would have offered and say
    # how to choose without it. Everything below reads `pick` and does
    # not care where it came from.
    print("\nipywidgets is not installed, so there is no dropdown. Here is")
    print("what is on disk instead. Put one of the values from the left")
    print("column into pick, in this cell, then run the notebook from the")
    print("top:")
    print('    pick = "P01_120000_adaptive"')
    catalogue()
    print("\nvalues pick accepts, one game then one session then one person:")
    for _label, _value in menu_options(cat):
        if _value is not None:
            print(f"   {str(_value):28} {_label.strip()}")
    print(f"\npick is {pick!r} right now.")
else:
    options = menu_options(cat)
    start = pick if any(v == pick for _, v in options) else None
    note = W.HTML("<span style='color:#64748b'>newest game until you "
                  "choose</span>" if start is None else
                  f"<span style='color:#16a34a'>selected {pick!r}</span>")
    menu = W.Dropdown(options=options, value=start,
                      description="Analyse:", layout=W.Layout(width="660px"),
                      style={"description_width": "70px"})
    rescan = W.Button(description="Rescan", icon="refresh",
                      tooltip="look for recordings made since this cell ran",
                      layout=W.Layout(width="130px"))

    def chosen(change):
        global pick
        if change["new"] is None:            # a heading row, not a save
            note.value = ("<span style='color:#b45309'>that row is a "
                          "heading, choose a save under it</span>")
            return
        pick = change["new"]
        # Everything below still shows the PREVIOUS selection until the
        # cells are re-run. Say so plainly rather than let the header name
        # one save while the tables show another.
        stale = (" Everything below is still the previous save until you "
                 "Run All.")
        note.value = (f"<span style='color:#16a34a'>selected {pick!r}.</span>"
                      f"<span style='color:#b45309'>{stale}</span>")

    def rescan_saves(_):
        """Look for new recordings without running anything.

        A block recorded while this notebook is open is invisible until
        the catalogue is rebuilt, and nothing on screen says the list is
        stale. The observer comes off while the options are swapped, so
        refreshing the menu does not read as a fresh choice and does not
        mark the results below stale on its own.
        """
        global cat
        cat = build_catalogue()
        keeping = menu.value
        menu.unobserve(chosen, names="value")
        menu.options = menu_options(cat)
        menu.value = (keeping if any(v == keeping for _, v in menu.options)
                      else None)
        menu.observe(chosen, names="value")
        gone = ("" if menu.value is not None else
                "<span style='color:#b45309'> the save that was selected is "
                "no longer on disk, choose another.</span>")
        note.value = (f"<span style='color:#16a34a'>rescanned, {len(cat)} "
                      f"game(s) on disk.</span>{gone}")

    menu.observe(chosen, names="value")
    rescan.on_click(rescan_saves)
    display(W.VBox([W.HBox([menu, rescan]), note]))

print("\nPick a save, then press Run All in the toolbar to fill in "
      "everything below.")

# ------------------------------------------------- references
# The sources behind the modes, the analysis and the thesis
# anchors. Verified against publisher and index pages before being
# listed; anything that could not be fully confirmed keeps only its
# confirmed parts and a visible [details to confirm] marker rather
# than a guessed detail. An invented reference found by a marker
# would cost more than any gap.
REFERENCES = [
    ('Reaction Time', [
        'Basner, M., and Dinges, D.F. (2011). Maximizing sensitivity of the psychomotor vigilance test (PVT) to sleep loss. Sleep, 34(5), 581-591.',
        'Basner, M., Mollicone, D., and Dinges, D.F. (2011). Validity and sensitivity of a brief psychomotor vigilance test (PVT-B) to total and partial sleep deprivation. Acta Astronautica, 69(11-12), 949-959.',
        'Dean, P.J.A., Seiss, E., and Sterr, A. (2012). Motor planning in chronic upper-limb hemiparesis: evidence from movement-related potentials. PLoS ONE, 7(10), e44558.',
        'Der, G., and Deary, I.J. (2006). Age and sex differences in reaction time in adulthood: results from the United Kingdom Health and Lifestyle Survey. Psychology and Aging, 21(1), 62-73.',
        'Dinges, D.F., and Powell, J.W. (1985). Microcomputer analyses of performance on a portable, simple visual RT task during sustained operations. Behavior Research Methods, Instruments, and Computers, 17(6), 652-655.',
        'Hick, W.E. (1952). On the rate of gain of information. Quarterly Journal of Experimental Psychology, 4(1), 11-26.',
        'Hyman, R. (1953). Stimulus information as a determinant of reaction time. Journal of Experimental Psychology, 45(3), 188-196.',
        'Luce, R.D. (1986). Response Times: Their Role in Inferring Elementary Mental Organization. Oxford University Press, New York.',
        'Niemi, P., and Naatanen, R. (1981). Foreperiod and simple reaction time. Psychological Bulletin, 89(1), 133-162.',
        'Owen, A.M., Hampshire, A., Grahn, J.A., Stenton, R., Dajani, S., Burns, A.S., Howard, R.J., and Ballard, C.G. (2010). Putting brain training to the test. Nature, 465(7299), 775-778.',
        'Ratcliff, R. (1993). Methods for dealing with reaction time outliers. Psychological Bulletin, 114(3), 510-532.',
        'Whelan, R. (2008). Effective analysis of reaction time data. The Psychological Record, 58(3), 475-482.',
        'Woods, D.L., Wyma, J.M., Yund, E.W., Herron, T.J., and Reed, B. (2015). Factors influencing the latency of simple reaction time. Frontiers in Human Neuroscience, 9, 131.',
    ]),
    ('Motor Sequence Learning', [
        'Boyd, L.A., and Winstein, C.J. (2003). Impact of explicit information on implicit motor-sequence learning following middle cerebral artery stroke. Physical Therapy, 83(11), 976-989.',
        'Boyd, L.A., and Winstein, C.J. (2004). Providing explicit information disrupts implicit motor learning after basal ganglia stroke. Learning and Memory, 11(4), 388-396.',
        'Destrebecqz, A., and Cleeremans, A. (2001). Can sequence learning be implicit? New evidence with the process dissociation procedure. Psychonomic Bulletin and Review, 8(2), 343-350.',
        'Kal, E., Winters, M., van der Kamp, J., Houdijk, H., Groet, E., van Bennekom, C., and Scherder, E. (2016). Is implicit motor learning preserved after stroke? A systematic review with meta-analysis. PLoS ONE, 11(12), e0166376.',
        'Nemeth, D., Janacsek, K., Londe, Z., Ullman, M.T., Howard, D.V., and Howard, J.H. Jr (2010). Sleep has no critical role in implicit motor sequence learning in young and old adults. Experimental Brain Research, 201(2), 351-358.',
        'Nissen, M.J., and Bullemer, P. (1987). Attentional requirements of learning: evidence from performance measures. Cognitive Psychology, 19(1), 1-32.',
        'Pan, S.C., and Rickard, T.C. (2015). Sleep and motor learning: is there room for consolidation? Psychological Bulletin, 141(4), 812-834.',
        'Reed, J., and Johnson, P. (1994). Assessing implicit learning with indirect tests: determining what is learned about sequence structure. Journal of Experimental Psychology: Learning, Memory, and Cognition, 20(3), 585-594.',
        'Robertson, E.M., Pascual-Leone, A., and Press, D.Z. (2004). Awareness modifies the skill-learning benefits of sleep. Current Biology, 14(3), 208-212.',
        'Romano, J.C., Howard, J.H. Jr, and Howard, D.V. (2010). One-year retention of general and sequence-specific skills in a probabilistic, serial reaction time task. Memory, 18(4), 427-441.',
        'Savion-Lemieux, T., and Penhune, V.B. (2005). The effects of practice and delay on motor skill learning and retention. Experimental Brain Research, 161(4), 423-431.',
        'Siengsukon, C.F., and Boyd, L.A. (2008). Sleep enhances implicit motor skill learning in individuals poststroke. Topics in Stroke Rehabilitation, 15(1), 1-12.',
        'Simmons, A.L., and Duke, R.A. (2006). Effects of sleep on performance of a keyboard melody. Journal of Research in Music Education, 54(3), 257-269.',
        'Trofimova, O., Mottaz, A., Allaman, L., Chauvigne, L.A.S., and Guggisberg, A.G. (2020). The "implicit" serial reaction time task induces rapid and temporary adaptation rather than implicit motor learning. Neurobiology of Learning and Memory, 175, 107297.',
        'Walker, M.P., Brakefield, T., Morgan, A., Hobson, J.A., and Stickgold, R. (2002). Practice with sleep makes perfect: sleep-dependent motor skill learning. Neuron, 35(1), 205-211.',
        'Willingham, D.B., and Dumas, J.A. (1997). Long-term retention of a motor skill: implicit sequence knowledge is not retained after a one-year delay. Psychological Research, 60(1-2), 113-119.',
    ]),
    ('Finger Individuation and Enslaving', [
        'Abolins, V., Stremoukhov, A., Walter, C., and Latash, M.L. (2020). On the origin of finger enslaving: control with referent coordinates and effects of visual feedback. Journal of Neurophysiology, 124(6), 1625-1636. [Cited throughout the project as "Cuadra and Latash 2021"; see notes.]',
        'Chiang, H., Slobounov, S.M., and Ray, W. (2004). Practice-related modulations of force enslaving and cortical activity as revealed by EEG. Clinical Neurophysiology, 115(5), 1033-1043.',
        'Danion, F., Latash, M.L., Li, Z.M., and Zatsiorsky, V.M. (2000). The effect of fatigue on multifinger co-ordination in force production tasks in humans. Journal of Physiology, 523(2), 523-532.',
        'Danion, F., Latash, M.L., Li, Z.M., and Zatsiorsky, V.M. (2001). The effect of a fatiguing exercise by the index finger on single- and multi-finger force production tasks. Experimental Brain Research, 138(3), 322-329.',
        'Hager-Ross, C., and Schieber, M.H. (2000). Quantifying the independence of human finger movements: comparisons of digits, hands, and movement frequencies. Journal of Neuroscience, 20(22), 8542-8550.',
        'Lang, C.E., and Schieber, M.H. (2003). Differential impairment of individuated finger movements in humans after damage to the motor cortex or the corticospinal tract. Journal of Neurophysiology, 90(2), 1160-1170.',
        'Lang, C.E., and Schieber, M.H. (2004). Reduced muscle selectivity during individuated finger movements in humans after damage to the motor cortex or corticospinal tract. Journal of Neurophysiology, 91(4), 1722-1734.',
        'Li, Z.M., Latash, M.L., and Zatsiorsky, V.M. (1998). Force sharing among fingers as a model of the redundancy problem. Experimental Brain Research, 119(3), 276-286.',
        'Xu, J., Ejaz, N., Hertler, B., Branscheidt, M., Widmer, M., Faria, A.V., Harran, M.D., Cortes, J.C., Kim, N., Celnik, P.A., Kitago, T., Luft, A.R., Krakauer, J.W., and Diedrichsen, J. (2017). Separable systems for recovery of finger strength and control after stroke. Journal of Neurophysiology, 118(2), 1151-1163.',
        'Zatsiorsky, V.M., Li, Z.M., and Latash, M.L. (2000). Enslaving effects in multi-finger force production. Experimental Brain Research, 131(2), 187-195.',
    ]),
    ('Phonological Awareness, Rhythm and Dyslexia', [
        'Anthony, J.L., and Francis, D.J. (2005). Development of phonological awareness. Current Directions in Psychological Science, 14(5), 255-259.',
        'Bhide, A., Power, A., and Goswami, U. (2013). A rhythmic musical intervention for poor readers: a comparison of efficacy with a letter-based intervention. Mind, Brain, and Education, 7(2), 113-123.',
        'Bradley, L., and Bryant, P.E. (1983). Categorizing sounds and learning to read: a causal connection. Nature, 301(5899), 419-421.',
        'de Jong, P.F., and Oude Vrielink, L. (2004). Rapid automatic naming: easy to measure, hard to improve (quickly). Annals of Dyslexia, 54(1), 65-88. [author name form to confirm]',
        'Descamps, M., Grossard, C., Pellerin, H., et al. (2025). Rhythm training improves word-reading in children with dyslexia. Scientific Reports, 15, 17631. [full author list to confirm]',
        'Drake, C., Jones, M.R., and Baruch, C. (2000). The development of rhythmic attending in auditory sequences: attunement, referent period, focal attending. Cognition, 77(3), 251-288.',
        'Dumont, E., Syurina, E.V., Feron, F.J.M., and van Hooren, S. (2017). Music interventions and child development: a critical review and further directions. Frontiers in Psychology, 8, 1694.',
        "Ehri, L.C., Nunes, S.R., Willows, D.M., Schuster, B.V., Yaghoub-Zadeh, Z., and Shanahan, T. (2001). Phonemic awareness instruction helps children learn to read: evidence from the National Reading Panel's meta-analysis. Reading Research Quarterly, 36(3), 250-287.",
        'Flaugnacco, E., Lopez, L., Terribili, C., Montico, M., Zoia, S., and Schon, D. (2015). Music training increases phonological awareness and reading skills in developmental dyslexia: a randomized control trial. PLoS ONE, 10(9), e0138715.',
        'Goswami, U. (2011). A temporal sampling framework for developmental dyslexia. Trends in Cognitive Sciences, 15(1), 3-10.',
        'Goswami, U., Gerson, D., and Astruc, L. (2010). Amplitude envelope perception, phonology and prosodic sensitivity in children with developmental dyslexia. Reading and Writing, 23, 995-1019.',
        'Liberman, I.Y., Shankweiler, D., Fischer, F.W., and Carter, B. (1974). Explicit syllable and phoneme segmentation in the young child. Journal of Experimental Child Psychology, 18(2), 201-212.',
        'McAuley, J.D., Jones, M.R., Holub, S., Johnston, H.M., and Miller, N.S. (2006). The time of our lives: life span development of timing and event tracking. Journal of Experimental Psychology: General, 135(3), 348-367. [volume/pages to confirm]',
        'Norton, E.S., and Wolf, M. (2012). Rapid automatized naming (RAN) and reading fluency: implications for understanding and treatment of reading disabilities. Annual Review of Psychology, 63, 427-452.',
        'Repp, B.H., and Su, Y.H. (2013). Sensorimotor synchronization: a review of recent research (2006-2012). Psychonomic Bulletin and Review, 20(3), 403-452.',
        'Stevens, E.A., Austin, C., Moore, C., Scammacca, N., Boucher, A.N., and Vaughn, S. (2021). Current state of the evidence: examining the effects of Orton-Gillingham reading interventions for students with or at risk for word-level reading disabilities. Exceptional Children, 87(4), 397-417.',
        'Thomson, J.M., and Goswami, U. (2008). Rhythmic processing in children with developmental dyslexia: auditory and motor rhythms link to reading and spelling. Journal of Physiology-Paris, 102(1-3), 120-129.',
        'Ziegler, J.C., and Goswami, U. (2005). Reading acquisition, developmental dyslexia, and skilled reading across languages: a psycholinguistic grain size theory. Psychological Bulletin, 131(1), 3-29.',
    ]),
    ('Motor Learning and Rehabilitation (challenge point, mirror therapy, FINGER robot, MusicGlove)', [
        'Altschuler, E.L., Wisdom, S.B., Stone, L., Foster, C., Galasko, D., Llewellyn, D.M., and Ramachandran, V.S. (1999). Rehabilitation of hemiparesis after stroke with a mirror. The Lancet, 353(9169), 2035-2036.',
        'Birkenmeier, R.L., Prager, E.M., and Lang, C.E. (2010). Translating animal doses of task-specific training to people with chronic stroke in 1-hour therapy sessions: a proof-of-concept study. Neurorehabilitation and Neural Repair, 24(7), 620-635.',
        'Guadagnoli, M.A., and Lee, T.D. (2004). Challenge point: a framework for conceptualizing the effects of various practice conditions in motor learning. Journal of Motor Behavior, 36(2), 212-224.',
        'Cauraugh, J.H., Lodha, N., Naik, S.K., and Summers, J.J. (2010). Bilateral movement training and stroke motor recovery progress: a structured review and meta-analysis. Human Movement Science, 29(5), 853-870. [publicly contested in a response letter; present both sides, not as settled]',
        'Ramachandran, V.S., and Rogers-Ramachandran, D. (1996). Synaesthesia in phantom limbs induced with mirrors. Proceedings of the Royal Society of London B, 263(1369), 377-386. [origin-of-idea citation only: mirror.py implements mirror-FREE bilateral synchronous training (Whitall/Cauraugh lineage below), not the mirror-visual-illusion protocol this paper describes]',
        'Whitall, J., McCombe Waller, S., Silver, K.H., and Macko, R.F. (2000). Repetitive bilateral arm training with rhythmic auditory cueing improves motor function in chronic hemiparetic stroke. Stroke, 31(10), 2390-2395.',
        'Rowe, J.B., Chan, V., Ingemanson, M.L., Cramer, S.C., Wolbrecht, E.T., and Reinkensmeyer, D.J. (2017). Robotic assistance for training finger movement using a Hebbian model: a randomized controlled trial. Neurorehabilitation and Neural Repair, 31(8), 769-780.',
        'Taheri, H., Rowe, J.B., Gardner, D., Chan, V., Gray, K., Bower, C., Reinkensmeyer, D.J., and Wolbrecht, E.T. (2014). Design and preliminary evaluation of the FINGER rehabilitation robot: controlling challenge and quantifying finger individuation during musical computer game play. Journal of NeuroEngineering and Rehabilitation, 11, 10.',
        'Thielbar, K.O., Lord, T.J., Fischer, H.C., Lazzaro, E.C., Barth, K.C., Stoykov, M.E., Triandafilou, K.M., and Kamper, D.G. (2014). Training finger individuation with a mechatronic-virtual reality system leads to improved fine motor control post-stroke. Journal of NeuroEngineering and Rehabilitation, 11, 171.',
        'Zondervan, D.K., Friedman, N., Chang, E., Zhao, X., Augsburger, R., Reinkensmeyer, D.J., and Cramer, S.C. (2016). Home-based hand rehabilitation after chronic stroke: randomized, controlled single-blind trial comparing the MusicGlove with a conventional exercise program. Journal of Rehabilitation Research and Development, 53(4), 457-472.',
    ]),
    ('Visuomotor Force Tracking (Force Pilot)', [
        'Archer, D.B., Kang, N., Misra, G., Marble, S., Patten, C., and Coombes, S.A. (2017). Visual feedback alters force control and functional activity in the visuomotor network after stroke. NeuroImage: Clinical, 17, 505-517.',
        "Davidson, S., Learman, K., Rosenfeldt, A.B., Zimmerman, E., and Alberts, J.L. (2026). Parkinson's disease impairs grip force release during a sinusoidal force tracking task. Experimental Brain Research, 244(4), 46.",
        'Kurillo, G., Gregoric, M., Goljar, N., and Bajd, T. (2005). Grip force tracking system for assessment and rehabilitation of hand function. Technology and Health Care, 13(3), 137-149.',
        'Lodha, N., Misra, G., Coombes, S.A., Christou, E.A., and Cauraugh, J.H. (2013). Increased force variability in chronic stroke: contributions of force modulation below 1 Hz. PLoS ONE, 8(12), e83468.',
        'Naik, S.K., Patten, C., Lodha, N., et al. (2011). Force control deficits in chronic stroke: grip formation and release phases. Experimental Brain Research, 211(1), 1-15. [full author list to confirm]',
        'Pennati, G.V., Plantin, J., Carment, L., et al. (2020). Recovery and prediction of dynamic precision grip force control after stroke. Stroke, 51(3), 944-951. [full author list to confirm]',
        'Taud, B., Lindenberg, R., Darkow, R., Wevers, J., Hofflin, D., Grittner, U., Meinzer, M., and Floel, A. (2021). Limited add-on effects of unilateral and bilateral transcranial direct current stimulation on visuo-motor grip force tracking task training outcome in chronic stroke: a randomized controlled trial. Frontiers in Neurology, 12. [article number to confirm]',
        'Wasaka, T., Ando, K., Nomura, M., Toshima, K., Tamaru, T., and Morita, Y. (2022). Visuomotor tracking task for enhancing activity in motor areas of stroke patients. Brain Sciences, 12(8), 1063.',
    ]),
    ('Precision Hold and Force Sense (Lighthouse)', [
        'Camacho-Villa, M.A., et al. (2025). Relationship between force steadiness and functionality in older adults: a systematic review with meta-analysis. Scandinavian Journal of Medicine and Science in Sports, 35(4), e70040. [full author list to confirm]',
        'Li, K., Evans, P.J., Seitz, W.H. Jr, and Li, Z.M. (2015). Carpal tunnel syndrome impairs sustained precision pinch performance. Clinical Neurophysiology, 126(1), 194-201.',
        'Lima, K.C.A., Borges, L.S., Hatanaka, E., Rolim, L.C., and de Freitas, P.B. (2017). Grip force control and hand dexterity are impaired in individuals with diabetic peripheral neuropathy. Neuroscience Letters, 659, 54-59.',
        'Marmon, A.R., Gould, J.R., and Enoka, R.M. (2011). Practicing a functional task improves steadiness with hand muscles in older adults. Medicine and Science in Sports and Exercise, 43(8). [pages to confirm]',
        'Peters, S., Page, M.J., Coppieters, M.W., Ross, M., and Johnston, V. (2016). Rehabilitation following carpal tunnel release. Cochrane Database of Systematic Reviews, CD004158.',
    ]),
    ('Tactile Perception and Sensory Retraining (Buzz Hunt)', [
        'Auld, M., et al. (2014). Determination of interventions for upper extremity tactile impairment in children with cerebral palsy: a systematic review. Developmental Medicine and Child Neurology. [author initials, volume and pages to confirm]',
        'Carey, L., Macdonell, R., and Matyas, T.A. (2011). SENSe: Study of the Effectiveness of Neurorehabilitation on Sensation: a randomized controlled trial. Neurorehabilitation and Neural Repair, 25(4), 304-313.',
        'Jerosch-Herold, C., Houghton, J., Miller, L., and Shepstone, L. (2016). Does sensory relearning improve tactile function after carpal tunnel decompression? A pragmatic, assessor-blinded, randomized clinical trial. Journal of Hand Surgery (European Volume). [volume and pages to confirm]',
        'Vikstrom, P., et al. (2017). Similar 2-point discrimination and stereognosia but better locognosia at long term with an independent home-based sensory reeducation program vs no reeducation after low-median nerve transection and repair. Journal of Hand Therapy. [full author list, volume and pages to confirm]',
        'Weber, M., Marshall, A., Timircan, R., McGlone, F., Watt, S.J., Onyekwelu, O., Booth, L., Jesudason, E., Lees, V., and Valyear, K.F. (2023). Touch localization after nerve repair in the hand: insights from a new measurement tool. Journal of Neurophysiology, 130(5), 1126-1141.',
        'Zeuner, K.E., et al. (2002). Sensory training for patients with focal hand dystonia. Annals of Neurology, 51. [full author list and pages to confirm]',
    ]),
    ('Explicit Visuospatial Span (Echo)', [
        'Berch, D.B., Krikorian, R., and Huha, E.M. (1998). The Corsi block-tapping task: methodological and theoretical considerations. Brain and Cognition, 38(3), 317-338.',
        'Brunetti, R., Del Gatto, C., and Delogu, F. (2014). eCorsi: implementation and testing of the Corsi block-tapping task for digital tablets. Frontiers in Psychology, 5, 939.',
        "Conway, A.R.A., Kane, M.J., Bunting, M.F., Hambrick, D.Z., Wilhelm, O., and Engle, R.W. (2005). Working memory span tasks: a methodological review and user's guide. Psychonomic Bulletin and Review, 12(5), 769-786.",
        'Corsi, P.M. (1972). Human memory and the medial temporal region of the brain. PhD thesis, McGill University.',
        'Couture, M., and Tremblay, S. (2007). Exploring the characteristics of the visuospatial Hebb repetition effect. Memory and Cognition, 35. [pages to confirm]',
        'Farrell Pagulayan, K., Busch, R.M., Medina, K.L., Bartok, J.A., and Krikorian, R. (2006). Developmental normative data for the Corsi Block-Tapping task. Journal of Clinical and Experimental Neuropsychology, 28(6), 1043-1052.',
        'Gendle, M.H., and Ransom, M.R. (2006). Use of the electronic game SIMON as a measure of working memory span in college age adults. Journal of Behavioral and Neuroscience Research, 4, 1-7.',
        'Gonthier, C. (2022). An easy way to improve scoring of memory span tasks: the edit distance. Behavior Research Methods. [volume and pages to confirm]',
        'Hebb, D.O. (1961). Distinctive features of learning in the higher animal. In Delafresnaye, J.F. (ed), Brain Mechanisms and Learning. Oxford University Press.',
        'Kessels, R.P.C., van Zandvoort, M.J.E., Postma, A., Kappelle, L.J., and de Haan, E.H.F. (2000). The Corsi Block-Tapping Task: standardization and normative data. Applied Neuropsychology, 7(4), 252-258.',
        'Shams, L., and Seitz, A.R. (2008). Benefits of multisensory learning. Trends in Cognitive Sciences, 12(11), 411-417.',
    ]),
    ('Measurement Methods', [
        'Goebl, W. (2001). Melody lead in piano performance: expressive device or artifact? Journal of the Acoustical Society of America, 110(1), 563-572.',
        'Plant, R.R., and Turner, G. (2009). Millisecond precision psychological research in a world of commodity computers: new hardware, new problems? Behavior Research Methods, 41(3), 598-614.',
        'Rasch, R.A. (1979). Synchronization in performed ensemble music. Acustica, 43(2), 121-131.',
        'Teasdale, N., Bard, C., Fleury, M., Young, D.E., and Proteau, L. (1993). Determining movement onsets from temporal series. Journal of Motor Behavior, 25(2), 97-106.',
        'Ulrich, R., and Giray, M. (1989). Time resolution of clocks: effects on reaction time measurement - good news for bad clocks. British Journal of Mathematical and Statistical Psychology, 42(1), 1-12.',
    ]),
    ('Project Theses (Curtin University, unpublished)', [
        'Lim, C. (2023). Development and Design of a Multimodal Finger Exercising Device. Unpublished final-year thesis, Mechatronic Engineering (BEng Hons) and Computer Science, Curtin University. Student 19754532. The 2023 foundation device.',
        'Lew, Y.X. (2024). Development of Software for a Multi-Sensory Finger Exercising Device for Enhancing Stroke Rehabilitation through Multi-Modal Learning. Unpublished final-year engineering thesis, Curtin University. Student 20158374.',
        'Palmer, C.J. (2024). Development and Design of a Multi-Modal, Neural Stimulating, Finger Exercising Device. Unpublished final-year engineering thesis, Curtin University. Student 20187925. The cue-comparison anchor: reaction time differed between an LED-only cue and all cues together.',
        'Saini, A. (2024). Development of a Multi-Sensory Finger Exercising Device for Enhancing Stroke Rehabilitation through Multi-Modal Learning. Unpublished final-year engineering thesis, Curtin University. Student 20162456.',
        'Demouche, I. (2025). Development of a Multi-Sensory Finger Exercising Device for Enhancing Stroke Rehabilitation through Multi-Modal-Learning. Unpublished final-year engineering thesis (Electrical role), Curtin University. Student 20574682. The force-measurement anchor: SingleTact sensor characterisation (crosstalk, noise, peaks, repeatability, baseline drift).',
        'Dixon, S.J. (2025). Development of a Multi-Sensory Finger Exercising Device for Enhancing Stroke Rehabilitation through Multi-Modal Learning. Unpublished final-year engineering thesis (Mechanical role), Curtin University. Student 20569235.',
        'Lee, H.E.C. (2025). Development of a Multi-Sensory Finger Exercising Device for Enhancing Stroke Rehabilitation through Multi-Modal Learning. Unpublished BEng Mechatronic Engineering thesis (Button role), School of Civil and Mechanical Engineering, Curtin University, Semester 2 2025. Student 20720126.',
        "Nakayama, S. (2025). Development of a multi-sensory finger exercising device for stroke rehabilitation through multi modal learning. Unpublished final-year engineering thesis (Software role), Curtin University. Student 20722630. The software thesis this project's codebase follows on from.",
        'Aiden [surname unknown] (2026). Arduino firmware for the current two-board rehabilitation device (one hand of 4 force sensors per board, device-side sample timing). Referenced in main.py, finger_rehab/hardware/multi_serial.py, finger_rehab/hardware/eeg.py and arduino/firmware_on_device/. [details to confirm: surname, and whether this is a thesis or team-member firmware contribution]',
    ]),
]


def print_references():
    """The reference list, grouped, wrapped so nothing runs off screen."""
    import textwrap
    for group, entries in REFERENCES:
        print(group.upper())
        print("-" * len(group))
        for entry in entries:
            lines = textwrap.wrap(entry, width=100,
                                  subsequent_indent="      ")
            for ln in lines:
                print(ln)
        print()
    n = sum(len(e) for _g, e in REFERENCES)
    todo = sum(1 for _g, es in REFERENCES for e in es
               if "details to confirm" in e)
    print(f"{n} references in {len(REFERENCES)} groups."
          + (f" {todo} marked [details to confirm]." if todo else ""))


def parse_waveform_params_cell(cell):
    """One trial's waveform_params cell back into a dict.

    The game packs parameters as "key=value;key=value" with sorted
    keys (finger_rehab.data.logger.pack_waveform_params); this is the
    matching reader, values as float where they parse. The notebook
    carries its own copy because it travels without the game package.
    """
    out = {}
    for part in str(cell or "").split(";"):
        if "=" not in part:
            continue
        key, _, text = part.partition("=")
        try:
            out[key] = float(text)
        except ValueError:
            out[key] = text
    return out


def parse_segment_times_cell(cell):
    """One trial's segment_times cell back into (name, start, end)
    tuples, start/end in raw-stream t_perf seconds. Malformed entries
    are dropped so a truncated row costs one trial, not the section."""
    out = []
    for part in str(cell or "").split(";"):
        bits = part.split(":")
        if len(bits) != 3:
            continue
        try:
            out.append((bits[0], float(bits[1]), float(bits[2])))
        except ValueError:
            continue
    return out


def cut_segment(raw, start, end):
    """The 200 Hz sample rows of one raw stream between two t_perf
    bounds. segment_times and raw.csv's t_perf share a clock, so this
    is a plain slice, no alignment step."""
    samples = raw[raw["event"].isna() | (raw["event"] == "")]
    return samples[(samples["t_perf"] >= start)
                   & (samples["t_perf"] <= end)]


def continuous_rows(trials):
    """Trials whose stimulus was a trajectory or timed pulse: any row
    with a non-empty waveform column. Old CSVs without the column
    yield an empty frame rather than a KeyError."""
    if "waveform" not in trials.columns:
        return trials.iloc[0:0]
    w = trials["waveform"].fillna("").astype(str)
    return trials[w != ""]


def segment_marker_check(folder, rows, tol_s=0.005,
                         respond_tol_s=0.02):
    """Cross-check segment_times against the segment_start /
    segment_end events the engine wrote into the same raw.csv. Both
    describe the same windows, so a mismatch means the raw stream and
    the trial row disagree and the cut cannot be trusted. Returns
    (checked, mismatched).

    buzz_hunt's "respond" segment is a documented exception: the
    packed segment_times cell opens respond at the target's own
    onset (buzz_hunt.py's _begin_play sets it to _target_on), while
    the raw-stream marker fires from the next display frame's tick,
    about one frame (~17 ms) later. That is a logging-convention
    offset, not a cut error, so it gets a wider tolerance rather
    than being flagged alongside real mismatches.
    """
    raw = load_raw(folder)
    if raw is None or "event" not in raw.columns:
        return 0, 0
    events = raw[raw["event"].isin(["segment_start", "segment_end"])]
    if events.empty:
        return 0, 0
    bounds = {}
    for _idx, ev in events.iterrows():
        detail = str(ev.get("detail", "") or "")
        fields = dict(p.partition("=")[::2] for p in detail.split(";")
                      if "=" in p)
        key = (fields.get("trial_id"), fields.get("segment"))
        side = 0 if ev["event"] == "segment_start" else 1
        bounds.setdefault(key, [None, None])[side] = float(ev["t_perf"])
    checked = mismatched = 0
    for _idx, row in rows.iterrows():
        waveform = str(row.get("waveform", "") or "")
        for name, start, end in parse_segment_times_cell(
                row.get("segment_times", "")):
            key = (str(row.get("trial", "")), name)
            got = bounds.get(key)
            if got is None or got[0] is None or got[1] is None:
                continue
            checked += 1
            tol = (respond_tol_s
                   if name == "respond" and waveform.startswith("buzz")
                   else tol_s)
            if abs(got[0] - start) > tol or abs(got[1] - end) > tol:
                mismatched += 1
    return checked, mismatched


def sec_continuous(folders, trials):
    """What the continuous-force trials asked of the patient.

    The foundation section: waveform mix, parameter conditions, seeds,
    and the segment cut check. Per-mode scoring (tracking error,
    lit-blind deltas, staircase thresholds) lands in its own section
    with each mode.
    """
    print("\n" + "=" * 62)
    print("CONTINUOUS FORCE TRIALS")
    print("=" * 62)
    rows = continuous_rows(trials)
    if rows.empty:
        _nothing("No continuous-force trials in this selection,",
                 "so there are no waveforms to rebuild here.",
                 "The threshold modes leave the waveform column empty.")
        return None
    counts = rows["waveform"].value_counts()
    print(f"{len(rows)} trials across {len(counts)} waveform type(s):")
    for name, n in counts.items():
        print(f"  {name}: {n}")
    seeds = rows["waveform_seed"].fillna("").astype(str)
    n_seeded = int((seeds != "").sum())
    if n_seeded:
        print(f"{n_seeded} trial(s) carry a waveform_seed "
              "(pseudorandom sections, regenerable).")
    conditions = (rows["waveform_params"].fillna("").astype(str)
                  .value_counts())
    print(f"{len(conditions)} distinct parameter condition(s); "
          "identical parameters pack to identical strings, so this is "
          "a grouping key, not a parse.")
    keys = set()
    for cell in conditions.index:
        keys.update(parse_waveform_params_cell(cell).keys())
    if keys:
        print("parameter keys seen: " + ", ".join(sorted(keys)))
    total_checked = total_bad = 0
    per_game_bad = []
    for f in folders:
        # Each folder's own rows only: rows carries every selected
        # game pooled, and trial numbers restart at 1 in every
        # block, so checking one folder's markers against every
        # selected game's rows reports mostly false mismatches from
        # trial-id collisions across games (audit finding #102).
        game_rows = rows[rows["folder"] == str(f)]
        checked, bad = segment_marker_check(f, game_rows)
        total_checked += checked
        total_bad += bad
        if bad:
            per_game_bad.append((game_key(f), checked, bad))
    if total_checked:
        print(f"segment cut check: {total_checked} segment(s) matched "
              f"against raw-stream markers, {total_bad} mismatched"
              + ("" if total_bad == 0 else "  <-- investigate before"
                 " scoring these traces"))
    else:
        print("segment cut check: no bracketing markers found in the "
              "selected raw streams.")
    if per_game_bad:
        print("mismatches by game (report per game, not pooled "
              "across the selection):")
        for gname, checked_g, bad_g in per_game_bad:
            print(f"   {gname}: {checked_g} checked, {bad_g} "
                  f"mismatched")
    return rows


# ===================================== continuous-force offline scoring
# Shared plumbing for the three chapters below (Force tracking,
# Precision hold and force sense, Tactile perception). All three modes
# make the same promise in their docstrings: the in-game numbers are
# feedback, and every research number is re-scored offline from the
# 200 Hz raw stream between the logged segment bounds. This is that
# re-scoring path. The trajectory builders are copies of the game's
# pure functions, carried here because the notebook travels without
# the game package; the copied logic is pinned by
# test_notebook_matches_software.py on the game side.

RAW_SAMPLE_CACHE = {}


def raw_sample_frame(folder):
    """The numeric 200 Hz sample rows of one game's raw.csv, cached
    per folder. Event rows are dropped once here so every per-trial
    cut below is a plain time slice on t_perf."""
    key = str(folder)
    if key not in RAW_SAMPLE_CACHE:
        raw = load_raw(Path(folder))
        if raw is None or "t_perf" not in raw.columns:
            RAW_SAMPLE_CACHE[key] = None
        else:
            ev = (raw["event"].fillna("").astype(str)
                  if "event" in raw.columns
                  else pd.Series("", index=raw.index))
            samp = raw[ev == ""].dropna(subset=["t_perf"])
            RAW_SAMPLE_CACHE[key] = samp.sort_values("t_perf")
    return RAW_SAMPLE_CACHE[key]


def lane_fsr_column(row):
    """Which raw column carries this trial's finger. Lanes are global
    (0..3 right, 4..7 left) and bilateral raw rows hold the right hand
    in fsr1-4 and the left in fsr5-8; a single-hand game's four
    sensors land in fsr1-4 whichever hand played."""
    lane0 = _lane0(row)
    if lane0 is None:
        return None
    if str(row.get("hand_mode", "right")) == "both":
        return f"fsr{int(lane0) + 1}"
    return f"fsr{int(lane0) % 4 + 1}"


def trial_tare(samp, col, t_start, back_s=1.0, gap_s=0.05):
    """The lane's raw tare level just before a trial's first scored
    segment. The modes re-tare while the hand rests in the announce
    phase (ForceView.rebaseline), so the offline reference is the
    median raw level over that same resting window. Falls back to the
    trial's own 5th percentile when the pre-window has no samples (a
    trimmed raw file), which biases errors low rather than dropping
    the trial."""
    if samp is None or col not in samp.columns:
        return None
    pre = samp[(samp["t_perf"] >= t_start - back_s)
               & (samp["t_perf"] < t_start - gap_s)]
    vals = pd.to_numeric(pre[col], errors="coerce").dropna()
    if len(vals) >= 20:
        return float(vals.median())
    win = samp[(samp["t_perf"] >= t_start)
               & (samp["t_perf"] <= t_start + 30.0)]
    vals = pd.to_numeric(win[col], errors="coerce").dropna()
    return float(vals.quantile(0.05)) if len(vals) else None


def percent_trace(samp, col, t0, t1, ref, max_counts):
    """(t, pct) arrays for one lane between two t_perf bounds, in
    percent of the trial's logged session max. Empty arrays when the
    slice has nothing, so callers can guard on length."""
    if samp is None or col not in samp.columns or not max_counts:
        return np.array([]), np.array([])
    w = samp[(samp["t_perf"] >= t0) & (samp["t_perf"] <= t1)]
    t = w["t_perf"].to_numpy(dtype=float)
    v = pd.to_numeric(w[col], errors="coerce").to_numpy(dtype=float)
    keep_m = np.isfinite(v)
    pct = (v[keep_m] - float(ref)) / float(max_counts) * 100.0
    return t[keep_m], pct


def num_or_nan(value):
    """A stimulus field as a float, NaN when absent or empty. The
    modes pack unset numbers as empty strings (hold rows before any
    dark window write delta=), so a bare float() here would crash on
    exactly the rows that are fine."""
    try:
        return float(value)
    except (TypeError, ValueError):
        return np.nan


def continuous_floor_note(kind):
    """The measurement floors, stated before any figure so no number
    below gets read past them. kind picks the mode-specific lines."""
    print("MEASUREMENT FLOOR, read before the figures")
    print("   Force is sampled at 200 Hz (5 ms grid) and the display")
    print("   runs at 60 Hz, so the patient saw the target quantised")
    print("   to 16.7 ms frames plus an unmeasured panel delay.")
    if kind == "tracking":
        print("   Tracking lag is estimated on the 5 ms sample grid,")
        print("   so single-run lags resolve to about +/-10 ms and the")
        print("   absolute lag includes the unmeasured display delay:")
        print("   compare lags within this device, never across rigs.")
        print("   A 25 to 30 s run gives a spectral resolution near")
        print("   0.04 Hz, enough to separate the 0.1-0.3 and 0.5-0.8")
        print("   Hz bands, and force fluctuations of interest sit")
        print("   well below the 100 Hz Nyquist limit.")
    if kind in ("tracking", "hold"):
        print("   SingleTact accuracy and drift at very low force is")
        print("   uncharacterised on this rig (the bench work is a")
        print("   thesis instrumentation task), and the probed max is")
        print("   a flat-finger press, not the grip or pinch MVC of")
        print("   the cited protocols: percent-of-max matches those")
        print("   studies in construct, not in newtons.")
    if kind == "tactile":
        print("   The ERM motors have a mechanical rise and stop time")
        print("   around 20 ms that is uncharacterised on this rig,")
        print("   requests below 20 ms are clamped, and each pulse")
        print("   stretches by up to about one display frame. Every")
        print("   duration and gap threshold inherits that bias, so")
        print("   thresholds here are within-person, within-device")
        print("   measures for tracking change, never comparable to")
        print("   published electrical-stimulus norms.")


# ======================================================= Force tracking
# Force Pilot re-scored offline. The corridor is rebuilt exactly from
# each row's waveform_params (the logging contract: waveform + params
# + seed suffice, to the 6-significant-digit precision of the packed
# cell), the raw trace is cut on segment_times, and every metric below
# comes from that cut, not from the in-game frame-rate score.

def fp_sections_from_params(p):
    """Notebook copy of the game's deterministic section builder
    (force_pilot.sections_from_params): the same plan from the same
    logged numbers, as plain dicts. Any drift between the two would
    mis-score every run, which is why the game side pins the contract
    in its tests."""
    base = float(p["base_pct"])
    plateau = float(p["plateau_pct"])
    rate = float(p["ramp_rate_pct_s"])
    ramp_s = max(0.1, (plateau - base) / max(0.1, rate))
    sine_amp = float(p["sine_amp_pct"])
    sine_f = float(p["sine_freq_hz"])
    sine_dur = max(1, int(round(float(p["sine_cycles"])))) / sine_f
    sos_amps = tuple(float(p[f"sos_a{i}_pct"]) for i in (1, 2, 3))
    sos_freqs = tuple(float(p[f"sos_f{i}_hz"]) for i in (1, 2, 3))
    sos_phases = tuple(float(p[f"sos_p{i}_rad"]) for i in (1, 2, 3))
    sine_mid = base + sine_amp
    sos_mid = base + sum(sos_amps)
    sos_v0 = sos_mid + sum(a * np.sin(ph)
                           for a, ph in zip(sos_amps, sos_phases))
    out = []
    t = 0.0

    def add(name, kind, dur, a, b=0.0, freqs=(), amps=(), phases=()):
        nonlocal t
        out.append({"name": name, "kind": kind, "start": t,
                    "dur": float(dur), "end": t + float(dur),
                    "a": float(a), "b": float(b), "freqs": freqs,
                    "amps": amps, "phases": phases})
        t += float(dur)

    add("hold_in", "hold", float(p["hold_in_s"]), base)
    add("ramp_up", "ramp", ramp_s, base, plateau)
    add("hold_top", "hold", float(p["hold_top_s"]), plateau)
    add("release", "ramp", ramp_s, plateau, base)
    add("sine", "osc", sine_dur, sine_mid,
        freqs=(sine_f,), amps=(sine_amp,), phases=(-np.pi / 2.0,))
    add("pre_assess", "ramp", float(p["pre_assess_s"]), base, sos_v0)
    add("assess_sos", "osc", float(p["sos_s"]), sos_mid,
        freqs=sos_freqs, amps=sos_amps, phases=sos_phases)
    return out


def fp_target_pct(sections, t):
    """The corridor centreline at run time t, percent of max. Clamped
    to the plan's ends, matching the game's renderer."""
    if not sections:
        return 0.0
    t = min(max(t, 0.0), sections[-1]["end"])
    for sec in sections:
        if t <= sec["end"] or sec is sections[-1]:
            tt = t - sec["start"]
            if sec["kind"] == "hold":
                return sec["a"]
            if sec["kind"] == "ramp":
                frac = 0.0 if sec["dur"] <= 0 else min(1.0, tt / sec["dur"])
                return sec["a"] + (sec["b"] - sec["a"]) * frac
            return sec["a"] + sum(
                a * np.sin(2.0 * np.pi * f * tt + ph)
                for f, a, ph in zip(sec["freqs"], sec["amps"],
                                    sec["phases"]))
    return sections[-1]["a"]


def fp_target_array(sections, ts):
    return np.array([fp_target_pct(sections, x) for x in ts])


def tracking_lag_ms(err_free_force, err_free_target, max_lag_s=1.5):
    """Tracking lag by cross-correlation of the mean-removed force and
    target, positive when the force trails the target. Returns
    (lag_ms, r_at_lag); NaNs when the traces are too short or flat."""
    a = np.asarray(err_free_force, dtype=float)
    b = np.asarray(err_free_target, dtype=float)
    n = min(len(a), len(b))
    if n < 400:
        return np.nan, np.nan
    a = a[:n] - a[:n].mean()
    b = b[:n] - b[:n].mean()
    denom = np.sqrt((a * a).sum() * (b * b).sum())
    if denom <= 0:
        return np.nan, np.nan
    max_k = int(max_lag_s * 200)
    xc = np.correlate(a, b, mode="full")[n - 1 - max_k:n + max_k]
    k = int(np.argmax(xc)) - max_k
    return k * 5.0, float(xc.max() / denom)


def lodha_bands(err, fs=200.0):
    """Normalised power in the 0.1-0.3 and 0.5-0.8 Hz bands of the
    tracking residual, each as a fraction of 0.05-4 Hz power. The
    bands are Lodha 2013's: chronic stroke shifted force modulation
    down into 0.1-0.3 Hz at the cost of 0.5-0.8 Hz, and that shift
    explained most of the elevated variability. One honest departure:
    Lodha measured constant-force holds, while this residual comes
    from the whole tracking run, so the ratio is the biomarker's
    shape, not its published operating point."""
    x = np.asarray(err, dtype=float)
    if len(x) < 1000:
        return np.nan, np.nan
    t = np.arange(len(x)) / fs
    x = x - np.polyval(np.polyfit(t, x, 1), t)     # detrend
    psd = np.abs(np.fft.rfft(x * np.hanning(len(x)))) ** 2
    f = np.fft.rfftfreq(len(x), 1.0 / fs)
    total = psd[(f >= 0.05) & (f <= 4.0)].sum()
    if total <= 0:
        return np.nan, np.nan
    low = psd[(f >= 0.1) & (f <= 0.3)].sum() / total
    high = psd[(f >= 0.5) & (f <= 0.8)].sum() / total
    return float(low), float(high)

def ramp_segmentation(t, pct, segs, min_pause_s=0.15,
                      rate_floor_pct_s=3.0):
    """Step count and mean pause duration on the ramp_up and
    release segments (Naik 2011's segmentation measures: chronic
    stroke and PD hands break a smooth ramp into discrete steps
    separated by pauses, and step count / pause duration are named
    a reliable PD discriminator in the cluster notes that scoped
    this mode -- named in the brief, never computed until now,
    audit finding #83).

    A pause is a contiguous stretch of the cut trace where the
    force rate of change stays under rate_floor_pct_s (percent of
    max per second) for at least min_pause_s. Both thresholds are
    a first cut, not calibrated against a reference dataset (no
    published operating point for this device and force range),
    so step counts are within-device / within-session comparisons
    only until validated, the same caveat the rest of this chapter
    carries for every absolute number.

    Returns {segment_name: {"steps": int, "mean_pause_s": float}}
    for whichever of ramp_up / release the row's segs contain;
    empty dict when the cut is too short to say anything.
    """
    out = {}
    t = np.asarray(t, dtype=float)
    pct = np.asarray(pct, dtype=float)
    if len(t) < 10:
        return out
    dt = np.diff(t)
    dpct = np.diff(pct)
    rate = np.divide(dpct, dt, out=np.zeros_like(dpct),
                      where=dt > 0)
    for name, a, b in segs:
        if name not in ("ramp_up", "release"):
            continue
        m = (t[:-1] >= a) & (t[:-1] <= b)
        if m.sum() < 5:
            continue
        seg_rate, seg_dt = rate[m], dt[m]
        pauses, run_s = [], 0.0
        for is_low, d in zip(np.abs(seg_rate) < rate_floor_pct_s,
                             seg_dt):
            if is_low:
                run_s += float(d)
            else:
                if run_s >= min_pause_s:
                    pauses.append(run_s)
                run_s = 0.0
        if run_s >= min_pause_s:
            pauses.append(run_s)
        out[name] = {
            "steps": len(pauses),
            "mean_pause_s": (float(np.mean(pauses)) if pauses
                              else 0.0),
        }
    return out



def force_tracking_runs(folders, trials, metas=None):
    """One row per corridor run, scored offline: rebuild the target,
    cut the trace, and derive every metric from the cut. Runs whose
    raw slice is missing are counted and dropped rather than scored
    from nothing.

    Each row carries a demo flag: a supervisor Test Mode block runs
    compressed holds and ramps (block_stats stores this precisely
    as block_summary.force_pilot.demo) and is not a patient
    measurement, so it must stay separable from real play rather
    than enter the per-finger tables, the release-generation
    aggregate or the learning slopes indistinguishably (audit
    finding #82). metas=None (old call sites) falls back to the
    scored-duration tell (demo runs land around 14 s against 23 to
    30 s for real runs) rather than losing the split."""
    rows = mode_rows(trials, "force_pilot")
    if rows.empty or "waveform" not in rows.columns:
        return pd.DataFrame(), 0
    rows = rows[rows["waveform"].fillna("").astype(str) == "corridor"]
    game_demo = {}
    if metas:
        for name, meta in _meta_items(metas):
            bs = (meta.get("block_summary", {})
                  or {}).get("force_pilot")
            if isinstance(bs, dict) and "demo" in bs:
                game_demo[name] = bool(bs.get("demo"))
    out, dropped = [], 0
    for _idx, row in rows.iterrows():
        p = parse_waveform_params_cell(row.get("waveform_params", ""))
        segs = parse_segment_times_cell(row.get("segment_times", ""))
        col = lane_fsr_column(row)
        maxc = p.get("max_press_counts")
        if not p or not segs or col is None or not maxc:
            dropped += 1
            continue
        samp = raw_sample_frame(row["folder"])
        t0, t1 = segs[0][1], segs[-1][2]
        ref = trial_tare(samp, col, t0)
        if ref is None:
            dropped += 1
            continue
        t, pct = percent_trace(samp, col, t0, t1, ref, maxc)
        if len(t) < 400:
            dropped += 1
            continue
        secs = fp_sections_from_params(p)
        tgt = fp_target_array(secs, t - t0)
        err = pct - tgt
        hw = float(p.get("hw_pct", 8.0))
        seg_mae = {}
        for name, a, b in segs:
            m = (t >= a) & (t <= b)
            if m.sum() >= 20:
                seg_mae[name] = float(np.abs(err[m]).mean())
        # CoV per hold segment, then averaged: the two holds sit at
        # different levels, so pooling them first would report the
        # level difference as steadiness.
        covs = []
        for name, a, b in segs:
            if name not in ("hold_in", "hold_top"):
                continue
            m = (t >= a) & (t <= b)
            if m.sum() >= 100 and pct[m].mean() > 1e-6:
                covs.append(float(pct[m].std() / pct[m].mean()))
        cov = float(np.mean(covs)) if covs else np.nan
        lag_ms, lag_r = tracking_lag_ms(pct, tgt)
        low_n, high_n = lodha_bands(err)
        seg_steps = ramp_segmentation(t, pct, segs)
        ramp_up_steps = seg_steps.get("ramp_up", {})
        release_steps = seg_steps.get("release", {})
        _head, kv = stimulus_parts(row.get("stimulus", ""))
        # The two holds together, for the cohort ordering check
        # (assess above sine above holds).
        hold_maes = [seg_mae[k] for k in ("hold_in", "hold_top")
                     if k in seg_mae]
        out.append({
            "game": row.get("game"), "session": row.get("session"),
            "folder": row.get("folder"),
            "hand": kv.get("hand", row.get("side", "right")),
            "finger": row.get("finger"), "lane": row.get("lane"),
            "trial": row.get("trial"),
            "level": int(float(p.get("lvl", 1))),
            "hw_pct": hw,
            "ramp_rate_pct_s": float(p.get("ramp_rate_pct_s", np.nan)),
            "scored_s": float(t[-1] - t[0]),
            "mae": float(np.abs(err).mean()),
            "rmse": float(np.sqrt((err ** 2).mean())),
            "tic": float((np.abs(err) <= hw).mean()),
            "cov_hold": cov,
            "lag_ms": lag_ms, "lag_r": lag_r,
            "low_n": low_n, "high_n": high_n,
            "press_mae": seg_mae.get("ramp_up", np.nan),
            "release_mae": seg_mae.get("release", np.nan),
            "sine_mae": seg_mae.get("sine", np.nan),
            "assess_mae": seg_mae.get("assess_sos", np.nan),
            "hold_mae": (float(np.mean(hold_maes)) if hold_maes
                         else np.nan),
            "ramp_up_steps": ramp_up_steps.get("steps", np.nan),
            "ramp_up_pause_s": ramp_up_steps.get("mean_pause_s",
                                                  np.nan),
            "release_steps": release_steps.get("steps", np.nan),
            "release_pause_s": release_steps.get("mean_pause_s",
                                                   np.nan),
            "game_tic": num_or_nan(kv.get("tic")),
            "game_mae": num_or_nan(kv.get("mae")),
            "demo": game_demo.get(row.get("game"),
                                  float(t[-1] - t[0]) < 18.0),
        })
    return pd.DataFrame(out), dropped


def sec_force_tracking(folders, trials, metas=None):
    """The Force Pilot chapter: offline tracking error, time in
    corridor, lag, the Lodha spectral ratio, release against
    generation, asymmetry, and the learning curve.

    Metric lineage, stated once: RMSE / CoV / time-in-target are the
    standard visuomotor tracking outcomes (Kurillo 2005 is the build
    template); the sub-1 Hz normalised power split is Lodha 2013's
    stroke biomarker; release-versus-generation scoring follows Naik
    2011's ramp segmentation, with Davidson 2026 making release the
    Parkinson's-sensitive half; longitudinal force-control tracking
    is the Pennati 2020 argument for measuring this at all.
    """
    print("\n" + "=" * 62)
    print("FORCE TRACKING (Force Pilot)")
    print("=" * 62)
    runs, dropped = force_tracking_runs(folders, trials, metas)
    if runs.empty:
        _nothing("No Force Pilot corridor runs in this selection, so",
                 "there is nothing to re-score. Play a Force Pilot",
                 "block (it needs the force pads, not the keyboard)",
                 "and this chapter fills in.")
        return None
    # Demo (supervisor Test Mode) runs are not patient data -- drop
    # them from every aggregate here rather than let a compressed
    # block sit indistinguishably inside the per-finger tables, the
    # release-generation aggregate and the learning slopes (finding
    # #82). Reported once, then never mentioned again: the rest of
    # this chapter reads `runs` as real play.
    n_demo = int(runs["demo"].sum()) if "demo" in runs.columns else 0
    if n_demo:
        print(f"\n{n_demo} demo (Test Mode) run(s) excluded from "
              "every table and curve below.")
        runs = runs[~runs["demo"]].copy()
    if runs.empty:
        _nothing("Every Force Pilot run in this selection was a demo",
                 "(Test Mode) block, so there is no patient data to",
                 "re-score.")
        return None
    continuous_floor_note("tracking")
    print(f"\n{len(runs)} run(s) re-scored offline from the raw stream"
          + (f"; {dropped} dropped for missing raw data" if dropped
             else ""))

    # The in-game score is feedback, not the result, but the two
    # should not disagree wildly; a large gap means the offline cut
    # or the tare is wrong, so it is checked before anything else.
    both = runs.dropna(subset=["game_tic"])
    if len(both):
        d_tic = (both["tic"] - both["game_tic"]).abs().median()
        d_mae = (both["mae"] - both["game_mae"]).abs().median()
        print(f"cross-check against the in-game score: median "
              f"|offline - game| = {d_tic:.3f} on time-in-corridor, "
              f"{d_mae:.2f}% on MAE"
              + ("" if d_tic < 0.1 else
                 "  <-- large gap, check the tare before trusting this"))

    # Grouped by level as well as hand/finger: a run at the easiest
    # corridor and a run at the hardest corridor are not the same
    # measurement, so pooling them here would blend two difficulties
    # into one number that describes neither.
    per = (runs.groupby(["hand", "finger", "level"])
           .agg(runs=("mae", "count"), mae=("mae", "mean"),
                rmse=("rmse", "mean"), tic=("tic", "mean"),
                cov_hold=("cov_hold", "mean"),
                lag_ms=("lag_ms", "median"))
           .round(3))
    print("\nper finger and level, offline (MAE / RMSE in % of max):")
    _show(per)

    # Segmentation: step count and mean pause duration on the ramp
    # up and release windows, split by the DRAWN ramp rate (slow
    # ramps are where segmentation shows up -- Naik 2011, and the
    # cluster notes name it a reliable PD discriminator). First cut
    # in this codebase (see ramp_segmentation's own caveat): read
    # step counts as within-device, within-session comparisons
    # only (audit finding #83).
    seg_cols = ["ramp_up_steps", "ramp_up_pause_s",
                "release_steps", "release_pause_s"]
    seg_rows = runs.dropna(subset=seg_cols, how="all")
    if len(seg_rows) and "ramp_rate_pct_s" in seg_rows.columns:
        seg = (seg_rows.groupby("ramp_rate_pct_s")
               .agg(runs=("mae", "count"),
                    ramp_up_steps=("ramp_up_steps", "mean"),
                    ramp_up_pause_s=("ramp_up_pause_s", "mean"),
                    release_steps=("release_steps", "mean"),
                    release_pause_s=("release_pause_s", "mean"))
               .round(3))
        print("\nramp segmentation, by drawn ramp rate (Naik 2011 "
              "measures; within-device comparisons only):")
        _show(seg)


    # Example run: the median-MAE run, so it is typical rather than
    # a best case. Target, corridor and trace together.
    mid = runs.sort_values("mae").iloc[len(runs) // 2]
    row = trials[(trials["folder"] == mid["folder"])
                 & (trials["mode"] == "force_pilot")
                 & (trials["trial"] == mid["trial"])].iloc[0]
    p = parse_waveform_params_cell(row["waveform_params"])
    segs = parse_segment_times_cell(row["segment_times"])
    samp = raw_sample_frame(row["folder"])
    col = lane_fsr_column(row)
    t0, t1 = segs[0][1], segs[-1][2]
    ref = trial_tare(samp, col, t0)
    t, pct = percent_trace(samp, col, t0, t1, ref,
                           p["max_press_counts"])
    secs = fp_sections_from_params(p)
    tgt = fp_target_array(secs, t - t0)
    hw = float(p.get("hw_pct", 8.0))
    fig, ax = plt.subplots(figsize=(11, 3.8))
    ax.fill_between(t - t0, tgt - hw, tgt + hw, color="#0ea5e9",
                    alpha=.15, label=f"corridor +/-{hw:.0f}%")
    ax.plot(t - t0, tgt, color="#0f172a", lw=1.5, label="target")
    ax.plot(t - t0, pct, color="#ea580c", lw=1, label="force")
    for name, a, b in segs:
        ax.axvline(a - t0, color="#94a3b8", lw=.6, ls=":")
        ax.text((a + b) / 2 - t0, 0.97, name, ha="center", va="top",
                fontsize=6, color="#64748b",
                transform=ax.get_xaxis_transform())
    ax.set_xlabel("run time (s)")
    ax.set_ylabel("force (% of max press)")
    ax.set_title(f"Median run: {mid['hand']} {mid['finger']}, "
                 f"MAE {mid['mae']:.2f}%, "
                 f"in corridor {mid['tic']:.0%}")
    ax.legend(frameon=False, fontsize=8)
    _save(fig, "force_tracking_example")
    plt.show()

    # Release against generation: the same ramp cut both ways.
    pr = runs.dropna(subset=["press_mae", "release_mae"])
    if len(pr) >= 4:
        d = pr["release_mae"] - pr["press_mae"]
        lo, hi = boot_ci(d.values)
        print(f"\nrelease minus generation error: "
              f"{d.mean():+.2f}% of max  [{lo:+.2f}, {hi:+.2f}] "
              f"across {len(pr)} runs")
        print("Positive means letting go tracks worse than pressing,")
        print("the release deficit Naik 2011 scored in stroke and")
        print("Davidson 2026 found disproportionate in Parkinson's.")
        fig, ax = plt.subplots(figsize=(7, 3.2))
        hands = list(dict.fromkeys(pr["hand"]))
        width = 0.35
        for i, hand in enumerate(hands):
            g = pr[pr["hand"] == hand]
            ax.bar(i - width / 2, g["press_mae"].mean(), width,
                   color="#0ea5e9",
                   label="generation (ramp up)" if i == 0 else None)
            ax.bar(i + width / 2, g["release_mae"].mean(), width,
                   color="#ea580c",
                   label="release (ramp down)" if i == 0 else None)
        ax.set_xticks(range(len(hands)))
        ax.set_xticklabels(hands)
        ax.set_ylabel("MAE (% of max)")
        ax.set_title("Generation against release, by hand")
        ax.legend(frameon=False, fontsize=8)
        _save(fig, "force_tracking_release")
        plt.show()

    # The Lodha ratio, labelled as what it is: the stroke biomarker.
    sp = runs.dropna(subset=["low_n", "high_n"])
    if len(sp):
        print("\nsub-1 Hz structure of the tracking residual (the")
        print("Lodha 2013 stroke biomarker: paretic hands shift")
        print("normalised power INTO 0.1-0.3 Hz and OUT of 0.5-0.8")
        print("Hz, and that shift, not the mean error, carried about")
        print("80 percent of the elevated variability):")
        tbl = (sp.groupby("hand")[["low_n", "high_n"]].mean()
               .assign(ratio=lambda x: x["low_n"] / x["high_n"])
               .round(3))
        _show(tbl)
        print("Computed on the whole-run residual rather than a")
        print("constant hold, so track the ratio within this device")
        print("over sessions; do not read it against the paper's")
        print("absolute values.")

    # Lag by cross-correlation, with its floor restated in context.
    lg = runs.dropna(subset=["lag_ms"])
    if len(lg):
        line = "   ".join(
            f"{h}: {g['lag_ms'].median():.0f} ms (r {g['lag_r'].median():.2f})"
            for h, g in lg.groupby("hand"))
        print(f"\ntracking lag, median per hand:   {line}")
        print("(5 ms grid, display latency included; within-device")
        print("comparisons only)")

    # Between-hand asymmetry, only measurable when both hands played.
    bi = runs[runs.groupby("game")["hand"].transform("nunique") > 1]
    if len(bi):
        wide = (bi.groupby(["finger", "hand"])["mae"].mean()
                .unstack("hand"))
        if {"left", "right"} <= set(wide.columns):
            wide = wide.dropna(subset=["left", "right"])
            wide["asym"] = ((wide["left"] - wide["right"])
                            / (wide["left"] + wide["right"]))
            print("\nbetween-hand asymmetry per finger, (L-R)/(L+R) "
                  "on MAE (positive = left worse):")
            _show(wide.round(3).reindex(
                [f for f in FINGERS if f in wide.index]))
            fig, ax = plt.subplots(figsize=(7, 3.2))
            order = [f for f in FINGERS if f in wide.index]
            xs = np.arange(len(order))
            ax.bar(xs - 0.18, [wide.loc[f, "right"] for f in order],
                   0.36, color=HAND_COLOUR["right"], label="right")
            ax.bar(xs + 0.18, [wide.loc[f, "left"] for f in order],
                   0.36, color=HAND_COLOUR["left"], label="left")
            ax.set_xticks(xs)
            ax.set_xticklabels(order)
            ax.set_ylabel("MAE (% of max)")
            ax.set_title("Tracking error per finger, hand against hand")
            ax.legend(frameon=False, fontsize=8)
            _save(fig, "force_tracking_asymmetry")
            plt.show()

    # Learning: error against run order. Within a session this is
    # familiarisation; across sessions it becomes the training curve
    # Pennati 2020 argues these metrics can carry after scales
    # saturate. Grouped by level, hand and finger: a slope pooled
    # across a difficulty change, or across two different fingers,
    # would read as a training effect what is really an easier
    # corridor or a different digit doing the pressing.
    runs = runs.sort_values(["session", "trial"])
    runs["run_idx"] = runs.groupby(
        ["session", "level", "hand", "finger"]).cumcount() + 1
    print("\nlearning within session(s), split by level/hand/finger "
          "(each needs >= 4 runs to fit a slope):")
    slope_rows = []
    for (lvl, hand, finger), g in runs.groupby(
            ["level", "hand", "finger"]):
        if len(g) < 4:
            continue
        slope, lo, hi = slope_ci(g["run_idx"].values, g["mae"].values)
        slope_rows.append({
            "level": lvl, "hand": hand, "finger": finger, "n": len(g),
            "mae_slope_pct_per_run": round(slope, 3),
            "ci_lo": round(lo, 3), "ci_hi": round(hi, 3)})
    if slope_rows:
        _show(pd.DataFrame(slope_rows))
    else:
        print("(not enough runs in any one level/hand/finger group "
              "yet; need >= 4)")
    n_sessions = runs["session"].nunique()
    if n_sessions >= 2:
        per_s = runs.groupby("session")["mae"].median()
        fig, ax = plt.subplots(figsize=(8, 3.2))
        xs = range(1, len(per_s) + 1)
        ax.plot(xs, per_s.values, "o-", lw=2, color="#0ea5e9")
        ax.set_xticks(list(xs))
        ax.set_xticklabels(list(per_s.index), rotation=20, fontsize=8)
        ax.set_ylabel("median MAE (% of max)")
        ax.set_title("Across sessions")
        _save(fig, "force_tracking_sessions")
        plt.show()
    else:
        fig, ax = plt.subplots(figsize=(8, 3.2))
        for sess, g in runs.groupby("session"):
            ax.plot(g["run_idx"], g["mae"], "o-", lw=1.5, ms=4,
                    label=str(sess))
        ax.set_xlabel("run within session")
        ax.set_ylabel("MAE (% of max)")
        ax.set_title("Learning curve, one session so far")
        ax.legend(frameon=False, fontsize=7)
        _save(fig, "force_tracking_learning")
        plt.show()
        print("One session per selection so far: the across-session")
        print("curve appears once repeat sessions exist.")

    stored = stored_mode_stats(metas, "force_pilot") if metas else {}
    if stored:
        def _level_final_repr(d):
            """Per (hand, finger) final level under the current
            block_stats shape ({"hand:finger": {start, final,
            trace}}); falls back to the legacy single-level
            {start, final, trace} shape for old metadata files."""
            lv = d.get("levels") or {}
            if not lv:
                return None
            if "final" in lv and not isinstance(lv.get("final"), dict):
                return lv.get("final")
            parts = [f"{k}:{v.get('final')}"
                     for k, v in sorted(lv.items())
                     if isinstance(v, dict)]
            return ", ".join(parts) if parts else None

        tbl = pd.DataFrame([{
            "game": gname,
            "stored_mae_pct": (d.get("overall") or {}).get("mae_pct"),
            "stored_tic": (d.get("overall") or {}).get(
                "time_in_corridor"),
            "level_final": _level_final_repr(d),
            "visual_gain": d.get("visual_gain"),
        } for gname, d in stored.items()])
        print("\nwhat each block stored about itself (block_summary."
              "force_pilot); visual gain must be constant within any")
        print("comparison (Archer 2017 moves error an order of "
              "magnitude with gain):")
        _show(tbl)

    return {"runs": runs,
            "n_runs": int(len(runs)),
            "mae_pct": round(float(runs["mae"].mean()), 2),
            "tic": round(float(runs["tic"].mean()), 3),
            "release_minus_press": (
                round(float((runs["release_mae"]
                             - runs["press_mae"]).dropna().mean()), 2)
                if runs["release_mae"].notna().any() else np.nan)}


# ======================================== Precision hold and force sense
# Lighthouse re-scored offline. Hold trials cut into their lit and
# dark windows straight from segment_times; echo trials cut into
# study / delay / reproduce. The lit-dark contrast is the reason the
# mode exists (Li 2015: visual feedback masks the CTS deficit,
# withdrawal exposes it), so it is the headline here, not the mean.

def precision_hold_rows(trials):
    rows = mode_rows(trials, "lighthouse")
    if rows.empty or "waveform" not in rows.columns:
        return pd.DataFrame(), pd.DataFrame()
    w = rows["waveform"].fillna("").astype(str)
    return rows[w == "hold"].copy(), rows[w == "reproduce"].copy()


def score_hold(row):
    """One hold, offline: per-window MAE / CoV / mean level, pooled
    lit and dark scores, and per-dark-window drift (rate and
    direction). None when the raw cut is unusable."""
    p = parse_waveform_params_cell(row.get("waveform_params", ""))
    segs = parse_segment_times_cell(row.get("segment_times", ""))
    col = lane_fsr_column(row)
    maxc = p.get("max_press_counts")
    target = p.get("target_pct")
    if not p or not segs or col is None or not maxc or target is None:
        return None
    samp = raw_sample_frame(row["folder"])
    names = [s[0] for s in segs]
    if "ignite" not in names:
        return None
    ig_start = segs[names.index("ignite")][1]
    ref = trial_tare(samp, col, ig_start)
    if ref is None:
        return None
    win_stats, drifts = [], []
    for name, a, b in segs:
        if name == "ignite":
            continue
        t, pct = percent_trace(samp, col, a, b, ref, maxc)
        if len(t) < 20:
            continue
        err = pct - float(target)
        stat = {"window": name,
                "kind": "dark" if name.startswith("dark") else "lit",
                "dur_s": float(b - a),
                "mae": float(np.abs(err).mean()),
                "rmse": float(np.sqrt((err ** 2).mean())),
                "cov": (float(pct.std() / pct.mean())
                        if pct.mean() > 1e-6 else np.nan),
                "mean_pct": float(pct.mean())}
        if name.startswith("dark") and len(t) >= 40:
            # Drift as the slope of the whole unseen stretch: less
            # sensitive to entry noise than an endpoint difference,
            # and signed so direction survives.
            rate = float(np.polyfit(t - a, pct, 1)[0])
            stat["drift_rate"] = rate
            drifts.append(rate)
        win_stats.append(stat)
    if not win_stats:
        return None
    ws = pd.DataFrame(win_stats)

    def pooled(kind, field):
        g = ws[ws["kind"] == kind].dropna(subset=[field])
        if g.empty:
            return np.nan
        return float(np.average(g[field], weights=g["dur_s"]))

    _head, kv = stimulus_parts(row.get("stimulus", ""))
    return {"game": row.get("game"), "session": row.get("session"),
            "who": row.get("participant"),
            "hand": kv.get("hand", row.get("side", "right")),
            "finger": row.get("finger"), "trial": row.get("trial"),
            "level": int(float(p.get("lvl", 1))),
            "target_pct": float(target),
            "lit_mae": pooled("lit", "mae"),
            "lit_rmse": pooled("lit", "rmse"),
            "lit_cov": pooled("lit", "cov"),
            "dark_mae": pooled("dark", "mae"),
            "dark_rmse": pooled("dark", "rmse"),
            "dark_cov": pooled("dark", "cov"),
            "drift_rate": (float(np.mean(drifts)) if drifts else np.nan),
            "n_dark": int((ws["kind"] == "dark").sum()),
            "game_delta": num_or_nan(kv.get("delta")),
            "windows": ws}


def score_echo(row):
    """One echo trial, offline: the reproduced level over the settle
    window against the studied target."""
    p = parse_waveform_params_cell(row.get("waveform_params", ""))
    segs = parse_segment_times_cell(row.get("segment_times", ""))
    col = lane_fsr_column(row)
    maxc = p.get("max_press_counts")
    target = p.get("target_pct")
    names = {s[0]: (s[1], s[2]) for s in segs}
    if (not p or col is None or not maxc or target is None
            or "reproduce" not in names):
        return None
    samp = raw_sample_frame(row["folder"])
    a, b = names["reproduce"]
    # Tare from the delay stretch when there is one: the hand rests
    # through it, and it sits nearer the reproduce press than the
    # trial's start, so slow drift between study and reproduction is
    # absorbed rather than scored as memory error.
    if "delay" in names:
        da, db = names["delay"]
        ref = trial_tare(samp, col, db, back_s=min(2.0, db - da),
                         gap_s=0.1)
    else:
        ref = trial_tare(samp, col, a)
    if ref is None:
        return None
    settle = float(p.get("settle_s", 2.0))
    t, pct = percent_trace(samp, col, max(a, b - settle), b, ref, maxc)
    if len(t) < 40:
        return None
    made = float(pct.mean())
    _head, kv = stimulus_parts(row.get("stimulus", ""))
    return {"game": row.get("game"), "session": row.get("session"),
            "hand": kv.get("hand", row.get("side", "right")),
            "finger": row.get("finger"),
            "delay_s": float(p.get("delay_s", np.nan)),
            "cross": bool(int(float(p.get("cross", 0)))),
            "target_pct": float(target), "made_pct": made,
            "err": made - float(target),
            "game_err": num_or_nan(kv.get("err"))}


def icc_two_one(mat):
    """ICC(2,1), two-way random effects, absolute agreement, single
    measure: the standard test-retest coefficient. mat is sessions in
    columns, targets (finger x metric) in rows, no NaNs."""
    m = np.asarray(mat, dtype=float)
    n, k = m.shape
    if n < 2 or k < 2:
        return np.nan
    grand = m.mean()
    row_m = m.mean(axis=1)
    col_m = m.mean(axis=0)
    ss_rows = k * ((row_m - grand) ** 2).sum()
    ss_cols = n * ((col_m - grand) ** 2).sum()
    ss_err = (((m - row_m[:, None] - col_m[None, :] + grand) ** 2)
              .sum())
    ms_rows = ss_rows / (n - 1)
    ms_cols = ss_cols / (k - 1)
    ms_err = ss_err / ((n - 1) * (k - 1))
    denom = (ms_rows + (k - 1) * ms_err
             + k * (ms_cols - ms_err) / n)
    return float((ms_rows - ms_err) / denom) if denom > 0 else np.nan


def sec_precision_hold(folders, trials, metas=None):
    """The Lighthouse chapter: lit and dark steadiness, the lit-dark
    delta as the headline, post-fade drift, and force reproduction by
    delay, with the ICC scaffold for when repeat sessions exist.

    Metric lineage: the feedback-withdrawal contrast is Li 2015's CTS
    discriminator (equal with vision, collapse without); low-force
    steadiness itself carries function (Camacho-Villa 2025, r = 0.58
    at 5 to 25 percent MVC); constant and variable error by delay is
    the standard force-sense reproduction analysis; and the Cochrane
    vacuum after carpal tunnel release (Peters 2016) is why a cheap
    objective tracker earns its place.
    """
    print("\n" + "=" * 62)
    print("PRECISION HOLD AND FORCE SENSE (Lighthouse)")
    print("=" * 62)
    hold_rows, echo_rows = precision_hold_rows(trials)
    if hold_rows.empty and echo_rows.empty:
        _nothing("No Lighthouse trials in this selection: nothing to",
                 "score. Play a Lighthouse block (force pads only)",
                 "and this chapter fills in.")
        return None
    continuous_floor_note("hold")

    holds = pd.DataFrame(
        [h for h in (score_hold(r) for _i, r in hold_rows.iterrows())
         if h is not None])
    echoes = pd.DataFrame(
        [e for e in (score_echo(r) for _i, r in echo_rows.iterrows())
         if e is not None])
    print(f"\n{len(holds)} hold(s) and {len(echoes)} echo trial(s) "
          f"scored offline "
          f"({len(hold_rows) - len(holds)} hold(s), "
          f"{len(echo_rows) - len(echoes)} echo(es) dropped for "
          f"missing raw data or gutters)")

    if len(holds):
        lit_only = holds[holds["n_dark"] == 0]
        with_dark = holds.dropna(subset=["dark_mae"])
        print("\nlit steadiness, every hold (the Camacho-Villa outcome,"
              " 5 to 25% of max):")
        per = (holds.groupby(["hand", "finger"])
               .agg(holds=("lit_mae", "count"),
                    lit_mae=("lit_mae", "mean"),
                    lit_rmse=("lit_rmse", "mean"),
                    lit_cov=("lit_cov", "mean")).round(3))
        _show(per)

        if len(with_dark):
            d = with_dark["dark_mae"] - with_dark["lit_mae"]
            lo, hi = boot_ci(d.values)
            print("\nHEADLINE  lit-dark delta (dark MAE minus lit MAE):"
                  f"  {d.mean():+.2f}% of max  [{lo:+.2f}, {hi:+.2f}]"
                  f"  across {len(with_dark)} hold(s)")
            print("This is the Li 2015 contrast run per finger: CTS")
            print("hands matched controls WITH feedback and fell apart")
            print("without it, so the delta, not the lit error, is the")
            print("carpal-tunnel-sensitive number.")
            for hand, g in with_dark.groupby("hand"):
                dd = g["dark_mae"] - g["lit_mae"]
                print(f"   {hand:6s} {dd.mean():+.2f}%  "
                      f"(n {len(g)}, dark CoV "
                      f"{g['dark_cov'].mean():.3f} vs lit "
                      f"{g['lit_cov'].mean():.3f})")

            fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
            order = [f for f in FINGERS
                     if f in set(with_dark["finger"])]
            for i, hand in enumerate(
                    dict.fromkeys(with_dark["hand"])):
                g = with_dark[with_dark["hand"] == hand]
                gm = g.groupby("finger")[["lit_mae", "dark_mae"]].mean()
                got = [f for f in order if f in gm.index]
                ax[0].plot([order.index(f) + (i - 0.5) * 0.12
                            for f in got],
                           gm.loc[got, "lit_mae"], "o",
                           color=HAND_COLOUR.get(hand, "#64748b"),
                           label=f"{hand} lit")
                ax[0].plot([order.index(f) + (i - 0.5) * 0.12
                            for f in got],
                           gm.loc[got, "dark_mae"], "s",
                           color=HAND_COLOUR.get(hand, "#64748b"),
                           mfc="none", label=f"{hand} dark")
            ax[0].set_xticks(range(len(order)))
            ax[0].set_xticklabels(order)
            ax[0].set_ylabel("MAE (% of max)")
            ax[0].set_title("Lit (filled) against dark (open)")
            ax[0].legend(frameon=False, fontsize=7)
            dr = holds.dropna(subset=["drift_rate"])
            if len(dr):
                gm = dr.groupby(["hand", "finger"])["drift_rate"].mean()
                labels = [f"{h[0].upper()}.{f}" for h, f in gm.index]
                colours = [HAND_COLOUR.get(h, "#64748b")
                           for h, _f in gm.index]
                ax[1].bar(range(len(gm)), gm.values, color=colours)
                ax[1].axhline(0, color="#0f172a", lw=.8)
                ax[1].set_xticks(range(len(gm)))
                ax[1].set_xticklabels(labels, fontsize=7, rotation=30)
                ax[1].set_ylabel("drift (%/s in the dark)")
                ax[1].set_title("Post-fade drift rate and direction")
            else:
                ax[1].axis("off")
            _save(fig, "precision_hold_delta")
            plt.show()
            dr_all = holds["drift_rate"].dropna()
            if len(dr_all):
                sag = (dr_all < 0).mean()
                print(f"post-fade drift: {sag:.0%} of dark windows "
                      f"sagged (negative drift), mean "
                      f"{dr_all.mean():+.3f}%/s")
        elif len(lit_only) == len(holds):
            print("\nEvery hold here ran fully lit (level 1), so there")
            print("is no lit-dark delta yet: the ladder introduces")
            print("dark windows from level 2.")

        # One typical hold drawn end to end, dark windows shaded.
        pick_h = (with_dark if len(with_dark) else holds).iloc[0]
        row = hold_rows[(hold_rows["trial"] == pick_h["trial"])
                        & (hold_rows["game"] == pick_h["game"])].iloc[0]
        p = parse_waveform_params_cell(row["waveform_params"])
        segs = parse_segment_times_cell(row["segment_times"])
        samp = raw_sample_frame(row["folder"])
        col = lane_fsr_column(row)
        names = [s[0] for s in segs]
        h0 = segs[names.index("ignite")][2] if "ignite" in names \
            else segs[0][1]
        ref = trial_tare(samp, col, segs[0][1])
        t, pct = percent_trace(samp, col, h0, segs[-1][2], ref,
                               p["max_press_counts"])
        if len(t):
            tol = float(p.get("tol_pct", 3.0))
            tgt = float(p["target_pct"])
            fig, ax = plt.subplots(figsize=(11, 3.4))
            ax.axhspan(tgt - tol, tgt + tol, color="#16a34a",
                       alpha=.12, label=f"band +/-{tol:.0f}%")
            ax.axhline(tgt, color="#0f172a", lw=1.2, label="target")
            for name, a, b in segs:
                if name.startswith("dark"):
                    ax.axvspan(a - h0, b - h0, color="#0f172a",
                               alpha=.18)
            ax.plot(t - h0, pct, color="#ea580c", lw=1, label="force")
            ax.set_xlabel("hold time (s)  (shaded = feedback off)")
            ax.set_ylabel("force (% of max)")
            ax.set_title(f"One hold: {pick_h['hand']} "
                         f"{pick_h['finger']}, target {tgt:.1f}%")
            ax.legend(frameon=False, fontsize=8)
            _save(fig, "precision_hold_example")
            plt.show()

    if len(echoes):
        print(f"\nforce reproduction ({int(echoes['cross'].sum())} of "
              f"{len(echoes)} cross-hand):")
        by_delay = (echoes.groupby("delay_s")["err"]
                    .agg(n="count", constant="mean", variable="std")
                    .round(2))
        _show(by_delay)
        print("Constant error is the signed bias (negative =")
        print("undershoot), variable error the spread: the standard")
        print("split for force-sense reproduction, with delay as the")
        print("memory load.")
        ok = echoes.dropna(subset=["game_err"])
        if len(ok):
            gap = (ok["err"] - ok["game_err"]).abs().median()
            print(f"cross-check against the in-game score: median "
                  f"|offline - game| = {gap:.2f}%"
                  + ("" if gap < 1.0 else
                     "  <-- large gap, check the tare"))
        if echoes["delay_s"].nunique() >= 2:
            fig, ax = plt.subplots(figsize=(8, 3.2))
            for hand, g in echoes.groupby("hand"):
                bd = g.groupby("delay_s")["err"]
                ax.errorbar(bd.mean().index, bd.mean().values,
                            yerr=bd.std().values, fmt="o-", lw=1.5,
                            capsize=3,
                            color=HAND_COLOUR.get(hand, "#64748b"),
                            label=hand)
            ax.axhline(0, color="#0f172a", lw=.8)
            ax.set_xlabel("delay (s)")
            ax.set_ylabel("reproduction error (% of max)")
            ax.set_title("Force memory: error against delay "
                         "(mean +/- sd)")
            ax.legend(frameon=False, fontsize=8)
            _save(fig, "precision_hold_echo")
            plt.show()

    # Test-retest scaffolding: ICC(2,1) runs as soon as the same
    # participant has the same metric in two or more sessions, and it
    # runs PER participant: test-retest is a within-person quantity,
    # and a mixed selection's session columns would otherwise belong
    # to different people.
    if len(holds):
        # Rows key on (session, hand, finger): a within-hand
        # finger name alone mixes a bilateral participant's two
        # hands into one row (audit finding #88), shrinking the
        # between-target variance the ICC is meant to read.
        who_ser = (holds["who"].astype(str) if "who" in holds.columns
                   else pd.Series("?", index=holds.index))
        printed_icc = False
        for who, hw in holds.groupby(who_ser):
            wide = (hw.groupby(["session", "hand", "finger"])["lit_mae"]
                    .mean()
                    .unstack("session").dropna())
            if wide.shape[1] >= 2:
                icc = icc_two_one(wide.values)
                lead = "" if who_ser.nunique() <= 1 else f"{who}, "
                print(f"\n{lead}test-retest ICC(2,1) on per-finger lit "
                      f"MAE across {wide.shape[1]} sessions: {icc:.2f}")
                printed_icc = True
        if printed_icc:
            print("Published pinch force-sense reliability sits around")
            print("0.6 to 0.9; below that band, treat single-session")
            print("numbers as noisy.")
        else:
            print("\ntest-retest ICC: scaffolded, waiting for repeat")
            print("sessions by the same participant (needs the same")
            print("fingers measured on two or more days; the published")
            print("force-sense bar to beat is ICC 0.6 to 0.9).")

    stored = stored_mode_stats(metas, "lighthouse") if metas else {}
    if stored:
        tbl = pd.DataFrame([{
            "game": gname,
            "stored_delta_pct": (d.get("overall") or {}).get(
                "lit_dark_delta_pct"),
            "stored_lit_cov": (d.get("overall") or {}).get("lit_cov"),
            "gutters": d.get("gutters"),
            "level_final": (d.get("levels") or {}).get("final"),
        } for gname, d in stored.items()])
        print("\nwhat each block stored about itself (block_summary."
              "lighthouse):")
        _show(tbl)

    return {"holds": holds, "echoes": echoes,
            "n_holds": int(len(holds)),
            "lit_dark_delta": (
                round(float((holds["dark_mae"]
                             - holds["lit_mae"]).dropna().mean()), 2)
                if len(holds) and holds["dark_mae"].notna().any()
                else np.nan)}


# ====================================================== Tactile perception
# Buzz Hunt scored from the trial rows plus the raw stream's reversal
# markers. No force trace to cut here: the analysis is psychophysics.
# The confusion matrix and the cross-talk heatmap are siblings on
# purpose: one maps force leaking onto quiet fingers, the other maps
# touch leaking onto neighbouring fingers, and they share the drawing
# style so the two pictures read side by side in the thesis.

BH_CHANCE_LAPSE = 0.02       # floor on the fitted lapse grid


def bh_frame(trials):
    """Buzz Hunt rows with the stimulus and params unpacked. Catch
    outcomes ride along flagged is_event True, exactly as reaction
    mode's event rows do."""
    rows = mode_rows(trials, "buzz_hunt")
    if rows.empty or "stimulus" not in rows.columns:
        return pd.DataFrame()
    heads, quads = [], []
    for cell in rows["stimulus"]:
        head, kv = stimulus_parts(cell)
        heads.append(head or "unknown")
        quads.append(kv)
    rows["stage"] = heads
    rows["kv"] = quads
    rows["params"] = [parse_waveform_params_cell(c)
                      for c in rows.get("waveform_params",
                                        pd.Series("", index=rows.index))]
    label = rows["early_late"].fillna("").astype(str)
    err = (rows["error_type"].fillna("").astype(str)
           if "error_type" in rows.columns
           else pd.Series("", index=rows.index))
    rows["is_event"] = (label == "CatchOk") | err.eq("catch_false_start")
    rows["hit"] = (label != "Miss") & ~rows["is_event"]
    return rows


def bh_lane_labels(lanes):
    """Short hand-qualified finger names for matrix axes: R.Index,
    L.Ring and so on, in global lane order."""
    out = []
    for lane in lanes:
        side = "L" if lane >= 4 else "R"
        out.append(f"{side}.{FINGERS[lane % 4]}")
    return out


def bh_confusion(rows):
    """(matrix, lanes): response counts per stimulus lane over the
    LOCALISATION trials only, with a trailing no-response column.
    Lanes are global and 0-based. Distractor-stage presses are
    excluded (audit finding #95): a decoy lure is a designed
    attention failure the player was told to expect and gate out,
    a different mechanism from the uncued localisation confusion
    this matrix is the Weber 2023 analogue of, and pooling the two
    inflates cross-hand cells and dilutes the adjacent-finger
    structure the matrix is meant to show."""
    loc = rows[(rows["stage"] == "loc") & ~rows["is_event"]]
    if loc.empty:
        return None, []
    lanes = sorted({int(l) - 1 for l in loc["lane"].dropna()})
    idx = {lane: i for i, lane in enumerate(lanes)}
    m = np.zeros((len(lanes), len(lanes) + 1))
    for _i, row in loc.iterrows():
        stim = int(row["lane"]) - 1
        if stim not in idx:
            continue
        keys = row.get("keys_pressed")
        # A no-response row reads back as NaN, which is truthy, so a
        # plain `or ""` would let the string "nan" through here.
        keys = "" if pd.isna(keys) else str(keys)
        first = keys.split(",")[0].strip()
        if not first:
            m[idx[stim], len(lanes)] += 1
            continue
        resp = int(float(first)) - 1
        m[idx[stim], idx.get(resp, len(lanes))] += 1
    return m, lanes


def fit_logistic(x, y, gamma):
    """Maximum-likelihood logistic psychometric fit with a lapse rate,
    by coarse-to-fine grid search (no scipy in this notebook). Model:
    p = gamma + (1 - gamma - lapse) / (1 + exp(-(x - mu) / s)).
    Returns (mu, s, lapse, threshold_707) with NaNs when the data
    cannot constrain a fit."""
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    if len(x) < 10 or len(set(np.round(x, 1))) < 3:
        return np.nan, np.nan, np.nan, np.nan
    lo, hi = x.min(), x.max()
    span = max(hi - lo, 1.0)
    best = (np.inf, np.nan, np.nan, np.nan)
    mus = np.linspace(lo - 0.3 * span, hi + 0.3 * span, 41)
    sigmas = np.geomspace(span / 50, span, 21)
    lapses = np.array([BH_CHANCE_LAPSE, 0.05, 0.10, 0.15])
    for _pass in range(2):
        for mu in mus:
            for s in sigmas:
                core = 1.0 / (1.0 + np.exp(-(x - mu) / s))
                for lam in lapses:
                    p = gamma + (1.0 - gamma - lam) * core
                    p = np.clip(p, 1e-6, 1 - 1e-6)
                    nll = -(y * np.log(p)
                            + (1 - y) * np.log(1 - p)).sum()
                    if nll < best[0]:
                        best = (nll, mu, s, lam)
        _nll, mu0, s0, _l0 = best
        mus = np.linspace(mu0 - 0.1 * span, mu0 + 0.1 * span, 21)
        sigmas = np.geomspace(max(s0 / 3, span / 100), s0 * 3, 15)
    _nll, mu, s, lam = best
    core_needed = (0.707 - gamma) / (1.0 - gamma - lam)
    thr = (mu + s * np.log(core_needed / (1.0 - core_needed))
           if 0.0 < core_needed < 1.0 else np.nan)
    return mu, s, lam, thr


def bh_reversals(folders):
    """Staircase reversal events from every selected raw stream, as
    one frame: stair kind, hand, level at reversal, reversal index.
    The mode logs these the moment they happen, so they are the
    ground truth for the threshold estimates."""
    out = []
    for folder in folders:
        raw = load_raw(Path(folder))
        if raw is None or "event" not in raw.columns:
            continue
        ev = raw[raw["event"] == "buzz_hunt_reversal"]
        for _i, row in ev.iterrows():
            fields = dict(pp.partition("=")[::2]
                          for pp in str(row.get("detail", "")).split(";")
                          if "=" in pp)
            try:
                out.append({"game": game_key(folder),
                            "stair": fields.get("stair", "?"),
                            "hand": fields.get("hand", "?"),
                            "level_ms": float(fields["level_ms"]),
                            "n": int(float(fields.get("n", 0)))})
            except (KeyError, ValueError):
                continue
    return pd.DataFrame(out)


def sec_tactile(folders, trials, metas=None):
    """The Buzz Hunt chapter: confusion matrix, staircase thresholds
    with reversal plots, psychometric fits, d-prime from the catch
    trials, the span curve with the Hebb slope, and the threshold
    learning scaffold.

    Metric lineage: graded discrimination just above threshold is the
    SENSe RCT's active ingredient (Carey 2011), and sensory-only
    training moving a motor scale is Zeuner 2002; the confusion
    matrix is the digital analogue of Weber 2023's misreferral map;
    2-down 1-up staircases converge on the 70.7 percent point; for
    post-release CTS this is measurement, not therapy, because the
    definitive sensory-relearning RCT was negative (Jerosch-Herold
    2016).
    """
    print("\n" + "=" * 62)
    print("TACTILE PERCEPTION (Buzz Hunt)")
    print("=" * 62)
    rows = bh_frame(trials)
    if rows.empty:
        _nothing("No Buzz Hunt trials in this selection: nothing to",
                 "score. Play a Buzz Hunt block (it needs the motors)",
                 "and this chapter fills in.")
        return None
    continuous_floor_note("tactile")
    stages = rows[~rows["is_event"]]["stage"].value_counts()
    print("\n" + f"{int(stages.sum())} scorable trials: "
          + ", ".join(f"{k} {v}" for k, v in stages.items()))

    # ---- confusion matrix, the sibling of the cross-talk heatmap ----
    m, lanes = bh_confusion(rows)
    if m is not None and m.sum() > 0:
        labels = bh_lane_labels(lanes)
        row_n = m.sum(axis=1, keepdims=True)
        pctm = np.where(row_n > 0, m / row_n * 100.0, np.nan)
        fig, ax = plt.subplots(
            figsize=(1.0 + 0.62 * (len(lanes) + 1),
                     0.9 + 0.58 * len(lanes)))
        ax.imshow(pctm, cmap="Reds", vmin=0, vmax=100)
        ax.set_xticks(range(len(lanes) + 1))
        ax.set_xticklabels(labels + ["none"], fontsize=7, rotation=45)
        ax.set_yticks(range(len(lanes)))
        ax.set_yticklabels(labels, fontsize=7)
        for i in range(len(lanes)):
            for j in range(len(lanes) + 1):
                v = pctm[i, j]
                if np.isnan(v) or v == 0:
                    continue
                dark = v > 55
                ax.text(j, i, f"{v:.0f}", ha="center", va="center",
                        fontsize=7,
                        color="white" if dark else "#0f172a")
        ax.set_ylabel("finger that buzzed")
        ax.set_xlabel("finger pressed (% of that row)")
        ax.set_title("Localisation confusion matrix")
        ax.grid(False)
        _save(fig, "tactile_confusion")
        plt.show()
        diag = np.array([pctm[i, i] for i in range(len(lanes))])
        print("This matrix is the cross-talk heatmap's sibling: that")
        print("one maps force leaking onto quiet fingers, this one")
        print("maps touch leaking onto neighbouring fingers, the")
        print("misreferral structure Weber 2023 measured after nerve")
        print("repair. Diagonal = correct localisation per finger:")
        print("   " + "   ".join(f"{l} {v:.0f}%"
                                 for l, v in zip(labels, diag)))
        off = m[:, :len(lanes)].copy()
        np.fill_diagonal(off, 0)
        neigh = 0
        for i, li in enumerate(lanes):
            for j, lj in enumerate(lanes):
                if i != j and abs(li - lj) == 1 and li // 4 == lj // 4:
                    neigh += off[i, j]
        if off.sum() > 0:
            print(f"{neigh / off.sum():.0%} of mislocalisations landed "
                  f"on an adjacent finger of the same hand, the")
            print("neighbour structure both siblings should show.")

    # ---- d-prime and criterion from the catch trials ----------------
    loc_all = rows[rows["stage"] == "loc"]
    catches = loc_all[loc_all["is_event"]]
    real = loc_all[~loc_all["is_event"]]
    if len(catches) and len(real):
        from statistics import NormalDist
        responded = (real["keys_pressed"].fillna("").astype(str)
                     .str.len() > 0)
        n_fa = int((catches["error_type"] == "catch_false_start").sum())
        # The log-linear correction keeps z finite at 0 or 100
        # percent, the standard small-n fix.
        hit_rate = (responded.sum() + 0.5) / (len(real) + 1.0)
        fa_rate = (n_fa + 0.5) / (len(catches) + 1.0)
        z = NormalDist().inv_cdf
        dprime = z(hit_rate) - z(fa_rate)
        crit = -0.5 * (z(hit_rate) + z(fa_rate))
        print(f"\ndetection from catch trials: {len(catches)} catch "
              f"trial(s), {n_fa} false alarm(s)")
        print(f"   d-prime {dprime:.2f}   criterion {crit:+.2f}  "
              f"(log-linear corrected; positive criterion =")
        print("   conservative, waiting unless sure)")
        if len(catches) < 10:
            print(f"   only {len(catches)} catch trials, so these are "
                  f"coarse: the catch rate prices guessing, it does")
            print("   not measure it precisely inside one block.")

    # ---- staircases: reversal plots and threshold estimates ---------
    rev = bh_reversals(folders)
    ests = []
    if len(rev):
        fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
        for ax, stair, title in (
                (axes[0], "duration", "Duration staircase reversals"),
                (axes[1], "gap", "Gap staircase reversals")):
            g = rev[rev["stair"] == stair]
            if g.empty:
                ax.axis("off")
                ax.set_title(f"No {stair} reversals logged")
                continue
            # Group by (hand, GAME) not hand alone: a reversal
            # index restarts at 1 in every session, so pooling
            # several sessions' reversal sequences under one hand
            # zigzags the line and blends the tail-of-reversals
            # threshold across games that were never the same run
            # (audit finding #96). The per-game learning-across-
            # sessions table further down already keys on
            # (game, hand); this headline plot now does too.
            n_games = g["game"].nunique()
            for (hand, game), gh in g.groupby(["hand", "game"]):
                gh = gh.sort_values("n")
                label = hand if n_games <= 1 else f"{hand} ({game})"
                ax.plot(gh["n"], gh["level_ms"], "o-", lw=1.5,
                        color=HAND_COLOUR.get(hand, "#64748b"),
                        label=label)
                tail = gh["level_ms"].tail(6)
                if len(tail) >= 2:
                    est = float(tail.mean())
                    ests.append({"stair": stair, "hand": hand,
                                 "game": game,
                                 "threshold_ms": round(est, 1),
                                 "n_reversals": int(len(gh))})
                    ax.axhline(est, ls="--", lw=.8,
                               color=HAND_COLOUR.get(hand, "#64748b"))
            ax.axhline(40, color="#dc2626", lw=.8, ls=":")
            ax.text(0.98, 40, "40 ms floor", fontsize=6,
                    color="#dc2626", va="bottom", ha="right",
                    transform=ax.get_yaxis_transform())
            ax.set_xlabel("reversal number")
            ax.set_ylabel("level (ms)")
            ax.set_title(title)
            ax.legend(frameon=False, fontsize=8)
        _save(fig, "tactile_staircases")
        plt.show()
    if ests:
        est_df = pd.DataFrame(ests)
        print("threshold = mean of the last reversals (the standard")
        print("2-down 1-up readout, converging on 70.7% correct):")
        _show(est_df)
        wide = est_df.pivot_table(index="stair", columns="hand",
                                  values="threshold_ms")
        if {"left", "right"} <= set(wide.columns):
            for stair, r in wide.iterrows():
                print(f"   {stair}: left {r['left']:.0f} ms vs right "
                      f"{r['right']:.0f} ms "
                      f"({r['left'] - r['right']:+.0f} ms)")
    else:
        print("\nNo staircase reversals in the selected raw streams,")
        print("so no threshold estimate; the psychometric fit below")
        print("still runs on the trial-level data.")

    # ---- psychometric fits with lapse rate ---------------------------
    loc_scored = rows[(rows["stage"] == "loc") & ~rows["is_event"]]
    n_resp = max(4, len({int(l) - 1 for l in rows["lane"].dropna()}))
    if len(loc_scored) >= 12:
        print("\nlogistic psychometric fits (guess floor fixed at "
              f"1/{n_resp} = chance over the active fingers, lapse "
              "fitted):")
        fig, ax = plt.subplots(figsize=(8, 3.4))
        for hand, g in loc_scored.groupby(
                loc_scored["kv"].map(lambda kv: kv.get("hand", "?"))):
            durs = g["kv"].map(
                lambda kv: num_or_nan(kv.get("dur_ms")))
            ok = g["hit"].astype(float)
            keep_m = durs.notna()
            mu, s, lam, thr = fit_logistic(durs[keep_m], ok[keep_m],
                                           gamma=1.0 / n_resp)
            colour = HAND_COLOUR.get(hand, "#64748b")
            binned = ok.groupby((durs // 40 * 40 + 20)).agg(
                ["mean", "count"])
            ax.scatter(binned.index, binned["mean"],
                       s=8 + 3 * binned["count"], color=colour,
                       alpha=.6)
            if np.isfinite(mu):
                xs = np.linspace(max(20, durs.min() - 40),
                                 durs.max() + 40, 120)
                ps = (1.0 / n_resp + (1 - 1.0 / n_resp - lam)
                      / (1 + np.exp(-(xs - mu) / s)))
                ax.plot(xs, ps, lw=2, color=colour, label=hand)
                thr_word = (f"{thr:.0f} ms" if np.isfinite(thr)
                            else "not reached")
                print(f"   {hand:6s} midpoint {mu:.0f} ms, lapse "
                      f"{lam:.2f}, 70.7% point {thr_word} "
                      f"({int(keep_m.sum())} trials)")
        ax.axhline(0.707, color="#b45309", lw=.8, ls="--")
        ax.axvline(40, color="#dc2626", lw=.8, ls=":")
        ax.set_ylim(0, 1.02)
        ax.set_xlabel("pulse duration (ms)")
        ax.set_ylabel("p(correct finger)")
        ax.set_title("Localisation psychometric functions "
                     "(dot size = trials)")
        ax.legend(frameon=False, fontsize=8)
        _save(fig, "tactile_psychometric")
        plt.show()
        print("The staircase estimate and the fitted 70.7% point")
        print("should roughly agree; a fit far below the 40 ms floor")
        print("is extrapolation, not measurement.")

    # Distractor stage: attention at fixed just-above-threshold level.
    # Each hand runs its decoy trials at ITS OWN staircase level, so
    # pooling both hands into one accuracy figure averages across two
    # different difficulties and represents neither hand. Group by
    # hand first; the per-trial hand lives in the packed stimulus (the
    # row-level "hand" column is the session's bilateral setting, e.g.
    # "both", not the trial's own side).
    dis = rows[(rows["stage"] == "distractor") & ~rows["is_event"]].copy()
    if len(dis):
        dis["dis_hand"] = dis["kv"].map(lambda kv: kv.get("hand", "?"))
        dis["lured"] = dis["kv"].map(
            lambda kv: str(kv.get("lured", "")) == "True")
        dis["dur_ms"] = dis["kv"].map(
            lambda kv: num_or_nan(kv.get("dur_ms")))
        print("\ndistractor stage (bilateral only), by hand (each hand")
        print("runs its decoys at its own staircase level, so the two")
        print("are not comparable pooled):")
        for hand, g in dis.groupby("dis_hand"):
            level = g["dur_ms"].dropna()
            level_txt = (f", level {level.iloc[0]:.0f} ms"
                         if len(level) else "")
            print(f"   {hand}: {g['hit'].mean():.0%} correct over "
                  f"{len(g)} trial(s){level_txt}, "
                  f"{int(g['lured'].sum())} lured to the decoy hand")
        print("These trials hold the staircase level fixed, so this is")
        print("selective attention at threshold, not a threshold.")

    # ---- gap detection ------------------------------------------------
    gap = rows[(rows["stage"] == "gap") & ~rows["is_event"]].copy()
    if len(gap):
        gap["gap_ms"] = gap["kv"].map(
            lambda kv: num_or_nan(kv.get("gap_ms")))
        gap["two"] = gap["kv"].map(
            lambda kv: str(kv.get("two", "0")) == "1")
        # A no-response trial says nothing about the percept -- the
        # mode's own gap staircase holds still on silence rather
        # than reading it as a wrong answer (buzz_hunt.py's
        # _close_gap). Counting no-response rows as false twos or
        # as fit failures contradicts that (audit finding #91):
        # filter both to RESPONDED trials and report no-response
        # as its own count, the way block_stats does.
        gap["taps"] = gap["kv"].map(
            lambda kv: int(float(kv.get("taps", 0))))
        gap["responded"] = gap["taps"] > 0
        n_no_resp = int((~gap["responded"]).sum())
        resp = gap[gap["responded"]]
        two = resp[resp["two"]]
        one = resp[~resp["two"]]
        fa = 1.0 - one["hit"].mean() if len(one) else np.nan
        print(f"\ngap detection: {len(two)} two-buzz and {len(one)} "
              f"one-buzz trial(s) with a response, {n_no_resp} "
              f"no-response; false 'two' on responded one-buzz "
              f"trials {fa:.0%}")
        if len(two) >= 12:
            # The true guess floor for p(correct | two-buzz) at zero
            # gap is the participant's own measured false-two bias
            # on one-buzz trials, not an arbitrary constant (audit
            # finding #97): a conservative and a liberal responder
            # get the same fixed gamma otherwise, biasing the
            # extracted threshold in opposite directions.
            gamma = fa if np.isfinite(fa) else BH_CHANCE_LAPSE
            gamma = min(max(gamma, BH_CHANCE_LAPSE), 0.5)
            mu, s, lam, thr = fit_logistic(
                two["gap_ms"], two["hit"].astype(float), gamma=gamma)
            if np.isfinite(thr):
                print(f"   fitted 70.7% gap threshold {thr:.0f} ms "
                      f"(midpoint {mu:.0f} ms, lapse {lam:.2f}, "
                      f"guess floor {gamma:.0%} from the measured "
                      f"false-two rate)")
            print("   the one-buzz control lasts exactly two shorts")
            print("   plus the gap, so total duration never gives the")
            print("   answer away; both thresholds sit on top of the")
            print("   motor's uncharacterised ~20 ms rise time.")

    # ---- span and the Hebb slope --------------------------------------
    span = rows[(rows["stage"] == "span") & ~rows["is_event"]].copy()
    if len(span):
        span["len"] = span["kv"].map(
            lambda kv: int(float(kv.get("len", 0))))
        span["hebb"] = span["kv"].map(
            lambda kv: str(kv.get("hebb", "0")) == "1")

        def item_acc(kv):
            played = str(kv.get("played", "")).split("-")
            pressed = str(kv.get("pressed", "")).split("-")
            if not played or played == [""]:
                return np.nan
            hits = sum(1 for a, b in zip(played, pressed) if a == b)
            return hits / len(played)

        span["item_acc"] = span["kv"].map(item_acc)
        best = span.loc[span["hit"], "len"].max() if span["hit"].any() \
            else 0
        print(f"\nsequence span: {len(span)} trial(s), longest "
              f"correctly replayed {best} item(s)")
        curve = span.groupby("len")["hit"].agg(["mean", "count"])
        _show(curve.round(2))
        hebb = span[span["hebb"]].reset_index(drop=True)
        novel = span[~span["hebb"]]
        if len(hebb):
            # hebb_sequence (finger_rehab/game/modes/buzz_hunt.py) draws a
            # DIFFERENT hidden sequence per span LENGTH, so pooling
            # hebb trials across lengths fits one slope over item
            # accuracies that were never repeats of each other
            # (audit finding #90). Fit within each length's own
            # recurrences only, and say plainly when a length has
            # too few repeats to fit.
            print("\nHebb repetition learning (fit within each")
            print("length's own repeats only -- the hidden")
            print("sequence differs by length, so trials of")
            print("different lengths are not repeats of the same")
            print("material):")
            any_usable = False
            for length, g in hebb.groupby("len"):
                g = g.reset_index(drop=True)
                hx = np.arange(1, len(g) + 1)
                hy = g["item_acc"].to_numpy(dtype=float)
                keep_m = np.isfinite(hy)
                if len(g) < 2 or keep_m.sum() < 2:
                    print(f"   length {int(length)}: repeated "
                          f"{len(g)} time(s), too few scored "
                          f"repeats to fit a slope -- unusable")
                    continue
                any_usable = True
                hslope = float(np.polyfit(hx[keep_m], hy[keep_m], 1)[0])
                nov_len = novel.loc[novel["len"] == length, "item_acc"]
                nov_txt = (f"{nov_len.mean():.2f}" if len(nov_len)
                          else "n/a")
                print(f"   length {int(length)}: repeated {len(g)} "
                      f"time(s); item accuracy slope {hslope:+.3f} "
                      f"per repeat against {nov_txt} on fresh "
                      f"length-{int(length)} sequences.")
            if any_usable:
                print("The material is derived from the participant")
                print("name, so each length's slope keeps")
                print("accumulating across sessions without the")
                print("player ever being told a sequence repeats.")

    # ---- learning across sessions -------------------------------------
    n_sessions = rows["session"].nunique()
    if n_sessions >= 2 and len(rev):
        print("\nthreshold learning across sessions:")
        for stair, g in rev.groupby("stair"):
            per_game = (g.groupby(["game", "hand"])["level_ms"]
                        .apply(lambda v: v.tail(6).mean()).round(1))
            _show(per_game.rename(f"{stair}_threshold_ms"))
    else:
        print("\nthreshold learning across sessions: scaffolded, one")
        print("session so far. SENSe-style gains (Carey 2011) and the")
        print("Zeuner 2002 result predict thresholds should fall with")
        print("training; this table fills in when repeat sessions")
        print("exist.")

    stored = stored_mode_stats(metas, "buzz_hunt") if metas else {}
    if stored:
        tbl = pd.DataFrame([{
            "game": gname,
            "loc_accuracy": (d.get("loc") or {}).get("accuracy"),
            "fa_rate": ((d.get("loc") or {}).get("catch")
                        or {}).get("fa_rate"),
            "dur_thr_right": ((d.get("threshold") or {})
                              .get("right") or {}).get("estimate_ms"),
            "dur_thr_left": ((d.get("threshold") or {})
                             .get("left") or {}).get("estimate_ms"),
            "span_max": (d.get("span") or {}).get("max_correct"),
        } for gname, d in stored.items()])
        print("\nwhat each block stored about itself (block_summary."
              "buzz_hunt), the on-device cross-check for the offline")
        print("numbers above:")
        _show(tbl)

    return {"rows": rows,
            "n_trials": int(len(rows[~rows["is_event"]])),
            "thresholds": ests}


# ========================================================= Echo mode
# The explicit span game: the rig plays a growing lane sequence (tile
# light plus a buzz on that finger), the patient replays it in order,
# and the Kessels ladder says when the block ends. Scored offline from
# the per-trial lane lists the rows carry.


def _levenshtein(a, b):
    """Plain edit distance between two lane lists. Sequences top out
    at nine items, so the quadratic table is nothing."""
    if not a:
        return len(b)
    if not b:
        return len(a)
    prev = list(range(len(b) + 1))
    for i, x in enumerate(a, 1):
        cur = [i]
        for j, y in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1,
                           prev[j - 1] + (x != y)))
        prev = cur
    return prev[-1]


def _echo_error(played, pressed, outcome):
    """The serial-recall taxonomy for one failed reproduction. The
    mode closes the attempt at the first wrong press, so the last
    entered lane IS the error: an item of the sequence in the wrong
    place is a transposition (the dominant spatial-span error), a
    lane the sequence never held is an intrusion, and a trial that
    timed out silent is an omission. Correct trials come back empty."""
    if outcome == "correct":
        return ""
    if outcome == "omission" or not pressed:
        return "omission"
    return "transposition" if pressed[-1] in played else "intrusion"


def echo_frame(trials):
    """Echo rows with the packed stimulus unpacked into the columns
    the span analyses read: length, hidden-repeat flag, played and
    pressed lane lists (global 0-based, the pack_lanes convention),
    outcome, reproduction press offsets in ms, partial credit
    (Conway 2005: proportion of items in their own serial position),
    the Gonthier 2022 edit-distance score, and the error class."""
    rows = mode_rows(trials, "echo")
    if rows.empty or "stimulus" not in rows.columns:
        return pd.DataFrame()
    heads, kvs = [], []
    for cell in rows["stimulus"]:
        head, kv = stimulus_parts(cell)
        heads.append(head or "unknown")
        kvs.append(kv)
    rows["kv"] = kvs
    rows = rows[[h == "echo" for h in heads]].copy()
    if rows.empty:
        return pd.DataFrame()

    def lanes_of(token):
        return [int(float(p)) for p in str(token).split("-")
                if p not in ("", "nan")]

    rows["len"] = rows["kv"].map(
        lambda kv: int(float(kv.get("len", 0))))
    rows["hebb"] = rows["kv"].map(
        lambda kv: str(kv.get("hebb", "0")) == "1")
    rows["outcome"] = rows["kv"].map(
        lambda kv: str(kv.get("outcome", "")))
    rows["played"] = rows["kv"].map(
        lambda kv: lanes_of(kv.get("played", "")))
    rows["pressed"] = rows["kv"].map(
        lambda kv: lanes_of(kv.get("pressed", "")))
    rows["press_ms"] = rows["kv"].map(
        lambda kv: [float(p) for p in str(kv.get("pt", "")).split("-")
                    if p not in ("", "nan")])
    rows["hit"] = rows["outcome"] == "correct"
    rows["partial"] = [
        (sum(1 for a, b in zip(p, q) if a == b) / len(p))
        if p else np.nan
        for p, q in zip(rows["played"], rows["pressed"])]
    rows["edit_score"] = [
        max(0.0, 1.0 - _levenshtein(p, q) / len(p)) if p else np.nan
        for p, q in zip(rows["played"], rows["pressed"])]
    # Named echo_error, not error_type: the CSV already has an
    # error_type column (catch outcomes and the like) and shadowing
    # it here would poison any later cross-mode read of these rows.
    rows["echo_error"] = [
        _echo_error(p, q, o) for p, q, o in
        zip(rows["played"], rows["pressed"], rows["outcome"])]
    rows["trial_n"] = pd.to_numeric(rows.get("trial"), errors="coerce")
    return rows


def sec_echo(trials, metas=None):
    """The Echo chapter: explicit visuospatial span on the finger
    lanes, scored offline from the per-trial lane lists.

    Metric lineage: the ladder and its headline numbers are the
    Kessels 2000 Corsi standard (span, total correct sequences, and
    the span x correct product, a compound with a better distribution
    than span alone); per-item partial credit over all-or-nothing is
    Conway 2005's recommendation for span tasks, with Gonthier 2022's
    edit distance as the refined variant; the hidden repeated
    sequence is Hebb 1961's every-third-trial schedule, confirmed in
    the visuospatial domain by Couture and Tremblay 2007; the error
    split is the standard serial-recall taxonomy, and spatial span is
    expected to lean on transpositions and omissions. Claim limits
    up front: four or eight lanes with revisits, bimodal light plus
    buzz presentation and a press response make these spans
    within-person tracking numbers, never comparable to Corsi norms
    (Kessels 2000; Farrell Pagulayan 2006), and the mode measures
    span rather than treating anything.
    """
    print("\n" + "=" * 62)
    print("ECHO (explicit span)")
    print("=" * 62)
    rows = echo_frame(trials)
    if rows.empty:
        _nothing("No Echo trials in this selection: nothing to score.",
                 "Play an Echo block (keyboard play is a full equal",
                 "there: the stimulus is on screen) and this chapter",
                 "fills in.")
        return None

    # Cumulative (classic Simon) blocks rehearse every prefix of one
    # growing sequence, so their spans are inflated by the game rule,
    # not the person: set them aside before anything comparative.
    stored = stored_mode_stats(metas, "echo") if metas else {}
    cum_games = {g for g, d in stored.items() if d.get("cumulative")}
    cum = rows[rows["game"].isin(cum_games)]
    rows = rows[~rows["game"].isin(cum_games)].copy()
    if len(cum):
        print(f"\n{cum['game'].nunique()} cumulative (classic Simon) "
              f"block(s), {len(cum)} trial(s), set aside: appended")
        print("material rehearses every prefix, so those spans are")
        print("play value and never pool with the ladder numbers.")
    if rows.empty:
        _nothing("Only cumulative blocks here: no ladder to score.")
        return None

    # Presentation parameters, per block. Berch 1998's review found
    # 25 years of Corsi results incomparable because labs drifted on
    # exactly these, so a block that deviates from the shipped
    # 500/1000 ms grid is named and its numbers read apart.
    rows["params"] = [parse_waveform_params_cell(c)
                      for c in rows.get("waveform_params",
                                        pd.Series("", index=rows.index))]

    per_game = []
    for game, g in rows.groupby("game"):
        ok = g[g["hit"]]
        novel_ok = ok[~ok["hebb"]]
        span_all = int(ok["len"].max()) if len(ok) else 0
        span_novel = int(novel_ok["len"].max()) if len(novel_ok) else 0
        ipis = [b - a for ms in g["press_ms"]
                for a, b in zip(ms, ms[1:])]
        p = next((pp for pp in g["params"] if pp), {})
        n_lanes = (8 if str(g["hand_mode"].iloc[0]) == "both" else 4)
        per_game.append({
            "game": game,
            "session": g["session"].iloc[0],
            "n_lanes": n_lanes,
            "trials": int(len(g)),
            "span": span_all,
            "span_novel": span_novel,
            "correct": int(g["hit"].sum()),
            "product": span_all * int(g["hit"].sum()),
            "partial": round(float(g["partial"].mean()), 2),
            "edit": round(float(g["edit_score"].mean()), 2),
            "omissions": int((g["outcome"] == "omission").sum()),
            "ipi_ms": (round(float(np.median(ipis)))
                       if ipis else np.nan),
            "item_on_ms": num_or_nan(p.get("pulse_ms")),
            "ioi_ms": num_or_nan(p.get("ioi_ms")),
        })
    pg = pd.DataFrame(per_game).sort_values(["n_lanes", "session"])
    print(f"\n{len(rows)} ladder trial(s) across {len(pg)} block(s). "
          "Per block (span = longest length with a correct")
    print("reproduction; product = span x correct, the Kessels total;")
    print("partial = mean proportion of items in their own serial")
    print("position; ipi = median gap between reproduction presses,")
    print("logged only, never scored):")
    _show(pg.set_index("game"))
    if pg["n_lanes"].nunique() > 1:
        print("Both 4-lane (unilateral) and 8-lane (bimanual) blocks")
        print("are present. A sequence that crosses hands is a harder,")
        print("different construct, so the two never pool; per-hand")
        print("span comes from unilateral blocks only.")
    dev = pg[(pg["item_on_ms"].notna() & (pg["item_on_ms"] != 500))
             | (pg["ioi_ms"].notna() & (pg["ioi_ms"] != 1000))]
    if len(dev):
        print("PROTOCOL DEVIATION: block(s) "
              + ", ".join(dev["game"]) + " ran off the standard")
        print("500/1000 ms presentation grid. Their numbers are only")
        print("comparable to blocks on the same grid (the Berch")
        print("lesson), so read them apart from the rest.")
    if (pg["span"] != pg["span_novel"]).any():
        print("span and span_novel differ somewhere above: a learned")
        print("hidden sequence carried a length the novel material")
        print("did not pass. Both are reported so the repetition")
        print("learning cannot silently inflate the span trajectory.")

    # Partial credit by length, novel against hidden repeats. The
    # per-length curve is the graded view the single span number
    # flattens (Conway 2005's argument in one plot).
    if len(rows) >= 4:
        fig, ax = plt.subplots(figsize=(7.5, 3.2))
        for flag, label, colour in ((False, "novel", "#2563eb"),
                                    (True, "hidden repeat", "#b45309")):
            g = rows[rows["hebb"] == flag]
            if g.empty:
                continue
            curve = g.groupby("len")["partial"].agg(["mean", "count"])
            ax.plot(curve.index, curve["mean"], "o-", lw=1.8,
                    color=colour, label=label)
            for length, r in curve.iterrows():
                ax.annotate(f"n={int(r['count'])}",
                            (length, r["mean"]), fontsize=6,
                            textcoords="offset points", xytext=(0, 6),
                            ha="center", color=colour)
        ax.set_xlabel("sequence length (items)")
        ax.set_ylabel("partial credit")
        ax.set_ylim(0, 1.05)
        ax.xaxis.set_major_locator(MaxNLocator(integer=True))
        ax.set_title("Echo: items in correct position, by length")
        ax.legend(frameon=False, fontsize=8)
        _save(fig, "echo_partial_by_length")
        plt.show()

    # Hebb repetition learning. Unlike buzz_hunt's span stage (a
    # DIFFERENT hidden sequence per length, so slopes fit within a
    # length only, audit finding #90), Echo's hidden material is one
    # prefix-stable stream: the length-L repeat is the first L items
    # of the same sequence, every item keeping its serial position
    # and neighbours. Exposures therefore pool across lengths here
    # by design, and the slope is per exposure of the same material.
    hebb = rows[rows["hebb"]].sort_values(
        ["session", "game", "trial_n"]).reset_index(drop=True)
    novel = rows[~rows["hebb"]]
    hebb_slope = None
    if len(hebb):
        y = hebb["partial"].to_numpy(dtype=float)
        x = np.arange(1, len(hebb) + 1, dtype=float)
        fin = np.isfinite(y)
        matched = novel[novel["len"].isin(hebb["len"].unique())]
        m_txt = (f"{matched['partial'].mean():.2f}" if len(matched)
                 else "n/a")
        print(f"\nhidden repeats: {len(hebb)} exposure(s) of the "
              f"participant stream; partial credit "
              f"{np.nanmean(y):.2f} against {m_txt} on novel")
        print("sequences at the same lengths.")
        if fin.sum() >= 3:
            hebb_slope = float(np.polyfit(x[fin], y[fin], 1)[0])
            print(f"   slope {hebb_slope:+.3f} per exposure. The "
                  "stream is derived from the participant name, so")
            print("   exposures keep accumulating across sessions")
            print("   without the player ever being told a sequence")
            print("   repeats; a positive slope pulling clear of the")
            print("   novel line is the Hebb repetition effect.")
        else:
            print("   too few scored exposures to fit a slope yet;")
            print("   it fills in as sessions accumulate.")

    # Error taxonomy over the failed reproductions.
    errs = rows.loc[~rows["hit"], "echo_error"]
    errs = errs[errs != ""]
    if len(errs):
        counts = errs.value_counts()
        print("\nfailed reproductions by error class (spatial span")
        print("leans on transpositions and omissions; a pile of")
        print("intrusions instead suggests guessing or motor slips):")
        for name, c in counts.items():
            print(f"   {name}: {c} ({c / len(errs):.0%})")
        for game, g in rows.groupby("game"):
            n_om = int((g["outcome"] == "omission").sum())
            n_wr = int((g["outcome"] == "wrong").sum())
            if n_om >= 2 and n_om > n_wr:
                print(f"   {game}: omissions outnumber wrong presses "
                      f"({n_om} to {n_wr}), so that block's span is")
                print("   likely motor- or fatigue-limited rather")
                print("   than memory-limited; read it with care.")

    # Reproduction pace, for context only: healthy adults replay a
    # span sequence at roughly 600 ms between taps (eCorsi, forward
    # condition). Reproduction here is self-paced and unscored, so
    # this is a free motor-fluency read, not a performance number.
    all_ipis = [b - a for ms in rows["press_ms"]
                for a, b in zip(ms, ms[1:])]
    if all_ipis:
        print(f"\nreproduction pace: median inter-press interval "
              f"{np.median(all_ipis):.0f} ms over "
              f"{len(all_ipis)} interval(s) (healthy-adult reference "
              "about 600 ms).")

    # The within-person trajectory, the primary outcome. span_novel
    # rides along so the Hebb inflation stays visible session over
    # session.
    n_sessions = rows["session"].nunique()
    if n_sessions >= 2:
        by_sess = (pg.groupby(["session", "n_lanes"])
                   [["span", "span_novel"]].max().reset_index())
        sessions = sorted(by_sess["session"].unique())
        xpos = {s: i for i, s in enumerate(sessions)}
        fig, ax = plt.subplots(figsize=(7.5, 3.0))
        for n_lanes, g in by_sess.groupby("n_lanes"):
            g = g.sort_values("session")
            xs = [xpos[s] for s in g["session"]]
            ax.plot(xs, g["span"], "o-", lw=1.8,
                    label=f"span ({n_lanes} lanes)")
            ax.plot(xs, g["span_novel"], "o--", lw=1.2,
                    label=f"span_novel ({n_lanes} lanes)")
        ax.set_xticks(range(len(sessions)))
        ax.set_xticklabels([str(s)[:10] for s in sessions],
                           fontsize=7, rotation=20)
        ax.set_ylabel("span (items)")
        ax.yaxis.set_major_locator(MaxNLocator(integer=True))
        ax.set_title("Echo span across sessions (within-person only)")
        ax.legend(frameon=False, fontsize=8)
        _save(fig, "echo_span_trajectory")
        plt.show()
    else:
        print("\nspan across sessions: scaffolded, one session so")
        print("far. Within-person change is the reportable quantity")
        print("for this mode; the trajectory plot fills in when")
        print("repeat sessions exist.")

    if stored:
        tbl = pd.DataFrame([{
            "game": gname,
            "span": d.get("span"),
            "correct": d.get("total_correct"),
            "product": d.get("product_score"),
            "n_lanes": d.get("n_lanes"),
            "item_on_ms": d.get("item_on_ms"),
            "ioi_ms": d.get("ioi_ms"),
            "cumulative": bool(d.get("cumulative")),
            "demo": bool(d.get("demo")),
        } for gname, d in stored.items()])
        print("\nwhat each block stored about itself (block_summary."
              "echo), the on-device cross-check for the offline")
        print("numbers above:")
        _show(tbl)

    print("\nClaim limits: these spans come from four or eight lanes")
    print("with revisits, bimodal light-plus-buzz presentation and a")
    print("press response, so they are within-person tracking numbers")
    print("and are never read against Corsi norms (Kessels 2000;")
    print("Farrell Pagulayan 2006). A wrong press can be a motor slip")
    print("rather than a memory error; the omission and pace numbers")
    print("above make that visible, not gone. Span games measure:")
    print("no therapy or transfer claim is made.")

    return {"rows": rows,
            "per_game": pg,
            "hebb_slope": hebb_slope,
            "n_trials": int(len(rows))}


# ========================================================= Cohort chapters
# The healthy baseline study (docs/research/healthy_baseline_study.txt,
# Section 4): every participant code in the sessions tree, both visits,
# both hands, read as one cohort. Everything above this line reads ONE
# selection; these sections read the whole tree on purpose, because a
# normative range or a test-retest coefficient is a property of the
# cohort, not of the save picked in the dropdown.
#
# The unit record is the long table: one row per (participant, visit,
# hand, mode, metric). Every table below is built from it, and it is
# the CSV that JASP or R reads (cohort_metrics.csv).
#
# Small-n honesty. The design analyses COHORT_N_DESIGN participants and
# sized its intervals for that number (Bonett 2002 for the ICC, paired
# power for the hand contrast). Every inferential statistic here states
# its n and refuses to print below COHORT_MIN_N, with a message that
# says how many are in and how many are missing, so a number computed
# on six people can never be read as a cohort result. Coverage counts
# still print at any n, because they are progress, not a finding.

import hashlib
from statistics import NormalDist

try:
    from scipy import stats as _sps
except ImportError:
    _sps = None

# Copy of finger_rehab/data/intake.CODE_RE: one to three letters then
# two to four digits. The notebook travels alone, so it cannot import
# the module; a change there has to be copied here.
COHORT_CODE_RE = re.compile(r"^([A-Za-z]{1,3})(\d{2,4})$")

# Section 2.1 of the design: N = 28 analysed. The refusal threshold is
# the same number so a preview can never masquerade as the result.
COHORT_N_DESIGN = 28
COHORT_MIN_N = 28

# The retest interval the design accepts, in days (Section 2.1).
COHORT_INTERVAL_DAYS = (5, 9)

# Bootstrap resamples for the paired dz interval (Section 4.4 asks
# for 10000; the seed is the notebook's own so a thesis figure
# reproduces).
COHORT_BOOT_N = 10000

# Where the cohort outputs go: beside the tree they describe, never
# inside one participant's folder. Anchored to SESSIONS_DIR for the
# same reason PATIENT_RESULTS is.
COHORT_RESULTS = Path(SESSIONS_DIR) / "cohort_results"
COHORT_CSV = "cohort_metrics.csv"

COHORT_LONG_COLS = ["participant", "visit", "day", "hand", "hand_role",
                    "mode", "metric", "value", "n_trials", "block_folder",
                    "config_hash"]

# Koo and Li 2016 bands, upper edges.
COHORT_ICC_BANDS = ((0.5, "poor"), (0.75, "moderate"), (0.9, "good"),
                    (1.01, "excellent"))

# The Buzz Hunt duration floor the staircases stop at (buzz_hunt.py
# LEVEL_FLOOR_MS, config buzz_hunt.floor_ms). A threshold sitting on
# it is censored, not measured.
COHORT_BH_FLOOR_MS = 40.0

# The equivalence margin for F3 (release error against press error),
# in percent of max. The design asks for an equivalence statement and
# names no margin, so this is a pre-set analysis constant, stated in
# the printed verdict.
COHORT_F3_MARGIN_PCT = 2.0

# The metric registry: what each number means, which direction is
# better (for the plain wording of a paired difference), the reference
# value a figure draws where one exists, and whether the design says
# the metric is poor by construction (its ICC prints with the reason
# and no MDC). Section 4.2 of the design is the source of the list.
COHORT_METRICS = {
    ("reaction", "median_rt_ms"): dict(
        unit="ms", better="lower", headline=True),
    ("reaction", "sd_rt_ms"): dict(unit="ms", better="lower"),
    ("reaction", "p10_rt_ms"): dict(unit="ms", better="lower"),
    ("reaction", "lapse_like_rate"): dict(unit="fraction", better="lower"),
    ("reaction", "false_start_rate"): dict(
        unit="fraction", better="lower", ref=0.10,
        ref_label="R1: under 10% of attempts"),
    ("reaction", "accuracy"): dict(unit="fraction", better="higher"),
    ("reaction", "rho_rt_vs_fp"): dict(
        unit="rho", better=None, ref=0.0,
        ref_label="R1: near zero (|rho| under 0.2)"),
    ("mirror", "mean_gap_ms"): dict(
        unit="ms", better="lower", headline=True, ref=60.0,
        ref_label="M1: expected under about 60 ms"),
    ("mirror", "hit_rate"): dict(unit="fraction", better="higher"),
    ("mirror", "rt_ms"): dict(unit="ms", better="lower"),
    ("rhythm", "asyn_mean_ms"): dict(
        unit="ms", better=None, headline=True, ref=0.0,
        ref_label="Rh1: zero, taps expected early (negative)"),
    ("rhythm", "asyn_sd_ms"): dict(
        unit="ms", better="lower", ref=39.0,
        ref_label="Rh2: about 39 ms (Rose 2019)"),
    ("rhythm", "asyn_abs_mean_ms"): dict(unit="ms", better="lower"),
    ("rhythm", "hit_rate"): dict(unit="fraction", better="higher"),
    ("rhythm", "interval_cv"): dict(unit="cv", better="lower"),
    ("echo", "span"): dict(
        unit="items", better="higher", headline=True, ref=6.2,
        ref_label="E1: Kessels 2000 Corsi mean 6.2 (plausibility only)"),
    ("echo", "total_correct"): dict(unit="sequences", better="higher"),
    ("echo", "product_score"): dict(unit="span x correct", better="higher"),
    ("echo", "hebb_minus_novel_acc"): dict(
        unit="fraction", better="higher", ref=0.0,
        ref_label="E2: above zero"),
    ("echo", "n_omissions"): dict(unit="trials", better="lower"),
    ("force_pilot", "mae_pct"): dict(
        unit="% of max", better="lower", headline=True),
    ("force_pilot", "rmse_pct"): dict(unit="% of max", better="lower"),
    ("force_pilot", "time_in_corridor"): dict(
        unit="fraction", better="higher", ref=0.8,
        ref_label="F4: the 0.8 promotion criterion"),
    ("force_pilot", "lag_ms"): dict(
        unit="ms", better=None, ref=200.0,
        ref_label="F2: of order 100 to 300 ms"),
    ("force_pilot", "press_mae_pct"): dict(unit="% of max", better="lower"),
    ("force_pilot", "release_mae_pct"): dict(
        unit="% of max", better="lower"),
    ("force_pilot", "hold_mae_pct"): dict(unit="% of max", better="lower"),
    ("force_pilot", "sine_mae_pct"): dict(unit="% of max", better="lower"),
    ("force_pilot", "assess_mae_pct"): dict(unit="% of max", better="lower"),
    ("force_pilot", "cov_hold"): dict(unit="cv", better="lower"),
    ("force_pilot", "lodha_ratio"): dict(unit="fraction", better=None),
    ("lighthouse", "lit_cov"): dict(unit="cv", better="lower"),
    ("lighthouse", "lit_mae_pct"): dict(
        unit="% of max", better="lower", headline=True),
    ("lighthouse", "dark_mae_pct"): dict(unit="% of max", better="lower"),
    ("lighthouse", "lit_dark_delta_pct"): dict(
        unit="% of max", better="lower", ref=0.0,
        ref_label="L1: above zero but small"),
    ("lighthouse", "drift_rate_pct_s"): dict(unit="%/s", better=None),
    ("lighthouse", "echo_abs_err_2s"): dict(unit="% of max", better="lower"),
    ("lighthouse", "echo_abs_err_10s"): dict(
        unit="% of max", better="lower"),
    ("lighthouse", "echo_const_err_pct"): dict(unit="% of max", better=None),
    ("lighthouse", "echo_var_err_pct"): dict(
        unit="% of max", better="lower"),
    ("chords", "median_er"): dict(
        unit="ratio", better="lower", headline=True, ref=0.15,
        ref_label="C1: under 0.15 (Abolins 2020)"),
    ("chords", "median_span_ms"): dict(unit="ms", better="lower"),
    ("chords", "clean_hit_rate"): dict(unit="fraction", better="higher"),
    ("chords", "hit_rate_mirror"): dict(unit="fraction", better="higher"),
    ("chords", "hit_rate_nonmirror"): dict(unit="fraction", better="higher"),
    ("chords", "median_lag_ms"): dict(unit="ms", better="lower"),
    ("buzz_hunt", "loc_accuracy"): dict(
        unit="fraction", better="higher", headline=True, ref=0.9,
        ref_label="B1: above 0.9",
        no_mdc="accuracy sits at ceiling, so its range is restricted"),
    ("buzz_hunt", "adjacent_error_share"): dict(
        unit="fraction", better=None, ref=0.5, ref_label="B2: above 0.5"),
    ("buzz_hunt", "catch_fa_rate"): dict(
        unit="fraction", better="lower", ref=0.10,
        ref_label="B3: under 0.10"),
    ("buzz_hunt", "d_prime"): dict(unit="d'", better="higher"),
    ("buzz_hunt", "threshold_final_ms"): dict(
        unit="ms", better="lower", ref=COHORT_BH_FLOOR_MS,
        ref_label="the 40 ms floor",
        no_mdc="censored at the 40 ms floor for most healthy hands"),
    ("buzz_hunt", "at_floor"): dict(unit="0/1", better=None,
                                    no_mdc="a censoring flag, not a score"),
    ("buzz_hunt", "span_max_correct"): dict(
        unit="items", better="higher", ref=4.0,
        ref_label="B4: around 4 items"),
    ("buzz_hunt", "hebb_minus_novel_acc"): dict(
        unit="fraction", better="higher", ref=0.0),
    ("pattern", "learning_score_ms"): dict(
        unit="ms", better="higher", headline=True, ref=0.0,
        ref_label="P1: above zero",
        no_mdc="individual SRTT scores pool at r = 0.28 "
               "(Oliveira 2023); group-level only"),
    ("pattern", "accuracy_rebound_pct"): dict(
        unit="% points", better="higher", ref=0.0, ref_label="P3: above zero",
        no_mdc="a probe difference, same construct as the learning score"),
    ("pattern", "random_take_rt_ms"): dict(unit="ms", better="lower"),
    ("pattern", "trained_take_rt_ms"): dict(unit="ms", better="lower"),
}

COHORT_MODES = ("reaction", "mirror", "rhythm", "echo", "force_pilot",
                "lighthouse", "chords", "buzz_hunt", "pattern")


# ------------------------------------------------------------ selection

def is_study_code(who) -> bool:
    return COHORT_CODE_RE.match(str(who or "").strip()) is not None


def cohort_hand_role(hand, dominant) -> str:
    """dominant / nondominant / both / unknown for a block's hand."""
    h = normalise_hand(hand)
    if str(hand or "").strip().lower() == "both":
        return "both"
    d = normalise_hand(dominant)
    if h is None or d is None:
        return "unknown"
    return "dominant" if h == d else "nondominant"


def cohort_config_hash(meta, mode) -> str:
    """A short hash of the config this block ran under.

    Only the mode's own section, the scoring windows and the cue
    switches go in: the session keys (participant, visit, age) differ
    for every person by design and would make every block its own
    group. A changed count, window or ladder key shows up as a split
    in the long table, the same idea pattern_consistency_groups
    already applies to one mode.
    """
    snap = meta.get("config_snapshot") or {}
    part = {"mode": snap.get(mode), "scoring": snap.get("scoring"),
            "cue": snap.get("cue")}
    text = json.dumps(part, sort_keys=True, default=str)
    return hashlib.sha256(text.encode("utf-8")).hexdigest()[:10]


def cohort_block_is_demo(meta) -> bool:
    """Test Mode blocks are a supervisor demo, never a measurement."""
    bs = meta.get("block_summary", {}) or {}
    for sub in bs.values():
        if isinstance(sub, dict) and sub.get("demo"):
            return True
    snap = meta.get("config_snapshot") or {}
    return bool((snap.get("game") or {}).get("test_mode_enabled"))


def cohort_catalogue(cat):
    """The catalogue rows that belong to the study, with the intake
    fields read off each block's metadata, plus the counts of what was
    dropped and why.

    A block is in when its who is a participant code AND its metadata
    carries a visit number: a code with no visit is a bench test that
    happened to use a code, and a name with a visit is not anonymised.
    Demo blocks and abandoned blocks are dropped and counted.
    """
    dropped = {"name_not_code": 0, "no_visit": 0, "demo": 0,
               "abandoned": 0}
    if cat is None or cat.empty:
        return pd.DataFrame(), dropped
    rows = []
    for _i, r in cat.iterrows():
        who = str(r["who"]).strip()
        if not is_study_code(who):
            dropped["name_not_code"] += 1
            continue
        meta = read_meta(Path(r["folder"]))
        visit = str(meta.get("visit", "") or "").strip()
        if not visit:
            dropped["no_visit"] += 1
            continue
        if cohort_block_is_demo(meta):
            dropped["demo"] += 1
            continue
        bs = meta.get("block_summary", {}) or {}
        if str(bs.get("status", "")) != "completed":
            dropped["abandoned"] += 1
            continue
        bat = meta.get("battery") or {}
        bat_cell = bat.get("cell") or {}
        cell_label = str(bat_cell.get("mode_order", ""))
        if bat_cell.get("hand_first") == "non_dominant":
            cell_label += " non-dominant first"
        rows.append({
            **{c: r[c] for c in cat.columns},
            "participant": who.upper() if who[:1].isalpha() else who,
            "visit": visit,
            "dominant_hand": normalise_hand(meta.get("dominant_hand")),
            "sex": str(meta.get("sex", "") or ""),
            "age": str(meta.get("age", "") or ""),
            "edinburgh_lq": str(meta.get("edinburgh_lq", "") or ""),
            "hand_length_mm": str(meta.get("hand_length_mm", "") or ""),
            "hand_breadth_mm": str(meta.get("hand_breadth_mm", "") or ""),
            "battery_position": int(bat.get("position", 0) or 0),
            "cell": cell_label,
        })
    sel = pd.DataFrame(rows)
    return sel, dropped


def cohort_people(sel) -> pd.DataFrame:
    """One row per participant: intake fields, visits and days."""
    if sel is None or sel.empty:
        return pd.DataFrame()
    out = []
    for who, g in sel.groupby("participant"):
        dom = g["dominant_hand"].dropna()
        dom = dom.mode().iloc[0] if len(dom) else None
        visits = sorted(g["visit"].unique(), key=lambda v: str(v))
        days = {v: sorted(g.loc[g["visit"] == v, "day"].unique())
                for v in visits}
        lq = pd.to_numeric(pd.Series(g["edinburgh_lq"].unique()),
                           errors="coerce").dropna()
        lq_val = float(lq.iloc[0]) if len(lq) else np.nan
        lq_hand = (None if pd.isna(lq_val) else
                   ("right" if lq_val > 0 else
                    "left" if lq_val < 0 else None))
        flags = []
        if pd.notna(lq_val) and -40 <= lq_val <= 40:
            flags.append("ambidextrous range (LQ within -40 to +40)")
        if lq_hand and dom and lq_hand != dom:
            flags.append(f"LQ says {lq_hand}, label says {dom}")
        if g["dominant_hand"].dropna().nunique() > 1:
            flags.append("dominant hand differs between blocks")
        interval = np.nan
        if len(visits) >= 2 and days[visits[0]] and days[visits[1]]:
            try:
                d1 = pd.Timestamp(days[visits[0]][0])
                d2 = pd.Timestamp(days[visits[1]][0])
                interval = float((d2 - d1).days)
            except (ValueError, TypeError):
                interval = np.nan
        out.append({
            "participant": who,
            "dominant_hand": dom,
            "sex": next((s for s in g["sex"] if s), ""),
            "age": next((a for a in g["age"] if a), ""),
            "edinburgh_lq": lq_val,
            "cell": next((c for c in g["cell"] if c), ""),
            "visits": len(visits),
            "days": "; ".join(f"v{v}: {', '.join(days[v])}"
                              for v in visits),
            "interval_days": interval,
            "blocks": int(len(g)),
            "flags": "; ".join(flags),
        })
    return pd.DataFrame(out).sort_values("participant").reset_index(
        drop=True)


# --------------------------------------------------- per-mode metric rows
# Each builder returns (hand, metric, value, n) tuples for one block.
# Values come from block_stats where the mode computes them and from
# the offline re-score functions above where those exist; the design
# (Section 4.1) names which is which per mode. None and NaN values are
# skipped at emit time so a missing number is a missing row, never a
# zero.

def _emit(out, hand, metric, value, n):
    try:
        v = float(value)
    except (TypeError, ValueError):
        return
    if not np.isfinite(v):
        return
    try:
        n_int = int(n) if n is not None and not pd.isna(n) else 0
    except (TypeError, ValueError):
        n_int = 0
    out.append((str(hand), metric, v, n_int))


def _cohort_reaction(block):
    out = []
    st = block["bs"].get("reaction") or {}
    hand = block["hand"]
    if hand in ("left", "right") and st:
        n_valid = int(st.get("n_valid") or 0)
        n_sc = int(st.get("n_scorable") or 0)
        n_att = int(st.get("n_attempts") or 0)
        for key, name, n in (("median_rt_ms", "median_rt_ms", n_valid),
                             ("sd_rt_ms", "sd_rt_ms", n_valid),
                             ("p10_rt_ms", "p10_rt_ms", n_valid),
                             ("lapse_like_rate", "lapse_like_rate", n_sc),
                             ("accuracy", "accuracy", n_sc),
                             ("spearman_rho_rt_vs_fp", "rho_rt_vs_fp",
                              n_valid)):
            _emit(out, hand, name, st.get(key), n)
        if n_att:
            _emit(out, hand, "false_start_rate",
                  float(st.get("n_false_start_total") or 0) / n_att, n_att)
        return out
    # A bilateral block pools both hands in block_stats, so the split
    # comes from the trial rows instead (the design never runs one,
    # but a free pick can).
    rx = reaction_frame(block["rows"])
    if rx.empty:
        return out
    scored = rx[~rx["is_event"]]
    for side, g in scored.groupby("side"):
        label = g["early_late"].fillna("").astype(str)
        rt = pd.to_numeric(g["time_difference_ms"], errors="coerce")
        valid = rt[(label != "Miss") & rt.notna() & (rt >= ANTICIPATION_MS)]
        if len(valid) >= 3:
            _emit(out, side, "median_rt_ms", valid.median(), len(valid))
            _emit(out, side, "sd_rt_ms", valid.std(ddof=1), len(valid))
            _emit(out, side, "p10_rt_ms", valid.quantile(0.10), len(valid))
            _emit(out, side, "lapse_like_rate",
                  float((valid > LAPSE_MS).mean()), len(valid))
        if len(g):
            _emit(out, side, "accuracy", float((label != "Miss").mean()),
                  len(g))
    return out


def _cohort_mirror(block):
    out = []
    bs = block["bs"]
    st = bs.get("mirror") or {}
    rows = block["rows"]
    _emit(out, "both", "mean_gap_ms", st.get("mean_gap_ms"),
          st.get("n_clean_pairs"))
    _emit(out, "both", "hit_rate", bs.get("hit_rate"), bs.get("trials"))
    for side, key, col in (("right", "right_hand_mean_rt_ms",
                            "mirror_right_rt_ms"),
                           ("left", "left_hand_mean_rt_ms",
                            "mirror_left_rt_ms")):
        n = (int(pd.to_numeric(rows[col], errors="coerce").notna().sum())
             if col in rows.columns else st.get("n_clean_pairs"))
        _emit(out, side, "rt_ms", st.get(key), n)
    return out


def _cohort_rhythm(block):
    out = []
    rows = block["rows"]
    hand = block["hand"]
    rhy = rhythm_rows(rows)
    allr = rows[rows["mode"] == "rhythm"] if "mode" in rows.columns \
        else rows
    sides = ["right", "left"] if hand == "both" else [hand]
    for side in sides:
        g = rhy[rhy["side"] == side] if hand == "both" else rhy
        ga = allr[allr["side"] == side] if hand == "both" else allr
        off = pd.to_numeric(g["time_difference_ms"], errors="coerce") \
            .dropna() if len(g) else pd.Series(dtype="float64")
        if len(off) >= 3:
            _emit(out, side, "asyn_mean_ms", off.mean(), len(off))
            _emit(out, side, "asyn_sd_ms", off.std(ddof=1), len(off))
            _emit(out, side, "asyn_abs_mean_ms", off.abs().mean(), len(off))
        if len(ga):
            label = ga["early_late"].fillna("").astype(str)
            _emit(out, side, "hit_rate", float((label != "Miss").mean()),
                  len(ga))
        if len(g) >= 3 and "song_time_s" in g.columns:
            press, _note = tap_series(g)
            cv = interval_cv(press)
            if cv is not None:
                _emit(out, side, "interval_cv", cv, len(press))
    return out


def _cohort_echo(block):
    out = []
    st = block["bs"].get("echo") or {}
    if st.get("cumulative"):
        # Classic Simon rehearses every prefix; its span is a game
        # rule, not the person (sec_echo says the same).
        return out
    ef = echo_frame(block["rows"])
    if ef.empty:
        return out
    hand = block["hand"]
    ok = ef[ef["hit"]]
    span = int(ok["len"].max()) if len(ok) else 0
    correct = int(ef["hit"].sum())
    _emit(out, hand, "span", span, len(ef))
    _emit(out, hand, "total_correct", correct, len(ef))
    _emit(out, hand, "product_score", span * correct, len(ef))
    _emit(out, hand, "n_omissions",
          int((ef["outcome"] == "omission").sum()), len(ef))
    hebb = ef[ef["hebb"]]
    novel = ef[~ef["hebb"] & ef["len"].isin(hebb["len"].unique())]
    if len(hebb) and len(novel):
        # Partial credit (Conway 2005), matched on length, so the
        # repeated material is scored against novel sequences of the
        # same load.
        _emit(out, hand, "hebb_minus_novel_acc",
              hebb["partial"].mean() - novel["partial"].mean(),
              len(hebb) + len(novel))
    block["extra"]["echo_errors"] = ef.loc[~ef["hit"], "echo_error"] \
        .tolist()
    return out


def _cohort_chords(block):
    out = []
    st = block["bs"].get("chords") or {}
    per_hand = st.get("per_hand") or {}
    for h, d in per_hand.items():
        if not isinstance(d, dict):
            continue
        _emit(out, h, "median_er", d.get("median_er"), d.get("n_chords"))
        _emit(out, h, "median_span_ms", d.get("median_span_ms"),
              d.get("n_chords"))
    cf = chord_frame(block["rows"], block.get("calset"))
    if len(cf):
        chords = cf[cf["kind"] == "chord"]
        for side, g in chords.groupby("side"):
            _emit(out, side, "clean_hit_rate", float(g["clean"].mean()),
                  len(g))
        block["extra"]["chord_rows"] = chords[
            ["side", "chord", "d", "clean", "er", "press_level",
             "leaks"]].copy()
    cross = st.get("cross") or {}
    if cross.get("n_chords"):
        n = int(cross.get("n_chords") or 0)
        _emit(out, "both", "hit_rate_mirror", cross.get("hit_rate_mirror"),
              n)
        _emit(out, "both", "hit_rate_nonmirror",
              cross.get("hit_rate_nonmirror"), n)
        _emit(out, "both", "median_lag_ms", cross.get("median_lag_ms"), n)
    per_chord = st.get("per_chord") or []
    if per_chord:
        block["extra"]["per_chord"] = [dict(p) for p in per_chord
                                       if isinstance(p, dict)]
    return out


def _cohort_force_pilot(block):
    out = []
    runs, _dropped = force_tracking_runs(
        [block["folder"]], block["rows"],
        metas={block["game"]: block["meta"]})
    if runs.empty:
        return out
    runs = runs[~runs["demo"].astype(bool)]
    for hand, g in runs.groupby("hand"):
        n = len(g)
        _emit(out, hand, "mae_pct", g["mae"].mean(), n)
        _emit(out, hand, "rmse_pct", g["rmse"].mean(), n)
        _emit(out, hand, "time_in_corridor", g["tic"].mean(), n)
        _emit(out, hand, "lag_ms", g["lag_ms"].mean(), n)
        _emit(out, hand, "press_mae_pct", g["press_mae"].mean(), n)
        _emit(out, hand, "release_mae_pct", g["release_mae"].mean(), n)
        _emit(out, hand, "sine_mae_pct", g["sine_mae"].mean(), n)
        _emit(out, hand, "assess_mae_pct", g["assess_mae"].mean(), n)
        if "hold_mae" in g.columns:
            _emit(out, hand, "hold_mae_pct", g["hold_mae"].mean(), n)
        _emit(out, hand, "cov_hold", g["cov_hold"].mean(), n)
        both = g["low_n"] + g["high_n"]
        ratio = (g["low_n"] / both).where(both > 0)
        _emit(out, hand, "lodha_ratio", ratio.mean(), int(ratio.notna().sum()))
    block["extra"]["force_runs"] = runs
    return out


def _cohort_lighthouse(block):
    out = []
    hold_rows, echo_rows = precision_hold_rows(block["rows"])
    holds = pd.DataFrame(
        [h for h in (score_hold(r) for _i, r in hold_rows.iterrows())
         if h is not None])
    echoes = pd.DataFrame(
        [e for e in (score_echo(r) for _i, r in echo_rows.iterrows())
         if e is not None])
    if len(holds):
        for hand, g in holds.groupby("hand"):
            n = len(g)
            _emit(out, hand, "lit_cov", g["lit_cov"].mean(), n)
            _emit(out, hand, "lit_mae_pct", g["lit_mae"].mean(), n)
            wd = g.dropna(subset=["dark_mae"])
            if len(wd):
                _emit(out, hand, "dark_mae_pct", wd["dark_mae"].mean(),
                      len(wd))
                _emit(out, hand, "lit_dark_delta_pct",
                      (wd["dark_mae"] - wd["lit_mae"]).mean(), len(wd))
            dr = g["drift_rate"].dropna()
            if len(dr):
                _emit(out, hand, "drift_rate_pct_s", dr.mean(), len(dr))
        block["extra"]["lighthouse_holds"] = holds
    if len(echoes):
        for hand, g in echoes.groupby("hand"):
            for delay, name in ((2.0, "echo_abs_err_2s"),
                                (10.0, "echo_abs_err_10s")):
                d = g[np.isclose(g["delay_s"], delay)]
                if len(d):
                    _emit(out, hand, name, d["err"].abs().mean(), len(d))
            _emit(out, hand, "echo_const_err_pct", g["err"].mean(), len(g))
            if len(g) >= 2:
                _emit(out, hand, "echo_var_err_pct", g["err"].std(ddof=1),
                      len(g))
    return out


def _cohort_buzz_hunt(block):
    out = []
    st = block["bs"].get("buzz_hunt") or {}
    bf = bh_frame(block["rows"])
    hand = block["hand"]
    snap_floor = ((block["meta"].get("config_snapshot") or {})
                  .get("buzz_hunt") or {}).get("floor_ms")
    floor = float(snap_floor) if snap_floor else COHORT_BH_FLOOR_MS
    if len(bf):
        loc = bf[bf["stage"] == "loc"].copy()
        loc["h"] = loc["kv"].map(lambda kv: str(kv.get("hand", hand)))
        scored = loc[~loc["is_event"]]
        catches = loc[loc["is_event"]]
        z = NormalDist().inv_cdf
        for h in sorted(loc["h"].unique()):
            s = scored[scored["h"] == h]
            c = catches[catches["h"] == h]
            if len(s):
                _emit(out, h, "loc_accuracy", float(s["hit"].mean()), len(s))
                adjacent, n_err = 0, 0
                for _i, r in s[~s["hit"]].iterrows():
                    keys = r.get("keys_pressed")
                    keys = "" if pd.isna(keys) else str(keys)
                    first = keys.split(",")[0].strip()
                    if not first or pd.isna(r.get("lane")):
                        continue
                    stim = int(r["lane"]) - 1
                    resp = int(float(first)) - 1
                    n_err += 1
                    if resp // 4 == stim // 4 and abs(resp - stim) == 1:
                        adjacent += 1
                if n_err:
                    _emit(out, h, "adjacent_error_share",
                          adjacent / n_err, n_err)
            if len(c):
                n_fa = int((c["error_type"].fillna("").astype(str)
                            == "catch_false_start").sum())
                _emit(out, h, "catch_fa_rate", n_fa / len(c), len(c))
                if len(s):
                    responded = (s["keys_pressed"].fillna("").astype(str)
                                 .str.strip() != "")
                    hit_rate = (responded.sum() + 0.5) / (len(s) + 1.0)
                    fa_rate = (n_fa + 0.5) / (len(c) + 1.0)
                    _emit(out, h, "d_prime", z(hit_rate) - z(fa_rate),
                          len(s) + len(c))
    thr = st.get("threshold") or {}
    for h, d in thr.items():
        if not isinstance(d, dict) or d.get("final_ms") is None:
            continue
        final = float(d["final_ms"])
        _emit(out, h, "threshold_final_ms", final, d.get("n_reversals"))
        _emit(out, h, "at_floor", 1.0 if final <= floor + 1e-6 else 0.0,
              d.get("n_reversals"))
    span = st.get("span") or {}
    if span.get("trials"):
        _emit(out, hand, "span_max_correct", span.get("max_correct"),
              span.get("trials"))
        hb = (span.get("hebb") or {}).get("accuracy")
        nv = (span.get("novel") or {}).get("accuracy")
        if hb is not None and nv is not None:
            _emit(out, hand, "hebb_minus_novel_acc", float(hb) - float(nv),
                  span.get("trials"))
    return out


def _cohort_pattern(block):
    out = []
    st = block["bs"].get("pattern") or {}
    hand = block["hand"]
    trims = {block["game"]: int(st.get("start_trim") or 0)}
    takes = pattern_take_table(block["rows"], start_trim_by_game=trims)
    if takes.empty:
        return out
    scores = pattern_learning_scores(takes)
    scores = scores[scores["learning_score_ms"].notna()] \
        if len(scores) else scores
    if len(scores):
        _emit(out, hand, "learning_score_ms",
              scores["learning_score_ms"].mean(), len(scores))
    reb = [d.get("accuracy_rebound_pct")
           for d in (st.get("probe_scores") or [])
           if isinstance(d, dict)
           and d.get("accuracy_rebound_pct") is not None]
    if reb:
        _emit(out, hand, "accuracy_rebound_pct", float(np.mean(reb)),
              len(reb))
    rnd = takes[takes["kind"] == "random"].dropna(subset=["rt_ms"])
    if len(rnd):
        _emit(out, hand, "random_take_rt_ms", rnd["rt_ms"].mean(),
              int(rnd["n_rt"].sum()))
    seq = takes[takes["kind"] == "seq"].dropna(subset=["rt_ms"])
    if len(seq):
        _emit(out, hand, "trained_take_rt_ms", seq["rt_ms"].mean(),
              int(seq["n_rt"].sum()))
    block["extra"]["pattern_takes"] = takes
    return out


COHORT_BUILDERS = {
    "reaction": _cohort_reaction,
    "mirror": _cohort_mirror,
    "rhythm": _cohort_rhythm,
    "echo": _cohort_echo,
    "chords": _cohort_chords,
    "force_pilot": _cohort_force_pilot,
    "lighthouse": _cohort_lighthouse,
    "buzz_hunt": _cohort_buzz_hunt,
    "pattern": _cohort_pattern,
}


def cohort_long_table(sel, trials, metas, calset=None):
    """The long table plus the per-mode frames the validity checks
    read (holds, chord rows, echo errors): one row per (participant,
    visit, hand, mode, metric), values as the builders above emit
    them. Two completed blocks of the same kind at one visit both
    stay in, told apart by block_folder; cohort_values averages them
    when a per-participant number is needed and says so."""
    frames = {}
    rows = []
    if sel is None or sel.empty:
        return pd.DataFrame(columns=COHORT_LONG_COLS), frames
    for _i, r in sel.iterrows():
        game = game_key(r["folder"])
        meta = metas.get(game, {}) or {}
        bs = meta.get("block_summary", {}) or {}
        mode = str(bs.get("block") or r["mode"])
        build = COHORT_BUILDERS.get(mode)
        if build is None:
            continue
        g = trials[trials["game"] == game] if not trials.empty \
            else trials
        block = {"game": game, "folder": Path(r["folder"]), "meta": meta,
                 "bs": bs, "rows": g, "hand": str(meta.get("hand", "?")),
                 "calset": calset, "extra": {}}
        try:
            emitted = build(block)
        except Exception as e:      # one broken block must not sink the cohort
            print(f"WARNING: {mode} block {game} could not be scored "
                  f"({type(e).__name__}: {e}); skipped.")
            continue
        for key, val in block["extra"].items():
            frames.setdefault(key, []).append(
                (r["participant"], str(r["visit"]), val))
        chash = cohort_config_hash(meta, mode)
        for hand, metric, value, n in emitted:
            rows.append({
                "participant": r["participant"],
                "visit": str(r["visit"]),
                "day": r["day"],
                "hand": hand,
                "hand_role": cohort_hand_role(hand, r["dominant_hand"]),
                "mode": mode,
                "metric": metric,
                "value": value,
                "n_trials": n,
                "block_folder": Path(r["folder"]).name,
                "config_hash": chash,
            })
    long = pd.DataFrame(rows, columns=COHORT_LONG_COLS)
    if not long.empty:
        long = long.sort_values(["mode", "metric", "participant", "visit",
                                 "hand"]).reset_index(drop=True)
    return long, frames


# ------------------------------------------------------------ statistics

def cohort_gate(n, min_n, what) -> bool:
    """True when n reaches the design minimum. Otherwise the refusal
    is printed in plain words and the caller prints no statistic."""
    if n >= min_n:
        return True
    print(f"{what}: {n} participant(s) with usable data; the design "
          f"analyses {min_n}, so no statistic is printed until "
          f"{min_n - n} more are in.")
    return False


def cohort_values(long, mode, metric, visit, hand_role=None):
    """One value per participant for a metric at a visit: the block's
    value, averaged when a participant has two completed blocks of
    the same kind. hand_role None pools the hands by averaging."""
    if long is None or long.empty:
        return pd.Series(dtype="float64")
    m = (long["mode"] == mode) & (long["metric"] == metric) \
        & (long["visit"].astype(str) == str(visit))
    if hand_role is not None:
        m &= long["hand_role"] == hand_role
    g = long[m]
    if g.empty:
        return pd.Series(dtype="float64")
    return g.groupby("participant")["value"].mean().astype(float)


def icc_ci(mat, alpha=0.05) -> dict:
    """ICC(2,1) and ICC(3,1) with exact F-based intervals.

    ICC(2,1): two-way random effects, absolute agreement, single
    measure (Shrout and Fleiss 1979 form 2,1; the same value
    icc_two_one returns). ICC(3,1): consistency. The interval
    formulae are McGraw and Wong 1996 (Table 7), the ones pingouin and
    the R psych package implement; the test suite checks them against
    the Shrout and Fleiss worked example (ICC(2,1) 0.29, 0.019 to
    0.761; ICC(3,1) 0.71, 0.342 to 0.946). mat is participants in
    rows, visits in columns, no NaNs.
    """
    from scipy import stats as sps
    m = np.asarray(mat, dtype=float)
    nan = dict(n=0, k=0, icc21=np.nan, lo21=np.nan, hi21=np.nan,
               icc31=np.nan, lo31=np.nan, hi31=np.nan, reason="")
    if m.ndim != 2:
        return dict(nan, reason="not a two-way table")
    n, k = m.shape
    if n < 2 or k < 2:
        return dict(nan, n=n, k=k, reason="needs two participants and "
                                           "two visits")
    grand = m.mean()
    row_m = m.mean(axis=1)
    col_m = m.mean(axis=0)
    msr = k * ((row_m - grand) ** 2).sum() / (n - 1)
    msc = n * ((col_m - grand) ** 2).sum() / (k - 1)
    mse = (((m - row_m[:, None] - col_m[None, :] + grand) ** 2).sum()
           / ((n - 1) * (k - 1)))
    denom21 = msr + (k - 1) * mse + k * (msc - mse) / n
    denom31 = msr + (k - 1) * mse
    if denom21 <= 0 or denom31 <= 0:
        # Every value equal: nothing between people to agree on.
        return dict(nan, n=n, k=k, reason="no variance at all")
    icc21 = (msr - mse) / denom21
    icc31 = (msr - mse) / denom31
    out = dict(n=n, k=k, icc21=float(icc21), icc31=float(icc31),
               lo21=np.nan, hi21=np.nan, lo31=np.nan, hi31=np.nan,
               reason="")
    if mse <= 0:
        # No error term: every participant repeated their own value
        # exactly (a coarse integer such as a span can do this). The
        # coefficients are defined, ICC(3,1) is 1 and ICC(2,1) is 1
        # unless a uniform shift separates the visits, but the F
        # interval divides by the error term, so both intervals are
        # the limit as that term goes to zero: exact at 1 for the
        # consistency form, and for the agreement form the same
        # limit whenever the shift is zero too. Reported with a
        # note rather than as a missing value, because a reader
        # would otherwise take "undefined" for "unreliable".
        out.update(lo31=1.0, hi31=1.0,
                   reason="every participant repeated their value "
                          "exactly, so the interval is the zero-error "
                          "limit")
        if icc21 >= 1.0 - 1e-12:
            out.update(icc21=1.0, lo21=1.0, hi21=1.0)
            return out
    if icc21 < 1:
        a = k * icc21 / (n * (1 - icc21))
        b = 1 + k * icc21 * (n - 1) / (n * (1 - icc21))
        v = ((a * msc + b * mse) ** 2
             / ((a * msc) ** 2 / (k - 1)
                + (b * mse) ** 2 / ((n - 1) * (k - 1))))
        f1 = sps.f.ppf(1 - alpha / 2, n - 1, v)
        f2 = sps.f.ppf(1 - alpha / 2, v, n - 1)
        out["lo21"] = float(n * (msr - f1 * mse)
                            / (f1 * (k * msc + (k * n - k - n) * mse)
                               + n * msr))
        out["hi21"] = float(n * (f2 * msr - mse)
                            / (k * msc + (k * n - k - n) * mse
                               + n * f2 * msr))
    if mse > 0:
        fobs = msr / mse
        fl = fobs / sps.f.ppf(1 - alpha / 2, n - 1, (n - 1) * (k - 1))
        fu = fobs * sps.f.ppf(1 - alpha / 2, (n - 1) * (k - 1), n - 1)
        out["lo31"] = float((fl - 1) / (fl + k - 1))
        out["hi31"] = float((fu - 1) / (fu + k - 1))
    return out


def icc_band(icc) -> str:
    if icc is None or not np.isfinite(icc):
        return ""
    for edge, name in COHORT_ICC_BANDS:
        if icc < edge:
            return name
    return "excellent"


def cohort_t_ci(values, alpha=0.05):
    """(mean, lo, hi) with the t interval; NaNs under three values."""
    v = np.asarray(pd.Series(values, dtype="float64").dropna(), dtype=float)
    if len(v) < 3:
        return (float(v.mean()) if len(v) else np.nan, np.nan, np.nan)
    sd = v.std(ddof=1)
    if _sps is None:
        return (float(v.mean()), np.nan, np.nan)
    h = _sps.t.ppf(1 - alpha / 2, len(v) - 1) * sd / np.sqrt(len(v))
    return (float(v.mean()), float(v.mean() - h), float(v.mean() + h))


def cohort_dz_ci(diff, n=COHORT_BOOT_N, seed=BOOT_SEED):
    """Cohen dz (mean over SD of the differences) with a percentile
    bootstrap interval, seeded."""
    d = np.asarray(pd.Series(diff, dtype="float64").dropna(), dtype=float)
    if len(d) < 3 or d.std(ddof=1) == 0:
        return (np.nan, np.nan, np.nan)
    dz = float(d.mean() / d.std(ddof=1))
    rng = _rng(seed)
    idx = rng.integers(0, len(d), size=(n, len(d)))
    draws = d[idx]
    sds = draws.std(axis=1, ddof=1)
    ok = sds > 0
    if ok.sum() < 20:
        return (dz, np.nan, np.nan)
    boot = draws[ok].mean(axis=1) / sds[ok]
    return (dz, float(np.quantile(boot, 0.025)),
            float(np.quantile(boot, 0.975)))


def cohort_rank_biserial(diff):
    """Matched-pairs rank-biserial correlation, the effect size that
    goes with the Wilcoxon signed-rank test."""
    d = np.asarray(pd.Series(diff, dtype="float64").dropna(), dtype=float)
    d = d[d != 0]
    if len(d) == 0 or _sps is None:
        return np.nan
    ranks = _sps.rankdata(np.abs(d))
    w_pos = ranks[d > 0].sum()
    w_neg = ranks[d < 0].sum()
    tot = w_pos + w_neg
    return float((w_pos - w_neg) / tot) if tot > 0 else np.nan


def cohort_paired(a, b, alternative="two-sided"):
    """One paired comparison a minus b on two Series aligned by index.

    Paired t with its interval and Cohen dz (bootstrap CI), Shapiro-Wilk
    on the differences, and the Wilcoxon signed-rank test with the
    rank-biserial effect size. `test` names which p the plain wording
    uses: Wilcoxon when the differences are non-normal (Shapiro p under
    0.05) or n is under 20, paired t otherwise. Both are always in the
    row so a reader sees when they disagree (Section 4.4).
    """
    pair = pd.concat([pd.Series(a, dtype="float64"),
                      pd.Series(b, dtype="float64")], axis=1,
                     keys=["a", "b"]).dropna()
    out = dict(n=int(len(pair)), mean_a=np.nan, mean_b=np.nan,
               diff=np.nan, ci_lo=np.nan, ci_hi=np.nan, t_p=np.nan,
               dz=np.nan, dz_lo=np.nan, dz_hi=np.nan, shapiro_p=np.nan,
               w_p=np.nan, rank_biserial=np.nan, test="", p=np.nan)
    if len(pair) < 3:
        return out
    d = pair["a"] - pair["b"]
    out["mean_a"] = float(pair["a"].mean())
    out["mean_b"] = float(pair["b"].mean())
    out["diff"], out["ci_lo"], out["ci_hi"] = cohort_t_ci(d)
    out["dz"], out["dz_lo"], out["dz_hi"] = cohort_dz_ci(d)
    if _sps is None:
        out["test"] = "scipy missing"
        return out
    try:
        out["t_p"] = float(_sps.ttest_rel(pair["a"], pair["b"],
                                          alternative=alternative).pvalue)
    except ValueError:
        pass
    if len(d) >= 3 and d.std(ddof=1) > 0:
        try:
            out["shapiro_p"] = float(_sps.shapiro(d).pvalue)
        except ValueError:
            pass
    if (d != 0).any():
        try:
            out["w_p"] = float(_sps.wilcoxon(
                d, alternative=alternative).pvalue)
        except ValueError:
            pass
        out["rank_biserial"] = cohort_rank_biserial(d)
    nonnormal = pd.notna(out["shapiro_p"]) and out["shapiro_p"] < 0.05
    if not (d != 0).any():
        # Every pair identical: nothing to test, and saying so beats a
        # row of NaNs.
        out["test"] = "no variation"
    elif nonnormal or len(pair) < 20:
        out["test"] = "wilcoxon"
        out["p"] = out["w_p"]
    else:
        out["test"] = "paired t"
        out["p"] = out["t_p"]
    return out


def cohort_stat_words(row) -> str:
    """'n 28, wilcoxon p 0.031, dz -0.62' or, when the pairs never
    differed, 'n 28, no variation between the pairs'."""
    n = int(row["n"])
    if row.get("test") == "no variation" or pd.isna(row.get("p")):
        return f"n {n}, no variation between the pairs"
    return f"n {n}, {row['test']} p {row['p']:.3f}, dz {row['dz']:+.2f}"


def _fmt(v) -> str:
    """Four significant figures, so 0.1996 does not print as 0.2 next
    to a 0.2 criterion."""
    try:
        f = float(v)
    except (TypeError, ValueError):
        return str(v)
    if not np.isfinite(f):
        return ""
    return f"{f:.4g}"


def cohort_one_sample(values, mu=0.0, alternative="two-sided"):
    """A one-sample test against mu, the same shape as cohort_paired."""
    v = pd.Series(values, dtype="float64").dropna()
    ref = pd.Series(mu, index=v.index, dtype="float64")
    return cohort_paired(v, ref, alternative=alternative)


def cohort_direction(mode, metric, diff, unit, label_a="dominant",
                     label_b="non-dominant") -> str:
    """Plain words for a difference a minus b: which side is higher, by
    how much, and whether that is better, from the metric registry."""
    if diff is None or not np.isfinite(diff):
        return "no difference computed"
    spec = COHORT_METRICS.get((mode, metric), {})
    better = spec.get("better")
    if abs(diff) < 1e-12:
        return f"{label_a} equals {label_b}"
    side = label_a if diff > 0 else label_b
    other = label_b if diff > 0 else label_a
    word = "higher"
    tail = ""
    if better == "lower":
        word = "higher"
        tail = f", so {other} is better"
    elif better == "higher":
        tail = f", so {side} is better"
    return f"{side} {word} than {other} by {abs(diff):.3g} {unit}{tail}"


def _cohort_frame_rows(frames, key):
    """Every stored frame under `key` as one DataFrame with participant
    and visit columns, or an empty frame."""
    parts = []
    for who, visit, val in frames.get(key, []):
        if isinstance(val, pd.DataFrame) and len(val):
            v = val.copy()
            v["participant"] = who
            v["visit"] = visit
            parts.append(v)
    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()


def _cohort_point_outputs(out_dir):
    """Send figures into the cohort folder, not into the last session
    picked. The capture records absolute figure paths, so the
    per-session report written afterwards still finds its own."""
    global FIGDIR, _FIGS_CLEARED, _COHORT_PREV_FIGDIR
    _COHORT_PREV_FIGDIR = FIGDIR
    FIGDIR = Path(out_dir) / "figures"
    FIGDIR.mkdir(parents=True, exist_ok=True)
    for old in FIGDIR.glob("*.png"):
        try:
            old.unlink()
        except OSError:
            pass
    _FIGS_CLEARED = True
    _CAPTURE["figs"] = _fig_state()


_COHORT_PREV_FIGDIR = None


# ------------------------------------------------------------- sections

def sec_cohort_selection(cat, root=None, min_n=None):
    """Select the cohort and build the long table.

    Everything the other cohort sections read comes back in one dict:
    the catalogue rows kept, the participants, the trials and metadata
    loaded for them, the long table and the per-mode frames the
    validity checks need. min_n overrides COHORT_MIN_N for a dry run
    on a pilot or a synthetic tree; the default is the design's
    analysed sample.
    """
    print("\n" + "=" * 62)
    print("COHORT: SELECTION AND THE LONG TABLE")
    print("=" * 62)
    min_n = int(min_n if min_n is not None else COHORT_MIN_N)
    out_dir = Path(root or SESSIONS_DIR) / "cohort_results"
    cohort = {"sel": pd.DataFrame(), "people": pd.DataFrame(),
              "trials": pd.DataFrame(), "metas": {}, "calset": None,
              "long": pd.DataFrame(columns=COHORT_LONG_COLS),
              "frames": {}, "dropped": {}, "min_n": min_n,
              "out_dir": out_dir, "tables": {}}
    sel, dropped = cohort_catalogue(cat)
    cohort["dropped"] = dropped
    print(f"design: N = {COHORT_N_DESIGN} analysed, two visits "
          f"{COHORT_INTERVAL_DAYS[0]} to {COHORT_INTERVAL_DAYS[1]} days "
          f"apart; statistics print from n = {min_n}.")
    print("dropped: "
          + ", ".join(f"{k.replace('_', ' ')} {v}"
                      for k, v in dropped.items()))
    if sel.empty:
        _nothing("No study blocks on disk: a block counts when its",
                 "participant is a code (P01, P02, ...), its metadata",
                 "carries a visit, it completed and it is not a demo.",
                 "Every cohort section below says so rather than",
                 "falling over.")
        return cohort
    _cohort_point_outputs(out_dir)
    people = cohort_people(sel)
    folders = [Path(p) for p in sel["folder"]]
    metas = load_metas(folders)
    trials = load_games(folders, cat)
    unit = force_unit(metas)
    calset = calibration_factors(metas, unit)
    trials = add_force_columns(trials, calset)
    long, frames = cohort_long_table(sel, trials, metas, calset)
    cohort.update({"sel": sel, "people": people, "trials": trials,
                   "metas": metas, "calset": calset, "long": long,
                   "frames": frames})

    both = people[people["visits"] >= 2]
    print(f"\n{len(people)} participant(s), {len(both)} with two visits, "
          f"{len(sel)} completed block(s), {len(trials)} trial(s).")
    if len(both):
        iv = both["interval_days"].dropna()
        lo, hi = COHORT_INTERVAL_DAYS
        off = both[(both["interval_days"] < lo) | (both["interval_days"] > hi)]
        if len(iv):
            print(f"retest interval: median {iv.median():.0f} days "
                  f"(range {iv.min():.0f} to {iv.max():.0f}); "
                  f"{len(off)} outside {lo} to {hi} days"
                  + (": " + ", ".join(off["participant"]) if len(off)
                     else ""))
    flagged = people[people["flags"] != ""]
    if len(flagged):
        print("intake flags:")
        for _i, r in flagged.iterrows():
            print(f"   {r['participant']}: {r['flags']}")
    show = people[["participant", "dominant_hand", "sex", "age",
                   "edinburgh_lq", "cell", "visits", "interval_days",
                   "blocks"]]
    _show(show.set_index("participant"))

    blocks = (sel.groupby(["mode", "visit"]).size().unstack("visit")
              .fillna(0).astype(int))
    print("\ncompleted blocks per mode and visit:")
    _show(blocks)
    missing = [m for m in COHORT_MODES if m not in set(sel["mode"])]
    if missing:
        print("no blocks yet for: " + ", ".join(missing))

    if long.empty:
        _nothing("\nThe blocks loaded but no metric could be read from",
                 "them. Check block_summary in metadata.json.")
        return cohort
    cover = (long.groupby(["mode", "metric", "visit"])["participant"]
             .nunique().unstack("visit").fillna(0).astype(int))
    print(f"\nlong table: {len(long)} row(s), one per participant, "
          f"visit, hand, mode and metric. Participants per metric and "
          f"visit (coverage, not a statistic):")
    _show(cover)
    splits = (long.groupby(["mode"])["config_hash"].nunique())
    splits = splits[splits > 1]
    if len(splits):
        print("\nCONFIG SPLIT: these modes ran under more than one config "
              "(counts, windows or ladder keys differ between blocks):")
        for mode, n in splits.items():
            print(f"   {mode}: {n} distinct configs. Read cohort_metrics."
                  "csv's config_hash before pooling them.")
    return cohort


def sec_cohort_describe(cohort):
    """Descriptives per metric and hand role, visit 1 and visit 2
    kept apart. The visit 1 table is the normative table the thesis
    prints; the percentiles are what a later patient is read
    against, so they only print at the design's n."""
    print("\n" + "=" * 62)
    print("COHORT: DESCRIPTIVES AND NORMATIVE RANGES")
    print("=" * 62)
    long = cohort.get("long")
    min_n = cohort.get("min_n", COHORT_MIN_N)
    if long is None or long.empty:
        _nothing("No cohort rows to describe.")
        return pd.DataFrame()
    rows = []
    for (mode, metric, role, visit), g in long.groupby(
            ["mode", "metric", "hand_role", "visit"]):
        v = g.groupby("participant")["value"].mean()
        spec = COHORT_METRICS.get((mode, metric), {})
        rows.append({
            "mode": mode, "metric": metric, "hand_role": role,
            "visit": visit, "unit": spec.get("unit", ""),
            "n": int(len(v)),
            "mean": float(v.mean()),
            "sd": float(v.std(ddof=1)) if len(v) >= 2 else np.nan,
            "median": float(v.median()),
            "q1": float(v.quantile(0.25)), "q3": float(v.quantile(0.75)),
            "p5": float(v.quantile(0.05)), "p95": float(v.quantile(0.95)),
            "reference": spec.get("ref", np.nan),
        })
    desc = pd.DataFrame(rows)
    cohort["tables"]["describe"] = desc
    v1 = desc[desc["visit"] == "1"]
    ready = v1[v1["n"] >= min_n]
    short = v1[v1["n"] < min_n]
    if len(ready):
        print(f"\nnormative table, visit 1 (n at or above {min_n}):")
        _show(ready.set_index(["mode", "metric", "hand_role"])
              [["n", "mean", "sd", "median", "q1", "q3", "p5", "p95",
                "unit", "reference"]].round(3))
        v2 = desc[(desc["visit"] == "2") & (desc["n"] >= min_n)]
        if len(v2):
            print("\nvisit 2, kept separate (the practice section reads "
                  "the difference):")
            _show(v2.set_index(["mode", "metric", "hand_role"])
                  [["n", "mean", "sd", "median", "p5", "p95"]].round(3))
    if len(short):
        print(f"\n{len(short)} metric row(s) at visit 1 are below "
              f"n = {min_n} and print as coverage only:")
        _show(short.set_index(["mode", "metric", "hand_role"])[["n"]])
    if not len(ready):
        return desc

    # One figure per mode, its headline metric only: one focal point.
    for mode in COHORT_MODES:
        head = next((m for (md, m), s in COHORT_METRICS.items()
                     if md == mode and s.get("headline")), None)
        if head is None:
            continue
        g = ready[(ready["mode"] == mode) & (ready["metric"] == head)]
        if g.empty:
            continue
        spec = COHORT_METRICS[(mode, head)]
        roles = list(g["hand_role"])
        data = [cohort_values(long, mode, head, "1", role).values
                for role in roles]
        fig, ax = plt.subplots(figsize=(6.5, 3.2))
        bp = ax.boxplot(data, tick_labels=roles, patch_artist=True,
                        widths=.55)
        for p, role in zip(bp["boxes"], roles):
            p.set_facecolor(HAND_COLOUR.get(
                "right" if role == "dominant" else "left", "#64748b"))
            p.set_alpha(.6)
        for m in bp["medians"]:
            m.set_color("white"); m.set_linewidth(2)
        for i, d in enumerate(data, start=1):
            ax.plot(np.full(len(d), i) + np.linspace(-.12, .12, len(d)),
                    d, "o", ms=3, color="#0f172a", alpha=.6)
        if spec.get("ref") is not None:
            ax.axhline(spec["ref"], color="#dc2626", lw=1.2, ls="--",
                       label=spec.get("ref_label", "reference"))
            ax.legend(frameon=False, fontsize=8)
        ax.set_ylabel(f"{head} ({spec.get('unit', '')})")
        ax.set_title(f"{mode}: {head} at visit 1, n = "
                     f"{int(g['n'].max())}")
        _save(fig, f"cohort_norm_{mode}")
        plt.show()
    return desc


def sec_cohort_hands(cohort):
    """Dominant against non-dominant, paired within participant, per
    metric and visit. R2 and M2 are one-sided (dominant faster), the
    rest two-sided with no claim beyond the interval."""
    print("\n" + "=" * 62)
    print("COHORT: DOMINANT AGAINST NON-DOMINANT")
    print("=" * 62)
    long = cohort.get("long")
    min_n = cohort.get("min_n", COHORT_MIN_N)
    if long is None or long.empty:
        _nothing("No cohort rows.")
        return pd.DataFrame()
    one_sided = {("reaction", "median_rt_ms"): "less",
                 ("mirror", "rt_ms"): "less"}
    rows = []
    for (mode, metric), spec in COHORT_METRICS.items():
        for visit in ("1", "2"):
            a = cohort_values(long, mode, metric, visit, "dominant")
            b = cohort_values(long, mode, metric, visit, "nondominant")
            both = a.index.intersection(b.index)
            if len(both) == 0:
                continue
            alt = one_sided.get((mode, metric), "two-sided")
            st = cohort_paired(a.loc[both], b.loc[both], alternative=alt)
            rows.append({"mode": mode, "metric": metric, "visit": visit,
                         "unit": spec.get("unit", ""),
                         "alternative": alt, **st,
                         "direction": cohort_direction(
                             mode, metric, st["diff"], spec.get("unit", ""))})
    tbl = pd.DataFrame(rows)
    cohort["tables"]["hands"] = tbl
    if tbl.empty:
        _nothing("No metric has both a dominant and a non-dominant value",
                 "for the same participant yet.")
        return tbl
    ready = tbl[tbl["n"] >= min_n]
    short = tbl[tbl["n"] < min_n]
    if len(short):
        cohort_gate(int(short["n"].max()), min_n,
                    f"dominant against non-dominant on {len(short)} "
                    f"metric row(s)")
    if ready.empty:
        return tbl
    print("\npaired difference = dominant minus non-dominant; dz = mean "
          "over SD of the differences (bootstrap CI); the test named in "
          "'test' is the one the wording uses (Wilcoxon when Shapiro p "
          "is under 0.05 or n under 20), the other is beside it:")
    _show(ready.set_index(["mode", "metric", "visit"])
          [["n", "mean_a", "mean_b", "diff", "ci_lo", "ci_hi", "dz",
            "dz_lo", "dz_hi", "shapiro_p", "t_p", "w_p", "rank_biserial",
            "test", "alternative"]].round(3))
    print("\nin words (visit 1; visit 2 is the replication):")
    for _i, r in ready[ready["visit"] == "1"].iterrows():
        agree = ""
        if pd.notna(r["t_p"]) and pd.notna(r["w_p"]):
            if (r["t_p"] < 0.05) != (r["w_p"] < 0.05):
                agree = "  (t and Wilcoxon disagree at 0.05)"
        print(f"   {r['mode']} {r['metric']}: {r['direction']}; "
              f"{cohort_stat_words(r)}{agree}")

    # Paired dot plots for the directional checks (R2, M2) and the
    # force metrics the design expects to be null.
    wanted = [("reaction", "median_rt_ms"), ("mirror", "rt_ms"),
              ("force_pilot", "mae_pct"), ("lighthouse", "lit_mae_pct"),
              ("chords", "median_er")]
    for mode, metric in wanted:
        r = ready[(ready["mode"] == mode) & (ready["metric"] == metric)
                  & (ready["visit"] == "1")]
        if r.empty:
            continue
        a = cohort_values(long, mode, metric, "1", "dominant")
        b = cohort_values(long, mode, metric, "1", "nondominant")
        both = a.index.intersection(b.index)
        fig, ax = plt.subplots(figsize=(4.2, 3.4))
        for who in both:
            ax.plot([0, 1], [a[who], b[who]], "-", color="#94a3b8", lw=.9)
        ax.plot(np.zeros(len(both)), a.loc[both], "o",
                color=HAND_COLOUR["right"], label="dominant")
        ax.plot(np.ones(len(both)), b.loc[both], "o",
                color=HAND_COLOUR["left"], label="non-dominant")
        ax.set_xticks([0, 1]); ax.set_xticklabels(["dominant", "non-dominant"])
        ax.set_xlim(-.4, 1.4)
        spec = COHORT_METRICS[(mode, metric)]
        ax.set_ylabel(f"{metric} ({spec.get('unit', '')})")
        ax.set_title(f"{mode}: dz {float(r['dz'].iloc[0]):+.2f}, "
                     f"n = {int(r['n'].iloc[0])}")
        _save(fig, f"cohort_paired_{mode}_{metric}")
        plt.show()
    return tbl


def sec_cohort_retest(cohort):
    """Test-retest per metric and hand role: ICC(2,1) with its exact
    95 percent CI, ICC(3,1) beside it, SEM, MDC95 (Weir 2005) and
    the Bland-Altman bias and limits between visit 1 and visit 2."""
    print("\n" + "=" * 62)
    print("COHORT: TEST-RETEST, SEM AND MDC")
    print("=" * 62)
    long = cohort.get("long")
    min_n = cohort.get("min_n", COHORT_MIN_N)
    if long is None or long.empty:
        _nothing("No cohort rows.")
        return pd.DataFrame()
    rows = []
    for (mode, metric), spec in COHORT_METRICS.items():
        roles = sorted(set(long.loc[(long["mode"] == mode)
                                    & (long["metric"] == metric),
                                    "hand_role"]))
        for role in roles:
            v1 = cohort_values(long, mode, metric, "1", role)
            v2 = cohort_values(long, mode, metric, "2", role)
            both = v1.index.intersection(v2.index)
            if len(both) == 0:
                continue
            a, b = v1.loc[both], v2.loc[both]
            row = {"mode": mode, "metric": metric, "hand_role": role,
                   "unit": spec.get("unit", ""), "n": int(len(both)),
                   "icc21": np.nan, "ci_lo": np.nan, "ci_hi": np.nan,
                   "band": "", "icc31": np.nan, "sem": np.nan,
                   "mdc95": np.nan, "mdc_over_sd1": np.nan,
                   "bias": np.nan, "bias_lo": np.nan, "bias_hi": np.nan,
                   "loa_lo": np.nan, "loa_hi": np.nan,
                   "practice_dz": np.nan, "note": spec.get("no_mdc", "")}
            if len(both) >= 2:
                icc = icc_ci(np.column_stack([a.values, b.values]))
                row.update(icc21=icc["icc21"], ci_lo=icc["lo21"],
                           ci_hi=icc["hi21"], icc31=icc["icc31"],
                           band=icc_band(icc["icc21"]))
                if icc.get("reason"):
                    head = ("ICC undefined" if not np.isfinite(icc["icc21"])
                            else "ICC exact")
                    row["note"] = (f"{head}: {icc['reason']}"
                                   + (f"; {row['note']}" if row["note"]
                                      else ""))
                pooled = np.concatenate([a.values, b.values]).std(ddof=1)
                sd1 = a.std(ddof=1)
                if (np.isfinite(icc["icc21"]) and not spec.get("no_mdc")
                        and icc["icc21"] < 1):
                    sem = pooled * np.sqrt(max(0.0, 1 - icc["icc21"]))
                    mdc = 1.96 * np.sqrt(2) * sem
                    row.update(sem=float(sem), mdc95=float(mdc),
                               mdc_over_sd1=(float(mdc / sd1)
                                             if sd1 > 0 else np.nan))
                d = b - a
                bias, lo, hi = cohort_t_ci(d)
                sd_d = d.std(ddof=1) if len(d) >= 2 else np.nan
                row.update(bias=bias, bias_lo=lo, bias_hi=hi,
                           loa_lo=bias - 1.96 * sd_d,
                           loa_hi=bias + 1.96 * sd_d,
                           practice_dz=cohort_dz_ci(d)[0])
            rows.append(row)
    tbl = pd.DataFrame(rows)
    cohort["tables"]["retest"] = tbl
    if tbl.empty:
        _nothing("No participant has the same metric at both visits yet;",
                 "the ICC table fills in when visit 2 sessions exist.")
        return tbl
    ready = tbl[tbl["n"] >= min_n]
    short = tbl[tbl["n"] < min_n]
    if len(short):
        n_max = int(short["n"].max())
        cohort_gate(n_max, min_n,
                    f"test-retest on {len(short)} metric row(s)")
    if ready.empty:
        return tbl
    print("\nICC(2,1): two-way random, absolute agreement, single "
          "measure, exact F interval (McGraw and Wong 1996); bands "
          "poor < 0.5, moderate, good, excellent > 0.9 (Koo and Li "
          "2016). ICC(3,1) is consistency: the gap between the two is "
          "how much of the disagreement is a uniform shift. SEM = "
          "SD_pooled x sqrt(1 - ICC); MDC95 = 1.96 x sqrt(2) x SEM "
          "(Weir 2005), also as a fraction of the visit 1 SD. Bias is "
          "visit 2 minus visit 1 with limits of agreement.")
    _show(ready.set_index(["mode", "metric", "hand_role"])
          [["n", "icc21", "ci_lo", "ci_hi", "band", "icc31", "sem",
            "mdc95", "mdc_over_sd1", "bias", "bias_lo", "bias_hi",
            "loa_lo", "loa_hi", "note"]].round(3))
    for _i, r in ready[ready["note"] != ""].iterrows():
        lead = f"   {r['mode']} {r['metric']} ({r['hand_role']}): "
        if pd.isna(r["icc21"]):
            print(lead + r["note"] + ".")
        else:
            print(lead + "ICC printed, no MDC: " + r["note"] + ".")

    # The forest plot: the chapter's headline figure.
    fp = ready.dropna(subset=["icc21"]).copy()
    if len(fp):
        fp["label"] = (fp["mode"] + " " + fp["metric"] + " ("
                       + fp["hand_role"] + ")")
        fp = fp.sort_values("icc21")
        fig, ax = plt.subplots(figsize=(7.5, max(3.0, 0.28 * len(fp) + 1)))
        for edge, colour in ((0.5, "#fee2e2"), (0.75, "#fef3c7"),
                             (0.9, "#dcfce7")):
            pass
        ax.axvspan(-1, 0.5, color="#fee2e2", alpha=.5, lw=0)
        ax.axvspan(0.5, 0.75, color="#fef3c7", alpha=.5, lw=0)
        ax.axvspan(0.75, 0.9, color="#dcfce7", alpha=.5, lw=0)
        ax.axvspan(0.9, 1.0, color="#bbf7d0", alpha=.5, lw=0)
        y = np.arange(len(fp))
        lo = (fp["icc21"] - fp["ci_lo"]).fillna(0).values
        hi = (fp["ci_hi"] - fp["icc21"]).fillna(0).values
        ax.errorbar(fp["icc21"], y, xerr=[lo, hi], fmt="o", color="#0f172a",
                    ecolor="#334155", capsize=3, lw=1.2)
        ax.set_yticks(y); ax.set_yticklabels(fp["label"], fontsize=7)
        ax.set_xlim(max(-0.5, float(np.nanmin(fp["ci_lo"].fillna(0))) - .05),
                    1.0)
        ax.set_xlabel("ICC(2,1) with 95% CI")
        ax.set_title(f"Test-retest reliability, n = {int(fp['n'].min())} "
                     f"to {int(fp['n'].max())}")
        _save(fig, "cohort_icc_forest")
        plt.show()

    # Bland-Altman for the headline metric of each mode.
    heads = [(md, m) for (md, m), s in COHORT_METRICS.items()
             if s.get("headline")]
    panels = []
    for mode, metric in heads:
        r = ready[(ready["mode"] == mode) & (ready["metric"] == metric)]
        if r.empty:
            continue
        role = ("dominant" if "dominant" in set(r["hand_role"])
                else r["hand_role"].iloc[0])
        v1 = cohort_values(long, mode, metric, "1", role)
        v2 = cohort_values(long, mode, metric, "2", role)
        both = v1.index.intersection(v2.index)
        rr = r[r["hand_role"] == role].iloc[0]
        panels.append((mode, metric, role, v1.loc[both], v2.loc[both], rr))
    if panels:
        cols = min(3, len(panels))
        nrows = int(np.ceil(len(panels) / cols))
        fig, axes = plt.subplots(nrows, cols, figsize=(3.6 * cols, 3.0 * nrows),
                                 squeeze=False)
        for ax, (mode, metric, role, a, b, rr) in zip(axes.flat, panels):
            mean = (a + b) / 2
            diff = b - a
            ax.plot(mean, diff, "o", ms=4, color="#0f172a")
            ax.axhline(rr["bias"], color="#dc2626", lw=1.2)
            ax.axhline(rr["loa_lo"], color="#94a3b8", lw=1, ls="--")
            ax.axhline(rr["loa_hi"], color="#94a3b8", lw=1, ls="--")
            ax.axhline(0, color="#0f172a", lw=.6)
            ax.set_title(f"{mode} {metric}\n{role}, bias "
                         f"{rr['bias']:+.2f}", fontsize=8)
            ax.set_xlabel("mean of visits", fontsize=7)
            ax.set_ylabel("visit 2 minus visit 1", fontsize=7)
        for ax in list(axes.flat)[len(panels):]:
            ax.axis("off")
        _save(fig, "cohort_bland_altman")
        plt.show()
    return tbl


def sec_cohort_practice(cohort):
    """Practice effects: visit 2 minus visit 1 per metric and hand
    role, as a paired difference with dz. The design expects none for
    reaction (R3), a positive shift for pattern (P2) and small gains
    elsewhere; the wording says which way each metric moved and
    whether that is better."""
    print("\n" + "=" * 62)
    print("COHORT: PRACTICE EFFECTS, VISIT 1 TO VISIT 2")
    print("=" * 62)
    long = cohort.get("long")
    min_n = cohort.get("min_n", COHORT_MIN_N)
    if long is None or long.empty:
        _nothing("No cohort rows.")
        return pd.DataFrame()
    rows = []
    for (mode, metric), spec in COHORT_METRICS.items():
        roles = sorted(set(long.loc[(long["mode"] == mode)
                                    & (long["metric"] == metric),
                                    "hand_role"]))
        for role in roles:
            v1 = cohort_values(long, mode, metric, "1", role)
            v2 = cohort_values(long, mode, metric, "2", role)
            both = v1.index.intersection(v2.index)
            if len(both) == 0:
                continue
            st = cohort_paired(v2.loc[both], v1.loc[both])
            sd1 = v1.loc[both].std(ddof=1) if len(both) >= 2 else np.nan
            rows.append({
                "mode": mode, "metric": metric, "hand_role": role,
                "unit": spec.get("unit", ""), **st,
                "change_over_sd1": (st["diff"] / sd1
                                    if sd1 and sd1 > 0 else np.nan),
                "direction": cohort_direction(mode, metric, st["diff"],
                                              spec.get("unit", ""),
                                              "visit 2", "visit 1"),
            })
    tbl = pd.DataFrame(rows)
    cohort["tables"]["practice"] = tbl
    if tbl.empty:
        _nothing("No participant has both visits yet.")
        return tbl
    ready = tbl[tbl["n"] >= min_n]
    short = tbl[tbl["n"] < min_n]
    if len(short):
        cohort_gate(int(short["n"].max()), min_n,
                    f"practice effects on {len(short)} metric row(s)")
    if ready.empty:
        return tbl
    print("\nchange = visit 2 minus visit 1 (mean, t interval), dz with "
          "bootstrap CI, change also in units of the visit 1 SD:")
    _show(ready.set_index(["mode", "metric", "hand_role"])
          [["n", "mean_b", "mean_a", "diff", "ci_lo", "ci_hi", "dz",
            "dz_lo", "dz_hi", "change_over_sd1", "test", "p"]]
          .rename(columns={"mean_b": "visit_1", "mean_a": "visit_2",
                           "diff": "change"}).round(3))
    print("\nin words:")
    for _i, r in ready.iterrows():
        zero_in = (pd.notna(r["ci_lo"]) and pd.notna(r["ci_hi"])
                   and r["ci_lo"] <= 0 <= r["ci_hi"])
        verdict = ("no detectable practice effect (interval includes zero)"
                   if zero_in else r["direction"])
        print(f"   {r['mode']} {r['metric']} ({r['hand_role']}): "
              f"{verdict}; {cohort_stat_words(r)}")

    fp = ready.dropna(subset=["change_over_sd1"]).copy()
    if len(fp):
        fp["label"] = (fp["mode"] + " " + fp["metric"] + " ("
                       + fp["hand_role"] + ")")
        fp = fp.sort_values("change_over_sd1")
        fig, ax = plt.subplots(figsize=(7.5, max(3.0, 0.28 * len(fp) + 1)))
        y = np.arange(len(fp))
        sd1 = (fp["diff"] / fp["change_over_sd1"]).replace(0, np.nan)
        lo = fp["ci_lo"] / sd1
        hi = fp["ci_hi"] / sd1
        ax.hlines(y, lo, hi, color="#94a3b8", lw=2)
        ax.plot(fp["change_over_sd1"], y, "o", color="#0f172a")
        ax.axvline(0, color="#dc2626", lw=1)
        ax.set_yticks(y); ax.set_yticklabels(fp["label"], fontsize=7)
        ax.set_xlabel("visit 2 minus visit 1, in visit 1 SDs (95% CI)")
        ax.set_title("Practice effect per metric")
        _save(fig, "cohort_practice")
        plt.show()
    return tbl


# The pre-specified checks, Section 1 and Section 4.6 of the design.
# One row each, decided before the data comes in; the verdict is pass,
# fail, or not testable (no data, or n under the design minimum).

def _verdict(ok):
    if ok is None:
        return "not testable"
    return "pass" if ok else "fail"


def _check_row(cid, mode, check, n, value, ci, reference, criterion,
               ok, detail=""):
    return {"id": cid, "mode": mode, "check": check, "n": int(n or 0),
            "value": value,
            "ci_lo": ci[0] if ci else np.nan,
            "ci_hi": ci[1] if ci else np.nan,
            "reference": reference, "criterion": criterion,
            "verdict": _verdict(ok), "detail": detail}


def _mean_check(long, mode, metric, visit, role, min_n, cid, check,
                ref, op, unit, use="mean", alternative=None):
    """A cohort mean or median against a fixed reference, with the
    one-sample test where the criterion is a direction from zero."""
    if role == "any":
        v = cohort_values(long, mode, metric, visit, None)
    else:
        v = cohort_values(long, mode, metric, visit, role)
    n = len(v)
    if n < max(min_n, 3):
        return _check_row(cid, mode, check, n, np.nan, None, ref,
                          f"{use} {op} {ref}", None,
                          "n under the design minimum" if n else "no data")
    stat = float(v.mean()) if use == "mean" else float(v.median())
    _m, lo, hi = cohort_t_ci(v)
    if op == "<":
        ok = stat < ref
    elif op == ">":
        ok = stat > ref
    elif op == "within":
        ok = ref[0] <= stat <= ref[1]
    else:
        ok = None
    detail = ""
    if alternative:
        st = cohort_one_sample(v, mu=ref if not isinstance(ref, tuple)
                               else 0.0, alternative=alternative)
        ok = ok and pd.notna(st["p"]) and st["p"] < 0.05
        detail = _stat_detail(st)
    return _check_row(cid, mode, check, n, stat, (lo, hi), ref,
                      f"{use} {op} {ref} {unit}", ok, detail)


def _stat_detail(st) -> str:
    """The test, p and dz of a paired or one-sample result in words,
    or the reason there is none."""
    if st.get("test") == "no variation" or pd.isna(st.get("p")):
        return "no variation to test"
    text = f"{st['test']} p {st['p']:.3f}, dz {st['dz']:+.2f}"
    if pd.notna(st.get("rank_biserial")):
        text += f", rank-biserial {st['rank_biserial']:+.2f}"
    return text


def _paired_check(long, mode, metric_a, metric_b, visit, role_a, role_b,
                  min_n, cid, check, alternative, unit, visit_b=None):
    a = cohort_values(long, mode, metric_a, visit, role_a)
    b = cohort_values(long, mode, metric_b, visit_b or visit, role_b)
    both = a.index.intersection(b.index)
    n = len(both)
    crit = {"less": "a below b", "greater": "a above b"}.get(
        alternative, "a differs from b")
    if n < max(min_n, 3):
        return _check_row(cid, mode, check, n, np.nan, None, 0.0, crit,
                          None, "n under the design minimum" if n
                          else "no data")
    st = cohort_paired(a.loc[both], b.loc[both], alternative=alternative)
    ok = pd.notna(st["p"]) and st["p"] < 0.05
    if alternative == "less":
        ok = ok and st["diff"] < 0
    elif alternative == "greater":
        ok = ok and st["diff"] > 0
    return _check_row(cid, mode, check, n, st["diff"],
                      (st["ci_lo"], st["ci_hi"]), 0.0,
                      f"{crit} ({unit})", ok, _stat_detail(st))


def _no_change_check(long, mode, metric, role, min_n, cid, check, unit):
    """A null claim: the interval of visit 2 minus visit 1 includes
    zero. Stated as such, not as evidence of no effect."""
    v1 = cohort_values(long, mode, metric, "1", role)
    v2 = cohort_values(long, mode, metric, "2", role)
    both = v1.index.intersection(v2.index)
    n = len(both)
    if n < max(min_n, 3):
        return _check_row(cid, mode, check, n, np.nan, None, 0.0,
                          "95% CI of visit 2 minus visit 1 includes zero",
                          None, "n under the design minimum" if n
                          else "no data")
    st = cohort_paired(v2.loc[both], v1.loc[both])
    ok = (pd.notna(st["ci_lo"]) and pd.notna(st["ci_hi"])
          and st["ci_lo"] <= 0 <= st["ci_hi"])
    return _check_row(cid, mode, check, n, st["diff"],
                      (st["ci_lo"], st["ci_hi"]), 0.0,
                      f"95% CI of the change includes zero ({unit})", ok,
                      f"dz {st['dz']:+.2f}")


def _spearman(x, y):
    if _sps is None or len(x) < 4:
        return np.nan, np.nan
    r = _sps.spearmanr(x, y)
    return float(r.statistic), float(r.pvalue)


def sec_cohort_validity(cohort):
    """The known-effect checks the design pre-specifies (Section 4.6),
    one row each, printed as plain verdicts with the number and the
    reference value. No other inferential test is run in this
    notebook; everything else is descriptive with intervals."""
    print("\n" + "=" * 62)
    print("COHORT: KNOWN-EFFECT VALIDITY CHECKS")
    print("=" * 62)
    long = cohort.get("long")
    frames = cohort.get("frames", {})
    min_n = cohort.get("min_n", COHORT_MIN_N)
    if long is None or long.empty:
        _nothing("No cohort rows.")
        return pd.DataFrame()
    rows = []

    # Reaction.
    rho = cohort_values(long, "reaction", "rho_rt_vs_fp", "1", None).abs()
    fs = cohort_values(long, "reaction", "false_start_rate", "1", None)
    n = min(len(rho), len(fs))
    if n < max(min_n, 3):
        rows.append(_check_row("R1", "reaction", "foreperiod works", n,
                               np.nan, None, 0.2, "median |rho| < 0.2 and "
                               "false starts < 10%", None,
                               "n under the design minimum" if n
                               else "no data"))
    else:
        ok = bool(rho.median() < 0.2 and fs.mean() < 0.10)
        rows.append(_check_row("R1", "reaction", "foreperiod works", n,
                               float(rho.median()), None, 0.2,
                               "median |rho| < 0.2 and false starts < 10%",
                               ok, f"false starts {fs.mean():.1%} of attempts"))
    rows.append(_paired_check(long, "reaction", "median_rt_ms",
                              "median_rt_ms", "1", "dominant", "nondominant",
                              min_n, "R2", "dominant hand faster", "less",
                              "ms"))
    rows.append(_no_change_check(long, "reaction", "median_rt_ms",
                                 "dominant", min_n, "R3",
                                 "no practice effect on median RT", "ms"))
    # Pattern.
    rows.append(_mean_check(long, "pattern", "learning_score_ms", "1",
                            "any", min_n, "P1", "learning score above zero",
                            0.0, ">", "ms", alternative="greater"))
    rows.append(_paired_check(long, "pattern", "learning_score_ms",
                              "learning_score_ms", "2", "dominant",
                              "dominant", min_n, "P2",
                              "larger learning score at visit 2",
                              "greater", "ms", visit_b="1"))
    rows.append(_mean_check(long, "pattern", "accuracy_rebound_pct", "1",
                            "any", min_n, "P3", "accuracy drops on probes",
                            0.0, ">", "% points", alternative="greater"))
    # Chords.
    rows.append(_mean_check(long, "chords", "median_er", "1", "any", min_n,
                            "C1", "cohort median ER under 0.15", 0.15, "<",
                            "ratio", use="median"))
    per_chord = [(who, v, p) for who, v, lst in frames.get("per_chord", [])
                 for p in lst if v == "1"]
    n_pc = len({who for who, _v, _p in per_chord})
    if n_pc < max(min_n, 3):
        rows.append(_check_row("C2", "chords", "difficulty rank D predicts "
                               "outcome", n_pc, np.nan, None, 0.0,
                               "rho(D, hit rate) < 0 and rho(D, ER) > 0",
                               None, "n under the design minimum" if n_pc
                               else "no data"))
    else:
        pc = pd.DataFrame([p for _w, _v, p in per_chord])
        pc = pc.dropna(subset=["d"])
        pooled = (pc.groupby("chord").apply(
            lambda g: pd.Series({
                "d": g["d"].iloc[0],
                "hit_rate": np.average(
                    pd.to_numeric(g["hit_rate"], errors="coerce").fillna(0),
                    weights=g["n"]) if g["n"].sum() else np.nan,
                "median_er": pd.to_numeric(g["median_er"],
                                           errors="coerce").median()}),
            include_groups=False))
        r_hit, p_hit = _spearman(pooled["d"], pooled["hit_rate"])
        r_er, p_er = _spearman(pooled["d"], pooled["median_er"].fillna(
            pooled["median_er"].median()))
        ok = (np.isfinite(r_hit) and r_hit < 0 and p_hit < 0.05
              and np.isfinite(r_er) and r_er > 0 and p_er < 0.05)
        rows.append(_check_row("C2", "chords", "difficulty rank D predicts "
                               "outcome", n_pc, r_hit, None, 0.0,
                               "rho(D, hit rate) < 0 and rho(D, ER) > 0",
                               ok, f"rho(D, hit) {r_hit:+.2f} p {p_hit:.3f}; "
                                   f"rho(D, ER) {r_er:+.2f} p {p_er:.3f}; "
                                   f"{len(pooled)} chord types"))
    chord_rows = _cohort_frame_rows(frames, "chord_rows")
    chord_rows = chord_rows[chord_rows["visit"] == "1"] \
        if len(chord_rows) else chord_rows
    n_cr = chord_rows["participant"].nunique() if len(chord_rows) else 0
    if n_cr < max(min_n, 3):
        rows.append(_check_row("C3", "chords", "ring finger most enslaved",
                               n_cr, np.nan, None, "Ring",
                               "largest mean leak on the ring column",
                               None, "n under the design minimum" if n_cr
                               else "no data"))
    else:
        leak = {}
        for _i, r in chord_rows.iterrows():
            pl = r.get("press_level")
            if not isinstance(r.get("leaks"), dict) or not pl or pl <= 0:
                continue
            for f, v in r["leaks"].items():
                leak.setdefault(f, []).append(float(v) / float(pl))
        means = {f: float(np.mean(v)) for f, v in leak.items() if v}
        top = max(means, key=means.get) if means else ""
        rows.append(_check_row("C3", "chords", "ring finger most enslaved",
                               n_cr, means.get("Ring", np.nan), None,
                               "Ring", "largest mean leak on the ring column",
                               (top == "Ring") if means else None,
                               ", ".join(f"{f} {m:.3f}"
                                         for f, m in sorted(means.items()))))
    rows.append(_paired_check(long, "chords", "hit_rate_mirror",
                              "hit_rate_nonmirror", "1", "both", "both",
                              min_n, "C4", "mirror chords hit more often",
                              "greater", "fraction"))
    # Rhythm.
    for role in ("dominant", "nondominant"):
        rows.append(_mean_check(long, "rhythm", "asyn_mean_ms", "1", role,
                                min_n, "Rh1", f"negative mean asynchrony, "
                                f"{role}", 0.0, "<", "ms",
                                alternative="less"))
    rows.append(_mean_check(long, "rhythm", "asyn_sd_ms", "1", "any", min_n,
                            "Rh2", "asynchrony SD in the tens of ms",
                            (10.0, 100.0), "within", "ms", use="median"))
    # Mirror.
    rows.append(_mean_check(long, "mirror", "mean_gap_ms", "1", "both",
                            min_n, "M1", "in-phase gap well under the gate",
                            60.0, "<", "ms"))
    rows.append(_paired_check(long, "mirror", "rt_ms", "rt_ms", "1",
                              "dominant", "nondominant", min_n, "M2",
                              "dominant press leads", "less", "ms"))
    # Force pilot.
    fa = cohort_values(long, "force_pilot", "assess_mae_pct", "1", None)
    fs_ = cohort_values(long, "force_pilot", "sine_mae_pct", "1", None)
    fh = cohort_values(long, "force_pilot", "hold_mae_pct", "1", None)
    both3 = fa.index.intersection(fs_.index).intersection(fh.index)
    n3 = len(both3)
    if n3 < max(min_n, 3):
        rows.append(_check_row("F1", "force_pilot", "error grows with "
                               "target bandwidth", n3, np.nan, None, 0.0,
                               "assess > sine > holds within participant",
                               None, "n under the design minimum" if n3
                               else "no data (hold MAE needs raw.csv)"))
    else:
        s1 = cohort_paired(fa.loc[both3], fs_.loc[both3], "greater")
        s2 = cohort_paired(fs_.loc[both3], fh.loc[both3], "greater")
        share = float(((fa.loc[both3] > fs_.loc[both3])
                       & (fs_.loc[both3] > fh.loc[both3])).mean())
        ok = (s1["p"] < 0.05 and s2["p"] < 0.05 and s1["diff"] > 0
              and s2["diff"] > 0)
        rows.append(_check_row("F1", "force_pilot", "error grows with "
                               "target bandwidth", n3, share, None, 1.0,
                               "assess > sine > holds within participant",
                               ok, f"ordering held in {share:.0%}; assess "
                                   f"minus sine {s1['diff']:+.2f} p "
                                   f"{s1['p']:.3f}; sine minus hold "
                                   f"{s2['diff']:+.2f} p {s2['p']:.3f}"))
    rows.append(_mean_check(long, "force_pilot", "lag_ms", "1", "any", min_n,
                            "F2", "visuomotor lag of order 100 to 300 ms",
                            (100.0, 300.0), "within", "ms", use="median"))
    ra_ = cohort_values(long, "force_pilot", "release_mae_pct", "1", None)
    pa_ = cohort_values(long, "force_pilot", "press_mae_pct", "1", None)
    bothf = ra_.index.intersection(pa_.index)
    if len(bothf) < max(min_n, 3):
        rows.append(_check_row("F3", "force_pilot", "release error equals "
                               "press error", len(bothf), np.nan, None,
                               COHORT_F3_MARGIN_PCT,
                               f"95% CI within +/- {COHORT_F3_MARGIN_PCT} "
                               f"% of max", None,
                               "n under the design minimum" if len(bothf)
                               else "no data"))
    else:
        st = cohort_paired(ra_.loc[bothf], pa_.loc[bothf])
        ok = (pd.notna(st["ci_lo"]) and st["ci_lo"] > -COHORT_F3_MARGIN_PCT
              and st["ci_hi"] < COHORT_F3_MARGIN_PCT)
        rows.append(_check_row("F3", "force_pilot", "release error equals "
                               "press error", len(bothf), st["diff"],
                               (st["ci_lo"], st["ci_hi"]),
                               COHORT_F3_MARGIN_PCT,
                               f"95% CI within +/- {COHORT_F3_MARGIN_PCT} "
                               f"% of max (equivalence)", ok,
                               f"release minus press, dz {st['dz']:+.2f}"))
    runs = _cohort_frame_rows(frames, "force_runs")
    runs = runs[runs["visit"] == "1"] if len(runs) else runs
    n_fr = runs["participant"].nunique() if len(runs) else 0
    if n_fr < max(min_n, 3):
        rows.append(_check_row("F4", "force_pilot", "time in corridor above "
                               "0.8 on most runs", n_fr, np.nan, None, 0.5,
                               "share of runs with tic >= 0.8 above 0.5",
                               None, "n under the design minimum" if n_fr
                               else "no data"))
    else:
        share = float((runs["tic"] >= 0.8).mean())
        rows.append(_check_row("F4", "force_pilot", "time in corridor above "
                               "0.8 on most runs", n_fr, share, None, 0.5,
                               "share of runs with tic >= 0.8 above 0.5",
                               share > 0.5, f"{len(runs)} run(s)"))
    # Lighthouse.
    rows.append(_paired_check(long, "lighthouse", "dark_mae_pct",
                              "lit_mae_pct", "1", "dominant", "dominant",
                              min_n, "L1", "dark error above lit error",
                              "greater", "% of max"))
    rows.append(_paired_check(long, "lighthouse", "echo_abs_err_10s",
                              "echo_abs_err_2s", "1", "dominant", "dominant",
                              min_n, "L2", "reproduction error grows with "
                              "delay", "greater", "% of max"))
    holds = _cohort_frame_rows(frames, "lighthouse_holds")
    holds = holds[holds["visit"] == "1"] if len(holds) else holds
    n_h = holds["participant"].nunique() if len(holds) else 0
    if n_h < max(min_n, 3):
        rows.append(_check_row("L3", "lighthouse", "steadiness worse at "
                               "lower targets", n_h, np.nan, None, 0.0,
                               "rho(target %, lit CoV) < 0", None,
                               "n under the design minimum" if n_h
                               else "no data"))
    else:
        h = holds.dropna(subset=["target_pct", "lit_cov"])
        r, p = _spearman(h["target_pct"], h["lit_cov"])
        rows.append(_check_row("L3", "lighthouse", "steadiness worse at "
                               "lower targets", n_h, r, None, 0.0,
                               "rho(target %, lit CoV) < 0",
                               np.isfinite(r) and r < 0 and p < 0.05,
                               f"{len(h)} hold(s), p {p:.3f}"))
    # Buzz Hunt.
    rows.append(_mean_check(long, "buzz_hunt", "loc_accuracy", "1", "any",
                            min_n, "B1", "localisation near ceiling", 0.9,
                            ">", "fraction"))
    rows.append(_mean_check(long, "buzz_hunt", "adjacent_error_share", "1",
                            "any", min_n, "B2", "errors on neighbouring "
                            "fingers", 0.5, ">", "fraction"))
    rows.append(_mean_check(long, "buzz_hunt", "catch_fa_rate", "1", "any",
                            min_n, "B3", "guessing priced", 0.10, "<",
                            "fraction"))
    rows.append(_mean_check(long, "buzz_hunt", "span_max_correct", "1",
                            "any", min_n, "B4", "tactile span around 4",
                            (3.0, 5.0), "within", "items", use="median"))
    # Echo.
    rows.append(_mean_check(long, "echo", "span", "1", "any", min_n, "E1",
                            "span in the Corsi region", (5.0, 8.0),
                            "within", "items (Kessels 6.2)", use="median"))
    rows.append(_mean_check(long, "echo", "hebb_minus_novel_acc", "1", "any",
                            min_n, "E2", "Hebb repetition learning", 0.0,
                            ">", "fraction", alternative="greater"))
    errs = [e for _w, v, lst in frames.get("echo_errors", [])
            for e in lst if v == "1" and e]
    n_e = len({w for w, v, _l in frames.get("echo_errors", []) if v == "1"})
    if n_e < max(min_n, 3):
        rows.append(_check_row("E3", "echo", "serial-order errors dominate",
                               n_e, np.nan, None, 0.5, "transpositions + "
                               "omissions > intrusions", None,
                               "n under the design minimum" if n_e
                               else "no data"))
    else:
        counts = pd.Series(errs).value_counts() if errs else pd.Series(
            dtype="int64")
        serial = int(counts.get("transposition", 0) + counts.get("omission", 0))
        intr = int(counts.get("intrusion", 0))
        share = serial / len(errs) if errs else np.nan
        rows.append(_check_row("E3", "echo", "serial-order errors dominate",
                               n_e, share, None, 0.5,
                               "transpositions + omissions > intrusions",
                               (serial > intr) if errs else None,
                               f"{serial} serial-order, {intr} intrusion(s), "
                               f"{len(errs)} failed reproduction(s)"))

    tbl = pd.DataFrame(rows)
    cohort["tables"]["validity"] = tbl
    print("\nverdicts (pass / fail against the pre-set criterion; not "
          "testable = no data or n under the design minimum):")
    for _i, r in tbl.iterrows():
        val = _fmt(r["value"])
        ci = ("" if pd.isna(r["ci_lo"]) else
              f" [{_fmt(r['ci_lo'])}, {_fmt(r['ci_hi'])}]")
        ref = r["reference"]
        ref_txt = (f"{ref[0]:g} to {ref[1]:g}" if isinstance(ref, tuple)
                   else f"{ref}")
        print(f"   {r['id']:<4}{r['verdict']:<13}{r['mode']} {r['check']}: "
              f"{val}{ci} (n {int(r['n'])}); reference {ref_txt}; "
              f"{r['criterion']}"
              + (f"; {r['detail']}" if r["detail"] else ""))
    counts = tbl["verdict"].value_counts()
    print("\n" + ", ".join(f"{v} {k}" for k, v in counts.items()))

    # Per-mode figures for the checks that have a picture worth one
    # focal point: the asynchrony histogram per hand with zero, the
    # span histogram against 6.2, and RT against foreperiod with rho.
    # Only for checks that were decided: a figure on a refused n
    # would show what the text just declined to state.
    decided = set(tbl.loc[tbl["verdict"] != "not testable", "id"])
    trials = cohort.get("trials")
    rhy = rhythm_rows(trials) if trials is not None and not trials.empty \
        else pd.DataFrame()
    if len(rhy) >= 20 and "Rh1" in decided:
        fig, ax = plt.subplots(figsize=(6.5, 3.2))
        for side, g in rhy.groupby("side"):
            off = pd.to_numeric(g["time_difference_ms"], errors="coerce") \
                .dropna()
            ax.hist(off, bins=_nbins(off), alpha=.55,
                    color=HAND_COLOUR.get(side, "#64748b"),
                    label=f"{side} (mean {off.mean():+.0f} ms)")
        ax.axvline(0, color="#dc2626", lw=1.5)
        ax.set_xlabel("asynchrony (ms, negative = before the beat)")
        ax.set_ylabel("notes")
        ax.set_title("Rh1: taps against the beat, every cohort note")
        ax.legend(frameon=False, fontsize=8)
        _save(fig, "cohort_rh1_asynchrony")
        plt.show()
    spans = cohort_values(long, "echo", "span", "1", None)
    if len(spans) >= 3 and "E1" in decided:
        fig, ax = plt.subplots(figsize=(6.0, 3.0))
        ax.hist(spans, bins=np.arange(spans.min() - .5, spans.max() + 1.5, 1),
                color="#2563eb", alpha=.7)
        ax.axvline(6.2, color="#dc2626", lw=1.5, ls="--",
                   label="Kessels 2000 mean 6.2 (plausibility only)")
        ax.set_xlabel("span (items)"); ax.set_ylabel("participants")
        ax.set_title(f"E1: echo span at visit 1, n = {len(spans)}")
        ax.legend(frameon=False, fontsize=8)
        _save(fig, "cohort_e1_span")
        plt.show()
    rx = reaction_frame(trials) if trials is not None and not trials.empty \
        else pd.DataFrame()
    if len(rx) >= 20 and "R1" in decided:
        v = rx[~rx["is_event"]]
        rt = pd.to_numeric(v["time_difference_ms"], errors="coerce")
        keep_m = rt.notna() & (rt >= ANTICIPATION_MS) & v["fp_s"].notna()
        if keep_m.sum() >= 10:
            r, p = _spearman(v.loc[keep_m, "fp_s"], rt[keep_m])
            fig, ax = plt.subplots(figsize=(6.0, 3.2))
            ax.plot(v.loc[keep_m, "fp_s"], rt[keep_m], "o", ms=2.5,
                    alpha=.35, color="#0f172a")
            ax.set_xlabel("foreperiod (s)"); ax.set_ylabel("RT (ms)")
            ax.set_title(f"R1: RT against foreperiod, pooled rho "
                         f"{r:+.2f}")
            _save(fig, "cohort_r1_foreperiod")
            plt.show()
    return tbl


def sec_cohort_export(cohort):
    """Write cohort_metrics.csv (the long table) and every cohort
    table computed above into the cohort folder."""
    print("\n" + "=" * 62)
    print("COHORT: EXPORT")
    print("=" * 62)
    long = cohort.get("long")
    out_dir = Path(cohort.get("out_dir") or COHORT_RESULTS)
    if long is None or long.empty:
        _nothing("Nothing to export: the long table is empty.")
        return []
    out_dir.mkdir(parents=True, exist_ok=True)
    written = []
    path = out_dir / COHORT_CSV
    long.to_csv(path, index=False)
    written.append(path)
    people = cohort.get("people")
    if people is not None and len(people):
        p = out_dir / "cohort_participants.csv"
        people.to_csv(p, index=False)
        written.append(p)
    for name, tbl in (cohort.get("tables") or {}).items():
        if isinstance(tbl, pd.DataFrame) and len(tbl):
            p = out_dir / f"cohort_{name}.csv"
            tbl.to_csv(p, index=False)
            written.append(p)
    for p in written:
        print(f"csv -> {p}")
    print("\ncohort_metrics.csv is the tidy long table: one row per "
          "participant, visit, hand, mode and metric, with the trial "
          "count the value rests on and the config hash. It is the file "
          "JASP or R reads; nothing in it is a name.")
    return written


def write_cohort_report(ctx, cohort):
    """The cohort sections as one HTML under the cohort folder. The
    per-session write_report leaves these sections out, and this one
    takes only them, so neither report carries the other's text."""
    global FIGDIR
    sections = [s for s in _CAPTURE["sections"]
                if s["name"].startswith("cohort_")]
    if not sections:
        print("Nothing captured for the cohort. Run the cohort cells "
              "from the top.")
        return None
    sel = (cohort or {}).get("sel")
    if sel is None or getattr(sel, "empty", True):
        # No cohort means no folder: writing one would plant an empty
        # sessions tree beside the notebook, the trap PATIENT_RESULTS
        # is anchored against.
        print("No study blocks were selected, so no cohort report is "
              "written.")
        return None
    out_dir = Path((cohort or {}).get("out_dir") or COHORT_RESULTS)
    out_dir.mkdir(parents=True, exist_ok=True)
    people = (cohort or {}).get("people")
    n = int(len(people)) if people is not None else 0
    both = (int((people["visits"] >= 2).sum())
            if people is not None and len(people) else 0)
    title = "Finger Rehab cohort analysis"
    subtitle = (f"{n} participant(s), {both} with two visits | run "
                f"{_CAPTURE['started']} | {len(sections)} section(s) | "
                f"design N = {COHORT_N_DESIGN}")
    page = _build_report_html(title, subtitle, sections)
    here = out_dir / "report.html"
    here.write_text(page, encoding="utf-8")
    print(f"report -> {here}")
    # Put the figure folder back so a per-session section re-run
    # after this lands beside its own session again.
    if _COHORT_PREV_FIGDIR is not None:
        FIGDIR = _COHORT_PREV_FIGDIR
    return here


SECTION_TITLES.update({
    "cohort_selection": "Cohort: selection and the long table",
    "cohort_describe": "Cohort: descriptives and normative ranges",
    "cohort_hands": "Cohort: dominant against non-dominant",
    "cohort_retest": "Cohort: test-retest, SEM and MDC",
    "cohort_practice": "Cohort: practice effects",
    "cohort_validity": "Cohort: known-effect validity checks",
    "cohort_export": "Cohort: export",
})


## Load the selection

In [ ]:
ctx = load_selection(pick)
(cat, sel, folders, metas, sessions,
 trials, unit, calset, on_task) = unpack(ctx)

## Calibration this data was recorded under

In [ ]:
check_selection(ctx, pick)
cal_tables = keep(ctx, "calibration",
                  sec_calibration(metas, sessions, calset))

## Overview

In [ ]:
check_selection(ctx, pick)
on_task = keep(ctx, "on_task", sec_overview(trials, folders, metas))

## Data quality

In [ ]:
check_selection(ctx, pick)
keep(ctx, "quality", sec_quality(trials, folders, metas))

## Comparing the games

In [ ]:
check_selection(ctx, pick)
comparison = keep(ctx, "compare", sec_compare(trials))

## Comparing the training modes

In [ ]:
check_selection(ctx, pick)
mode_compare = keep(ctx, "mode_compare", sec_mode_comparison(trials))

## Fixed cadence against RAS

In [ ]:
check_selection(ctx, pick)
cadence_ras = keep(ctx, "cadence_ras", sec_cadence_vs_ras(trials, metas))

## Reaction time

In [ ]:
check_selection(ctx, pick)
rt = keep(ctx, "rt", sec_reaction_time(trials))

## Reaction mode

In [ ]:
check_selection(ctx, pick)
reaction_mode = keep(ctx, "reaction_mode",
                     sec_reaction_mode(trials, metas))

## Muscle memory (patterns)

In [ ]:
check_selection(ctx, pick)
srtt = keep(ctx, "srtt", sec_pattern_srtt(trials, metas))

## Fatigue across blocks

In [ ]:
check_selection(ctx, pick)
fatigue = keep(ctx, "fatigue", sec_fatigue(trials, metas))

## Accuracy and the challenge point

In [ ]:
check_selection(ctx, pick)
accuracy = keep(ctx, "accuracy", sec_accuracy(trials))

## Timing judgement bands

In [ ]:
check_selection(ctx, pick)
judgements = keep(ctx, "judgements", sec_judgement_bands(trials, metas))

## Outcome rates per finger

In [ ]:
check_selection(ctx, pick)
outcome_rates = keep(ctx, "outcome_rates", sec_outcome_rates(trials, metas))

## Force

In [ ]:
check_selection(ctx, pick)
force = keep(ctx, "force", sec_force(trials, unit, calset))

## Baseline drift

Rayan's raw-versus-zeroed check reproduced on this device's own logs. Force at every stim and press is computed twice: minus one session-static offset (the role his flat `-255` plays on his rig; here the pre-first-stim resting median, because each of our pads idles at its own level) and minus the mean of the 250 ms immediately before the press, his look-back zero. Both series are regressed on time in his plot style: faded raw, opaque zeroed, trend lines with the equations printed. The chapter also states which tare every downstream force number already rides, and measures the gap between the game's capture-time EMA baseline and his flat 50-sample mean instead of asserting they agree. Newtons come primarily from each session's own calibration; his flat 51.2 counts per newton is printed alongside so his and Welber's numbers read directly against ours.


In [ ]:
check_selection(ctx, pick)
drift = keep(ctx, "drift", sec_baseline_drift(folders, metas, calset))


## Finger individuation

In [ ]:
check_selection(ctx, pick)
ind = keep(ctx, "ind", sec_individuation(trials, calset))

## Chord mode

Within-hand chords (individuation and enslaving), the enslaving
matrices, then the cross-hand (bimanual) chords bilateral sessions
add: mirror versus non-mirror cost, between-hand lag, per-hand ER
and the bilateral deficit ratio. The two scopes never pool.

Sessions recorded from 2026-09 have no single-finger probe trials:
play is chords only (2 to 4 fingers), and the single-press reference
under every normalised force is the quick-cal light-press capture.
Older sessions still carry probe rows and probe-based matrices; the
chapter reads both.


In [ ]:
check_selection(ctx, pick)
chords = keep(ctx, "chords", sec_chords(trials, metas, calset))

## Cross-talk between fingers

In [ ]:
check_selection(ctx, pick)
crosstalk = keep(ctx, "crosstalk", sec_crosstalk(trials, calset))

## Rhythm

In [ ]:
check_selection(ctx, pick)
rhythm = keep(ctx, "rhythm", sec_rhythm(trials))

## Tap variability and beat phase

In [ ]:
check_selection(ctx, pick)
tap_variability = keep(ctx, "tap_variability",
                       sec_tap_variability(trials, metas))

## Syllable beats

In [ ]:
check_selection(ctx, pick)
syllables = keep(ctx, "syllables", sec_syllables(trials, metas, calset))

## Both hands

In [ ]:
check_selection(ctx, pick)
bilateral = keep(ctx, "bilateral", sec_bilateral(trials, unit, calset))

## Inter-hand correlation

In [ ]:
check_selection(ctx, pick)
inter_hand = keep(ctx, "inter_hand", sec_inter_hand(folders, metas, trials))

## Affected against unaffected hand

In [ ]:
check_selection(ctx, pick)
affected = keep(ctx, "affected",
                sec_affected_side(trials, metas, unit, calset))

## Raw sample stream

In [ ]:
check_selection(ctx, pick)
raw_stream = keep(ctx, "raw_stream",
                  sec_raw(folders, unit, calset))

## Continuous force modes
What the continuous-force trials asked for: waveform mix, parameter conditions, seeds, and whether the trial rows' scored segments line up with the markers in the raw stream.

In [ ]:
check_selection(ctx, pick)
continuous = keep(ctx, "continuous", sec_continuous(folders, trials))

## Force tracking
Force Pilot re-scored offline from the 200 Hz raw stream: the corridor is rebuilt exactly from each row's logged waveform, then RMSE, time in corridor, tracking lag, the sub-1 Hz spectral ratio (the Lodha stroke biomarker), and release against generation error, per finger and per hand.

In [ ]:
check_selection(ctx, pick)
force_tracking = keep(ctx, "force_tracking", sec_force_tracking(folders, trials, metas))

## Precision hold and force sense
Lighthouse holds and echo trials scored offline: lit against dark steadiness, the lit-dark delta headlined as the carpal-tunnel-sensitive contrast (Li 2015), post-fade drift, and force reproduction error by delay, with the test-retest ICC scaffold.

In [ ]:
check_selection(ctx, pick)
precision_hold = keep(ctx, "precision_hold", sec_precision_hold(folders, trials, metas))

## Tactile perception
Buzz Hunt scored offline: the per-finger confusion matrix (sibling of the enslavement heatmap), staircase thresholds with reversal plots, psychometric fits with a lapse rate, d-prime from the catch trials, and the span curve with the Hebb repetition slope.

In [ ]:
check_selection(ctx, pick)
tactile = keep(ctx, "tactile", sec_tactile(folders, trials, metas))

## Echo span
Echo scored offline: the Kessels ladder numbers (span, total correct, span x correct product), Conway 2005 partial credit with the Gonthier edit-distance refinement, span with and without the hidden Hebb repeats, the repetition-learning slope on the prefix-stable material, the serial-recall error taxonomy, and the reproduction pace. Four-lane and eight-lane blocks report apart, and every number is within-person only, never Corsi-norm comparable.

In [ ]:
check_selection(ctx, pick)
echo_span = keep(ctx, "echo_span", sec_echo(trials, metas))

## Movement onset and rate of force development

One onset detector now, not three: the exact port of the algorithm in Rayan's `process_force_peaks.py` (the one already run over Welber's earlier data), canonical in `finger_rehab/analytics/signal.py`, copied verbatim into this notebook's setup cell, and pinned by tests to his file and to the package copy. The chapter compares onset RT against the game's logged RT trial by trial, and checks his onset-anchored four-finger peak window against the cue-anchored `force_window_peaks` convention the game logs.


In [ ]:
check_selection(ctx, pick)
onset = keep(ctx, "onset", sec_onset(folders, trials, unit, calset))

## Objective 1, per-finger hit rate

In [ ]:
check_selection(ctx, pick)
objective_one = keep(ctx, "objective_one",
                     sec_objective_one(trials, calset=calset))

## Trial exclusions

In [ ]:
check_selection(ctx, pick)
flagged = keep(ctx, "flagged", sec_exclusions(trials))

## Pretest to aftertest

In [ ]:
check_selection(ctx, pick)
phases = keep(ctx, "phase", sec_phase(trials))

## Press thresholds in newtons

In [ ]:
check_selection(ctx, pick)
thresholds = keep(ctx, "threshold_audit",
                  sec_threshold_audit(metas=metas, calset=calset))

## Cue modality

In [ ]:
check_selection(ctx, pick)
cues = keep(ctx, "cues", sec_cue_modality(trials, calset))

## Dose

In [ ]:
check_selection(ctx, pick)
on_task = need(ctx, "on_task")["on_task"]
dose = keep(ctx, "dose", sec_dose(trials, on_task / 60))

## Sampling

In [ ]:
check_selection(ctx, pick)
sampling = keep(ctx, "sampling_note", sec_sampling_note(folders))

## Startup latency

In [ ]:
check_selection(ctx, pick)
startup = keep(ctx, "startup", sec_startup_latency(folders, metas))

## Constraint budget from the logs

In [ ]:
check_selection(ctx, pick)
constraints = keep(ctx, "constraints", sec_constraints(folders, metas))

## Progress per participant

In [ ]:
check_selection(ctx, pick)
progress = keep(ctx, "participant_progress",
                sec_participant_progress(cat=cat))

## Convergence across sessions

In [ ]:
check_selection(ctx, pick)
convergence = keep(ctx, "convergence", sec_convergence(cat=cat))

## Hand size against the sizing range

In [ ]:
check_selection(ctx, pick)
hand_size = keep(ctx, "hand_size", sec_hand_size(cat=cat))

## Intervals and tests

In [ ]:
check_selection(ctx, pick)
statistics = keep(ctx, "statistics", sec_statistics(trials))

## Headline numbers

In [ ]:
check_selection(ctx, pick)
summary = run_summary(ctx, trials, unit, calset, folders)
pd.DataFrame([summary]).T.rename(columns={0: "value"})

## Save the tables

In [ ]:
check_selection(ctx, pick)
run_exports(ctx, trials, calset)

## Cohort
The healthy baseline study read as one group: every participant code on disk, both visits, both hands. These sections ignore the dropdown on purpose and read the whole sessions tree, because a normative range or a test-retest coefficient belongs to the cohort, not to one save. Outputs land in `sessions/cohort_results/`; `cohort_metrics.csv` there is the long table JASP or R reads. Statistics print from the design's n (28); below it the sections show coverage only.


In [ ]:
cohort = keep(ctx, "cohort_selection", sec_cohort_selection(cat))


## Cohort descriptives and normative ranges
Per metric and hand role, visit 1 and visit 2 kept apart. The visit 1 table is the normative table; the headline metric of each mode is drawn with its reference line.


In [ ]:
keep(ctx, "cohort_describe", sec_cohort_describe(cohort))


## Cohort: dominant against non-dominant
Paired within participant: difference, interval, paired t, Cohen dz with a bootstrap interval, Shapiro-Wilk, Wilcoxon with the rank-biserial effect size. R2 and M2 are one-sided; everything else two-sided with no claim beyond the interval.


In [ ]:
keep(ctx, "cohort_hands", sec_cohort_hands(cohort))


## Cohort: test-retest, SEM and MDC
ICC(2,1) with its exact 95 percent interval, ICC(3,1) beside it, SEM and MDC95 (Weir 2005), Bland-Altman bias and limits. The forest plot is the chapter's headline figure.


In [ ]:
keep(ctx, "cohort_retest", sec_cohort_retest(cohort))


## Cohort: practice effects
Visit 2 minus visit 1 per metric, as a paired difference with dz and in units of the visit 1 SD.


In [ ]:
keep(ctx, "cohort_practice", sec_cohort_practice(cohort))


## Cohort: known-effect validity checks
The pre-specified checks from Section 1 of the study design, one row each, printed as plain verdicts with the number and the reference value. No other inferential test runs in this notebook.


In [ ]:
keep(ctx, "cohort_validity", sec_cohort_validity(cohort))


## Cohort export
`cohort_metrics.csv`, the tables above as CSV, and the cohort report as one HTML with the figures embedded.


In [ ]:
keep(ctx, "cohort_export", sec_cohort_export(cohort))
write_cohort_report(ctx, cohort)


## References
Everything the modes, the analysis and the thesis anchors lean on, verified before listing.

In [ ]:
print_references()

## Export report
Everything above as one HTML file, figures included.


In [ ]:
check_selection(ctx, pick)
write_report(ctx)
